# TokenMM Trading PnL Notebook

One-off exploratory notebook for fill-only trading PnL.

Methodology:
- Source of truth is `execution_fill` from local SQLite.
- Default scope excludes `BITGET` venue fills and `EXTERNAL` rows.
- Headline PnL nets all selected fills into one global fungible PLUME book.
- Local tables break the same tape out by `venue + symbol + product` and by `strategy_id`.
- This notebook does **not** include fees, funding, or any trading-vs-delta attribution split.
- Opening inventory defaults to zero at the selected window start; ending open inventory is reported separately.


In [1]:
from __future__ import annotations

import math
import re
import sqlite3
from pathlib import Path

import pandas as pd

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 120
pd.options.display.width = 240

NUMERIC_PATTERN = re.compile(r"[-+]?\d+(?:\.\d+)?")
INSTRUMENT_PRODUCTS = ("SPOT", "LINEAR", "PERP", "SWAP")


In [2]:
TELEMETRY_DIR = Path("/var/lib/nautilus/telemetry/tokenmm")
FILLS_DB_PATH = TELEMETRY_DIR / "fills.sqlite"

LOOKBACK_DAYS = 21
START_UTC = None
END_UTC = None
EXCLUDED_VENUES = {"BITGET"}
EXCLUDED_STRATEGY_IDS = {"EXTERNAL"}
OPENING_QTY = 0.0
OPENING_AVG_PX = None


def numeric(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series.astype("float64"), errors="coerce")
    extracted = series.astype("string").str.extract(f"({NUMERIC_PATTERN.pattern})", expand=False)
    return pd.to_numeric(extracted, errors="coerce")


def ns_to_utc(series: pd.Series) -> pd.Series:
    return pd.to_datetime(pd.to_numeric(series, errors="coerce"), unit="ns", utc=True)


def parse_instrument_id(instrument_id: str | None) -> dict[str, str | None]:
    text = (instrument_id or "").strip()
    venue = None
    body = text
    if "." in text:
        body, venue = text.rsplit(".", 1)
    symbol = body
    product = None
    for candidate in INSTRUMENT_PRODUCTS:
        suffix = f"-{candidate}"
        if body.endswith(suffix):
            symbol = body[: -len(suffix)]
            product = candidate
            break
    return {"symbol": symbol or None, "venue": venue, "product": product}


def format_utc(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S UTC").fillna("n/a")


def format_number(series: pd.Series, digits: int = 6) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return values.map(lambda value: f"{value:,.{digits}f}" if pd.notna(value) else "n/a")


def format_signed_number(series: pd.Series, digits: int = 6) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return values.map(lambda value: f"{value:+,.{digits}f}" if pd.notna(value) else "n/a")


def load_fills(path: Path) -> pd.DataFrame:
    with sqlite3.connect(path) as conn:
        fills = pd.read_sql_query("SELECT * FROM execution_fill", conn)
    fills["fill_px_num"] = numeric(fills["last_px"])
    qty_base = numeric(fills["last_qty_base"]) if "last_qty_base" in fills.columns else pd.Series(index=fills.index, dtype="float64")
    qty_raw = numeric(fills["last_qty"])
    fills["fill_qty_num"] = qty_base.combine_first(qty_raw)
    fills["fill_notional"] = fills["fill_qty_num"].abs() * fills["fill_px_num"]
    fills["fill_ts_utc"] = ns_to_utc(fills["ts_event"])
    fills["fill_ts_ms"] = (numeric(fills["ts_event"]) // 1_000_000).astype("Int64")
    parsed = pd.DataFrame([parse_instrument_id(value) for value in fills["instrument_id"]])
    fills["symbol"] = parsed["symbol"]
    fills["venue"] = parsed["venue"]
    fills["product"] = parsed["product"]
    fills["order_side"] = fills["order_side"].astype(str).str.upper()
    fills["fill_key"] = fills["trader_id"].astype(str) + "|" + fills["event_id"].astype(str)
    return fills


def filter_fills(fills: pd.DataFrame) -> pd.DataFrame:
    frame = fills.copy()
    frame = frame.loc[
        frame["fill_qty_num"].notna()
        & frame["fill_px_num"].notna()
        & frame["fill_ts_utc"].notna()
        & frame["order_side"].isin(["BUY", "SELL"])
    ].copy()
    end_ts = pd.Timestamp(END_UTC, tz="UTC") if END_UTC else frame["fill_ts_utc"].max()
    start_ts = pd.Timestamp(START_UTC, tz="UTC") if START_UTC else end_ts - pd.Timedelta(days=LOOKBACK_DAYS)
    frame = frame.loc[(frame["fill_ts_utc"] >= start_ts) & (frame["fill_ts_utc"] <= end_ts)].copy()
    if EXCLUDED_VENUES:
        frame = frame.loc[~frame["venue"].isin(sorted(EXCLUDED_VENUES))].copy()
    if EXCLUDED_STRATEGY_IDS:
        frame = frame.loc[~frame["strategy_id"].isin(sorted(EXCLUDED_STRATEGY_IDS))].copy()
    return frame.sort_values(["fill_ts_ms", "fill_key"]).reset_index(drop=True)


In [3]:
def build_trading_ledger(
    fills: pd.DataFrame,
    group_cols: tuple[str, ...] = (),
    *,
    opening_qty: float = 0.0,
    opening_avg_px: float | None = None,
) -> pd.DataFrame:
    records: list[dict[str, object]] = []
    groups = [((), fills)] if not group_cols else fills.groupby(list(group_cols), dropna=False, sort=True)

    for group_key, group in groups:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        position_qty = float(opening_qty)
        avg_cost = float(opening_avg_px) if opening_avg_px is not None and abs(opening_qty) > 1e-12 else None
        cumulative_realized = 0.0

        for row in group.itertuples(index=False):
            fill_qty = float(row.fill_qty_num)
            fill_px = float(row.fill_px_num)
            signed_fill_qty = fill_qty if row.order_side == "BUY" else -fill_qty
            position_qty_before = position_qty
            avg_cost_before = avg_cost
            realized_pnl = 0.0
            closing_qty = 0.0
            opening_fill_qty = 0.0

            if abs(position_qty) < 1e-12:
                position_qty = signed_fill_qty
                avg_cost = fill_px if abs(position_qty) > 1e-12 else None
                opening_fill_qty = abs(signed_fill_qty)
            elif position_qty * signed_fill_qty > 0:
                new_qty = position_qty + signed_fill_qty
                avg_cost = ((abs(position_qty) * float(avg_cost)) + (abs(signed_fill_qty) * fill_px)) / abs(new_qty)
                position_qty = new_qty
                opening_fill_qty = abs(signed_fill_qty)
            else:
                closing_qty = min(abs(position_qty), abs(signed_fill_qty))
                realized_pnl = ((fill_px - float(avg_cost)) * closing_qty) if position_qty > 0 else ((float(avg_cost) - fill_px) * closing_qty)
                residual = abs(signed_fill_qty) - closing_qty
                position_qty = position_qty + signed_fill_qty
                if abs(position_qty) < 1e-12:
                    position_qty = 0.0
                    avg_cost = None
                elif residual > 1e-12 and abs(signed_fill_qty) > abs(position_qty_before):
                    avg_cost = fill_px
                    opening_fill_qty = residual

            cumulative_realized += realized_pnl
            record = row._asdict()
            for column, value in zip(group_cols, group_key):
                record[column] = value
            record.update(
                {
                    "signed_fill_qty": signed_fill_qty,
                    "position_qty_before": position_qty_before,
                    "position_qty_after": position_qty,
                    "avg_cost_before": avg_cost_before,
                    "avg_cost_after": avg_cost,
                    "closing_qty": closing_qty,
                    "opening_fill_qty": opening_fill_qty,
                    "realized_pnl": realized_pnl,
                    "cumulative_realized_pnl": cumulative_realized,
                }
            )
            records.append(record)

    ledger = pd.DataFrame(records)
    ledger["fill_date"] = ledger["fill_ts_utc"].dt.strftime("%Y-%m-%d")
    return ledger


def summarize_ledger(ledger: pd.DataFrame, group_cols: tuple[str, ...] = ()) -> pd.DataFrame:
    records: list[dict[str, object]] = []
    groups = [((), ledger)] if not group_cols else ledger.groupby(list(group_cols), dropna=False, sort=True)
    for group_key, group in groups:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = {column: value for column, value in zip(group_cols, group_key)}
        row.update(
            {
                "fill_count": int(group["fill_key"].nunique()),
                "gross_buy_notional": float(group.loc[group["order_side"] == "BUY", "fill_notional"].sum()),
                "gross_sell_notional": float(group.loc[group["order_side"] == "SELL", "fill_notional"].sum()),
                "realized_pnl": float(group["realized_pnl"].sum()),
                "ending_open_qty": float(group["position_qty_after"].iloc[-1]),
                "ending_avg_cost": group["avg_cost_after"].iloc[-1],
                "ending_open_cost_basis": float(abs(group["position_qty_after"].iloc[-1]) * group["avg_cost_after"].iloc[-1]) if pd.notna(group["avg_cost_after"].iloc[-1]) else math.nan,
                "first_fill_utc": group["fill_ts_utc"].min(),
                "last_fill_utc": group["fill_ts_utc"].max(),
            }
        )
        records.append(row)
    return pd.DataFrame(records)


fills_all = load_fills(FILLS_DB_PATH)
fills_selected = filter_fills(fills_all)
global_ledger = build_trading_ledger(fills_selected, opening_qty=OPENING_QTY, opening_avg_px=OPENING_AVG_PX)
global_summary = summarize_ledger(global_ledger)
local_ledger = build_trading_ledger(fills_selected, group_cols=("venue", "symbol", "product"), opening_qty=OPENING_QTY, opening_avg_px=OPENING_AVG_PX)
local_summary = summarize_ledger(local_ledger, group_cols=("venue", "symbol", "product")).sort_values("realized_pnl", ascending=False)
strategy_summary = summarize_ledger(global_ledger, group_cols=("strategy_id",)).sort_values("realized_pnl", ascending=False)


## Telemetry Heartbeat And Global Summary

In [4]:
requested_end = pd.Timestamp(END_UTC, tz="UTC") if END_UTC else fills_all["fill_ts_utc"].max()
requested_start = pd.Timestamp(START_UTC, tz="UTC") if START_UTC else requested_end - pd.Timedelta(days=LOOKBACK_DAYS)

telemetry_overview = pd.DataFrame(
    [
        {
            "dataset": "fills_all",
            "rows": len(fills_all),
            "first_fill_utc": fills_all["fill_ts_utc"].min(),
            "last_fill_utc": fills_all["fill_ts_utc"].max(),
        },
        {
            "dataset": "fills_selected",
            "rows": len(fills_selected),
            "first_fill_utc": fills_selected["fill_ts_utc"].min(),
            "last_fill_utc": fills_selected["fill_ts_utc"].max(),
        },
    ]
)
telemetry_overview_display = telemetry_overview.assign(
    first_fill_utc=format_utc(telemetry_overview["first_fill_utc"]),
    last_fill_utc=format_utc(telemetry_overview["last_fill_utc"]),
)
telemetry_overview_display

window_summary = pd.DataFrame(
    [
        {
            "requested_start_utc": requested_start,
            "requested_end_utc": requested_end,
            "actual_start_utc": fills_selected["fill_ts_utc"].min() if not fills_selected.empty else pd.NaT,
            "actual_end_utc": fills_selected["fill_ts_utc"].max() if not fills_selected.empty else pd.NaT,
            "excluded_venues": ", ".join(sorted(EXCLUDED_VENUES)) or "none",
            "excluded_strategy_ids": ", ".join(sorted(EXCLUDED_STRATEGY_IDS)) or "none",
        }
    ]
)
window_summary_display = window_summary.assign(
    requested_start_utc=format_utc(window_summary["requested_start_utc"]),
    requested_end_utc=format_utc(window_summary["requested_end_utc"]),
    actual_start_utc=format_utc(window_summary["actual_start_utc"]),
    actual_end_utc=format_utc(window_summary["actual_end_utc"]),
)
window_summary_display

global_summary_display = global_summary.assign(
    gross_buy_notional=format_number(global_summary["gross_buy_notional"], digits=2),
    gross_sell_notional=format_number(global_summary["gross_sell_notional"], digits=2),
    realized_pnl=format_signed_number(global_summary["realized_pnl"], digits=6),
    ending_open_qty=format_signed_number(global_summary["ending_open_qty"], digits=4),
    ending_avg_cost=format_number(global_summary["ending_avg_cost"], digits=8),
    ending_open_cost_basis=format_number(global_summary["ending_open_cost_basis"], digits=2),
    first_fill_utc=format_utc(global_summary["first_fill_utc"]),
    last_fill_utc=format_utc(global_summary["last_fill_utc"]),
)
global_summary_display


,fill_count,gross_buy_notional,gross_sell_notional,realized_pnl,ending_open_qty,ending_avg_cost,ending_open_cost_basis,first_fill_utc,last_fill_utc
0,3684,"10,952.09","11,074.18",+108.984382,"-1,308.5000",0.01001971,13.11,2026-03-17 08:23:30 UTC,2026-03-26 16:54:39 UTC


## Local Books, Strategy Summary, And Daily Realized PnL

In [5]:
local_summary_display = local_summary.assign(
    gross_buy_notional=format_number(local_summary["gross_buy_notional"], digits=2),
    gross_sell_notional=format_number(local_summary["gross_sell_notional"], digits=2),
    realized_pnl=format_signed_number(local_summary["realized_pnl"], digits=6),
    ending_open_qty=format_signed_number(local_summary["ending_open_qty"], digits=4),
    ending_avg_cost=format_number(local_summary["ending_avg_cost"], digits=8),
    ending_open_cost_basis=format_number(local_summary["ending_open_cost_basis"], digits=2),
    first_fill_utc=format_utc(local_summary["first_fill_utc"]),
    last_fill_utc=format_utc(local_summary["last_fill_utc"]),
)
local_summary_display

strategy_summary_display = strategy_summary.assign(
    gross_buy_notional=format_number(strategy_summary["gross_buy_notional"], digits=2),
    gross_sell_notional=format_number(strategy_summary["gross_sell_notional"], digits=2),
    realized_pnl=format_signed_number(strategy_summary["realized_pnl"], digits=6),
    ending_open_qty=format_signed_number(strategy_summary["ending_open_qty"], digits=4),
    ending_avg_cost=format_number(strategy_summary["ending_avg_cost"], digits=8),
    ending_open_cost_basis=format_number(strategy_summary["ending_open_cost_basis"], digits=2),
    first_fill_utc=format_utc(strategy_summary["first_fill_utc"]),
    last_fill_utc=format_utc(strategy_summary["last_fill_utc"]),
)
strategy_summary_display

daily_realized = (
    global_ledger.groupby("fill_date", dropna=False)
    .agg(realized_pnl=("realized_pnl", "sum"), fill_count=("fill_key", "nunique"))
    .reset_index()
)
daily_realized_display = daily_realized.assign(
    realized_pnl=format_signed_number(daily_realized["realized_pnl"], digits=6),
)
daily_realized_display


,fill_date,realized_pnl,fill_count
0,2026-03-17,+16.583106,155
1,2026-03-18,+11.998319,335
2,2026-03-19,+18.234605,281
3,2026-03-20,+11.267298,311
4,2026-03-21,+13.410115,362
5,2026-03-22,+24.992950,223
6,2026-03-23,+5.847016,489
7,2026-03-24,+12.062346,1320
8,2026-03-25,+1.190393,154
9,2026-03-26,-6.601765,54


## Recent Realized Fills And Ending Open Inventory

In [6]:
recent_realized = global_ledger.loc[
    global_ledger["realized_pnl"] != 0,
    [
        "fill_ts_utc",
        "strategy_id",
        "venue",
        "symbol",
        "product",
        "order_side",
        "fill_qty_num",
        "fill_px_num",
        "realized_pnl",
        "position_qty_after",
        "avg_cost_after",
    ],
].tail(25)
recent_realized_display = recent_realized.assign(
    fill_ts_utc=format_utc(recent_realized["fill_ts_utc"]),
    fill_qty_num=format_number(recent_realized["fill_qty_num"], digits=2),
    fill_px_num=format_number(recent_realized["fill_px_num"], digits=8),
    realized_pnl=format_signed_number(recent_realized["realized_pnl"], digits=6),
    position_qty_after=format_signed_number(recent_realized["position_qty_after"], digits=4),
    avg_cost_after=format_number(recent_realized["avg_cost_after"], digits=8),
)
recent_realized_display

ending_open_inventory = local_summary.loc[local_summary["ending_open_qty"] != 0].copy()
ending_open_inventory_display = ending_open_inventory.assign(
    ending_open_qty=format_signed_number(ending_open_inventory["ending_open_qty"], digits=4),
    ending_avg_cost=format_number(ending_open_inventory["ending_avg_cost"], digits=8),
    ending_open_cost_basis=format_number(ending_open_inventory["ending_open_cost_basis"], digits=2),
    first_fill_utc=format_utc(ending_open_inventory["first_fill_utc"]),
    last_fill_utc=format_utc(ending_open_inventory["last_fill_utc"]),
)
ending_open_inventory_display


,venue,symbol,product,fill_count,gross_buy_notional,gross_sell_notional,realized_pnl,ending_open_qty,ending_avg_cost,ending_open_cost_basis,first_fill_utc,last_fill_utc
3,BYBIT,PLUMEUSDT,SPOT,874,4137.615362,4996.615700,120.358280,"-69,788.5000",0.01058401,738.64,2026-03-17 08:23:30 UTC,2026-03-25 03:57:54 UTC
1,BINANCE_SPOT,PLUMEUSDT,NaN,43,441.580000,0.000000,0.000000,"+42,000.0000",0.01051381,441.58,2026-03-24 11:29:03 UTC,2026-03-24 19:45:34 UTC
4,OKX,PLUME-USDT,SWAP,1731,943.728505,1000.716709,-1.369606,"-5,794.0000",0.01007211,58.36,2026-03-17 18:01:29 UTC,2026-03-26 16:54:39 UTC
0,BINANCE_PERP,PLUMEUSDT,PERP,215,1252.410000,987.120000,-3.187929,"+25,000.0000",0.01048408,262.10,2026-03-24 10:39:19 UTC,2026-03-26 14:35:23 UTC
2,BYBIT,PLUMEUSDT,LINEAR,821,4176.754138,4089.730765,-12.854862,"+7,274.0000",0.01019639,74.17,2026-03-17 12:45:00 UTC,2026-03-26 16:50:06 UTC


## Fees, Funding, And Net PnL

Fees are exact from `execution_fill.commission`. Funding is a frozen one-off pull from live venue ledgers and is embedded directly into this notebook so reruns stay offline.

This frozen dataset covers the current non-Bitget fill window only. If you move `START_UTC` or `END_UTC` outside that window, the funding section will not magically backfill new rows.


In [ ]:
FROZEN_FEE_ROWS = [
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735810037,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735810041,
    "fee_amount": 0.00534,
    "fee_currency": "USDT",
    "commission": "0.00534000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735810044,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735810193,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735827392,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735827779,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735827910,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735828071,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735920778,
    "fee_amount": 0.00534,
    "fee_currency": "USDT",
    "commission": "0.00534000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735920781,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735920985,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735921685,
    "fee_amount": 0.005344,
    "fee_currency": "USDT",
    "commission": "0.00534400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735982072,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735982303,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735982332,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735982532,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735982729,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735983027,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773735998397,
    "fee_amount": 0.005336,
    "fee_currency": "USDT",
    "commission": "0.00533600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773736049168,
    "fee_amount": 0.005332,
    "fee_currency": "USDT",
    "commission": "0.00533200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773736092084,
    "fee_amount": 0.00532,
    "fee_currency": "USDT",
    "commission": "0.00532000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773751500211,
    "fee_amount": 0.0012737,
    "fee_currency": "USDT",
    "commission": "0.00127370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773751500216,
    "fee_amount": 0.0012736,
    "fee_currency": "USDT",
    "commission": "0.00127360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773751500467,
    "fee_amount": 0.00127223,
    "fee_currency": "USDT",
    "commission": "0.00127223 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773751505696,
    "fee_amount": 0.00061775,
    "fee_currency": "USDT",
    "commission": "0.00061775 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773751507105,
    "fee_amount": 0.00065596,
    "fee_currency": "USDT",
    "commission": "0.00065596 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773751508406,
    "fee_amount": 0.0012736,
    "fee_currency": "USDT",
    "commission": "0.00127360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012943,
    "fee_currency": "USDT",
    "commission": "0.00129430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012946,
    "fee_currency": "USDT",
    "commission": "0.00129460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012949,
    "fee_currency": "USDT",
    "commission": "0.00129490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012953,
    "fee_currency": "USDT",
    "commission": "0.00129530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012956,
    "fee_currency": "USDT",
    "commission": "0.00129560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012957,
    "fee_currency": "USDT",
    "commission": "0.00129570 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012959,
    "fee_currency": "USDT",
    "commission": "0.00129590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757049625,
    "fee_amount": 0.0012962,
    "fee_currency": "USDT",
    "commission": "0.00129620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773757049665,
    "fee_amount": 0.005188,
    "fee_currency": "USDT",
    "commission": "0.00518800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757100488,
    "fee_amount": 0.0012877,
    "fee_currency": "USDT",
    "commission": "0.00128770 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757100492,
    "fee_amount": 0.0012873,
    "fee_currency": "USDT",
    "commission": "0.00128730 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773757100494,
    "fee_amount": 0.0012867,
    "fee_currency": "USDT",
    "commission": "0.00128670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773757100520,
    "fee_amount": 0.3728,
    "fee_currency": "USDT",
    "commission": "0.37280000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773757100689,
    "fee_amount": 0.0272,
    "fee_currency": "USDT",
    "commission": "0.02720000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773759252080,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773759252237,
    "fee_amount": 0.00128782,
    "fee_currency": "USDT",
    "commission": "0.00128782 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773759683650,
    "fee_amount": 0.0012934,
    "fee_currency": "USDT",
    "commission": "0.00129340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773760666706,
    "fee_amount": 0.0012828,
    "fee_currency": "USDT",
    "commission": "0.00128280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773760824042,
    "fee_amount": 0.005152,
    "fee_currency": "USDT",
    "commission": "0.00515200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773760824041,
    "fee_amount": 0.0012859,
    "fee_currency": "USDT",
    "commission": "0.00128590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773760824043,
    "fee_amount": 0.0012865,
    "fee_currency": "USDT",
    "commission": "0.00128650 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773760824053,
    "fee_amount": 0.0012872,
    "fee_currency": "USDT",
    "commission": "0.00128720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773760824085,
    "fee_amount": 0.005156,
    "fee_currency": "USDT",
    "commission": "0.00515600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773760824096,
    "fee_amount": 0.00096714,
    "fee_currency": "USDT",
    "commission": "0.00096714 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773760824196,
    "fee_amount": 0.00032067,
    "fee_currency": "USDT",
    "commission": "0.00032067 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773761898885,
    "fee_amount": 0.0012778,
    "fee_currency": "USDT",
    "commission": "0.00127780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773761898885,
    "fee_amount": 0.0012774,
    "fee_currency": "USDT",
    "commission": "0.00127740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773761898885,
    "fee_amount": 0.0012771,
    "fee_currency": "USDT",
    "commission": "0.00127710 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773761898885,
    "fee_amount": 0.0012768,
    "fee_currency": "USDT",
    "commission": "0.00127680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773761898896,
    "fee_amount": 0.0012765,
    "fee_currency": "USDT",
    "commission": "0.00127650 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773761898896,
    "fee_amount": 0.0012761,
    "fee_currency": "USDT",
    "commission": "0.00127610 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773761898896,
    "fee_amount": 0.0012759,
    "fee_currency": "USDT",
    "commission": "0.00127590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773763005828,
    "fee_amount": 0.0012831,
    "fee_currency": "USDT",
    "commission": "0.00128310 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773763005828,
    "fee_amount": 0.0012824,
    "fee_currency": "USDT",
    "commission": "0.00128240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773763005837,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773763005828,
    "fee_amount": 0.0012821,
    "fee_currency": "USDT",
    "commission": "0.00128210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773763005828,
    "fee_amount": 0.001282,
    "fee_currency": "USDT",
    "commission": "0.00128200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773763005828,
    "fee_amount": 0.0012815,
    "fee_currency": "USDT",
    "commission": "0.00128150 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773763005828,
    "fee_amount": 0.0012814,
    "fee_currency": "USDT",
    "commission": "0.00128140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773763005828,
    "fee_amount": 0.0012808,
    "fee_currency": "USDT",
    "commission": "0.00128080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773763680486,
    "fee_amount": 0.00516,
    "fee_currency": "USDT",
    "commission": "0.00516000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773764705811,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773764705807,
    "fee_amount": 0.0012827,
    "fee_currency": "USDT",
    "commission": "0.00128270 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773764705818,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773764705812,
    "fee_amount": 0.0012821,
    "fee_currency": "USDT",
    "commission": "0.00128210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773764705823,
    "fee_amount": 0.0012817,
    "fee_currency": "USDT",
    "commission": "0.00128170 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773764705863,
    "fee_amount": 0.0012814,
    "fee_currency": "USDT",
    "commission": "0.00128140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773765277008,
    "fee_amount": 0.0012812,
    "fee_currency": "USDT",
    "commission": "0.00128120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773765277071,
    "fee_amount": 0.0012812,
    "fee_currency": "USDT",
    "commission": "0.00128120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773765277108,
    "fee_amount": 0.0012818,
    "fee_currency": "USDT",
    "commission": "0.00128180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773765277116,
    "fee_amount": 0.00027049,
    "fee_currency": "USDT",
    "commission": "0.00027049 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773765277116,
    "fee_amount": 0.00101142,
    "fee_currency": "USDT",
    "commission": "0.00101142 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773765300082,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766517310,
    "fee_amount": 0.001283,
    "fee_currency": "USDT",
    "commission": "0.00128300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766517310,
    "fee_amount": 0.0012834,
    "fee_currency": "USDT",
    "commission": "0.00128340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766518742,
    "fee_amount": 0.0012843,
    "fee_currency": "USDT",
    "commission": "0.00128430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766518742,
    "fee_amount": 0.0012845,
    "fee_currency": "USDT",
    "commission": "0.00128450 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766520019,
    "fee_amount": 0.0012856,
    "fee_currency": "USDT",
    "commission": "0.00128560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766521250,
    "fee_amount": 0.0012862,
    "fee_currency": "USDT",
    "commission": "0.00128620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766521250,
    "fee_amount": 0.0012866,
    "fee_currency": "USDT",
    "commission": "0.00128660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773766521250,
    "fee_amount": 0.0012869,
    "fee_currency": "USDT",
    "commission": "0.00128690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773770489487,
    "fee_amount": 0.0012843,
    "fee_currency": "USDT",
    "commission": "0.00128430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773770489487,
    "fee_amount": 0.0012847,
    "fee_currency": "USDT",
    "commission": "0.00128470 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773770489487,
    "fee_amount": 0.0012849,
    "fee_currency": "USDT",
    "commission": "0.00128490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773770489487,
    "fee_amount": 0.0012853,
    "fee_currency": "USDT",
    "commission": "0.00128530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773770489487,
    "fee_amount": 0.0012856,
    "fee_currency": "USDT",
    "commission": "0.00128560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773770489516,
    "fee_amount": 0.0025672,
    "fee_currency": "USDT",
    "commission": "0.00256720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773770489532,
    "fee_amount": 0.005144,
    "fee_currency": "USDT",
    "commission": "0.00514400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773770489568,
    "fee_amount": 0.0025686,
    "fee_currency": "USDT",
    "commission": "0.00256860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773775441245,
    "fee_amount": 0.0012841,
    "fee_currency": "USDT",
    "commission": "0.00128410 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773776007737,
    "fee_amount": 0.0012903,
    "fee_currency": "USDT",
    "commission": "0.00129030 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773776007735,
    "fee_amount": 0.005172,
    "fee_currency": "USDT",
    "commission": "0.00517200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773776007756,
    "fee_amount": 0.005172,
    "fee_currency": "USDT",
    "commission": "0.00517200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773776007745,
    "fee_amount": 0.002582,
    "fee_currency": "USDT",
    "commission": "0.00258200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773776007769,
    "fee_amount": 0.0025832,
    "fee_currency": "USDT",
    "commission": "0.00258320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012782,
    "fee_currency": "USDT",
    "commission": "0.00127820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012776,
    "fee_currency": "USDT",
    "commission": "0.00127760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012773,
    "fee_currency": "USDT",
    "commission": "0.00127730 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.001277,
    "fee_currency": "USDT",
    "commission": "0.00127700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012769,
    "fee_currency": "USDT",
    "commission": "0.00127690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012767,
    "fee_currency": "USDT",
    "commission": "0.00127670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012766,
    "fee_currency": "USDT",
    "commission": "0.00127660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012763,
    "fee_currency": "USDT",
    "commission": "0.00127630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230820,
    "fee_amount": 0.0012759,
    "fee_currency": "USDT",
    "commission": "0.00127590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773777230893,
    "fee_amount": 0.0012776,
    "fee_currency": "USDT",
    "commission": "0.00127760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773777230880,
    "fee_amount": 0.002558,
    "fee_currency": "USDT",
    "commission": "0.00255800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773783292776,
    "fee_amount": 0.001297,
    "fee_currency": "USDT",
    "commission": "0.00129700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773783625479,
    "fee_amount": 0.0026012,
    "fee_currency": "USDT",
    "commission": "0.00260120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773783626889,
    "fee_amount": 0.0025976,
    "fee_currency": "USDT",
    "commission": "0.00259760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773783626889,
    "fee_amount": 0.002597,
    "fee_currency": "USDT",
    "commission": "0.00259700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773784316605,
    "fee_amount": 0.0025926,
    "fee_currency": "USDT",
    "commission": "0.00259260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773785068048,
    "fee_amount": 0.0025646,
    "fee_currency": "USDT",
    "commission": "0.00256460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773785068053,
    "fee_amount": 0.0025634,
    "fee_currency": "USDT",
    "commission": "0.00256340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773785068066,
    "fee_amount": 0.0012816,
    "fee_currency": "USDT",
    "commission": "0.00128160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773785068066,
    "fee_amount": 0.001281,
    "fee_currency": "USDT",
    "commission": "0.00128100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773785068076,
    "fee_amount": 0.0012804,
    "fee_currency": "USDT",
    "commission": "0.00128040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773786359244,
    "fee_amount": 0.0025582,
    "fee_currency": "USDT",
    "commission": "0.00255820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773786425430,
    "fee_amount": 0.002556,
    "fee_currency": "USDT",
    "commission": "0.00255600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773787056284,
    "fee_amount": 0.0025458,
    "fee_currency": "USDT",
    "commission": "0.00254580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787056320,
    "fee_amount": 0.0012724,
    "fee_currency": "USDT",
    "commission": "0.00127240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787056320,
    "fee_amount": 0.0012718,
    "fee_currency": "USDT",
    "commission": "0.00127180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787056320,
    "fee_amount": 0.0012711,
    "fee_currency": "USDT",
    "commission": "0.00127110 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787056320,
    "fee_amount": 0.0012707,
    "fee_currency": "USDT",
    "commission": "0.00127070 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787056320,
    "fee_amount": 0.0012705,
    "fee_currency": "USDT",
    "commission": "0.00127050 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773787094841,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787094841,
    "fee_amount": 0.0012681,
    "fee_currency": "USDT",
    "commission": "0.00126810 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773787094829,
    "fee_amount": 0.0025374,
    "fee_currency": "USDT",
    "commission": "0.00253740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773787094829,
    "fee_amount": 0.0025362,
    "fee_currency": "USDT",
    "commission": "0.00253620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787095070,
    "fee_amount": 0.0012675,
    "fee_currency": "USDT",
    "commission": "0.00126750 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773787095076,
    "fee_amount": 0.0012669,
    "fee_currency": "USDT",
    "commission": "0.00126690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789640201,
    "fee_amount": 0.0012623,
    "fee_currency": "USDT",
    "commission": "0.00126230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789677436,
    "fee_amount": 0.0012589,
    "fee_currency": "USDT",
    "commission": "0.00125890 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789677440,
    "fee_amount": 0.0012583,
    "fee_currency": "USDT",
    "commission": "0.00125830 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773789677427,
    "fee_amount": 0.0025188,
    "fee_currency": "USDT",
    "commission": "0.00251880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773789677427,
    "fee_amount": 0.0025176,
    "fee_currency": "USDT",
    "commission": "0.00251760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773789677430,
    "fee_amount": 0.0025164,
    "fee_currency": "USDT",
    "commission": "0.00251640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773789677432,
    "fee_amount": 0.002515,
    "fee_currency": "USDT",
    "commission": "0.00251500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773789677434,
    "fee_amount": 0.0025138,
    "fee_currency": "USDT",
    "commission": "0.00251380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789677444,
    "fee_amount": 0.0012576,
    "fee_currency": "USDT",
    "commission": "0.00125760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789677444,
    "fee_amount": 0.0012573,
    "fee_currency": "USDT",
    "commission": "0.00125730 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789677445,
    "fee_amount": 0.001257,
    "fee_currency": "USDT",
    "commission": "0.00125700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789677445,
    "fee_amount": 0.0012566,
    "fee_currency": "USDT",
    "commission": "0.00125660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773789677446,
    "fee_amount": 0.0012564,
    "fee_currency": "USDT",
    "commission": "0.00125640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773789677440,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773789677443,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773789677444,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773789677454,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773789677455,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773792990524,
    "fee_amount": 0.0012367,
    "fee_currency": "USDT",
    "commission": "0.00123670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773792990524,
    "fee_amount": 0.0012365,
    "fee_currency": "USDT",
    "commission": "0.00123650 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773792990524,
    "fee_amount": 0.0012363,
    "fee_currency": "USDT",
    "commission": "0.00123630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773792990524,
    "fee_amount": 0.0012361,
    "fee_currency": "USDT",
    "commission": "0.00123610 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773792990528,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773794725843,
    "fee_amount": 0.0024654,
    "fee_currency": "USDT",
    "commission": "0.00246540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795175346,
    "fee_amount": 0.0012312,
    "fee_currency": "USDT",
    "commission": "0.00123120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773795175346,
    "fee_amount": 0.004932,
    "fee_currency": "USDT",
    "commission": "0.00493200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773795175362,
    "fee_amount": 0.004936,
    "fee_currency": "USDT",
    "commission": "0.00493600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795175359,
    "fee_amount": 0.0012318,
    "fee_currency": "USDT",
    "commission": "0.00123180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773795175350,
    "fee_amount": 0.0024628,
    "fee_currency": "USDT",
    "commission": "0.00246280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773795301835,
    "fee_amount": 0.00494,
    "fee_currency": "USDT",
    "commission": "0.00494000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795301845,
    "fee_amount": 0.0012334,
    "fee_currency": "USDT",
    "commission": "0.00123340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773795301795,
    "fee_amount": 0.002466,
    "fee_currency": "USDT",
    "commission": "0.00246600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795301847,
    "fee_amount": 0.001234,
    "fee_currency": "USDT",
    "commission": "0.00123400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795301847,
    "fee_amount": 0.0012344,
    "fee_currency": "USDT",
    "commission": "0.00123440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773795301833,
    "fee_amount": 0.0024672,
    "fee_currency": "USDT",
    "commission": "0.00246720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773795301919,
    "fee_amount": 0.004944,
    "fee_currency": "USDT",
    "commission": "0.00494400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795307060,
    "fee_amount": 0.0012385,
    "fee_currency": "USDT",
    "commission": "0.00123850 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773795313439,
    "fee_amount": 0.004968,
    "fee_currency": "USDT",
    "commission": "0.00496800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795313446,
    "fee_amount": 0.0012393,
    "fee_currency": "USDT",
    "commission": "0.00123930 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795313450,
    "fee_amount": 0.0012395,
    "fee_currency": "USDT",
    "commission": "0.00123950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795313450,
    "fee_amount": 0.0012397,
    "fee_currency": "USDT",
    "commission": "0.00123970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773795578269,
    "fee_amount": 0.0012449,
    "fee_currency": "USDT",
    "commission": "0.00124490 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773795578304,
    "fee_amount": 0.0024908,
    "fee_currency": "USDT",
    "commission": "0.00249080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773795578304,
    "fee_amount": 0.002492,
    "fee_currency": "USDT",
    "commission": "0.00249200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773796157396,
    "fee_amount": 0.001251,
    "fee_currency": "USDT",
    "commission": "0.00125100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773796157398,
    "fee_amount": 0.0025032,
    "fee_currency": "USDT",
    "commission": "0.00250320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773796157407,
    "fee_amount": 0.0025044,
    "fee_currency": "USDT",
    "commission": "0.00250440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773796157436,
    "fee_amount": 0.0012516,
    "fee_currency": "USDT",
    "commission": "0.00125160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773796157522,
    "fee_amount": 0.005016,
    "fee_currency": "USDT",
    "commission": "0.00501600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773796499390,
    "fee_amount": 0.0012542,
    "fee_currency": "USDT",
    "commission": "0.00125420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773796499398,
    "fee_amount": 0.0012558,
    "fee_currency": "USDT",
    "commission": "0.00125580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773796499378,
    "fee_amount": 0.00077798,
    "fee_currency": "USDT",
    "commission": "0.00077798 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773796499378,
    "fee_amount": 0.00173162,
    "fee_currency": "USDT",
    "commission": "0.00173162 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773796499393,
    "fee_amount": 0.005028,
    "fee_currency": "USDT",
    "commission": "0.00502800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773796499438,
    "fee_amount": 0.005032,
    "fee_currency": "USDT",
    "commission": "0.00503200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773796499413,
    "fee_amount": 0.0012561,
    "fee_currency": "USDT",
    "commission": "0.00125610 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773796499413,
    "fee_amount": 0.0012564,
    "fee_currency": "USDT",
    "commission": "0.00125640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773796499416,
    "fee_amount": 0.0012567,
    "fee_currency": "USDT",
    "commission": "0.00125670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052008,
    "fee_amount": 0.0012575,
    "fee_currency": "USDT",
    "commission": "0.00125750 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797051996,
    "fee_amount": 0.0025162,
    "fee_currency": "USDT",
    "commission": "0.00251620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797051996,
    "fee_amount": 0.0025176,
    "fee_currency": "USDT",
    "commission": "0.00251760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797051998,
    "fee_amount": 0.0025188,
    "fee_currency": "USDT",
    "commission": "0.00251880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797052000,
    "fee_amount": 0.00252,
    "fee_currency": "USDT",
    "commission": "0.00252000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773797052008,
    "fee_amount": 0.00504,
    "fee_currency": "USDT",
    "commission": "0.00504000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773797052009,
    "fee_amount": 0.005044,
    "fee_currency": "USDT",
    "commission": "0.00504400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773797052055,
    "fee_amount": 0.005048,
    "fee_currency": "USDT",
    "commission": "0.00504800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797052041,
    "fee_amount": 0.0025212,
    "fee_currency": "USDT",
    "commission": "0.00252120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052008,
    "fee_amount": 0.0012581,
    "fee_currency": "USDT",
    "commission": "0.00125810 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052011,
    "fee_amount": 0.0012588,
    "fee_currency": "USDT",
    "commission": "0.00125880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052011,
    "fee_amount": 0.0012592,
    "fee_currency": "USDT",
    "commission": "0.00125920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052014,
    "fee_amount": 0.0012594,
    "fee_currency": "USDT",
    "commission": "0.00125940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052016,
    "fee_amount": 0.0012598,
    "fee_currency": "USDT",
    "commission": "0.00125980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052029,
    "fee_amount": 0.00126,
    "fee_currency": "USDT",
    "commission": "0.00126000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052103,
    "fee_amount": 0.0012603,
    "fee_currency": "USDT",
    "commission": "0.00126030 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797052106,
    "fee_amount": 0.0012604,
    "fee_currency": "USDT",
    "commission": "0.00126040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773797052266,
    "fee_amount": 0.005056,
    "fee_currency": "USDT",
    "commission": "0.00505600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797052974,
    "fee_amount": 0.0025216,
    "fee_currency": "USDT",
    "commission": "0.00252160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797052974,
    "fee_amount": 0.0025214,
    "fee_currency": "USDT",
    "commission": "0.00252140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797705312,
    "fee_amount": 0.002524,
    "fee_currency": "USDT",
    "commission": "0.00252400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797885314,
    "fee_amount": 0.002521,
    "fee_currency": "USDT",
    "commission": "0.00252100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797908322,
    "fee_amount": 0.002519,
    "fee_currency": "USDT",
    "commission": "0.00251900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797956548,
    "fee_amount": 0.0025156,
    "fee_currency": "USDT",
    "commission": "0.00251560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773797956569,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797956571,
    "fee_amount": 0.0012569,
    "fee_currency": "USDT",
    "commission": "0.00125690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773797956688,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773797956704,
    "fee_amount": 0.0012563,
    "fee_currency": "USDT",
    "commission": "0.00125630 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773797956688,
    "fee_amount": 0.002513,
    "fee_currency": "USDT",
    "commission": "0.00251300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773803790247,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773803790248,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773803790247,
    "fee_amount": 0.0012288,
    "fee_currency": "USDT",
    "commission": "0.00122880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773803790254,
    "fee_amount": 0.001227,
    "fee_currency": "USDT",
    "commission": "0.00122700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773803790255,
    "fee_amount": 0.0012266,
    "fee_currency": "USDT",
    "commission": "0.00122660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773803790256,
    "fee_amount": 0.0012263,
    "fee_currency": "USDT",
    "commission": "0.00122630 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773803790234,
    "fee_amount": 0.0024584,
    "fee_currency": "USDT",
    "commission": "0.00245840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773803790234,
    "fee_amount": 0.0024584,
    "fee_currency": "USDT",
    "commission": "0.00245840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773803790235,
    "fee_amount": 0.0024572,
    "fee_currency": "USDT",
    "commission": "0.00245720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773803790235,
    "fee_amount": 0.002456,
    "fee_currency": "USDT",
    "commission": "0.00245600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773803790242,
    "fee_amount": 0.0024548,
    "fee_currency": "USDT",
    "commission": "0.00245480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773807422420,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773808005366,
    "fee_amount": 0.0012184,
    "fee_currency": "USDT",
    "commission": "0.00121840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773808005355,
    "fee_amount": 0.0024374,
    "fee_currency": "USDT",
    "commission": "0.00243740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773808005376,
    "fee_amount": 0.0012178,
    "fee_currency": "USDT",
    "commission": "0.00121780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773808213149,
    "fee_amount": 0.002441,
    "fee_currency": "USDT",
    "commission": "0.00244100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773810361494,
    "fee_amount": 0.004952,
    "fee_currency": "USDT",
    "commission": "0.00495200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773810361496,
    "fee_amount": 0.0012359,
    "fee_currency": "USDT",
    "commission": "0.00123590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1773810361498,
    "fee_amount": 0.0012363,
    "fee_currency": "USDT",
    "commission": "0.00123630 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773822595077,
    "fee_amount": 0.0024174,
    "fee_currency": "USDT",
    "commission": "0.00241740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773823207260,
    "fee_amount": 0.004856,
    "fee_currency": "USDT",
    "commission": "0.00485600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773823207232,
    "fee_amount": 0.0024234,
    "fee_currency": "USDT",
    "commission": "0.00242340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773823207246,
    "fee_amount": 0.0024246,
    "fee_currency": "USDT",
    "commission": "0.00242460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773823866549,
    "fee_amount": 0.0024218,
    "fee_currency": "USDT",
    "commission": "0.00242180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773823992314,
    "fee_amount": 0.0024232,
    "fee_currency": "USDT",
    "commission": "0.00242320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773829166746,
    "fee_amount": 0.0024414,
    "fee_currency": "USDT",
    "commission": "0.00244140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773829166778,
    "fee_amount": 0.0024426,
    "fee_currency": "USDT",
    "commission": "0.00244260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773829166794,
    "fee_amount": 0.0024438,
    "fee_currency": "USDT",
    "commission": "0.00244380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773830197733,
    "fee_amount": 0.0024408,
    "fee_currency": "USDT",
    "commission": "0.00244080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773830197796,
    "fee_amount": 0.004888,
    "fee_currency": "USDT",
    "commission": "0.00488800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773832168808,
    "fee_amount": 0.0024092,
    "fee_currency": "USDT",
    "commission": "0.00240920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773832168808,
    "fee_amount": 0.002408,
    "fee_currency": "USDT",
    "commission": "0.00240800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773832168819,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773832168834,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773832168834,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773832168812,
    "fee_amount": 0.0024068,
    "fee_currency": "USDT",
    "commission": "0.00240680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773832168814,
    "fee_amount": 0.0024056,
    "fee_currency": "USDT",
    "commission": "0.00240560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773838405002,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773839341629,
    "fee_amount": 0.0023746,
    "fee_currency": "USDT",
    "commission": "0.00237460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773839341637,
    "fee_amount": 0.0023758,
    "fee_currency": "USDT",
    "commission": "0.00237580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773839964812,
    "fee_amount": 0.004744,
    "fee_currency": "USDT",
    "commission": "0.00474400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773839964813,
    "fee_amount": 0.004748,
    "fee_currency": "USDT",
    "commission": "0.00474800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773839964813,
    "fee_amount": 0.004752,
    "fee_currency": "USDT",
    "commission": "0.00475200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773839964830,
    "fee_amount": 0.004756,
    "fee_currency": "USDT",
    "commission": "0.00475600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773839964799,
    "fee_amount": 0.002369,
    "fee_currency": "USDT",
    "commission": "0.00236900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773839964801,
    "fee_amount": 0.0023702,
    "fee_currency": "USDT",
    "commission": "0.00237020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773839964801,
    "fee_amount": 0.0023714,
    "fee_currency": "USDT",
    "commission": "0.00237140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773839964801,
    "fee_amount": 0.0023726,
    "fee_currency": "USDT",
    "commission": "0.00237260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773839964802,
    "fee_amount": 0.0023736,
    "fee_currency": "USDT",
    "commission": "0.00237360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773840483799,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840483770,
    "fee_amount": 0.002372,
    "fee_currency": "USDT",
    "commission": "0.00237200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773840563897,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840563960,
    "fee_amount": 0.0023736,
    "fee_currency": "USDT",
    "commission": "0.00237360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840563960,
    "fee_amount": 0.0023728,
    "fee_currency": "USDT",
    "commission": "0.00237280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840676165,
    "fee_amount": 0.0023906,
    "fee_currency": "USDT",
    "commission": "0.00239060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840679151,
    "fee_amount": 0.0023904,
    "fee_currency": "USDT",
    "commission": "0.00239040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773840698393,
    "fee_amount": 0.005995,
    "fee_currency": "USDT",
    "commission": "0.00599500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840708035,
    "fee_amount": 0.0023942,
    "fee_currency": "USDT",
    "commission": "0.00239420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840725638,
    "fee_amount": 0.0024022,
    "fee_currency": "USDT",
    "commission": "0.00240220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840725645,
    "fee_amount": 0.0024002,
    "fee_currency": "USDT",
    "commission": "0.00240020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840730480,
    "fee_amount": 0.0023968,
    "fee_currency": "USDT",
    "commission": "0.00239680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840730483,
    "fee_amount": 0.0023958,
    "fee_currency": "USDT",
    "commission": "0.00239580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840734157,
    "fee_amount": 0.0023946,
    "fee_currency": "USDT",
    "commission": "0.00239460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840734157,
    "fee_amount": 0.0023936,
    "fee_currency": "USDT",
    "commission": "0.00239360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773840736713,
    "fee_amount": 0.004816,
    "fee_currency": "USDT",
    "commission": "0.00481600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773840757350,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840789930,
    "fee_amount": 0.0023998,
    "fee_currency": "USDT",
    "commission": "0.00239980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773840794627,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840794607,
    "fee_amount": 0.002397,
    "fee_currency": "USDT",
    "commission": "0.00239700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840794609,
    "fee_amount": 0.0023966,
    "fee_currency": "USDT",
    "commission": "0.00239660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773840925753,
    "fee_amount": 0.0024004,
    "fee_currency": "USDT",
    "commission": "0.00240040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773841942183,
    "fee_amount": 0.0023756,
    "fee_currency": "USDT",
    "commission": "0.00237560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773842967726,
    "fee_amount": 0.0023592,
    "fee_currency": "USDT",
    "commission": "0.00235920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773843348116,
    "fee_amount": 0.0023574,
    "fee_currency": "USDT",
    "commission": "0.00235740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773843355270,
    "fee_amount": 0.0023628,
    "fee_currency": "USDT",
    "commission": "0.00236280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773843355270,
    "fee_amount": 9.454e-05,
    "fee_currency": "USDT",
    "commission": "0.00009454 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773843355422,
    "fee_amount": 0.004736,
    "fee_currency": "USDT",
    "commission": "0.00473600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773843377957,
    "fee_amount": 0.09228,
    "fee_currency": "USDT",
    "commission": "0.09228000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773843377957,
    "fee_amount": 0.30772,
    "fee_currency": "USDT",
    "commission": "0.30772000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773843377951,
    "fee_amount": 0.0023606,
    "fee_currency": "USDT",
    "commission": "0.00236060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773843390364,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773844492134,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773844844372,
    "fee_amount": 0.0023272,
    "fee_currency": "USDT",
    "commission": "0.00232720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773844844376,
    "fee_amount": 0.0023282,
    "fee_currency": "USDT",
    "commission": "0.00232820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773844872853,
    "fee_amount": 0.0023304,
    "fee_currency": "USDT",
    "commission": "0.00233040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773844935538,
    "fee_amount": 0.0023294,
    "fee_currency": "USDT",
    "commission": "0.00232940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845012534,
    "fee_amount": 0.0023238,
    "fee_currency": "USDT",
    "commission": "0.00232380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845012538,
    "fee_amount": 0.0023248,
    "fee_currency": "USDT",
    "commission": "0.00232480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773845062272,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773845062272,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773845062284,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845062260,
    "fee_amount": 0.0023146,
    "fee_currency": "USDT",
    "commission": "0.00231460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845062260,
    "fee_amount": 0.0023134,
    "fee_currency": "USDT",
    "commission": "0.00231340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845062260,
    "fee_amount": 0.0023124,
    "fee_currency": "USDT",
    "commission": "0.00231240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845062264,
    "fee_amount": 0.0023112,
    "fee_currency": "USDT",
    "commission": "0.00231120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845088397,
    "fee_amount": 0.0023138,
    "fee_currency": "USDT",
    "commission": "0.00231380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845100412,
    "fee_amount": 0.0023148,
    "fee_currency": "USDT",
    "commission": "0.00231480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845123006,
    "fee_amount": 0.002311,
    "fee_currency": "USDT",
    "commission": "0.00231100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845123135,
    "fee_amount": 0.0023122,
    "fee_currency": "USDT",
    "commission": "0.00231220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845123135,
    "fee_amount": 0.002313,
    "fee_currency": "USDT",
    "commission": "0.00231300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845123135,
    "fee_amount": 0.0023132,
    "fee_currency": "USDT",
    "commission": "0.00231320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773845123282,
    "fee_amount": 0.0023134,
    "fee_currency": "USDT",
    "commission": "0.00231340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846144852,
    "fee_amount": 0.0022958,
    "fee_currency": "USDT",
    "commission": "0.00229580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846144852,
    "fee_amount": 0.0022954,
    "fee_currency": "USDT",
    "commission": "0.00229540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846144852,
    "fee_amount": 0.0022948,
    "fee_currency": "USDT",
    "commission": "0.00229480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773846165243,
    "fee_amount": 0.004616,
    "fee_currency": "USDT",
    "commission": "0.00461600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846165221,
    "fee_amount": 0.0023048,
    "fee_currency": "USDT",
    "commission": "0.00230480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846177537,
    "fee_amount": 0.0023068,
    "fee_currency": "USDT",
    "commission": "0.00230680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773846249079,
    "fee_amount": 0.00464,
    "fee_currency": "USDT",
    "commission": "0.00464000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846249033,
    "fee_amount": 0.0023172,
    "fee_currency": "USDT",
    "commission": "0.00231720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773846417764,
    "fee_amount": 0.004628,
    "fee_currency": "USDT",
    "commission": "0.00462800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773846456344,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846456333,
    "fee_amount": 0.002305,
    "fee_currency": "USDT",
    "commission": "0.00230500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773846456361,
    "fee_amount": 0.0023038,
    "fee_currency": "USDT",
    "commission": "0.00230380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847232587,
    "fee_amount": 0.0022824,
    "fee_currency": "USDT",
    "commission": "0.00228240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847298487,
    "fee_amount": 0.0022824,
    "fee_currency": "USDT",
    "commission": "0.00228240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847531232,
    "fee_amount": 0.0022814,
    "fee_currency": "USDT",
    "commission": "0.00228140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773847532336,
    "fee_amount": 0.004576,
    "fee_currency": "USDT",
    "commission": "0.00457600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847532324,
    "fee_amount": 0.0022854,
    "fee_currency": "USDT",
    "commission": "0.00228540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847532324,
    "fee_amount": 0.0022858,
    "fee_currency": "USDT",
    "commission": "0.00228580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847827929,
    "fee_amount": 0.0022914,
    "fee_currency": "USDT",
    "commission": "0.00229140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847930375,
    "fee_amount": 0.0022904,
    "fee_currency": "USDT",
    "commission": "0.00229040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773847999182,
    "fee_amount": 0.0022936,
    "fee_currency": "USDT",
    "commission": "0.00229360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848104004,
    "fee_amount": 0.0022906,
    "fee_currency": "USDT",
    "commission": "0.00229060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848162849,
    "fee_amount": 0.002296,
    "fee_currency": "USDT",
    "commission": "0.00229600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848162860,
    "fee_amount": 0.0022968,
    "fee_currency": "USDT",
    "commission": "0.00229680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848162861,
    "fee_amount": 2.297e-05,
    "fee_currency": "USDT",
    "commission": "0.00002297 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848162888,
    "fee_amount": 0.00227423,
    "fee_currency": "USDT",
    "commission": "0.00227423 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848162910,
    "fee_amount": 0.0022982,
    "fee_currency": "USDT",
    "commission": "0.00229820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848170715,
    "fee_amount": 0.0022934,
    "fee_currency": "USDT",
    "commission": "0.00229340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848170719,
    "fee_amount": 0.0022946,
    "fee_currency": "USDT",
    "commission": "0.00229460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848170791,
    "fee_amount": 0.004592,
    "fee_currency": "USDT",
    "commission": "0.00459200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848616226,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848616211,
    "fee_amount": 0.002303,
    "fee_currency": "USDT",
    "commission": "0.00230300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848616467,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848709183,
    "fee_amount": 0.002301,
    "fee_currency": "USDT",
    "commission": "0.00230100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848709186,
    "fee_amount": 0.0022998,
    "fee_currency": "USDT",
    "commission": "0.00229980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848779020,
    "fee_amount": 0.004632,
    "fee_currency": "USDT",
    "commission": "0.00463200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848779020,
    "fee_amount": 0.004636,
    "fee_currency": "USDT",
    "commission": "0.00463600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848798000,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848816160,
    "fee_amount": 0.0023056,
    "fee_currency": "USDT",
    "commission": "0.00230560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848819534,
    "fee_amount": 0.0023056,
    "fee_currency": "USDT",
    "commission": "0.00230560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848822398,
    "fee_amount": 0.0023044,
    "fee_currency": "USDT",
    "commission": "0.00230440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773848822398,
    "fee_amount": 0.0023042,
    "fee_currency": "USDT",
    "commission": "0.00230420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848823217,
    "fee_amount": 0.004628,
    "fee_currency": "USDT",
    "commission": "0.00462800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773848823217,
    "fee_amount": 0.004632,
    "fee_currency": "USDT",
    "commission": "0.00463200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773849375715,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773849375715,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773849616322,
    "fee_amount": 0.004612,
    "fee_currency": "USDT",
    "commission": "0.00461200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773849616306,
    "fee_amount": 0.0023026,
    "fee_currency": "USDT",
    "commission": "0.00230260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773849616306,
    "fee_amount": 0.0023034,
    "fee_currency": "USDT",
    "commission": "0.00230340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773849616306,
    "fee_amount": 0.0023036,
    "fee_currency": "USDT",
    "commission": "0.00230360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773849705253,
    "fee_amount": 0.004644,
    "fee_currency": "USDT",
    "commission": "0.00464400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773849935109,
    "fee_amount": 0.004628,
    "fee_currency": "USDT",
    "commission": "0.00462800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773849935100,
    "fee_amount": 0.0023114,
    "fee_currency": "USDT",
    "commission": "0.00231140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850022141,
    "fee_amount": 0.004628,
    "fee_currency": "USDT",
    "commission": "0.00462800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850027873,
    "fee_amount": 0.004636,
    "fee_currency": "USDT",
    "commission": "0.00463600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850136470,
    "fee_amount": 0.004652,
    "fee_currency": "USDT",
    "commission": "0.00465200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773850152182,
    "fee_amount": 0.0023174,
    "fee_currency": "USDT",
    "commission": "0.00231740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773850153657,
    "fee_amount": 0.0023154,
    "fee_currency": "USDT",
    "commission": "0.00231540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773850153662,
    "fee_amount": 0.0023142,
    "fee_currency": "USDT",
    "commission": "0.00231420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773850153662,
    "fee_amount": 0.002314,
    "fee_currency": "USDT",
    "commission": "0.00231400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773850325428,
    "fee_amount": 0.0023126,
    "fee_currency": "USDT",
    "commission": "0.00231260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850325448,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773850325462,
    "fee_amount": 0.0023114,
    "fee_currency": "USDT",
    "commission": "0.00231140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850326709,
    "fee_amount": 0.004636,
    "fee_currency": "USDT",
    "commission": "0.00463600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850326743,
    "fee_amount": 0.00464,
    "fee_currency": "USDT",
    "commission": "0.00464000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850416519,
    "fee_amount": 0.004648,
    "fee_currency": "USDT",
    "commission": "0.00464800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773850485280,
    "fee_amount": 0.004628,
    "fee_currency": "USDT",
    "commission": "0.00462800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773851128997,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773851134101,
    "fee_amount": 0.0022968,
    "fee_currency": "USDT",
    "commission": "0.00229680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773851161197,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773852302913,
    "fee_amount": 0.004616,
    "fee_currency": "USDT",
    "commission": "0.00461600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773852366695,
    "fee_amount": 0.00462,
    "fee_currency": "USDT",
    "commission": "0.00462000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773852936036,
    "fee_amount": 0.004632,
    "fee_currency": "USDT",
    "commission": "0.00463200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773852936043,
    "fee_amount": 0.004636,
    "fee_currency": "USDT",
    "commission": "0.00463600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773852936024,
    "fee_amount": 0.002311,
    "fee_currency": "USDT",
    "commission": "0.00231100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773852936027,
    "fee_amount": 0.0023122,
    "fee_currency": "USDT",
    "commission": "0.00231220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773852936027,
    "fee_amount": 0.0023134,
    "fee_currency": "USDT",
    "commission": "0.00231340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773852936569,
    "fee_amount": 0.00464,
    "fee_currency": "USDT",
    "commission": "0.00464000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773853601711,
    "fee_amount": 0.0023128,
    "fee_currency": "USDT",
    "commission": "0.00231280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773853601719,
    "fee_amount": 0.0023116,
    "fee_currency": "USDT",
    "commission": "0.00231160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773853739517,
    "fee_amount": 0.00464,
    "fee_currency": "USDT",
    "commission": "0.00464000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773853741916,
    "fee_amount": 0.004644,
    "fee_currency": "USDT",
    "commission": "0.00464400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773853933262,
    "fee_amount": 0.0023144,
    "fee_currency": "USDT",
    "commission": "0.00231440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773854112387,
    "fee_amount": 0.004644,
    "fee_currency": "USDT",
    "commission": "0.00464400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773854359200,
    "fee_amount": 0.0023136,
    "fee_currency": "USDT",
    "commission": "0.00231360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773855245576,
    "fee_amount": 0.0023318,
    "fee_currency": "USDT",
    "commission": "0.00233180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773855819581,
    "fee_amount": 0.004704,
    "fee_currency": "USDT",
    "commission": "0.00470400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773855819788,
    "fee_amount": 0.002348,
    "fee_currency": "USDT",
    "commission": "0.00234800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773855853570,
    "fee_amount": 0.004724,
    "fee_currency": "USDT",
    "commission": "0.00472400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773855901044,
    "fee_amount": 0.0023528,
    "fee_currency": "USDT",
    "commission": "0.00235280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773855901044,
    "fee_amount": 0.0023518,
    "fee_currency": "USDT",
    "commission": "0.00235180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773856508214,
    "fee_amount": 0.0023358,
    "fee_currency": "USDT",
    "commission": "0.00233580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773856508236,
    "fee_amount": 0.00468,
    "fee_currency": "USDT",
    "commission": "0.00468000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773856508220,
    "fee_amount": 0.002337,
    "fee_currency": "USDT",
    "commission": "0.00233700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773856508277,
    "fee_amount": 0.0023382,
    "fee_currency": "USDT",
    "commission": "0.00233820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773856835976,
    "fee_amount": 0.0023412,
    "fee_currency": "USDT",
    "commission": "0.00234120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773856837533,
    "fee_amount": 0.002341,
    "fee_currency": "USDT",
    "commission": "0.00234100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773857236284,
    "fee_amount": 0.004716,
    "fee_currency": "USDT",
    "commission": "0.00471600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773857981348,
    "fee_amount": 0.0023368,
    "fee_currency": "USDT",
    "commission": "0.00233680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773857981379,
    "fee_amount": 0.0023356,
    "fee_currency": "USDT",
    "commission": "0.00233560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773858620382,
    "fee_amount": 0.0023394,
    "fee_currency": "USDT",
    "commission": "0.00233940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773859328751,
    "fee_amount": 0.0023288,
    "fee_currency": "USDT",
    "commission": "0.00232880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773859367530,
    "fee_amount": 0.004672,
    "fee_currency": "USDT",
    "commission": "0.00467200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864707903,
    "fee_amount": 0.0022728,
    "fee_currency": "USDT",
    "commission": "0.00227280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864707903,
    "fee_amount": 0.0022716,
    "fee_currency": "USDT",
    "commission": "0.00227160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864707912,
    "fee_amount": 0.0022704,
    "fee_currency": "USDT",
    "commission": "0.00227040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864707912,
    "fee_amount": 0.0022694,
    "fee_currency": "USDT",
    "commission": "0.00226940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864740536,
    "fee_amount": 0.004568,
    "fee_currency": "USDT",
    "commission": "0.00456800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864740642,
    "fee_amount": 0.004568,
    "fee_currency": "USDT",
    "commission": "0.00456800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864745414,
    "fee_amount": 0.004576,
    "fee_currency": "USDT",
    "commission": "0.00457600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864773522,
    "fee_amount": 0.00458,
    "fee_currency": "USDT",
    "commission": "0.00458000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864975124,
    "fee_amount": 0.0046,
    "fee_currency": "USDT",
    "commission": "0.00460000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864975124,
    "fee_amount": 0.004604,
    "fee_currency": "USDT",
    "commission": "0.00460400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864975124,
    "fee_amount": 0.004608,
    "fee_currency": "USDT",
    "commission": "0.00460800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864975111,
    "fee_amount": 0.0022964,
    "fee_currency": "USDT",
    "commission": "0.00229640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864975111,
    "fee_amount": 0.0022974,
    "fee_currency": "USDT",
    "commission": "0.00229740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864975112,
    "fee_amount": 0.0022984,
    "fee_currency": "USDT",
    "commission": "0.00229840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773864975114,
    "fee_amount": 0.0022996,
    "fee_currency": "USDT",
    "commission": "0.00229960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773864975452,
    "fee_amount": 0.00462,
    "fee_currency": "USDT",
    "commission": "0.00462000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773865342323,
    "fee_amount": 0.0023066,
    "fee_currency": "USDT",
    "commission": "0.00230660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773865342360,
    "fee_amount": 0.0023078,
    "fee_currency": "USDT",
    "commission": "0.00230780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773865342298,
    "fee_amount": 0.00462,
    "fee_currency": "USDT",
    "commission": "0.00462000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773865342304,
    "fee_amount": 0.004624,
    "fee_currency": "USDT",
    "commission": "0.00462400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773865342374,
    "fee_amount": 0.004628,
    "fee_currency": "USDT",
    "commission": "0.00462800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773865397101,
    "fee_amount": 0.002309,
    "fee_currency": "USDT",
    "commission": "0.00230900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773865397111,
    "fee_amount": 0.0023082,
    "fee_currency": "USDT",
    "commission": "0.00230820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773865635804,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773865635771,
    "fee_amount": 0.0023088,
    "fee_currency": "USDT",
    "commission": "0.00230880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773865635778,
    "fee_amount": 0.0023076,
    "fee_currency": "USDT",
    "commission": "0.00230760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773866056137,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773866056137,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773866056142,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773866056208,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056152,
    "fee_amount": 0.0023274,
    "fee_currency": "USDT",
    "commission": "0.00232740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056152,
    "fee_amount": 0.0023262,
    "fee_currency": "USDT",
    "commission": "0.00232620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056154,
    "fee_amount": 0.002325,
    "fee_currency": "USDT",
    "commission": "0.00232500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056156,
    "fee_amount": 0.0023238,
    "fee_currency": "USDT",
    "commission": "0.00232380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056196,
    "fee_amount": 0.0023228,
    "fee_currency": "USDT",
    "commission": "0.00232280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056266,
    "fee_amount": 0.00185856,
    "fee_currency": "USDT",
    "commission": "0.00185856 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056268,
    "fee_amount": 0.00046464,
    "fee_currency": "USDT",
    "commission": "0.00046464 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866056270,
    "fee_amount": 0.002323,
    "fee_currency": "USDT",
    "commission": "0.00232300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773866108093,
    "fee_amount": 0.004676,
    "fee_currency": "USDT",
    "commission": "0.00467600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773866108105,
    "fee_amount": 0.00468,
    "fee_currency": "USDT",
    "commission": "0.00468000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866108082,
    "fee_amount": 0.0023328,
    "fee_currency": "USDT",
    "commission": "0.00233280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773866592066,
    "fee_amount": 0.004672,
    "fee_currency": "USDT",
    "commission": "0.00467200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866592218,
    "fee_amount": 0.0023312,
    "fee_currency": "USDT",
    "commission": "0.00233120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773866930747,
    "fee_amount": 0.0023324,
    "fee_currency": "USDT",
    "commission": "0.00233240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773867951197,
    "fee_amount": 0.004696,
    "fee_currency": "USDT",
    "commission": "0.00469600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773868640031,
    "fee_amount": 0.0023462,
    "fee_currency": "USDT",
    "commission": "0.00234620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773868645682,
    "fee_amount": 0.002344,
    "fee_currency": "USDT",
    "commission": "0.00234400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773868645684,
    "fee_amount": 0.002343,
    "fee_currency": "USDT",
    "commission": "0.00234300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773868727003,
    "fee_amount": 0.0023396,
    "fee_currency": "USDT",
    "commission": "0.00233960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773870227522,
    "fee_amount": 0.0023314,
    "fee_currency": "USDT",
    "commission": "0.00233140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773870227522,
    "fee_amount": 0.0023304,
    "fee_currency": "USDT",
    "commission": "0.00233040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773870273838,
    "fee_amount": 0.0023292,
    "fee_currency": "USDT",
    "commission": "0.00232920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773870273855,
    "fee_amount": 0.002328,
    "fee_currency": "USDT",
    "commission": "0.00232800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773870362585,
    "fee_amount": 0.002335,
    "fee_currency": "USDT",
    "commission": "0.00233500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773870362631,
    "fee_amount": 0.0023362,
    "fee_currency": "USDT",
    "commission": "0.00233620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773870961931,
    "fee_amount": 0.004696,
    "fee_currency": "USDT",
    "commission": "0.00469600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773871083916,
    "fee_amount": 0.00016359,
    "fee_currency": "USDT",
    "commission": "0.00016359 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773871732589,
    "fee_amount": 0.0023202,
    "fee_currency": "USDT",
    "commission": "0.00232020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773871771303,
    "fee_amount": 0.0023242,
    "fee_currency": "USDT",
    "commission": "0.00232420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773876595858,
    "fee_amount": 0.0023064,
    "fee_currency": "USDT",
    "commission": "0.00230640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773877510273,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773880222987,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773881630633,
    "fee_amount": 0.0022946,
    "fee_currency": "USDT",
    "commission": "0.00229460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773881630633,
    "fee_amount": 0.0022958,
    "fee_currency": "USDT",
    "commission": "0.00229580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773883963462,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773883963444,
    "fee_amount": 0.0022678,
    "fee_currency": "USDT",
    "commission": "0.00226780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773883963475,
    "fee_amount": 0.0022666,
    "fee_currency": "USDT",
    "commission": "0.00226660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773883963480,
    "fee_amount": 0.0022654,
    "fee_currency": "USDT",
    "commission": "0.00226540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773883963480,
    "fee_amount": 0.0022644,
    "fee_currency": "USDT",
    "commission": "0.00226440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773883963480,
    "fee_amount": 0.0022632,
    "fee_currency": "USDT",
    "commission": "0.00226320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773885723422,
    "fee_amount": 0.002257,
    "fee_currency": "USDT",
    "commission": "0.00225700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773885746570,
    "fee_amount": 0.002262,
    "fee_currency": "USDT",
    "commission": "0.00226200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773885782837,
    "fee_amount": 0.0022654,
    "fee_currency": "USDT",
    "commission": "0.00226540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773888685003,
    "fee_amount": 0.0022674,
    "fee_currency": "USDT",
    "commission": "0.00226740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773889232498,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773889232511,
    "fee_amount": 0.002266,
    "fee_currency": "USDT",
    "commission": "0.00226600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773889591435,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773889591624,
    "fee_amount": 0.0022616,
    "fee_currency": "USDT",
    "commission": "0.00226160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773890374929,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773890702764,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773890916180,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773890916180,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773892751973,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773892751973,
    "fee_amount": 0.088,
    "fee_currency": "USDT",
    "commission": "0.08800000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773892751973,
    "fee_amount": 0.312,
    "fee_currency": "USDT",
    "commission": "0.31200000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773892751961,
    "fee_amount": 0.0022582,
    "fee_currency": "USDT",
    "commission": "0.00225820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773892751994,
    "fee_amount": 0.0022572,
    "fee_currency": "USDT",
    "commission": "0.00225720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773892752093,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773893040372,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773893096757,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773897874037,
    "fee_amount": 0.0022524,
    "fee_currency": "USDT",
    "commission": "0.00225240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773898852452,
    "fee_amount": 0.004528,
    "fee_currency": "USDT",
    "commission": "0.00452800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773898861662,
    "fee_amount": 0.004532,
    "fee_currency": "USDT",
    "commission": "0.00453200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773898861695,
    "fee_amount": 0.0022606,
    "fee_currency": "USDT",
    "commission": "0.00226060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773900271644,
    "fee_amount": 0.0022678,
    "fee_currency": "USDT",
    "commission": "0.00226780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900271668,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900445739,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900445921,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900447234,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773900772325,
    "fee_amount": 0.0022662,
    "fee_currency": "USDT",
    "commission": "0.00226620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773900772325,
    "fee_amount": 0.0022674,
    "fee_currency": "USDT",
    "commission": "0.00226740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773900772325,
    "fee_amount": 0.0022684,
    "fee_currency": "USDT",
    "commission": "0.00226840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773900772328,
    "fee_amount": 0.0022696,
    "fee_currency": "USDT",
    "commission": "0.00226960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773900772349,
    "fee_amount": 0.0022708,
    "fee_currency": "USDT",
    "commission": "0.00227080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900772360,
    "fee_amount": 0.004544,
    "fee_currency": "USDT",
    "commission": "0.00454400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900889962,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900980645,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773900980645,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773900980635,
    "fee_amount": 0.0022602,
    "fee_currency": "USDT",
    "commission": "0.00226020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901080151,
    "fee_amount": 0.002259,
    "fee_currency": "USDT",
    "commission": "0.00225900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773901169073,
    "fee_amount": 0.00452,
    "fee_currency": "USDT",
    "commission": "0.00452000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901169057,
    "fee_amount": 0.002255,
    "fee_currency": "USDT",
    "commission": "0.00225500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901169057,
    "fee_amount": 0.0022562,
    "fee_currency": "USDT",
    "commission": "0.00225620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901169061,
    "fee_amount": 0.002257,
    "fee_currency": "USDT",
    "commission": "0.00225700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901169061,
    "fee_amount": 0.0022574,
    "fee_currency": "USDT",
    "commission": "0.00225740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901169061,
    "fee_amount": 0.0022582,
    "fee_currency": "USDT",
    "commission": "0.00225820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773901169105,
    "fee_amount": 0.00133865,
    "fee_currency": "USDT",
    "commission": "0.00133865 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773901169355,
    "fee_amount": 0.004528,
    "fee_currency": "USDT",
    "commission": "0.00452800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773901427503,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773901670191,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901670179,
    "fee_amount": 0.0022492,
    "fee_currency": "USDT",
    "commission": "0.00224920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773901707868,
    "fee_amount": 0.002247,
    "fee_currency": "USDT",
    "commission": "0.00224700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773902251340,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773902356716,
    "fee_amount": 0.39996,
    "fee_currency": "USDT",
    "commission": "0.39996000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903614280,
    "fee_amount": 0.0022706,
    "fee_currency": "USDT",
    "commission": "0.00227060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903614316,
    "fee_amount": 0.0022696,
    "fee_currency": "USDT",
    "commission": "0.00226960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903654466,
    "fee_amount": 0.0022704,
    "fee_currency": "USDT",
    "commission": "0.00227040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903654498,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903654492,
    "fee_amount": 0.0022692,
    "fee_currency": "USDT",
    "commission": "0.00226920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903675074,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903675092,
    "fee_amount": 0.0022748,
    "fee_currency": "USDT",
    "commission": "0.00227480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903678255,
    "fee_amount": 0.004572,
    "fee_currency": "USDT",
    "commission": "0.00457200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903698446,
    "fee_amount": 0.0022738,
    "fee_currency": "USDT",
    "commission": "0.00227380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903706490,
    "fee_amount": 0.004568,
    "fee_currency": "USDT",
    "commission": "0.00456800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903706942,
    "fee_amount": 0.004572,
    "fee_currency": "USDT",
    "commission": "0.00457200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903716121,
    "fee_amount": 0.0022788,
    "fee_currency": "USDT",
    "commission": "0.00227880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903721860,
    "fee_amount": 0.004568,
    "fee_currency": "USDT",
    "commission": "0.00456800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903723022,
    "fee_amount": 0.004572,
    "fee_currency": "USDT",
    "commission": "0.00457200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903725305,
    "fee_amount": 0.00458,
    "fee_currency": "USDT",
    "commission": "0.00458000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903725809,
    "fee_amount": 0.004584,
    "fee_currency": "USDT",
    "commission": "0.00458400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903738453,
    "fee_amount": 0.0022822,
    "fee_currency": "USDT",
    "commission": "0.00228220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903739528,
    "fee_amount": 0.002281,
    "fee_currency": "USDT",
    "commission": "0.00228100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903741366,
    "fee_amount": 0.00458,
    "fee_currency": "USDT",
    "commission": "0.00458000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903741374,
    "fee_amount": 0.004584,
    "fee_currency": "USDT",
    "commission": "0.00458400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903780177,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903780178,
    "fee_amount": 0.002282,
    "fee_currency": "USDT",
    "commission": "0.00228200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903780181,
    "fee_amount": 0.0022808,
    "fee_currency": "USDT",
    "commission": "0.00228080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903780217,
    "fee_amount": 0.00072947,
    "fee_currency": "USDT",
    "commission": "0.00072947 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903780219,
    "fee_amount": 0.00155013,
    "fee_currency": "USDT",
    "commission": "0.00155013 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903780225,
    "fee_amount": 0.0022786,
    "fee_currency": "USDT",
    "commission": "0.00227860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903781469,
    "fee_amount": 0.00458,
    "fee_currency": "USDT",
    "commission": "0.00458000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903781779,
    "fee_amount": 0.004584,
    "fee_currency": "USDT",
    "commission": "0.00458400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903782371,
    "fee_amount": 0.004588,
    "fee_currency": "USDT",
    "commission": "0.00458800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903800626,
    "fee_amount": 0.004596,
    "fee_currency": "USDT",
    "commission": "0.00459600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903810812,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903814762,
    "fee_amount": 0.004588,
    "fee_currency": "USDT",
    "commission": "0.00458800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903815409,
    "fee_amount": 0.004592,
    "fee_currency": "USDT",
    "commission": "0.00459200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903823751,
    "fee_amount": 0.0022842,
    "fee_currency": "USDT",
    "commission": "0.00228420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903825297,
    "fee_amount": 0.0022832,
    "fee_currency": "USDT",
    "commission": "0.00228320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903826651,
    "fee_amount": 0.004588,
    "fee_currency": "USDT",
    "commission": "0.00458800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903827422,
    "fee_amount": 0.004592,
    "fee_currency": "USDT",
    "commission": "0.00459200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903829745,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903829745,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903829749,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903829733,
    "fee_amount": 0.002283,
    "fee_currency": "USDT",
    "commission": "0.00228300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903829734,
    "fee_amount": 0.002282,
    "fee_currency": "USDT",
    "commission": "0.00228200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903829736,
    "fee_amount": 0.002281,
    "fee_currency": "USDT",
    "commission": "0.00228100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773903829887,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773903993164,
    "fee_amount": 0.0022752,
    "fee_currency": "USDT",
    "commission": "0.00227520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773904051834,
    "fee_amount": 0.0022744,
    "fee_currency": "USDT",
    "commission": "0.00227440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773904089076,
    "fee_amount": 0.004556,
    "fee_currency": "USDT",
    "commission": "0.00455600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773904089065,
    "fee_amount": 0.0022734,
    "fee_currency": "USDT",
    "commission": "0.00227340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773905430369,
    "fee_amount": 0.0022452,
    "fee_currency": "USDT",
    "commission": "0.00224520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773906717564,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773906717569,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773906717559,
    "fee_amount": 9.022e-05,
    "fee_currency": "USDT",
    "commission": "0.00009022 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773906717588,
    "fee_amount": 0.00216518,
    "fee_currency": "USDT",
    "commission": "0.00216518 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773907524497,
    "fee_amount": 0.002241,
    "fee_currency": "USDT",
    "commission": "0.00224100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773907524497,
    "fee_amount": 0.00224,
    "fee_currency": "USDT",
    "commission": "0.00224000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773907524497,
    "fee_amount": 0.0022388,
    "fee_currency": "USDT",
    "commission": "0.00223880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773908179972,
    "fee_amount": 0.0022568,
    "fee_currency": "USDT",
    "commission": "0.00225680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773908188403,
    "fee_amount": 0.0022568,
    "fee_currency": "USDT",
    "commission": "0.00225680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773908196713,
    "fee_amount": 0.0022568,
    "fee_currency": "USDT",
    "commission": "0.00225680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773908196718,
    "fee_amount": 0.0022572,
    "fee_currency": "USDT",
    "commission": "0.00225720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773908204037,
    "fee_amount": 0.002257,
    "fee_currency": "USDT",
    "commission": "0.00225700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773908232432,
    "fee_amount": 4.514e-05,
    "fee_currency": "USDT",
    "commission": "0.00004514 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773908232454,
    "fee_amount": 4.514e-05,
    "fee_currency": "USDT",
    "commission": "0.00004514 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773912641742,
    "fee_amount": 0.0022636,
    "fee_currency": "USDT",
    "commission": "0.00226360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773913200768,
    "fee_amount": 0.0022728,
    "fee_currency": "USDT",
    "commission": "0.00227280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773913404953,
    "fee_amount": 0.002279,
    "fee_currency": "USDT",
    "commission": "0.00227900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773913405008,
    "fee_amount": 0.004568,
    "fee_currency": "USDT",
    "commission": "0.00456800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773913844082,
    "fee_amount": 0.004572,
    "fee_currency": "USDT",
    "commission": "0.00457200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773913844082,
    "fee_amount": 0.004576,
    "fee_currency": "USDT",
    "commission": "0.00457600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773913866250,
    "fee_amount": 0.0022812,
    "fee_currency": "USDT",
    "commission": "0.00228120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773913866269,
    "fee_amount": 0.004572,
    "fee_currency": "USDT",
    "commission": "0.00457200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773913901760,
    "fee_amount": 0.004576,
    "fee_currency": "USDT",
    "commission": "0.00457600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773913947926,
    "fee_amount": 0.004576,
    "fee_currency": "USDT",
    "commission": "0.00457600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773913947913,
    "fee_amount": 0.0022832,
    "fee_currency": "USDT",
    "commission": "0.00228320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773913986713,
    "fee_amount": 0.0022852,
    "fee_currency": "USDT",
    "commission": "0.00228520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773913986912,
    "fee_amount": 0.00458,
    "fee_currency": "USDT",
    "commission": "0.00458000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773913986895,
    "fee_amount": 0.0022864,
    "fee_currency": "USDT",
    "commission": "0.00228640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773913986929,
    "fee_amount": 0.0022886,
    "fee_currency": "USDT",
    "commission": "0.00228860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914014198,
    "fee_amount": 0.0022796,
    "fee_currency": "USDT",
    "commission": "0.00227960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773914014219,
    "fee_amount": 0.004568,
    "fee_currency": "USDT",
    "commission": "0.00456800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914155440,
    "fee_amount": 0.0022776,
    "fee_currency": "USDT",
    "commission": "0.00227760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914220427,
    "fee_amount": 0.0022778,
    "fee_currency": "USDT",
    "commission": "0.00227780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773914220569,
    "fee_amount": 0.004564,
    "fee_currency": "USDT",
    "commission": "0.00456400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914220688,
    "fee_amount": 0.0022788,
    "fee_currency": "USDT",
    "commission": "0.00227880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773914289165,
    "fee_amount": 0.004564,
    "fee_currency": "USDT",
    "commission": "0.00456400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914289152,
    "fee_amount": 0.002278,
    "fee_currency": "USDT",
    "commission": "0.00227800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914289183,
    "fee_amount": 0.0022792,
    "fee_currency": "USDT",
    "commission": "0.00227920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914289183,
    "fee_amount": 0.00228,
    "fee_currency": "USDT",
    "commission": "0.00228000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914427897,
    "fee_amount": 0.0022806,
    "fee_currency": "USDT",
    "commission": "0.00228060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914492406,
    "fee_amount": 0.0022808,
    "fee_currency": "USDT",
    "commission": "0.00228080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914839591,
    "fee_amount": 0.0022752,
    "fee_currency": "USDT",
    "commission": "0.00227520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773914843713,
    "fee_amount": 0.0022752,
    "fee_currency": "USDT",
    "commission": "0.00227520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773915302676,
    "fee_amount": 0.0022714,
    "fee_currency": "USDT",
    "commission": "0.00227140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773915713761,
    "fee_amount": 0.0022738,
    "fee_currency": "USDT",
    "commission": "0.00227380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773915827203,
    "fee_amount": 0.002274,
    "fee_currency": "USDT",
    "commission": "0.00227400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773915873472,
    "fee_amount": 0.0022744,
    "fee_currency": "USDT",
    "commission": "0.00227440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773916481613,
    "fee_amount": 0.002271,
    "fee_currency": "USDT",
    "commission": "0.00227100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773916590510,
    "fee_amount": 0.0022712,
    "fee_currency": "USDT",
    "commission": "0.00227120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773916590530,
    "fee_amount": 0.0022724,
    "fee_currency": "USDT",
    "commission": "0.00227240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773918303425,
    "fee_amount": 0.00205897,
    "fee_currency": "USDT",
    "commission": "0.00205897 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773921720702,
    "fee_amount": 0.004532,
    "fee_currency": "USDT",
    "commission": "0.00453200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921720689,
    "fee_amount": 0.0022616,
    "fee_currency": "USDT",
    "commission": "0.00226160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921720692,
    "fee_amount": 0.0022626,
    "fee_currency": "USDT",
    "commission": "0.00226260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773921720709,
    "fee_amount": 0.004536,
    "fee_currency": "USDT",
    "commission": "0.00453600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921720700,
    "fee_amount": 0.0022638,
    "fee_currency": "USDT",
    "commission": "0.00226380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921720789,
    "fee_amount": 0.0022634,
    "fee_currency": "USDT",
    "commission": "0.00226340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921720791,
    "fee_amount": 0.0022636,
    "fee_currency": "USDT",
    "commission": "0.00226360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921724895,
    "fee_amount": 0.0022596,
    "fee_currency": "USDT",
    "commission": "0.00225960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921724895,
    "fee_amount": 0.0022586,
    "fee_currency": "USDT",
    "commission": "0.00225860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921724902,
    "fee_amount": 0.0022574,
    "fee_currency": "USDT",
    "commission": "0.00225740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921724904,
    "fee_amount": 0.0022562,
    "fee_currency": "USDT",
    "commission": "0.00225620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773921724908,
    "fee_amount": 0.0022552,
    "fee_currency": "USDT",
    "commission": "0.00225520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773923442486,
    "fee_amount": 0.0022312,
    "fee_currency": "USDT",
    "commission": "0.00223120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773923471341,
    "fee_amount": 0.002235,
    "fee_currency": "USDT",
    "commission": "0.00223500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773924047343,
    "fee_amount": 0.0022288,
    "fee_currency": "USDT",
    "commission": "0.00222880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773925042545,
    "fee_amount": 0.00446,
    "fee_currency": "USDT",
    "commission": "0.00446000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925042544,
    "fee_amount": 0.0022278,
    "fee_currency": "USDT",
    "commission": "0.00222780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773925362144,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773925362172,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773925362181,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773925834437,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773925834437,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925834422,
    "fee_amount": 0.0022128,
    "fee_currency": "USDT",
    "commission": "0.00221280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925834424,
    "fee_amount": 0.0022116,
    "fee_currency": "USDT",
    "commission": "0.00221160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925834425,
    "fee_amount": 0.0022104,
    "fee_currency": "USDT",
    "commission": "0.00221040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925834428,
    "fee_amount": 0.0022094,
    "fee_currency": "USDT",
    "commission": "0.00220940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773925961613,
    "fee_amount": 0.00442,
    "fee_currency": "USDT",
    "commission": "0.00442000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925987174,
    "fee_amount": 0.0021978,
    "fee_currency": "USDT",
    "commission": "0.00219780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925987178,
    "fee_amount": 0.0021976,
    "fee_currency": "USDT",
    "commission": "0.00219760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925987178,
    "fee_amount": 0.0021968,
    "fee_currency": "USDT",
    "commission": "0.00219680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925987178,
    "fee_amount": 0.0021966,
    "fee_currency": "USDT",
    "commission": "0.00219660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773925987179,
    "fee_amount": 0.0021954,
    "fee_currency": "USDT",
    "commission": "0.00219540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773926000066,
    "fee_amount": 0.002198,
    "fee_currency": "USDT",
    "commission": "0.00219800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773926229173,
    "fee_amount": 0.004412,
    "fee_currency": "USDT",
    "commission": "0.00441200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773926229164,
    "fee_amount": 0.0022024,
    "fee_currency": "USDT",
    "commission": "0.00220240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773926234042,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773926234082,
    "fee_amount": 0.0021978,
    "fee_currency": "USDT",
    "commission": "0.00219780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773926234085,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773926234465,
    "fee_amount": 0.0021946,
    "fee_currency": "USDT",
    "commission": "0.00219460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773926235857,
    "fee_amount": 0.0021972,
    "fee_currency": "USDT",
    "commission": "0.00219720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773926431097,
    "fee_amount": 0.004448,
    "fee_currency": "USDT",
    "commission": "0.00444800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773926431097,
    "fee_amount": 0.004452,
    "fee_currency": "USDT",
    "commission": "0.00445200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773926431157,
    "fee_amount": 0.0022204,
    "fee_currency": "USDT",
    "commission": "0.00222040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773927231535,
    "fee_amount": 0.004476,
    "fee_currency": "USDT",
    "commission": "0.00447600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773928000272,
    "fee_amount": 0.002231,
    "fee_currency": "USDT",
    "commission": "0.00223100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773928000278,
    "fee_amount": 0.0022322,
    "fee_currency": "USDT",
    "commission": "0.00223220 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773928008538,
    "fee_amount": 0.004472,
    "fee_currency": "USDT",
    "commission": "0.00447200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773928037232,
    "fee_amount": 0.0022326,
    "fee_currency": "USDT",
    "commission": "0.00223260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773928040066,
    "fee_amount": 0.0022346,
    "fee_currency": "USDT",
    "commission": "0.00223460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773928040182,
    "fee_amount": 0.004476,
    "fee_currency": "USDT",
    "commission": "0.00447600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773928477305,
    "fee_amount": 0.004484,
    "fee_currency": "USDT",
    "commission": "0.00448400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773930422517,
    "fee_amount": 0.00106032,
    "fee_currency": "USDT",
    "commission": "0.00106032 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773930422564,
    "fee_amount": 0.00114868,
    "fee_currency": "USDT",
    "commission": "0.00114868 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773933470352,
    "fee_amount": 0.0022058,
    "fee_currency": "USDT",
    "commission": "0.00220580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773936549781,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773939626770,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773939626968,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773939840591,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773940867303,
    "fee_amount": 0.002189,
    "fee_currency": "USDT",
    "commission": "0.00218900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773944370564,
    "fee_amount": 0.08532,
    "fee_currency": "USDT",
    "commission": "0.08532000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773944370568,
    "fee_amount": 0.31468,
    "fee_currency": "USDT",
    "commission": "0.31468000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773944370552,
    "fee_amount": 0.0021984,
    "fee_currency": "USDT",
    "commission": "0.00219840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773944370592,
    "fee_amount": 0.0021972,
    "fee_currency": "USDT",
    "commission": "0.00219720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773944370608,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773944370596,
    "fee_amount": 0.0021962,
    "fee_currency": "USDT",
    "commission": "0.00219620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773944370615,
    "fee_amount": 0.002195,
    "fee_currency": "USDT",
    "commission": "0.00219500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773944370615,
    "fee_amount": 0.002194,
    "fee_currency": "USDT",
    "commission": "0.00219400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773944370760,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773946350988,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773946516107,
    "fee_amount": 0.004364,
    "fee_currency": "USDT",
    "commission": "0.00436400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773946516117,
    "fee_amount": 0.002178,
    "fee_currency": "USDT",
    "commission": "0.00217800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773946516122,
    "fee_amount": 0.002179,
    "fee_currency": "USDT",
    "commission": "0.00217900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773946516172,
    "fee_amount": 0.004368,
    "fee_currency": "USDT",
    "commission": "0.00436800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773946564902,
    "fee_amount": 0.00438,
    "fee_currency": "USDT",
    "commission": "0.00438000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773946874051,
    "fee_amount": 0.0021898,
    "fee_currency": "USDT",
    "commission": "0.00218980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947200420,
    "fee_amount": 0.0022036,
    "fee_currency": "USDT",
    "commission": "0.00220360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947205860,
    "fee_amount": 0.0022034,
    "fee_currency": "USDT",
    "commission": "0.00220340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947230452,
    "fee_amount": 0.0022022,
    "fee_currency": "USDT",
    "commission": "0.00220220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947235894,
    "fee_amount": 0.0022014,
    "fee_currency": "USDT",
    "commission": "0.00220140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947235898,
    "fee_amount": 0.00063835,
    "fee_currency": "USDT",
    "commission": "0.00063835 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947235900,
    "fee_amount": 0.00156285,
    "fee_currency": "USDT",
    "commission": "0.00156285 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947269336,
    "fee_amount": 0.002199,
    "fee_currency": "USDT",
    "commission": "0.00219900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947400332,
    "fee_amount": 0.0022008,
    "fee_currency": "USDT",
    "commission": "0.00220080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947490449,
    "fee_amount": 0.0022106,
    "fee_currency": "USDT",
    "commission": "0.00221060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773947547115,
    "fee_amount": 0.0022106,
    "fee_currency": "USDT",
    "commission": "0.00221060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773951784759,
    "fee_amount": 0.0021804,
    "fee_currency": "USDT",
    "commission": "0.00218040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773957335686,
    "fee_amount": 0.0021876,
    "fee_currency": "USDT",
    "commission": "0.00218760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773958099786,
    "fee_amount": 0.0021836,
    "fee_currency": "USDT",
    "commission": "0.00218360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773958152912,
    "fee_amount": 0.0021858,
    "fee_currency": "USDT",
    "commission": "0.00218580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773958264661,
    "fee_amount": 0.002188,
    "fee_currency": "USDT",
    "commission": "0.00218800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773958264661,
    "fee_amount": 0.002189,
    "fee_currency": "USDT",
    "commission": "0.00218900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773958457903,
    "fee_amount": 0.004404,
    "fee_currency": "USDT",
    "commission": "0.00440400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773958457919,
    "fee_amount": 0.0021982,
    "fee_currency": "USDT",
    "commission": "0.00219820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773959597592,
    "fee_amount": 0.0022008,
    "fee_currency": "USDT",
    "commission": "0.00220080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773959597592,
    "fee_amount": 0.0022028,
    "fee_currency": "USDT",
    "commission": "0.00220280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773959597592,
    "fee_amount": 0.0022038,
    "fee_currency": "USDT",
    "commission": "0.00220380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773959597592,
    "fee_amount": 0.0022046,
    "fee_currency": "USDT",
    "commission": "0.00220460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773959597592,
    "fee_amount": 0.002205,
    "fee_currency": "USDT",
    "commission": "0.00220500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773960533606,
    "fee_amount": 0.0022094,
    "fee_currency": "USDT",
    "commission": "0.00220940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773960533707,
    "fee_amount": 0.0022106,
    "fee_currency": "USDT",
    "commission": "0.00221060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773960599157,
    "fee_amount": 0.0022158,
    "fee_currency": "USDT",
    "commission": "0.00221580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773960599211,
    "fee_amount": 0.004432,
    "fee_currency": "USDT",
    "commission": "0.00443200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773960599427,
    "fee_amount": 0.0022168,
    "fee_currency": "USDT",
    "commission": "0.00221680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773960844411,
    "fee_amount": 0.004436,
    "fee_currency": "USDT",
    "commission": "0.00443600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773960844398,
    "fee_amount": 0.002216,
    "fee_currency": "USDT",
    "commission": "0.00221600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773960844403,
    "fee_amount": 0.0022172,
    "fee_currency": "USDT",
    "commission": "0.00221720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773960993913,
    "fee_amount": 0.0022184,
    "fee_currency": "USDT",
    "commission": "0.00221840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773961690551,
    "fee_amount": 0.0022426,
    "fee_currency": "USDT",
    "commission": "0.00224260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773961991049,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773961991104,
    "fee_amount": 0.00223,
    "fee_currency": "USDT",
    "commission": "0.00223000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773962492610,
    "fee_amount": 0.0022246,
    "fee_currency": "USDT",
    "commission": "0.00222460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773962655187,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773962655239,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773962655376,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773963374086,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773964440207,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773964511570,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773964511595,
    "fee_amount": 0.0022152,
    "fee_currency": "USDT",
    "commission": "0.00221520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773964947446,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773964947434,
    "fee_amount": 0.002198,
    "fee_currency": "USDT",
    "commission": "0.00219800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773964947435,
    "fee_amount": 0.0021968,
    "fee_currency": "USDT",
    "commission": "0.00219680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773964947437,
    "fee_amount": 0.0021956,
    "fee_currency": "USDT",
    "commission": "0.00219560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773964947460,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773964947466,
    "fee_amount": 0.0021946,
    "fee_currency": "USDT",
    "commission": "0.00219460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773966121920,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773966121924,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773966121953,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773966121938,
    "fee_amount": 0.002196,
    "fee_currency": "USDT",
    "commission": "0.00219600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773966121974,
    "fee_amount": 0.002195,
    "fee_currency": "USDT",
    "commission": "0.00219500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773966121981,
    "fee_amount": 0.0021938,
    "fee_currency": "USDT",
    "commission": "0.00219380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773966121983,
    "fee_amount": 0.0021928,
    "fee_currency": "USDT",
    "commission": "0.00219280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773966330520,
    "fee_amount": 0.002198,
    "fee_currency": "USDT",
    "commission": "0.00219800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773967329563,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773967557261,
    "fee_amount": 0.004396,
    "fee_currency": "USDT",
    "commission": "0.00439600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773967557248,
    "fee_amount": 0.002195,
    "fee_currency": "USDT",
    "commission": "0.00219500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773967557249,
    "fee_amount": 0.002196,
    "fee_currency": "USDT",
    "commission": "0.00219600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773967557263,
    "fee_amount": 0.0044,
    "fee_currency": "USDT",
    "commission": "0.00440000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773967557249,
    "fee_amount": 0.0021962,
    "fee_currency": "USDT",
    "commission": "0.00219620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773967557252,
    "fee_amount": 0.0021972,
    "fee_currency": "USDT",
    "commission": "0.00219720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773967557325,
    "fee_amount": 0.0021984,
    "fee_currency": "USDT",
    "commission": "0.00219840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773968405021,
    "fee_amount": 0.002204,
    "fee_currency": "USDT",
    "commission": "0.00220400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773968461280,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773968984005,
    "fee_amount": 0.0022092,
    "fee_currency": "USDT",
    "commission": "0.00220920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773969102797,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773969102801,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773969102810,
    "fee_amount": 0.00198396,
    "fee_currency": "USDT",
    "commission": "0.00198396 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773969412454,
    "fee_amount": 0.004424,
    "fee_currency": "USDT",
    "commission": "0.00442400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773969412465,
    "fee_amount": 0.002209,
    "fee_currency": "USDT",
    "commission": "0.00220900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773969412465,
    "fee_amount": 0.00221,
    "fee_currency": "USDT",
    "commission": "0.00221000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773969412489,
    "fee_amount": 0.004428,
    "fee_currency": "USDT",
    "commission": "0.00442800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773969412470,
    "fee_amount": 0.0022112,
    "fee_currency": "USDT",
    "commission": "0.00221120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773969412472,
    "fee_amount": 0.0022122,
    "fee_currency": "USDT",
    "commission": "0.00221220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773969412509,
    "fee_amount": 0.0022134,
    "fee_currency": "USDT",
    "commission": "0.00221340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773971298886,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773971298879,
    "fee_amount": 0.002218,
    "fee_currency": "USDT",
    "commission": "0.00221800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773971300941,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773971868556,
    "fee_amount": 0.0022176,
    "fee_currency": "USDT",
    "commission": "0.00221760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773972126071,
    "fee_amount": 0.0022098,
    "fee_currency": "USDT",
    "commission": "0.00220980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773972126073,
    "fee_amount": 0.0022108,
    "fee_currency": "USDT",
    "commission": "0.00221080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773972255941,
    "fee_amount": 0.0022182,
    "fee_currency": "USDT",
    "commission": "0.00221820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773972907996,
    "fee_amount": 0.0022214,
    "fee_currency": "USDT",
    "commission": "0.00222140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773973264096,
    "fee_amount": 0.0022242,
    "fee_currency": "USDT",
    "commission": "0.00222420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773973415665,
    "fee_amount": 0.0022244,
    "fee_currency": "USDT",
    "commission": "0.00222440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773973415668,
    "fee_amount": 0.0022254,
    "fee_currency": "USDT",
    "commission": "0.00222540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773973927191,
    "fee_amount": 0.002238,
    "fee_currency": "USDT",
    "commission": "0.00223800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773973927336,
    "fee_amount": 0.0022386,
    "fee_currency": "USDT",
    "commission": "0.00223860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773973927338,
    "fee_amount": 0.0022392,
    "fee_currency": "USDT",
    "commission": "0.00223920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773974741497,
    "fee_amount": 0.0022342,
    "fee_currency": "USDT",
    "commission": "0.00223420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773975497372,
    "fee_amount": 0.0022242,
    "fee_currency": "USDT",
    "commission": "0.00222420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773975497372,
    "fee_amount": 0.002223,
    "fee_currency": "USDT",
    "commission": "0.00222300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773975497372,
    "fee_amount": 0.0022218,
    "fee_currency": "USDT",
    "commission": "0.00222180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773975497372,
    "fee_amount": 0.0022208,
    "fee_currency": "USDT",
    "commission": "0.00222080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773975497372,
    "fee_amount": 0.0022196,
    "fee_currency": "USDT",
    "commission": "0.00221960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773976149792,
    "fee_amount": 0.0022282,
    "fee_currency": "USDT",
    "commission": "0.00222820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773976269243,
    "fee_amount": 0.0022306,
    "fee_currency": "USDT",
    "commission": "0.00223060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773976923479,
    "fee_amount": 0.002234,
    "fee_currency": "USDT",
    "commission": "0.00223400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977162281,
    "fee_amount": 0.0022398,
    "fee_currency": "USDT",
    "commission": "0.00223980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977165394,
    "fee_amount": 0.0022376,
    "fee_currency": "USDT",
    "commission": "0.00223760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977241895,
    "fee_amount": 0.0022396,
    "fee_currency": "USDT",
    "commission": "0.00223960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977379238,
    "fee_amount": 0.0022434,
    "fee_currency": "USDT",
    "commission": "0.00224340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773977379269,
    "fee_amount": 0.004496,
    "fee_currency": "USDT",
    "commission": "0.00449600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977379301,
    "fee_amount": 0.0022444,
    "fee_currency": "USDT",
    "commission": "0.00224440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977521008,
    "fee_amount": 0.0022528,
    "fee_currency": "USDT",
    "commission": "0.00225280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977525749,
    "fee_amount": 0.0022508,
    "fee_currency": "USDT",
    "commission": "0.00225080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977547844,
    "fee_amount": 0.0022598,
    "fee_currency": "USDT",
    "commission": "0.00225980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977547844,
    "fee_amount": 0.0022608,
    "fee_currency": "USDT",
    "commission": "0.00226080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977547844,
    "fee_amount": 0.0022618,
    "fee_currency": "USDT",
    "commission": "0.00226180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977553846,
    "fee_amount": 4.518e-05,
    "fee_currency": "USDT",
    "commission": "0.00004518 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977559070,
    "fee_amount": 0.0022584,
    "fee_currency": "USDT",
    "commission": "0.00225840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977700575,
    "fee_amount": 0.002258,
    "fee_currency": "USDT",
    "commission": "0.00225800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773977700575,
    "fee_amount": 0.0022558,
    "fee_currency": "USDT",
    "commission": "0.00225580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773977700593,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773978234463,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773978234450,
    "fee_amount": 0.0022676,
    "fee_currency": "USDT",
    "commission": "0.00226760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773978234451,
    "fee_amount": 0.0022666,
    "fee_currency": "USDT",
    "commission": "0.00226660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773978271685,
    "fee_amount": 0.0022652,
    "fee_currency": "USDT",
    "commission": "0.00226520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773978271703,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773978271688,
    "fee_amount": 0.0022642,
    "fee_currency": "USDT",
    "commission": "0.00226420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773978271693,
    "fee_amount": 0.00196881,
    "fee_currency": "USDT",
    "commission": "0.00196881 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773978271693,
    "fee_amount": 0.00029419,
    "fee_currency": "USDT",
    "commission": "0.00029419 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773978271696,
    "fee_amount": 0.0022618,
    "fee_currency": "USDT",
    "commission": "0.00226180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773979278119,
    "fee_amount": 0.0022624,
    "fee_currency": "USDT",
    "commission": "0.00226240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773979278147,
    "fee_amount": 0.004532,
    "fee_currency": "USDT",
    "commission": "0.00453200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773979278240,
    "fee_amount": 0.0022632,
    "fee_currency": "USDT",
    "commission": "0.00226320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773979278242,
    "fee_amount": 0.0022636,
    "fee_currency": "USDT",
    "commission": "0.00226360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773979468217,
    "fee_amount": 0.0022488,
    "fee_currency": "USDT",
    "commission": "0.00224880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773979750086,
    "fee_amount": 0.0022516,
    "fee_currency": "USDT",
    "commission": "0.00225160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980124502,
    "fee_amount": 0.004528,
    "fee_currency": "USDT",
    "commission": "0.00452800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980124502,
    "fee_amount": 0.004528,
    "fee_currency": "USDT",
    "commission": "0.00452800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980124502,
    "fee_amount": 0.004532,
    "fee_currency": "USDT",
    "commission": "0.00453200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980124490,
    "fee_amount": 0.0022596,
    "fee_currency": "USDT",
    "commission": "0.00225960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980124492,
    "fee_amount": 0.0022608,
    "fee_currency": "USDT",
    "commission": "0.00226080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980124493,
    "fee_amount": 0.002262,
    "fee_currency": "USDT",
    "commission": "0.00226200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980124496,
    "fee_amount": 0.002263,
    "fee_currency": "USDT",
    "commission": "0.00226300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980124508,
    "fee_amount": 0.0022642,
    "fee_currency": "USDT",
    "commission": "0.00226420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980124534,
    "fee_amount": 0.004536,
    "fee_currency": "USDT",
    "commission": "0.00453600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980124537,
    "fee_amount": 0.00454,
    "fee_currency": "USDT",
    "commission": "0.00454000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980124808,
    "fee_amount": 0.004552,
    "fee_currency": "USDT",
    "commission": "0.00455200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980124808,
    "fee_amount": 0.004556,
    "fee_currency": "USDT",
    "commission": "0.00455600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980131898,
    "fee_amount": 0.002266,
    "fee_currency": "USDT",
    "commission": "0.00226600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980131902,
    "fee_amount": 0.002265,
    "fee_currency": "USDT",
    "commission": "0.00226500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980131902,
    "fee_amount": 0.0022648,
    "fee_currency": "USDT",
    "commission": "0.00226480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980238590,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980451812,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980451799,
    "fee_amount": 0.0022638,
    "fee_currency": "USDT",
    "commission": "0.00226380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980451802,
    "fee_amount": 0.0022626,
    "fee_currency": "USDT",
    "commission": "0.00226260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980451808,
    "fee_amount": 0.0022616,
    "fee_currency": "USDT",
    "commission": "0.00226160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980470580,
    "fee_amount": 0.0022612,
    "fee_currency": "USDT",
    "commission": "0.00226120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980470586,
    "fee_amount": 0.00226,
    "fee_currency": "USDT",
    "commission": "0.00226000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980470586,
    "fee_amount": 0.0022592,
    "fee_currency": "USDT",
    "commission": "0.00225920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980470617,
    "fee_amount": 0.0022588,
    "fee_currency": "USDT",
    "commission": "0.00225880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980470636,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980470662,
    "fee_amount": 0.0022594,
    "fee_currency": "USDT",
    "commission": "0.00225940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980625527,
    "fee_amount": 0.0022642,
    "fee_currency": "USDT",
    "commission": "0.00226420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980625531,
    "fee_amount": 0.0022652,
    "fee_currency": "USDT",
    "commission": "0.00226520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773980625559,
    "fee_amount": 0.0022662,
    "fee_currency": "USDT",
    "commission": "0.00226620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773980625679,
    "fee_amount": 0.004536,
    "fee_currency": "USDT",
    "commission": "0.00453600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773982532095,
    "fee_amount": 0.0022384,
    "fee_currency": "USDT",
    "commission": "0.00223840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773982532095,
    "fee_amount": 0.0022374,
    "fee_currency": "USDT",
    "commission": "0.00223740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773982532098,
    "fee_amount": 0.0022362,
    "fee_currency": "USDT",
    "commission": "0.00223620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773982532100,
    "fee_amount": 0.0022352,
    "fee_currency": "USDT",
    "commission": "0.00223520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773982532107,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773982532113,
    "fee_amount": 0.002234,
    "fee_currency": "USDT",
    "commission": "0.00223400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773982532113,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773982911961,
    "fee_amount": 0.01192,
    "fee_currency": "USDT",
    "commission": "0.01192000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773984497782,
    "fee_amount": 0.0022214,
    "fee_currency": "USDT",
    "commission": "0.00222140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773984778855,
    "fee_amount": 0.0022252,
    "fee_currency": "USDT",
    "commission": "0.00222520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773985348185,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773985348177,
    "fee_amount": 0.002233,
    "fee_currency": "USDT",
    "commission": "0.00223300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773985350086,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773985350072,
    "fee_amount": 0.0022298,
    "fee_currency": "USDT",
    "commission": "0.00222980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773985350086,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773985350072,
    "fee_amount": 0.0022288,
    "fee_currency": "USDT",
    "commission": "0.00222880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773985350075,
    "fee_amount": 0.0022278,
    "fee_currency": "USDT",
    "commission": "0.00222780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773985350075,
    "fee_amount": 0.0022266,
    "fee_currency": "USDT",
    "commission": "0.00222660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773985350115,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773988715261,
    "fee_amount": 0.0022126,
    "fee_currency": "USDT",
    "commission": "0.00221260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773988910308,
    "fee_amount": 0.00221,
    "fee_currency": "USDT",
    "commission": "0.00221000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773988921864,
    "fee_amount": 0.002213,
    "fee_currency": "USDT",
    "commission": "0.00221300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989207233,
    "fee_amount": 0.002211,
    "fee_currency": "USDT",
    "commission": "0.00221100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989249848,
    "fee_amount": 0.0022112,
    "fee_currency": "USDT",
    "commission": "0.00221120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773989632275,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989632263,
    "fee_amount": 0.0021986,
    "fee_currency": "USDT",
    "commission": "0.00219860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989632265,
    "fee_amount": 0.0021974,
    "fee_currency": "USDT",
    "commission": "0.00219740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989632268,
    "fee_amount": 0.0021964,
    "fee_currency": "USDT",
    "commission": "0.00219640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989690724,
    "fee_amount": 0.0021988,
    "fee_currency": "USDT",
    "commission": "0.00219880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989709229,
    "fee_amount": 0.0022,
    "fee_currency": "USDT",
    "commission": "0.00220000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773989856071,
    "fee_amount": 0.002203,
    "fee_currency": "USDT",
    "commission": "0.00220300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773990082370,
    "fee_amount": 0.0021852,
    "fee_currency": "USDT",
    "commission": "0.00218520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773990085907,
    "fee_amount": 0.0021852,
    "fee_currency": "USDT",
    "commission": "0.00218520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773990085907,
    "fee_amount": 0.0021862,
    "fee_currency": "USDT",
    "commission": "0.00218620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773990091550,
    "fee_amount": 0.0021836,
    "fee_currency": "USDT",
    "commission": "0.00218360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773990121624,
    "fee_amount": 0.0021808,
    "fee_currency": "USDT",
    "commission": "0.00218080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773990215124,
    "fee_amount": 0.0021778,
    "fee_currency": "USDT",
    "commission": "0.00217780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773991516105,
    "fee_amount": 0.0021962,
    "fee_currency": "USDT",
    "commission": "0.00219620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773991516107,
    "fee_amount": 0.0021972,
    "fee_currency": "USDT",
    "commission": "0.00219720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773992039896,
    "fee_amount": 0.0022058,
    "fee_currency": "USDT",
    "commission": "0.00220580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773992039972,
    "fee_amount": 0.004416,
    "fee_currency": "USDT",
    "commission": "0.00441600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773993525821,
    "fee_amount": 0.002215,
    "fee_currency": "USDT",
    "commission": "0.00221500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773993815021,
    "fee_amount": 0.004448,
    "fee_currency": "USDT",
    "commission": "0.00444800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773993815020,
    "fee_amount": 0.0022212,
    "fee_currency": "USDT",
    "commission": "0.00222120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773993815059,
    "fee_amount": 0.004452,
    "fee_currency": "USDT",
    "commission": "0.00445200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773993815080,
    "fee_amount": 0.00022224,
    "fee_currency": "USDT",
    "commission": "0.00022224 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773993815082,
    "fee_amount": 0.00200016,
    "fee_currency": "USDT",
    "commission": "0.00200016 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773993815379,
    "fee_amount": 0.00446,
    "fee_currency": "USDT",
    "commission": "0.00446000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773994037871,
    "fee_amount": 0.00448,
    "fee_currency": "USDT",
    "commission": "0.00448000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994037851,
    "fee_amount": 0.0022358,
    "fee_currency": "USDT",
    "commission": "0.00223580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994037860,
    "fee_amount": 0.002237,
    "fee_currency": "USDT",
    "commission": "0.00223700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773994046411,
    "fee_amount": 0.004488,
    "fee_currency": "USDT",
    "commission": "0.00448800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994056779,
    "fee_amount": 0.0022406,
    "fee_currency": "USDT",
    "commission": "0.00224060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773994286037,
    "fee_amount": 0.004488,
    "fee_currency": "USDT",
    "commission": "0.00448800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994419784,
    "fee_amount": 0.0022406,
    "fee_currency": "USDT",
    "commission": "0.00224060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773994539007,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994538990,
    "fee_amount": 0.0022364,
    "fee_currency": "USDT",
    "commission": "0.00223640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994538992,
    "fee_amount": 0.0022356,
    "fee_currency": "USDT",
    "commission": "0.00223560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994538992,
    "fee_amount": 0.00084938,
    "fee_currency": "USDT",
    "commission": "0.00084938 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994538994,
    "fee_amount": 0.00138582,
    "fee_currency": "USDT",
    "commission": "0.00138582 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773994538994,
    "fee_amount": 0.0022342,
    "fee_currency": "USDT",
    "commission": "0.00223420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773995635812,
    "fee_amount": 0.0008711,
    "fee_currency": "USDT",
    "commission": "0.00087110 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773995635812,
    "fee_amount": 0.0013625,
    "fee_currency": "USDT",
    "commission": "0.00136250 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773995635827,
    "fee_amount": 0.0022348,
    "fee_currency": "USDT",
    "commission": "0.00223480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773996276466,
    "fee_amount": 0.002229,
    "fee_currency": "USDT",
    "commission": "0.00222900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773996276466,
    "fee_amount": 0.0022278,
    "fee_currency": "USDT",
    "commission": "0.00222780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773996276466,
    "fee_amount": 0.0022268,
    "fee_currency": "USDT",
    "commission": "0.00222680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773996494711,
    "fee_amount": 0.0022224,
    "fee_currency": "USDT",
    "commission": "0.00222240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773997201864,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773997201869,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997201856,
    "fee_amount": 0.00124835,
    "fee_currency": "USDT",
    "commission": "0.00124835 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997201856,
    "fee_amount": 0.00098085,
    "fee_currency": "USDT",
    "commission": "0.00098085 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997201858,
    "fee_amount": 0.002228,
    "fee_currency": "USDT",
    "commission": "0.00222800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997201858,
    "fee_amount": 0.002227,
    "fee_currency": "USDT",
    "commission": "0.00222700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997201858,
    "fee_amount": 0.0022258,
    "fee_currency": "USDT",
    "commission": "0.00222580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997226304,
    "fee_amount": 0.0022176,
    "fee_currency": "USDT",
    "commission": "0.00221760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1773997226317,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997226309,
    "fee_amount": 0.0022164,
    "fee_currency": "USDT",
    "commission": "0.00221640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997226340,
    "fee_amount": 0.00137355,
    "fee_currency": "USDT",
    "commission": "0.00137355 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773997226341,
    "fee_amount": 0.00084185,
    "fee_currency": "USDT",
    "commission": "0.00084185 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1773999156522,
    "fee_amount": 0.002209,
    "fee_currency": "USDT",
    "commission": "0.00220900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774000295135,
    "fee_amount": 0.0022036,
    "fee_currency": "USDT",
    "commission": "0.00220360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774001261610,
    "fee_amount": 0.0021964,
    "fee_currency": "USDT",
    "commission": "0.00219640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774002985659,
    "fee_amount": 0.0022022,
    "fee_currency": "USDT",
    "commission": "0.00220220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774002985666,
    "fee_amount": 0.00037454,
    "fee_currency": "USDT",
    "commission": "0.00037454 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774006209695,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774006428260,
    "fee_amount": 0.0021956,
    "fee_currency": "USDT",
    "commission": "0.00219560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774006428261,
    "fee_amount": 0.0021944,
    "fee_currency": "USDT",
    "commission": "0.00219440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774006428261,
    "fee_amount": 0.0021934,
    "fee_currency": "USDT",
    "commission": "0.00219340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774006428273,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774006428273,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774006821373,
    "fee_amount": 0.0021956,
    "fee_currency": "USDT",
    "commission": "0.00219560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774007310149,
    "fee_amount": 0.0021978,
    "fee_currency": "USDT",
    "commission": "0.00219780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774007341317,
    "fee_amount": 0.0021998,
    "fee_currency": "USDT",
    "commission": "0.00219980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774007341318,
    "fee_amount": 0.002201,
    "fee_currency": "USDT",
    "commission": "0.00220100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774007341327,
    "fee_amount": 0.002202,
    "fee_currency": "USDT",
    "commission": "0.00220200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774007341349,
    "fee_amount": 0.004408,
    "fee_currency": "USDT",
    "commission": "0.00440800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774007893771,
    "fee_amount": 0.0022146,
    "fee_currency": "USDT",
    "commission": "0.00221460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774008423553,
    "fee_amount": 0.0022348,
    "fee_currency": "USDT",
    "commission": "0.00223480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774012894287,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774012896296,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774012904529,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774013696202,
    "fee_amount": 0.002184,
    "fee_currency": "USDT",
    "commission": "0.00218400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774013696202,
    "fee_amount": 0.0021828,
    "fee_currency": "USDT",
    "commission": "0.00218280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774014213384,
    "fee_amount": 0.0021954,
    "fee_currency": "USDT",
    "commission": "0.00219540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774014829590,
    "fee_amount": 0.0021864,
    "fee_currency": "USDT",
    "commission": "0.00218640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774014829619,
    "fee_amount": 0.00438,
    "fee_currency": "USDT",
    "commission": "0.00438000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774014829593,
    "fee_amount": 0.0021876,
    "fee_currency": "USDT",
    "commission": "0.00218760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774014829600,
    "fee_amount": 0.0021886,
    "fee_currency": "USDT",
    "commission": "0.00218860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774014829602,
    "fee_amount": 0.0021898,
    "fee_currency": "USDT",
    "commission": "0.00218980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774014829671,
    "fee_amount": 0.004384,
    "fee_currency": "USDT",
    "commission": "0.00438400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774014829751,
    "fee_amount": 0.0021908,
    "fee_currency": "USDT",
    "commission": "0.00219080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774015296988,
    "fee_amount": 0.0021766,
    "fee_currency": "USDT",
    "commission": "0.00217660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774017713491,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774018125218,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774018129904,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774018936896,
    "fee_amount": 0.0021746,
    "fee_currency": "USDT",
    "commission": "0.00217460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774018936896,
    "fee_amount": 0.0021736,
    "fee_currency": "USDT",
    "commission": "0.00217360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774018936896,
    "fee_amount": 0.0021724,
    "fee_currency": "USDT",
    "commission": "0.00217240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774018936896,
    "fee_amount": 0.0021714,
    "fee_currency": "USDT",
    "commission": "0.00217140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774018936896,
    "fee_amount": 0.0021702,
    "fee_currency": "USDT",
    "commission": "0.00217020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019028058,
    "fee_amount": 0.0021648,
    "fee_currency": "USDT",
    "commission": "0.00216480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019191773,
    "fee_amount": 0.0021656,
    "fee_currency": "USDT",
    "commission": "0.00216560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774019191790,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019191783,
    "fee_amount": 0.0021646,
    "fee_currency": "USDT",
    "commission": "0.00216460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774019191829,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019191814,
    "fee_amount": 0.0021634,
    "fee_currency": "USDT",
    "commission": "0.00216340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019194934,
    "fee_amount": 0.002159,
    "fee_currency": "USDT",
    "commission": "0.00215900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019194934,
    "fee_amount": 0.002158,
    "fee_currency": "USDT",
    "commission": "0.00215800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019194934,
    "fee_amount": 0.0021578,
    "fee_currency": "USDT",
    "commission": "0.00215780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019194934,
    "fee_amount": 0.0021568,
    "fee_currency": "USDT",
    "commission": "0.00215680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019194934,
    "fee_amount": 0.0021558,
    "fee_currency": "USDT",
    "commission": "0.00215580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774019214399,
    "fee_amount": 0.002159,
    "fee_currency": "USDT",
    "commission": "0.00215900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774020212716,
    "fee_amount": 0.0021692,
    "fee_currency": "USDT",
    "commission": "0.00216920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774020212716,
    "fee_amount": 0.0021694,
    "fee_currency": "USDT",
    "commission": "0.00216940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774020216333,
    "fee_amount": 0.0021698,
    "fee_currency": "USDT",
    "commission": "0.00216980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774021986520,
    "fee_amount": 0.0021656,
    "fee_currency": "USDT",
    "commission": "0.00216560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774022463279,
    "fee_amount": 0.0021708,
    "fee_currency": "USDT",
    "commission": "0.00217080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774022998036,
    "fee_amount": 0.004348,
    "fee_currency": "USDT",
    "commission": "0.00434800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774022998021,
    "fee_amount": 0.00217,
    "fee_currency": "USDT",
    "commission": "0.00217000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774022998043,
    "fee_amount": 0.004352,
    "fee_currency": "USDT",
    "commission": "0.00435200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774022998024,
    "fee_amount": 0.002171,
    "fee_currency": "USDT",
    "commission": "0.00217100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774022998027,
    "fee_amount": 0.0021722,
    "fee_currency": "USDT",
    "commission": "0.00217220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774022998029,
    "fee_amount": 0.0021732,
    "fee_currency": "USDT",
    "commission": "0.00217320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774022998034,
    "fee_amount": 0.0021742,
    "fee_currency": "USDT",
    "commission": "0.00217420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774023306672,
    "fee_amount": 0.004344,
    "fee_currency": "USDT",
    "commission": "0.00434400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774023306659,
    "fee_amount": 0.002169,
    "fee_currency": "USDT",
    "commission": "0.00216900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774023306673,
    "fee_amount": 0.004348,
    "fee_currency": "USDT",
    "commission": "0.00434800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774023306660,
    "fee_amount": 0.0021702,
    "fee_currency": "USDT",
    "commission": "0.00217020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774023306660,
    "fee_amount": 0.0021712,
    "fee_currency": "USDT",
    "commission": "0.00217120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774023306662,
    "fee_amount": 0.0021724,
    "fee_currency": "USDT",
    "commission": "0.00217240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774023306664,
    "fee_amount": 0.0021734,
    "fee_currency": "USDT",
    "commission": "0.00217340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774023438765,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774023565387,
    "fee_amount": 0.002168,
    "fee_currency": "USDT",
    "commission": "0.00216800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774024436804,
    "fee_amount": 0.0021485,
    "fee_currency": "USDT",
    "commission": "0.00214850 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774024436841,
    "fee_amount": 2.17e-05,
    "fee_currency": "USDT",
    "commission": "0.00002170 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774024436882,
    "fee_amount": 0.0021712,
    "fee_currency": "USDT",
    "commission": "0.00217120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774026105213,
    "fee_amount": 0.004376,
    "fee_currency": "USDT",
    "commission": "0.00437600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774026105200,
    "fee_amount": 0.0021846,
    "fee_currency": "USDT",
    "commission": "0.00218460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774026105200,
    "fee_amount": 0.0021856,
    "fee_currency": "USDT",
    "commission": "0.00218560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774026105290,
    "fee_amount": 0.0021868,
    "fee_currency": "USDT",
    "commission": "0.00218680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774026951791,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774027011721,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774034651225,
    "fee_amount": 0.0021542,
    "fee_currency": "USDT",
    "commission": "0.00215420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774034651253,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774035244932,
    "fee_amount": 0.002152,
    "fee_currency": "USDT",
    "commission": "0.00215200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774035266066,
    "fee_amount": 0.002148,
    "fee_currency": "USDT",
    "commission": "0.00214800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774035643274,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774037719288,
    "fee_amount": 0.0021696,
    "fee_currency": "USDT",
    "commission": "0.00216960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774037725571,
    "fee_amount": 0.0021686,
    "fee_currency": "USDT",
    "commission": "0.00216860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774041600676,
    "fee_amount": 0.0021992,
    "fee_currency": "USDT",
    "commission": "0.00219920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774041600676,
    "fee_amount": 0.0021982,
    "fee_currency": "USDT",
    "commission": "0.00219820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774041600676,
    "fee_amount": 0.0021972,
    "fee_currency": "USDT",
    "commission": "0.00219720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774041600676,
    "fee_amount": 0.0021962,
    "fee_currency": "USDT",
    "commission": "0.00219620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774041600676,
    "fee_amount": 0.002195,
    "fee_currency": "USDT",
    "commission": "0.00219500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774043436465,
    "fee_amount": 0.0021878,
    "fee_currency": "USDT",
    "commission": "0.00218780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774044053045,
    "fee_amount": 0.0021962,
    "fee_currency": "USDT",
    "commission": "0.00219620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774044806550,
    "fee_amount": 0.0022036,
    "fee_currency": "USDT",
    "commission": "0.00220360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774045319178,
    "fee_amount": 0.0021974,
    "fee_currency": "USDT",
    "commission": "0.00219740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774045386843,
    "fee_amount": 0.0005054,
    "fee_currency": "USDT",
    "commission": "0.00050540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774045387366,
    "fee_amount": 0.001692,
    "fee_currency": "USDT",
    "commission": "0.00169200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774045388415,
    "fee_amount": 0.00177892,
    "fee_currency": "USDT",
    "commission": "0.00177892 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774046345700,
    "fee_amount": 0.0021992,
    "fee_currency": "USDT",
    "commission": "0.00219920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774046345722,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774046345706,
    "fee_amount": 0.002198,
    "fee_currency": "USDT",
    "commission": "0.00219800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774046971794,
    "fee_amount": 0.00438,
    "fee_currency": "USDT",
    "commission": "0.00438000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774046971794,
    "fee_amount": 0.004384,
    "fee_currency": "USDT",
    "commission": "0.00438400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774047903300,
    "fee_amount": 0.0021798,
    "fee_currency": "USDT",
    "commission": "0.00217980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774047917037,
    "fee_amount": 0.0021788,
    "fee_currency": "USDT",
    "commission": "0.00217880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774047988216,
    "fee_amount": 0.0021786,
    "fee_currency": "USDT",
    "commission": "0.00217860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774047988260,
    "fee_amount": 0.0021776,
    "fee_currency": "USDT",
    "commission": "0.00217760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774047988316,
    "fee_amount": 0.0021764,
    "fee_currency": "USDT",
    "commission": "0.00217640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774051413166,
    "fee_amount": 0.00202628,
    "fee_currency": "USDT",
    "commission": "0.00202628 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774051413168,
    "fee_amount": 0.00015252,
    "fee_currency": "USDT",
    "commission": "0.00015252 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774051413186,
    "fee_amount": 0.004368,
    "fee_currency": "USDT",
    "commission": "0.00436800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774051663985,
    "fee_amount": 0.0021814,
    "fee_currency": "USDT",
    "commission": "0.00218140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774051663989,
    "fee_amount": 0.0021824,
    "fee_currency": "USDT",
    "commission": "0.00218240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774051663989,
    "fee_amount": 0.0021836,
    "fee_currency": "USDT",
    "commission": "0.00218360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774051663992,
    "fee_amount": 0.0021846,
    "fee_currency": "USDT",
    "commission": "0.00218460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774051664009,
    "fee_amount": 0.004372,
    "fee_currency": "USDT",
    "commission": "0.00437200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774051664048,
    "fee_amount": 0.004372,
    "fee_currency": "USDT",
    "commission": "0.00437200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774051664073,
    "fee_amount": 0.002184,
    "fee_currency": "USDT",
    "commission": "0.00218400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052228713,
    "fee_amount": 0.002206,
    "fee_currency": "USDT",
    "commission": "0.00220600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052270749,
    "fee_amount": 0.0009936,
    "fee_currency": "USDT",
    "commission": "0.00099360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774052303305,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774052303308,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052303294,
    "fee_amount": 0.0022048,
    "fee_currency": "USDT",
    "commission": "0.00220480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052303306,
    "fee_amount": 0.0022036,
    "fee_currency": "USDT",
    "commission": "0.00220360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052303358,
    "fee_amount": 0.0022038,
    "fee_currency": "USDT",
    "commission": "0.00220380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774052620262,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052621517,
    "fee_amount": 0.0022162,
    "fee_currency": "USDT",
    "commission": "0.00221620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052820507,
    "fee_amount": 0.0022194,
    "fee_currency": "USDT",
    "commission": "0.00221940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774052820515,
    "fee_amount": 0.0022206,
    "fee_currency": "USDT",
    "commission": "0.00222060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053138803,
    "fee_amount": 0.002222,
    "fee_currency": "USDT",
    "commission": "0.00222200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053141520,
    "fee_amount": 0.002225,
    "fee_currency": "USDT",
    "commission": "0.00222500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053141541,
    "fee_amount": 0.0022268,
    "fee_currency": "USDT",
    "commission": "0.00222680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053141541,
    "fee_amount": 0.002227,
    "fee_currency": "USDT",
    "commission": "0.00222700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053450305,
    "fee_amount": 0.0022314,
    "fee_currency": "USDT",
    "commission": "0.00223140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053450473,
    "fee_amount": 0.0022316,
    "fee_currency": "USDT",
    "commission": "0.00223160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053450473,
    "fee_amount": 0.0022326,
    "fee_currency": "USDT",
    "commission": "0.00223260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053547772,
    "fee_amount": 0.002231,
    "fee_currency": "USDT",
    "commission": "0.00223100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053557925,
    "fee_amount": 0.002235,
    "fee_currency": "USDT",
    "commission": "0.00223500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053591741,
    "fee_amount": 0.002235,
    "fee_currency": "USDT",
    "commission": "0.00223500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053694746,
    "fee_amount": 0.0022298,
    "fee_currency": "USDT",
    "commission": "0.00222980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774053694747,
    "fee_amount": 0.0022286,
    "fee_currency": "USDT",
    "commission": "0.00222860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774054204006,
    "fee_amount": 0.0022394,
    "fee_currency": "USDT",
    "commission": "0.00223940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774054204006,
    "fee_amount": 0.0022404,
    "fee_currency": "USDT",
    "commission": "0.00224040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774054296060,
    "fee_amount": 0.0022456,
    "fee_currency": "USDT",
    "commission": "0.00224560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774054307346,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774054307441,
    "fee_amount": 0.0022336,
    "fee_currency": "USDT",
    "commission": "0.00223360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774054307441,
    "fee_amount": 0.0022328,
    "fee_currency": "USDT",
    "commission": "0.00223280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774054307489,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774054307500,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774054514986,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774054579525,
    "fee_amount": 0.002225,
    "fee_currency": "USDT",
    "commission": "0.00222500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055311005,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055311000,
    "fee_amount": 0.0022282,
    "fee_currency": "USDT",
    "commission": "0.00222820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055311019,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055446204,
    "fee_amount": 0.004476,
    "fee_currency": "USDT",
    "commission": "0.00447600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055446191,
    "fee_amount": 0.0022348,
    "fee_currency": "USDT",
    "commission": "0.00223480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055446192,
    "fee_amount": 0.0022358,
    "fee_currency": "USDT",
    "commission": "0.00223580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055446212,
    "fee_amount": 0.00448,
    "fee_currency": "USDT",
    "commission": "0.00448000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055446192,
    "fee_amount": 0.002237,
    "fee_currency": "USDT",
    "commission": "0.00223700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055446195,
    "fee_amount": 0.002238,
    "fee_currency": "USDT",
    "commission": "0.00223800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055446203,
    "fee_amount": 0.0022392,
    "fee_currency": "USDT",
    "commission": "0.00223920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055446318,
    "fee_amount": 0.002239,
    "fee_currency": "USDT",
    "commission": "0.00223900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055446346,
    "fee_amount": 0.00448,
    "fee_currency": "USDT",
    "commission": "0.00448000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774055446346,
    "fee_amount": 0.0022392,
    "fee_currency": "USDT",
    "commission": "0.00223920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055843640,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055843662,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774055973548,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774056332009,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774056332016,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774056332000,
    "fee_amount": 0.0022246,
    "fee_currency": "USDT",
    "commission": "0.00222460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774056332000,
    "fee_amount": 0.0022236,
    "fee_currency": "USDT",
    "commission": "0.00222360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774056735089,
    "fee_amount": 0.002235,
    "fee_currency": "USDT",
    "commission": "0.00223500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774057003259,
    "fee_amount": 0.0022372,
    "fee_currency": "USDT",
    "commission": "0.00223720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774057006881,
    "fee_amount": 0.004484,
    "fee_currency": "USDT",
    "commission": "0.00448400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774058818097,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774058818091,
    "fee_amount": 0.0022364,
    "fee_currency": "USDT",
    "commission": "0.00223640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774059732956,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774059732949,
    "fee_amount": 0.00077707,
    "fee_currency": "USDT",
    "commission": "0.00077707 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774059732986,
    "fee_amount": 0.00144313,
    "fee_currency": "USDT",
    "commission": "0.00144313 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774061120389,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774063335205,
    "fee_amount": 0.0021998,
    "fee_currency": "USDT",
    "commission": "0.00219980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774063335205,
    "fee_amount": 0.0021988,
    "fee_currency": "USDT",
    "commission": "0.00219880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774063335205,
    "fee_amount": 0.0021976,
    "fee_currency": "USDT",
    "commission": "0.00219760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774063335223,
    "fee_amount": 0.0021966,
    "fee_currency": "USDT",
    "commission": "0.00219660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774063335223,
    "fee_amount": 0.0021954,
    "fee_currency": "USDT",
    "commission": "0.00219540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774063380390,
    "fee_amount": 0.0021972,
    "fee_currency": "USDT",
    "commission": "0.00219720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774064814189,
    "fee_amount": 0.004408,
    "fee_currency": "USDT",
    "commission": "0.00440800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774066981965,
    "fee_amount": 0.0022158,
    "fee_currency": "USDT",
    "commission": "0.00221580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774067897501,
    "fee_amount": 0.0022028,
    "fee_currency": "USDT",
    "commission": "0.00220280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774067897506,
    "fee_amount": 0.0022016,
    "fee_currency": "USDT",
    "commission": "0.00220160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774068676814,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774068676796,
    "fee_amount": 0.0021984,
    "fee_currency": "USDT",
    "commission": "0.00219840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774068676874,
    "fee_amount": 0.0021954,
    "fee_currency": "USDT",
    "commission": "0.00219540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774069430910,
    "fee_amount": 0.0021958,
    "fee_currency": "USDT",
    "commission": "0.00219580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774075175799,
    "fee_amount": 0.004416,
    "fee_currency": "USDT",
    "commission": "0.00441600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774075175804,
    "fee_amount": 0.00442,
    "fee_currency": "USDT",
    "commission": "0.00442000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774078587225,
    "fee_amount": 0.004412,
    "fee_currency": "USDT",
    "commission": "0.00441200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774078587208,
    "fee_amount": 0.002203,
    "fee_currency": "USDT",
    "commission": "0.00220300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774078587229,
    "fee_amount": 0.004416,
    "fee_currency": "USDT",
    "commission": "0.00441600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774078587214,
    "fee_amount": 0.0022042,
    "fee_currency": "USDT",
    "commission": "0.00220420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774078587360,
    "fee_amount": 0.004416,
    "fee_currency": "USDT",
    "commission": "0.00441600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774078587342,
    "fee_amount": 0.0022052,
    "fee_currency": "USDT",
    "commission": "0.00220520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774078587370,
    "fee_amount": 0.00442,
    "fee_currency": "USDT",
    "commission": "0.00442000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774078605955,
    "fee_amount": 0.004428,
    "fee_currency": "USDT",
    "commission": "0.00442800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774078605989,
    "fee_amount": 0.004432,
    "fee_currency": "USDT",
    "commission": "0.00443200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774078607026,
    "fee_amount": 0.004436,
    "fee_currency": "USDT",
    "commission": "0.00443600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774079676504,
    "fee_amount": 0.0021976,
    "fee_currency": "USDT",
    "commission": "0.00219760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774079713980,
    "fee_amount": 0.004412,
    "fee_currency": "USDT",
    "commission": "0.00441200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774079713970,
    "fee_amount": 0.0022022,
    "fee_currency": "USDT",
    "commission": "0.00220220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774079713972,
    "fee_amount": 0.0022032,
    "fee_currency": "USDT",
    "commission": "0.00220320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774079713982,
    "fee_amount": 0.0022044,
    "fee_currency": "USDT",
    "commission": "0.00220440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774079713999,
    "fee_amount": 0.0022054,
    "fee_currency": "USDT",
    "commission": "0.00220540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774079714052,
    "fee_amount": 0.004416,
    "fee_currency": "USDT",
    "commission": "0.00441600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080052780,
    "fee_amount": 0.004412,
    "fee_currency": "USDT",
    "commission": "0.00441200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080052774,
    "fee_amount": 0.002203,
    "fee_currency": "USDT",
    "commission": "0.00220300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080052816,
    "fee_amount": 0.004416,
    "fee_currency": "USDT",
    "commission": "0.00441600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080052821,
    "fee_amount": 0.00442,
    "fee_currency": "USDT",
    "commission": "0.00442000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080052802,
    "fee_amount": 0.0022042,
    "fee_currency": "USDT",
    "commission": "0.00220420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080052802,
    "fee_amount": 0.0022052,
    "fee_currency": "USDT",
    "commission": "0.00220520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080052826,
    "fee_amount": 0.004424,
    "fee_currency": "USDT",
    "commission": "0.00442400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080052826,
    "fee_amount": 0.004428,
    "fee_currency": "USDT",
    "commission": "0.00442800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080052802,
    "fee_amount": 0.0022064,
    "fee_currency": "USDT",
    "commission": "0.00220640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080052802,
    "fee_amount": 0.0022074,
    "fee_currency": "USDT",
    "commission": "0.00220740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080052889,
    "fee_amount": 0.004424,
    "fee_currency": "USDT",
    "commission": "0.00442400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080053528,
    "fee_amount": 0.0022074,
    "fee_currency": "USDT",
    "commission": "0.00220740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080093829,
    "fee_amount": 0.0022054,
    "fee_currency": "USDT",
    "commission": "0.00220540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080093830,
    "fee_amount": 0.0022044,
    "fee_currency": "USDT",
    "commission": "0.00220440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080246075,
    "fee_amount": 0.0022128,
    "fee_currency": "USDT",
    "commission": "0.00221280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080298506,
    "fee_amount": 0.004452,
    "fee_currency": "USDT",
    "commission": "0.00445200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080298489,
    "fee_amount": 0.0022228,
    "fee_currency": "USDT",
    "commission": "0.00222280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080308158,
    "fee_amount": 0.002219,
    "fee_currency": "USDT",
    "commission": "0.00221900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080320951,
    "fee_amount": 0.0022168,
    "fee_currency": "USDT",
    "commission": "0.00221680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080606238,
    "fee_amount": 0.004468,
    "fee_currency": "USDT",
    "commission": "0.00446800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080606240,
    "fee_amount": 0.0022294,
    "fee_currency": "USDT",
    "commission": "0.00222940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080606297,
    "fee_amount": 0.0022306,
    "fee_currency": "USDT",
    "commission": "0.00223060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080610558,
    "fee_amount": 0.0022332,
    "fee_currency": "USDT",
    "commission": "0.00223320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080633076,
    "fee_amount": 0.002231,
    "fee_currency": "USDT",
    "commission": "0.00223100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080760257,
    "fee_amount": 0.002231,
    "fee_currency": "USDT",
    "commission": "0.00223100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080760257,
    "fee_amount": 0.0022298,
    "fee_currency": "USDT",
    "commission": "0.00222980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080832593,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080832575,
    "fee_amount": 0.0022286,
    "fee_currency": "USDT",
    "commission": "0.00222860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080832595,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080832581,
    "fee_amount": 0.0022266,
    "fee_currency": "USDT",
    "commission": "0.00222660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774080832581,
    "fee_amount": 0.0022246,
    "fee_currency": "USDT",
    "commission": "0.00222460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080832602,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774080832676,
    "fee_amount": 0.32,
    "fee_currency": "USDT",
    "commission": "0.32000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774081717253,
    "fee_amount": 0.00446,
    "fee_currency": "USDT",
    "commission": "0.00446000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774081809853,
    "fee_amount": 8.934e-05,
    "fee_currency": "USDT",
    "commission": "0.00008934 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774081809855,
    "fee_amount": 0.00214406,
    "fee_currency": "USDT",
    "commission": "0.00214406 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774082109510,
    "fee_amount": 0.0022294,
    "fee_currency": "USDT",
    "commission": "0.00222940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774082245498,
    "fee_amount": 0.0022238,
    "fee_currency": "USDT",
    "commission": "0.00222380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774082245515,
    "fee_amount": 0.002223,
    "fee_currency": "USDT",
    "commission": "0.00222300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774082597510,
    "fee_amount": 0.00435215,
    "fee_currency": "USDT",
    "commission": "0.00435215 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774082597566,
    "fee_amount": 0.00011985,
    "fee_currency": "USDT",
    "commission": "0.00011985 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774082597688,
    "fee_amount": 0.004472,
    "fee_currency": "USDT",
    "commission": "0.00447200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774082597722,
    "fee_amount": 0.004476,
    "fee_currency": "USDT",
    "commission": "0.00447600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774082609318,
    "fee_amount": 0.004468,
    "fee_currency": "USDT",
    "commission": "0.00446800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774083206880,
    "fee_amount": 0.004456,
    "fee_currency": "USDT",
    "commission": "0.00445600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774083819616,
    "fee_amount": 0.004452,
    "fee_currency": "USDT",
    "commission": "0.00445200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774083822587,
    "fee_amount": 0.004456,
    "fee_currency": "USDT",
    "commission": "0.00445600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774085663682,
    "fee_amount": 0.004436,
    "fee_currency": "USDT",
    "commission": "0.00443600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774085663843,
    "fee_amount": 0.002213,
    "fee_currency": "USDT",
    "commission": "0.00221300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774086959095,
    "fee_amount": 0.004444,
    "fee_currency": "USDT",
    "commission": "0.00444400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774086959086,
    "fee_amount": 0.0022168,
    "fee_currency": "USDT",
    "commission": "0.00221680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774086959235,
    "fee_amount": 0.004448,
    "fee_currency": "USDT",
    "commission": "0.00444800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774086959215,
    "fee_amount": 0.002218,
    "fee_currency": "USDT",
    "commission": "0.00221800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774087202517,
    "fee_amount": 0.0022074,
    "fee_currency": "USDT",
    "commission": "0.00220740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774087202517,
    "fee_amount": 0.0022066,
    "fee_currency": "USDT",
    "commission": "0.00220660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774087202517,
    "fee_amount": 0.0022064,
    "fee_currency": "USDT",
    "commission": "0.00220640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774089119950,
    "fee_amount": 0.004464,
    "fee_currency": "USDT",
    "commission": "0.00446400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774089119944,
    "fee_amount": 0.002228,
    "fee_currency": "USDT",
    "commission": "0.00222800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774089416231,
    "fee_amount": 0.00446,
    "fee_currency": "USDT",
    "commission": "0.00446000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774089416268,
    "fee_amount": 0.0022252,
    "fee_currency": "USDT",
    "commission": "0.00222520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774089416311,
    "fee_amount": 0.004464,
    "fee_currency": "USDT",
    "commission": "0.00446400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774089416291,
    "fee_amount": 0.0022264,
    "fee_currency": "USDT",
    "commission": "0.00222640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774089676155,
    "fee_amount": 0.0022268,
    "fee_currency": "USDT",
    "commission": "0.00222680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774089875099,
    "fee_amount": 0.0022346,
    "fee_currency": "USDT",
    "commission": "0.00223460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774089918114,
    "fee_amount": 0.004492,
    "fee_currency": "USDT",
    "commission": "0.00449200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774089918210,
    "fee_amount": 0.004496,
    "fee_currency": "USDT",
    "commission": "0.00449600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774089918210,
    "fee_amount": 0.00109935,
    "fee_currency": "USDT",
    "commission": "0.00109935 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090187449,
    "fee_amount": 0.004488,
    "fee_currency": "USDT",
    "commission": "0.00448800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090187529,
    "fee_amount": 0.004492,
    "fee_currency": "USDT",
    "commission": "0.00449200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090187534,
    "fee_amount": 0.004496,
    "fee_currency": "USDT",
    "commission": "0.00449600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090187660,
    "fee_amount": 0.0045,
    "fee_currency": "USDT",
    "commission": "0.00450000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090187660,
    "fee_amount": 0.004504,
    "fee_currency": "USDT",
    "commission": "0.00450400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090187660,
    "fee_amount": 0.004508,
    "fee_currency": "USDT",
    "commission": "0.00450800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090201152,
    "fee_amount": 0.0045,
    "fee_currency": "USDT",
    "commission": "0.00450000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090203169,
    "fee_amount": 0.0022412,
    "fee_currency": "USDT",
    "commission": "0.00224120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090209142,
    "fee_amount": 0.00224,
    "fee_currency": "USDT",
    "commission": "0.00224000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090211485,
    "fee_amount": 0.002238,
    "fee_currency": "USDT",
    "commission": "0.00223800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090211487,
    "fee_amount": 0.0022378,
    "fee_currency": "USDT",
    "commission": "0.00223780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090211489,
    "fee_amount": 0.0022368,
    "fee_currency": "USDT",
    "commission": "0.00223680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090211595,
    "fee_amount": 0.0022366,
    "fee_currency": "USDT",
    "commission": "0.00223660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090211595,
    "fee_amount": 0.0022354,
    "fee_currency": "USDT",
    "commission": "0.00223540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090211640,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090288174,
    "fee_amount": 0.004492,
    "fee_currency": "USDT",
    "commission": "0.00449200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090288174,
    "fee_amount": 0.004496,
    "fee_currency": "USDT",
    "commission": "0.00449600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090288174,
    "fee_amount": 0.0045,
    "fee_currency": "USDT",
    "commission": "0.00450000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090288174,
    "fee_amount": 0.004504,
    "fee_currency": "USDT",
    "commission": "0.00450400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090288174,
    "fee_amount": 0.004508,
    "fee_currency": "USDT",
    "commission": "0.00450800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090288259,
    "fee_amount": 0.004496,
    "fee_currency": "USDT",
    "commission": "0.00449600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090291373,
    "fee_amount": 0.0022352,
    "fee_currency": "USDT",
    "commission": "0.00223520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090291374,
    "fee_amount": 0.00216776,
    "fee_currency": "USDT",
    "commission": "0.00216776 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090291376,
    "fee_amount": 6.704e-05,
    "fee_currency": "USDT",
    "commission": "0.00006704 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090291476,
    "fee_amount": 0.0022342,
    "fee_currency": "USDT",
    "commission": "0.00223420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090347154,
    "fee_amount": 0.004488,
    "fee_currency": "USDT",
    "commission": "0.00448800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090347154,
    "fee_amount": 0.004492,
    "fee_currency": "USDT",
    "commission": "0.00449200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090347448,
    "fee_amount": 0.004492,
    "fee_currency": "USDT",
    "commission": "0.00449200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090347448,
    "fee_amount": 0.004496,
    "fee_currency": "USDT",
    "commission": "0.00449600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090354157,
    "fee_amount": 0.004492,
    "fee_currency": "USDT",
    "commission": "0.00449200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090354157,
    "fee_amount": 0.004496,
    "fee_currency": "USDT",
    "commission": "0.00449600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090475406,
    "fee_amount": 0.004488,
    "fee_currency": "USDT",
    "commission": "0.00448800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090475489,
    "fee_amount": 0.004488,
    "fee_currency": "USDT",
    "commission": "0.00448800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090475489,
    "fee_amount": 0.004492,
    "fee_currency": "USDT",
    "commission": "0.00449200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090534472,
    "fee_amount": 0.00074682,
    "fee_currency": "USDT",
    "commission": "0.00074682 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090534474,
    "fee_amount": 0.00373318,
    "fee_currency": "USDT",
    "commission": "0.00373318 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090537341,
    "fee_amount": 0.002227,
    "fee_currency": "USDT",
    "commission": "0.00222700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090537341,
    "fee_amount": 0.0022262,
    "fee_currency": "USDT",
    "commission": "0.00222620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090537495,
    "fee_amount": 0.0022242,
    "fee_currency": "USDT",
    "commission": "0.00222420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090537495,
    "fee_amount": 0.002224,
    "fee_currency": "USDT",
    "commission": "0.00222400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090580862,
    "fee_amount": 0.0022174,
    "fee_currency": "USDT",
    "commission": "0.00221740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090580874,
    "fee_amount": 0.0022162,
    "fee_currency": "USDT",
    "commission": "0.00221620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090580878,
    "fee_amount": 0.0022154,
    "fee_currency": "USDT",
    "commission": "0.00221540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090580878,
    "fee_amount": 0.0022152,
    "fee_currency": "USDT",
    "commission": "0.00221520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090580906,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090580894,
    "fee_amount": 0.002214,
    "fee_currency": "USDT",
    "commission": "0.00221400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090652116,
    "fee_amount": 0.0022044,
    "fee_currency": "USDT",
    "commission": "0.00220440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090690521,
    "fee_amount": 0.0022042,
    "fee_currency": "USDT",
    "commission": "0.00220420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090690529,
    "fee_amount": 0.002203,
    "fee_currency": "USDT",
    "commission": "0.00220300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090690529,
    "fee_amount": 0.002202,
    "fee_currency": "USDT",
    "commission": "0.00220200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090749770,
    "fee_amount": 0.0021998,
    "fee_currency": "USDT",
    "commission": "0.00219980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090888877,
    "fee_amount": 0.004404,
    "fee_currency": "USDT",
    "commission": "0.00440400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090888862,
    "fee_amount": 0.0021942,
    "fee_currency": "USDT",
    "commission": "0.00219420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090888865,
    "fee_amount": 0.0021952,
    "fee_currency": "USDT",
    "commission": "0.00219520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774090888865,
    "fee_amount": 0.0021964,
    "fee_currency": "USDT",
    "commission": "0.00219640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090889019,
    "fee_amount": 0.004404,
    "fee_currency": "USDT",
    "commission": "0.00440400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774090980301,
    "fee_amount": 0.004404,
    "fee_currency": "USDT",
    "commission": "0.00440400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774091058147,
    "fee_amount": 0.004396,
    "fee_currency": "USDT",
    "commission": "0.00439600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774091058147,
    "fee_amount": 0.004396,
    "fee_currency": "USDT",
    "commission": "0.00439600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774091101824,
    "fee_amount": 0.002193,
    "fee_currency": "USDT",
    "commission": "0.00219300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774091103321,
    "fee_amount": 0.0044,
    "fee_currency": "USDT",
    "commission": "0.00440000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774091303543,
    "fee_amount": 0.0021864,
    "fee_currency": "USDT",
    "commission": "0.00218640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774091303543,
    "fee_amount": 0.0021852,
    "fee_currency": "USDT",
    "commission": "0.00218520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774091303543,
    "fee_amount": 0.0021842,
    "fee_currency": "USDT",
    "commission": "0.00218420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774091303543,
    "fee_amount": 0.002183,
    "fee_currency": "USDT",
    "commission": "0.00218300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774091341779,
    "fee_amount": 0.004388,
    "fee_currency": "USDT",
    "commission": "0.00438800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774092815029,
    "fee_amount": 0.0022026,
    "fee_currency": "USDT",
    "commission": "0.00220260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093289948,
    "fee_amount": 0.002204,
    "fee_currency": "USDT",
    "commission": "0.00220400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093493060,
    "fee_amount": 0.002203,
    "fee_currency": "USDT",
    "commission": "0.00220300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093943131,
    "fee_amount": 0.002207,
    "fee_currency": "USDT",
    "commission": "0.00220700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093944524,
    "fee_amount": 0.0022092,
    "fee_currency": "USDT",
    "commission": "0.00220920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093946466,
    "fee_amount": 0.00132612,
    "fee_currency": "USDT",
    "commission": "0.00132612 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093946466,
    "fee_amount": 0.00088408,
    "fee_currency": "USDT",
    "commission": "0.00088408 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093946467,
    "fee_amount": 0.0022104,
    "fee_currency": "USDT",
    "commission": "0.00221040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774093946471,
    "fee_amount": 0.0022114,
    "fee_currency": "USDT",
    "commission": "0.00221140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774097166704,
    "fee_amount": 0.0022056,
    "fee_currency": "USDT",
    "commission": "0.00220560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774097676044,
    "fee_amount": 0.0022016,
    "fee_currency": "USDT",
    "commission": "0.00220160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774097916711,
    "fee_amount": 0.00153986,
    "fee_currency": "USDT",
    "commission": "0.00153986 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774097916711,
    "fee_amount": 0.00065994,
    "fee_currency": "USDT",
    "commission": "0.00065994 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774097916714,
    "fee_amount": 0.0022008,
    "fee_currency": "USDT",
    "commission": "0.00220080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774097916819,
    "fee_amount": 0.004412,
    "fee_currency": "USDT",
    "commission": "0.00441200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774097916830,
    "fee_amount": 0.002202,
    "fee_currency": "USDT",
    "commission": "0.00220200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774098022043,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774098022027,
    "fee_amount": 0.0021956,
    "fee_currency": "USDT",
    "commission": "0.00219560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774098022027,
    "fee_amount": 0.0021944,
    "fee_currency": "USDT",
    "commission": "0.00219440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774099116533,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774099116533,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774099716594,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774099716594,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774099716594,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774099777374,
    "fee_amount": 0.0021968,
    "fee_currency": "USDT",
    "commission": "0.00219680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774099777374,
    "fee_amount": 0.002197,
    "fee_currency": "USDT",
    "commission": "0.00219700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774099777382,
    "fee_amount": 0.0021978,
    "fee_currency": "USDT",
    "commission": "0.00219780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774099777392,
    "fee_amount": 0.002198,
    "fee_currency": "USDT",
    "commission": "0.00219800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774099777574,
    "fee_amount": 0.002199,
    "fee_currency": "USDT",
    "commission": "0.00219900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774099917702,
    "fee_amount": 0.0022044,
    "fee_currency": "USDT",
    "commission": "0.00220440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774100274943,
    "fee_amount": 0.0021986,
    "fee_currency": "USDT",
    "commission": "0.00219860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774101280634,
    "fee_amount": 0.0021948,
    "fee_currency": "USDT",
    "commission": "0.00219480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774101280639,
    "fee_amount": 0.0021958,
    "fee_currency": "USDT",
    "commission": "0.00219580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774101300272,
    "fee_amount": 0.004408,
    "fee_currency": "USDT",
    "commission": "0.00440800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774101300258,
    "fee_amount": 0.0021992,
    "fee_currency": "USDT",
    "commission": "0.00219920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774101300259,
    "fee_amount": 0.0022002,
    "fee_currency": "USDT",
    "commission": "0.00220020 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774104101591,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774104101654,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774104900416,
    "fee_amount": 0.004416,
    "fee_currency": "USDT",
    "commission": "0.00441600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774104900587,
    "fee_amount": 0.0022034,
    "fee_currency": "USDT",
    "commission": "0.00220340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774106987673,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774106987657,
    "fee_amount": 0.0022066,
    "fee_currency": "USDT",
    "commission": "0.00220660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774110860145,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774110860145,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774110860132,
    "fee_amount": 0.0021704,
    "fee_currency": "USDT",
    "commission": "0.00217040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774110860132,
    "fee_amount": 0.0021694,
    "fee_currency": "USDT",
    "commission": "0.00216940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774110860133,
    "fee_amount": 0.0021682,
    "fee_currency": "USDT",
    "commission": "0.00216820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774111782117,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774111782136,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774112403674,
    "fee_amount": 0.0021672,
    "fee_currency": "USDT",
    "commission": "0.00216720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774114364792,
    "fee_amount": 0.0021812,
    "fee_currency": "USDT",
    "commission": "0.00218120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774114372895,
    "fee_amount": 0.0021832,
    "fee_currency": "USDT",
    "commission": "0.00218320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774114406870,
    "fee_amount": 0.0021844,
    "fee_currency": "USDT",
    "commission": "0.00218440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774114531812,
    "fee_amount": 0.0021846,
    "fee_currency": "USDT",
    "commission": "0.00218460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774114851465,
    "fee_amount": 0.0021768,
    "fee_currency": "USDT",
    "commission": "0.00217680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774114851465,
    "fee_amount": 0.0021778,
    "fee_currency": "USDT",
    "commission": "0.00217780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774114851465,
    "fee_amount": 0.0021788,
    "fee_currency": "USDT",
    "commission": "0.00217880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774115029242,
    "fee_amount": 0.004388,
    "fee_currency": "USDT",
    "commission": "0.00438800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774115029242,
    "fee_amount": 0.004392,
    "fee_currency": "USDT",
    "commission": "0.00439200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774115029217,
    "fee_amount": 0.0021884,
    "fee_currency": "USDT",
    "commission": "0.00218840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774115029217,
    "fee_amount": 0.0021892,
    "fee_currency": "USDT",
    "commission": "0.00218920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774115029217,
    "fee_amount": 0.0021904,
    "fee_currency": "USDT",
    "commission": "0.00219040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774115029217,
    "fee_amount": 0.0021914,
    "fee_currency": "USDT",
    "commission": "0.00219140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774117089383,
    "fee_amount": 0.004384,
    "fee_currency": "USDT",
    "commission": "0.00438400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774117925376,
    "fee_amount": 0.0021934,
    "fee_currency": "USDT",
    "commission": "0.00219340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774117925376,
    "fee_amount": 0.0021924,
    "fee_currency": "USDT",
    "commission": "0.00219240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774125190160,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774126652604,
    "fee_amount": 0.0021798,
    "fee_currency": "USDT",
    "commission": "0.00217980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774126807064,
    "fee_amount": 0.0021818,
    "fee_currency": "USDT",
    "commission": "0.00218180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774131504179,
    "fee_amount": 0.002174,
    "fee_currency": "USDT",
    "commission": "0.00217400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774135502230,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774136614922,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774136615429,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774136616446,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774136616446,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774136674547,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774136676577,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774136677038,
    "fee_amount": 0.0021576,
    "fee_currency": "USDT",
    "commission": "0.00215760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774136678114,
    "fee_amount": 0.0021578,
    "fee_currency": "USDT",
    "commission": "0.00215780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774136907225,
    "fee_amount": 0.0021382,
    "fee_currency": "USDT",
    "commission": "0.00213820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137132871,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137133411,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137133607,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137133653,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137133668,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137133696,
    "fee_amount": 0.002111,
    "fee_currency": "USDT",
    "commission": "0.00211100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137133812,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137133779,
    "fee_amount": 0.0021098,
    "fee_currency": "USDT",
    "commission": "0.00210980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137133892,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137133873,
    "fee_amount": 0.0021088,
    "fee_currency": "USDT",
    "commission": "0.00210880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137134021,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137135134,
    "fee_amount": 0.0021134,
    "fee_currency": "USDT",
    "commission": "0.00211340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137136067,
    "fee_amount": 0.0021164,
    "fee_currency": "USDT",
    "commission": "0.00211640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137136293,
    "fee_amount": 0.0021176,
    "fee_currency": "USDT",
    "commission": "0.00211760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137138676,
    "fee_amount": 6.356e-05,
    "fee_currency": "USDT",
    "commission": "0.00006356 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137138720,
    "fee_amount": 0.00078388,
    "fee_currency": "USDT",
    "commission": "0.00078388 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137138723,
    "fee_amount": 0.00127116,
    "fee_currency": "USDT",
    "commission": "0.00127116 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137139119,
    "fee_amount": 0.0021196,
    "fee_currency": "USDT",
    "commission": "0.00211960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137141638,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137142188,
    "fee_amount": 0.0021128,
    "fee_currency": "USDT",
    "commission": "0.00211280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137143723,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137143893,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137146247,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137146913,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137149255,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137152113,
    "fee_amount": 0.0021076,
    "fee_currency": "USDT",
    "commission": "0.00210760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137152150,
    "fee_amount": 0.0021078,
    "fee_currency": "USDT",
    "commission": "0.00210780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137152236,
    "fee_amount": 0.0021086,
    "fee_currency": "USDT",
    "commission": "0.00210860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137152518,
    "fee_amount": 0.0021096,
    "fee_currency": "USDT",
    "commission": "0.00210960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137152750,
    "fee_amount": 0.0021118,
    "fee_currency": "USDT",
    "commission": "0.00211180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137166565,
    "fee_amount": 0.0021182,
    "fee_currency": "USDT",
    "commission": "0.00211820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137282038,
    "fee_amount": 0.00038117,
    "fee_currency": "USDT",
    "commission": "0.00038117 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137282085,
    "fee_amount": 0.00173643,
    "fee_currency": "USDT",
    "commission": "0.00173643 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137282379,
    "fee_amount": 0.0021186,
    "fee_currency": "USDT",
    "commission": "0.00211860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137312552,
    "fee_amount": 0.002119,
    "fee_currency": "USDT",
    "commission": "0.00211900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137316592,
    "fee_amount": 0.00212,
    "fee_currency": "USDT",
    "commission": "0.00212000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137319883,
    "fee_amount": 0.0021252,
    "fee_currency": "USDT",
    "commission": "0.00212520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137398708,
    "fee_amount": 0.0021104,
    "fee_currency": "USDT",
    "commission": "0.00211040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137411310,
    "fee_amount": 0.0021064,
    "fee_currency": "USDT",
    "commission": "0.00210640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137411395,
    "fee_amount": 0.0021074,
    "fee_currency": "USDT",
    "commission": "0.00210740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137463319,
    "fee_amount": 0.0020982,
    "fee_currency": "USDT",
    "commission": "0.00209820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137463543,
    "fee_amount": 0.0020962,
    "fee_currency": "USDT",
    "commission": "0.00209620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137518997,
    "fee_amount": 0.0021054,
    "fee_currency": "USDT",
    "commission": "0.00210540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137574883,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137667569,
    "fee_amount": 0.002101,
    "fee_currency": "USDT",
    "commission": "0.00210100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137667576,
    "fee_amount": 0.0021002,
    "fee_currency": "USDT",
    "commission": "0.00210020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137667729,
    "fee_amount": 0.0021,
    "fee_currency": "USDT",
    "commission": "0.00210000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137689332,
    "fee_amount": 0.0020968,
    "fee_currency": "USDT",
    "commission": "0.00209680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137689465,
    "fee_amount": 0.0020956,
    "fee_currency": "USDT",
    "commission": "0.00209560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137697800,
    "fee_amount": 0.002095,
    "fee_currency": "USDT",
    "commission": "0.00209500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137704112,
    "fee_amount": 0.0020972,
    "fee_currency": "USDT",
    "commission": "0.00209720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137704175,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137704145,
    "fee_amount": 0.0020982,
    "fee_currency": "USDT",
    "commission": "0.00209820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137705130,
    "fee_amount": 0.0020908,
    "fee_currency": "USDT",
    "commission": "0.00209080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137705165,
    "fee_amount": 0.0020896,
    "fee_currency": "USDT",
    "commission": "0.00208960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137705256,
    "fee_amount": 0.0020886,
    "fee_currency": "USDT",
    "commission": "0.00208860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137715816,
    "fee_amount": 0.004196,
    "fee_currency": "USDT",
    "commission": "0.00419600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137715989,
    "fee_amount": 0.002096,
    "fee_currency": "USDT",
    "commission": "0.00209600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774137717414,
    "fee_amount": 0.0020906,
    "fee_currency": "USDT",
    "commission": "0.00209060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774137729233,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774138155889,
    "fee_amount": 0.002138,
    "fee_currency": "USDT",
    "commission": "0.00213800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774138621870,
    "fee_amount": 0.0021422,
    "fee_currency": "USDT",
    "commission": "0.00214220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774138621980,
    "fee_amount": 0.0021434,
    "fee_currency": "USDT",
    "commission": "0.00214340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774138622047,
    "fee_amount": 0.004292,
    "fee_currency": "USDT",
    "commission": "0.00429200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774139100493,
    "fee_amount": 0.0021442,
    "fee_currency": "USDT",
    "commission": "0.00214420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774139100854,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774139462396,
    "fee_amount": 0.004288,
    "fee_currency": "USDT",
    "commission": "0.00428800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774139462474,
    "fee_amount": 0.0021406,
    "fee_currency": "USDT",
    "commission": "0.00214060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774139484990,
    "fee_amount": 0.0021412,
    "fee_currency": "USDT",
    "commission": "0.00214120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774139775230,
    "fee_amount": 0.0021514,
    "fee_currency": "USDT",
    "commission": "0.00215140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774139996915,
    "fee_amount": 0.0021482,
    "fee_currency": "USDT",
    "commission": "0.00214820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140017648,
    "fee_amount": 0.0021462,
    "fee_currency": "USDT",
    "commission": "0.00214620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774140343186,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774140343186,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774140343222,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140343172,
    "fee_amount": 0.002142,
    "fee_currency": "USDT",
    "commission": "0.00214200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140343174,
    "fee_amount": 0.002141,
    "fee_currency": "USDT",
    "commission": "0.00214100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140343301,
    "fee_amount": 0.0021398,
    "fee_currency": "USDT",
    "commission": "0.00213980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140866609,
    "fee_amount": 0.002115,
    "fee_currency": "USDT",
    "commission": "0.00211500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774140871920,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140871906,
    "fee_amount": 0.0021128,
    "fee_currency": "USDT",
    "commission": "0.00211280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140871906,
    "fee_amount": 0.002111,
    "fee_currency": "USDT",
    "commission": "0.00211100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774140871907,
    "fee_amount": 0.00211,
    "fee_currency": "USDT",
    "commission": "0.00211000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774141695572,
    "fee_amount": 0.0021122,
    "fee_currency": "USDT",
    "commission": "0.00211220 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774141818398,
    "fee_amount": 0.004232,
    "fee_currency": "USDT",
    "commission": "0.00423200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774141818401,
    "fee_amount": 0.004236,
    "fee_currency": "USDT",
    "commission": "0.00423600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774141818401,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774141818394,
    "fee_amount": 0.0021126,
    "fee_currency": "USDT",
    "commission": "0.00211260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774141818394,
    "fee_amount": 0.0021136,
    "fee_currency": "USDT",
    "commission": "0.00211360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774141842739,
    "fee_amount": 0.0021106,
    "fee_currency": "USDT",
    "commission": "0.00211060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142032601,
    "fee_amount": 0.0021094,
    "fee_currency": "USDT",
    "commission": "0.00210940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142685384,
    "fee_amount": 0.0021124,
    "fee_currency": "USDT",
    "commission": "0.00211240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142796246,
    "fee_amount": 0.0021062,
    "fee_currency": "USDT",
    "commission": "0.00210620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142796246,
    "fee_amount": 0.0021052,
    "fee_currency": "USDT",
    "commission": "0.00210520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142796246,
    "fee_amount": 0.002104,
    "fee_currency": "USDT",
    "commission": "0.00210400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774142975923,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774142975925,
    "fee_amount": 0.004228,
    "fee_currency": "USDT",
    "commission": "0.00422800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774142975956,
    "fee_amount": 0.00039865,
    "fee_currency": "USDT",
    "commission": "0.00039865 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774142975961,
    "fee_amount": 0.00383335,
    "fee_currency": "USDT",
    "commission": "0.00383335 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142975913,
    "fee_amount": 0.0021084,
    "fee_currency": "USDT",
    "commission": "0.00210840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142975916,
    "fee_amount": 0.0021094,
    "fee_currency": "USDT",
    "commission": "0.00210940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774142975918,
    "fee_amount": 0.0021104,
    "fee_currency": "USDT",
    "commission": "0.00211040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774144884943,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774146532965,
    "fee_amount": 0.0043,
    "fee_currency": "USDT",
    "commission": "0.00430000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774147364697,
    "fee_amount": 0.002131,
    "fee_currency": "USDT",
    "commission": "0.00213100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774147364701,
    "fee_amount": 0.002132,
    "fee_currency": "USDT",
    "commission": "0.00213200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774147364774,
    "fee_amount": 0.0021318,
    "fee_currency": "USDT",
    "commission": "0.00213180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774147364799,
    "fee_amount": 0.00081024,
    "fee_currency": "USDT",
    "commission": "0.00081024 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774147364801,
    "fee_amount": 0.00132196,
    "fee_currency": "USDT",
    "commission": "0.00132196 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774149001820,
    "fee_amount": 0.0021178,
    "fee_currency": "USDT",
    "commission": "0.00211780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774149001820,
    "fee_amount": 0.002117,
    "fee_currency": "USDT",
    "commission": "0.00211700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774149001820,
    "fee_amount": 0.0021168,
    "fee_currency": "USDT",
    "commission": "0.00211680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774149060073,
    "fee_amount": 0.0021252,
    "fee_currency": "USDT",
    "commission": "0.00212520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774149060143,
    "fee_amount": 0.0021264,
    "fee_currency": "USDT",
    "commission": "0.00212640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774149060143,
    "fee_amount": 0.0021274,
    "fee_currency": "USDT",
    "commission": "0.00212740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774150085951,
    "fee_amount": 0.004296,
    "fee_currency": "USDT",
    "commission": "0.00429600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774150525370,
    "fee_amount": 0.0021488,
    "fee_currency": "USDT",
    "commission": "0.00214880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774150525399,
    "fee_amount": 0.004304,
    "fee_currency": "USDT",
    "commission": "0.00430400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774152432416,
    "fee_amount": 0.0021472,
    "fee_currency": "USDT",
    "commission": "0.00214720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774152493095,
    "fee_amount": 0.002144,
    "fee_currency": "USDT",
    "commission": "0.00214400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774153276255,
    "fee_amount": 0.0021278,
    "fee_currency": "USDT",
    "commission": "0.00212780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774154182840,
    "fee_amount": 0.0021374,
    "fee_currency": "USDT",
    "commission": "0.00213740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774154182859,
    "fee_amount": 0.004284,
    "fee_currency": "USDT",
    "commission": "0.00428400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774154182970,
    "fee_amount": 0.0021384,
    "fee_currency": "USDT",
    "commission": "0.00213840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774154295595,
    "fee_amount": 0.0021372,
    "fee_currency": "USDT",
    "commission": "0.00213720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774155117061,
    "fee_amount": 0.0021416,
    "fee_currency": "USDT",
    "commission": "0.00214160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774156415725,
    "fee_amount": 0.0021392,
    "fee_currency": "USDT",
    "commission": "0.00213920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774156440953,
    "fee_amount": 0.002137,
    "fee_currency": "USDT",
    "commission": "0.00213700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774156441340,
    "fee_amount": 0.002136,
    "fee_currency": "USDT",
    "commission": "0.00213600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774157020366,
    "fee_amount": 0.0021288,
    "fee_currency": "USDT",
    "commission": "0.00212880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774157286564,
    "fee_amount": 0.0021206,
    "fee_currency": "USDT",
    "commission": "0.00212060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774157286564,
    "fee_amount": 0.0021196,
    "fee_currency": "USDT",
    "commission": "0.00211960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774157286564,
    "fee_amount": 0.0021184,
    "fee_currency": "USDT",
    "commission": "0.00211840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774160534674,
    "fee_amount": 0.0021182,
    "fee_currency": "USDT",
    "commission": "0.00211820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774161366852,
    "fee_amount": 0.0021324,
    "fee_currency": "USDT",
    "commission": "0.00213240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774161366908,
    "fee_amount": 0.004272,
    "fee_currency": "USDT",
    "commission": "0.00427200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774161366899,
    "fee_amount": 0.0021336,
    "fee_currency": "USDT",
    "commission": "0.00213360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774161559007,
    "fee_amount": 0.0021328,
    "fee_currency": "USDT",
    "commission": "0.00213280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774161559007,
    "fee_amount": 0.0021338,
    "fee_currency": "USDT",
    "commission": "0.00213380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774161559007,
    "fee_amount": 0.0021348,
    "fee_currency": "USDT",
    "commission": "0.00213480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774166565101,
    "fee_amount": 0.0020994,
    "fee_currency": "USDT",
    "commission": "0.00209940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774168463680,
    "fee_amount": 0.002105,
    "fee_currency": "USDT",
    "commission": "0.00210500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774172122263,
    "fee_amount": 0.0021042,
    "fee_currency": "USDT",
    "commission": "0.00210420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774172122265,
    "fee_amount": 0.002105,
    "fee_currency": "USDT",
    "commission": "0.00210500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774172122267,
    "fee_amount": 0.0021052,
    "fee_currency": "USDT",
    "commission": "0.00210520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774174219445,
    "fee_amount": 0.002083,
    "fee_currency": "USDT",
    "commission": "0.00208300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774174926214,
    "fee_amount": 0.0020934,
    "fee_currency": "USDT",
    "commission": "0.00209340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774174926214,
    "fee_amount": 0.0020944,
    "fee_currency": "USDT",
    "commission": "0.00209440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774174926214,
    "fee_amount": 0.0020954,
    "fee_currency": "USDT",
    "commission": "0.00209540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774175788701,
    "fee_amount": 0.0020792,
    "fee_currency": "USDT",
    "commission": "0.00207920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774175788806,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774175788850,
    "fee_amount": 0.0020782,
    "fee_currency": "USDT",
    "commission": "0.00207820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774175788853,
    "fee_amount": 0.0020772,
    "fee_currency": "USDT",
    "commission": "0.00207720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774176784973,
    "fee_amount": 0.0020852,
    "fee_currency": "USDT",
    "commission": "0.00208520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774179054551,
    "fee_amount": 0.002089,
    "fee_currency": "USDT",
    "commission": "0.00208900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774179230305,
    "fee_amount": 0.0020888,
    "fee_currency": "USDT",
    "commission": "0.00208880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774179230307,
    "fee_amount": 0.0020878,
    "fee_currency": "USDT",
    "commission": "0.00208780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774179230323,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774179230336,
    "fee_amount": 0.0020868,
    "fee_currency": "USDT",
    "commission": "0.00208680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774179230336,
    "fee_amount": 0.0020856,
    "fee_currency": "USDT",
    "commission": "0.00208560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774179230337,
    "fee_amount": 0.0020846,
    "fee_currency": "USDT",
    "commission": "0.00208460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774179940292,
    "fee_amount": 0.0020864,
    "fee_currency": "USDT",
    "commission": "0.00208640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774181240793,
    "fee_amount": 0.004188,
    "fee_currency": "USDT",
    "commission": "0.00418800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774181246599,
    "fee_amount": 0.004188,
    "fee_currency": "USDT",
    "commission": "0.00418800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774181638813,
    "fee_amount": 0.004188,
    "fee_currency": "USDT",
    "commission": "0.00418800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774181638806,
    "fee_amount": 0.0020906,
    "fee_currency": "USDT",
    "commission": "0.00209060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774181638855,
    "fee_amount": 0.0020904,
    "fee_currency": "USDT",
    "commission": "0.00209040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774181638886,
    "fee_amount": 0.0020916,
    "fee_currency": "USDT",
    "commission": "0.00209160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774181823200,
    "fee_amount": 0.002093,
    "fee_currency": "USDT",
    "commission": "0.00209300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774182526910,
    "fee_amount": 0.0021104,
    "fee_currency": "USDT",
    "commission": "0.00211040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774183839677,
    "fee_amount": 0.0020894,
    "fee_currency": "USDT",
    "commission": "0.00208940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774183839679,
    "fee_amount": 0.0020904,
    "fee_currency": "USDT",
    "commission": "0.00209040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774190561518,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774198871773,
    "fee_amount": 0.0020524,
    "fee_currency": "USDT",
    "commission": "0.00205240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774200283153,
    "fee_amount": 0.0020592,
    "fee_currency": "USDT",
    "commission": "0.00205920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774201610706,
    "fee_amount": 0.0020656,
    "fee_currency": "USDT",
    "commission": "0.00206560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774201610722,
    "fee_amount": 0.0020666,
    "fee_currency": "USDT",
    "commission": "0.00206660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774201614449,
    "fee_amount": 0.0020606,
    "fee_currency": "USDT",
    "commission": "0.00206060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774201614449,
    "fee_amount": 0.0020596,
    "fee_currency": "USDT",
    "commission": "0.00205960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774201810879,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774202401086,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774202401074,
    "fee_amount": 0.0020484,
    "fee_currency": "USDT",
    "commission": "0.00204840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774202401075,
    "fee_amount": 0.0020472,
    "fee_currency": "USDT",
    "commission": "0.00204720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774204940636,
    "fee_amount": 0.0020396,
    "fee_currency": "USDT",
    "commission": "0.00203960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774204940751,
    "fee_amount": 0.0020406,
    "fee_currency": "USDT",
    "commission": "0.00204060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774211948934,
    "fee_amount": 2.031e-05,
    "fee_currency": "USDT",
    "commission": "0.00002031 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774212665772,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774212665770,
    "fee_amount": 0.002032,
    "fee_currency": "USDT",
    "commission": "0.00203200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774212665823,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774212665867,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774212670219,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774212670247,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213216728,
    "fee_amount": 0.004088,
    "fee_currency": "USDT",
    "commission": "0.00408800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213216728,
    "fee_amount": 0.004092,
    "fee_currency": "USDT",
    "commission": "0.00409200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213216728,
    "fee_amount": 0.004096,
    "fee_currency": "USDT",
    "commission": "0.00409600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213216728,
    "fee_amount": 0.0041,
    "fee_currency": "USDT",
    "commission": "0.00410000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213216779,
    "fee_amount": 0.0020418,
    "fee_currency": "USDT",
    "commission": "0.00204180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213216919,
    "fee_amount": 0.004092,
    "fee_currency": "USDT",
    "commission": "0.00409200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213216919,
    "fee_amount": 0.004096,
    "fee_currency": "USDT",
    "commission": "0.00409600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213217011,
    "fee_amount": 0.00198365,
    "fee_currency": "USDT",
    "commission": "0.00198365 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213217090,
    "fee_amount": 6.135e-05,
    "fee_currency": "USDT",
    "commission": "0.00006135 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213408166,
    "fee_amount": 0.0020402,
    "fee_currency": "USDT",
    "commission": "0.00204020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213514460,
    "fee_amount": 0.0020382,
    "fee_currency": "USDT",
    "commission": "0.00203820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213514490,
    "fee_amount": 2.037e-05,
    "fee_currency": "USDT",
    "commission": "0.00002037 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213514575,
    "fee_amount": 0.00201683,
    "fee_currency": "USDT",
    "commission": "0.00201683 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213516153,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213516141,
    "fee_amount": 0.002035,
    "fee_currency": "USDT",
    "commission": "0.00203500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213516221,
    "fee_amount": 0.002034,
    "fee_currency": "USDT",
    "commission": "0.00203400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213516266,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213516252,
    "fee_amount": 0.002033,
    "fee_currency": "USDT",
    "commission": "0.00203300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213516253,
    "fee_amount": 0.002032,
    "fee_currency": "USDT",
    "commission": "0.00203200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528093,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528149,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213528158,
    "fee_amount": 0.0020222,
    "fee_currency": "USDT",
    "commission": "0.00202220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213528216,
    "fee_amount": 0.0020212,
    "fee_currency": "USDT",
    "commission": "0.00202120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528239,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528257,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528276,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213528308,
    "fee_amount": 0.002021,
    "fee_currency": "USDT",
    "commission": "0.00202100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213528355,
    "fee_amount": 0.0020202,
    "fee_currency": "USDT",
    "commission": "0.00202020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213528355,
    "fee_amount": 0.00202,
    "fee_currency": "USDT",
    "commission": "0.00202000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528443,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528443,
    "fee_amount": 0.22084,
    "fee_currency": "USDT",
    "commission": "0.22084000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528453,
    "fee_amount": 0.17916,
    "fee_currency": "USDT",
    "commission": "0.17916000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213528667,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774213600251,
    "fee_amount": 0.0020314,
    "fee_currency": "USDT",
    "commission": "0.00203140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774213616269,
    "fee_amount": 0.00408,
    "fee_currency": "USDT",
    "commission": "0.00408000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774214433354,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214433342,
    "fee_amount": 0.0020064,
    "fee_currency": "USDT",
    "commission": "0.00200640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214433345,
    "fee_amount": 0.0020054,
    "fee_currency": "USDT",
    "commission": "0.00200540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774214433459,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774214433459,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774214437508,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774214437510,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214437499,
    "fee_amount": 0.0019998,
    "fee_currency": "USDT",
    "commission": "0.00199980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214437503,
    "fee_amount": 0.0019988,
    "fee_currency": "USDT",
    "commission": "0.00199880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214437503,
    "fee_amount": 0.0019978,
    "fee_currency": "USDT",
    "commission": "0.00199780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214440603,
    "fee_amount": 0.0020012,
    "fee_currency": "USDT",
    "commission": "0.00200120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214443709,
    "fee_amount": 0.0020032,
    "fee_currency": "USDT",
    "commission": "0.00200320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214446940,
    "fee_amount": 0.0020034,
    "fee_currency": "USDT",
    "commission": "0.00200340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774214574642,
    "fee_amount": 0.0020078,
    "fee_currency": "USDT",
    "commission": "0.00200780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774215483159,
    "fee_amount": 0.0020214,
    "fee_currency": "USDT",
    "commission": "0.00202140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774215484931,
    "fee_amount": 0.0020224,
    "fee_currency": "USDT",
    "commission": "0.00202240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774215529553,
    "fee_amount": 0.0020246,
    "fee_currency": "USDT",
    "commission": "0.00202460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774215529652,
    "fee_amount": 0.004056,
    "fee_currency": "USDT",
    "commission": "0.00405600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774215534650,
    "fee_amount": 0.004056,
    "fee_currency": "USDT",
    "commission": "0.00405600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774215539650,
    "fee_amount": 0.004056,
    "fee_currency": "USDT",
    "commission": "0.00405600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774215594650,
    "fee_amount": 0.004056,
    "fee_currency": "USDT",
    "commission": "0.00405600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774215600337,
    "fee_amount": 0.002025,
    "fee_currency": "USDT",
    "commission": "0.00202500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774215600425,
    "fee_amount": 0.002026,
    "fee_currency": "USDT",
    "commission": "0.00202600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774215600646,
    "fee_amount": 0.004056,
    "fee_currency": "USDT",
    "commission": "0.00405600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774216098269,
    "fee_amount": 0.0020134,
    "fee_currency": "USDT",
    "commission": "0.00201340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774216098270,
    "fee_amount": 0.0020144,
    "fee_currency": "USDT",
    "commission": "0.00201440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774216315374,
    "fee_amount": 0.0020198,
    "fee_currency": "USDT",
    "commission": "0.00201980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774216315377,
    "fee_amount": 0.0020208,
    "fee_currency": "USDT",
    "commission": "0.00202080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774216922717,
    "fee_amount": 0.002016,
    "fee_currency": "USDT",
    "commission": "0.00201600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774216956835,
    "fee_amount": 0.002011,
    "fee_currency": "USDT",
    "commission": "0.00201100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774218601134,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774218601122,
    "fee_amount": 0.0020456,
    "fee_currency": "USDT",
    "commission": "0.00204560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774218601125,
    "fee_amount": 0.0020444,
    "fee_currency": "USDT",
    "commission": "0.00204440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774218601135,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774218601127,
    "fee_amount": 0.0020434,
    "fee_currency": "USDT",
    "commission": "0.00204340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774218601127,
    "fee_amount": 0.0020424,
    "fee_currency": "USDT",
    "commission": "0.00204240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774218601127,
    "fee_amount": 0.0020414,
    "fee_currency": "USDT",
    "commission": "0.00204140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774220570265,
    "fee_amount": 0.002057,
    "fee_currency": "USDT",
    "commission": "0.00205700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774220944423,
    "fee_amount": 0.002053,
    "fee_currency": "USDT",
    "commission": "0.00205300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774221180626,
    "fee_amount": 0.0020602,
    "fee_currency": "USDT",
    "commission": "0.00206020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774224949229,
    "fee_amount": 0.0020284,
    "fee_currency": "USDT",
    "commission": "0.00202840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774226673757,
    "fee_amount": 0.0020268,
    "fee_currency": "USDT",
    "commission": "0.00202680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774227871416,
    "fee_amount": 0.002024,
    "fee_currency": "USDT",
    "commission": "0.00202400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774230990298,
    "fee_amount": 0.0020202,
    "fee_currency": "USDT",
    "commission": "0.00202020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774231324592,
    "fee_amount": 0.0020254,
    "fee_currency": "USDT",
    "commission": "0.00202540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774231324621,
    "fee_amount": 0.004056,
    "fee_currency": "USDT",
    "commission": "0.00405600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774231324610,
    "fee_amount": 0.00044581,
    "fee_currency": "USDT",
    "commission": "0.00044581 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774231324612,
    "fee_amount": 0.00158059,
    "fee_currency": "USDT",
    "commission": "0.00158059 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774231324629,
    "fee_amount": 0.00406,
    "fee_currency": "USDT",
    "commission": "0.00406000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774234971171,
    "fee_amount": 0.002036,
    "fee_currency": "USDT",
    "commission": "0.00203600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774235697128,
    "fee_amount": 0.0020232,
    "fee_currency": "USDT",
    "commission": "0.00202320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774235697137,
    "fee_amount": 0.0020242,
    "fee_currency": "USDT",
    "commission": "0.00202420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774241456441,
    "fee_amount": 0.0020082,
    "fee_currency": "USDT",
    "commission": "0.00200820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774243865899,
    "fee_amount": 0.0019966,
    "fee_currency": "USDT",
    "commission": "0.00199660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774246093703,
    "fee_amount": 0.0019864,
    "fee_currency": "USDT",
    "commission": "0.00198640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774246093679,
    "fee_amount": 0.003976,
    "fee_currency": "USDT",
    "commission": "0.00397600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774246093683,
    "fee_amount": 0.00398,
    "fee_currency": "USDT",
    "commission": "0.00398000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774246093933,
    "fee_amount": 0.003984,
    "fee_currency": "USDT",
    "commission": "0.00398400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774246132559,
    "fee_amount": 0.003996,
    "fee_currency": "USDT",
    "commission": "0.00399600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774246132576,
    "fee_amount": 0.0019948,
    "fee_currency": "USDT",
    "commission": "0.00199480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774246156479,
    "fee_amount": 0.004004,
    "fee_currency": "USDT",
    "commission": "0.00400400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774246272775,
    "fee_amount": 0.001996,
    "fee_currency": "USDT",
    "commission": "0.00199600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774246354033,
    "fee_amount": 0.0019938,
    "fee_currency": "USDT",
    "commission": "0.00199380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774246354058,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774246354046,
    "fee_amount": 0.0019928,
    "fee_currency": "USDT",
    "commission": "0.00199280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774247610900,
    "fee_amount": 0.0019744,
    "fee_currency": "USDT",
    "commission": "0.00197440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774247693894,
    "fee_amount": 0.0019822,
    "fee_currency": "USDT",
    "commission": "0.00198220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774248161984,
    "fee_amount": 0.001969,
    "fee_currency": "USDT",
    "commission": "0.00196900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249287921,
    "fee_amount": 0.00192914,
    "fee_currency": "USDT",
    "commission": "0.00192914 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249287923,
    "fee_amount": 5.966e-05,
    "fee_currency": "USDT",
    "commission": "0.00005966 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774249397422,
    "fee_amount": 0.003992,
    "fee_currency": "USDT",
    "commission": "0.00399200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249397410,
    "fee_amount": 0.0019918,
    "fee_currency": "USDT",
    "commission": "0.00199180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249397410,
    "fee_amount": 0.0019928,
    "fee_currency": "USDT",
    "commission": "0.00199280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774249397430,
    "fee_amount": 0.003996,
    "fee_currency": "USDT",
    "commission": "0.00399600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249437243,
    "fee_amount": 0.001998,
    "fee_currency": "USDT",
    "commission": "0.00199800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249440158,
    "fee_amount": 0.001997,
    "fee_currency": "USDT",
    "commission": "0.00199700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249608497,
    "fee_amount": 0.0019936,
    "fee_currency": "USDT",
    "commission": "0.00199360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774249623825,
    "fee_amount": 0.00012906,
    "fee_currency": "USDT",
    "commission": "0.00012906 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774249623825,
    "fee_amount": 0.00387894,
    "fee_currency": "USDT",
    "commission": "0.00387894 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774249866019,
    "fee_amount": 0.0019994,
    "fee_currency": "USDT",
    "commission": "0.00199940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774252041587,
    "fee_amount": 0.0019948,
    "fee_currency": "USDT",
    "commission": "0.00199480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774252041592,
    "fee_amount": 0.0019938,
    "fee_currency": "USDT",
    "commission": "0.00199380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774252380103,
    "fee_amount": 1.99e-05,
    "fee_currency": "USDT",
    "commission": "0.00001990 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774252546343,
    "fee_amount": 0.0019902,
    "fee_currency": "USDT",
    "commission": "0.00199020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774253220471,
    "fee_amount": 0.0020032,
    "fee_currency": "USDT",
    "commission": "0.00200320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774253486848,
    "fee_amount": 0.0019982,
    "fee_currency": "USDT",
    "commission": "0.00199820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774257968412,
    "fee_amount": 0.0019822,
    "fee_currency": "USDT",
    "commission": "0.00198220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774257968412,
    "fee_amount": 0.0019812,
    "fee_currency": "USDT",
    "commission": "0.00198120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774257968412,
    "fee_amount": 0.0019802,
    "fee_currency": "USDT",
    "commission": "0.00198020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774258910236,
    "fee_amount": 0.0019952,
    "fee_currency": "USDT",
    "commission": "0.00199520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774258910236,
    "fee_amount": 0.0019962,
    "fee_currency": "USDT",
    "commission": "0.00199620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774258910236,
    "fee_amount": 0.001997,
    "fee_currency": "USDT",
    "commission": "0.00199700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774261813240,
    "fee_amount": 0.0019962,
    "fee_currency": "USDT",
    "commission": "0.00199620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263415745,
    "fee_amount": 0.001996,
    "fee_currency": "USDT",
    "commission": "0.00199600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263415748,
    "fee_amount": 0.001995,
    "fee_currency": "USDT",
    "commission": "0.00199500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263917339,
    "fee_amount": 0.002005,
    "fee_currency": "USDT",
    "commission": "0.00200500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263917376,
    "fee_amount": 0.00402,
    "fee_currency": "USDT",
    "commission": "0.00402000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263917410,
    "fee_amount": 0.004024,
    "fee_currency": "USDT",
    "commission": "0.00402400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263917520,
    "fee_amount": 0.004028,
    "fee_currency": "USDT",
    "commission": "0.00402800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263917603,
    "fee_amount": 0.004028,
    "fee_currency": "USDT",
    "commission": "0.00402800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263917603,
    "fee_amount": 0.004032,
    "fee_currency": "USDT",
    "commission": "0.00403200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263917588,
    "fee_amount": 0.0020104,
    "fee_currency": "USDT",
    "commission": "0.00201040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263917760,
    "fee_amount": 2.011e-05,
    "fee_currency": "USDT",
    "commission": "0.00002011 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263917832,
    "fee_amount": 0.00199129,
    "fee_currency": "USDT",
    "commission": "0.00199129 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263917835,
    "fee_amount": 0.0020124,
    "fee_currency": "USDT",
    "commission": "0.00201240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263917982,
    "fee_amount": 0.00052328,
    "fee_currency": "USDT",
    "commission": "0.00052328 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263918420,
    "fee_amount": 0.0020152,
    "fee_currency": "USDT",
    "commission": "0.00201520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263918420,
    "fee_amount": 0.0020162,
    "fee_currency": "USDT",
    "commission": "0.00201620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263935684,
    "fee_amount": 0.004076,
    "fee_currency": "USDT",
    "commission": "0.00407600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263938071,
    "fee_amount": 0.0020318,
    "fee_currency": "USDT",
    "commission": "0.00203180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263938105,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263938225,
    "fee_amount": 0.0020306,
    "fee_currency": "USDT",
    "commission": "0.00203060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263940060,
    "fee_amount": 0.004084,
    "fee_currency": "USDT",
    "commission": "0.00408400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263940047,
    "fee_amount": 0.002039,
    "fee_currency": "USDT",
    "commission": "0.00203900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263940050,
    "fee_amount": 0.0020396,
    "fee_currency": "USDT",
    "commission": "0.00203960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263940072,
    "fee_amount": 0.004088,
    "fee_currency": "USDT",
    "commission": "0.00408800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263940055,
    "fee_amount": 0.0020408,
    "fee_currency": "USDT",
    "commission": "0.00204080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263940055,
    "fee_amount": 0.0020418,
    "fee_currency": "USDT",
    "commission": "0.00204180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263941947,
    "fee_amount": 0.0020332,
    "fee_currency": "USDT",
    "commission": "0.00203320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263944233,
    "fee_amount": 0.002044,
    "fee_currency": "USDT",
    "commission": "0.00204400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263944242,
    "fee_amount": 0.00098131,
    "fee_currency": "USDT",
    "commission": "0.00098131 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263944242,
    "fee_amount": 0.00106309,
    "fee_currency": "USDT",
    "commission": "0.00106309 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263944339,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263944326,
    "fee_amount": 0.002036,
    "fee_currency": "USDT",
    "commission": "0.00203600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263944370,
    "fee_amount": 0.002035,
    "fee_currency": "USDT",
    "commission": "0.00203500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263944687,
    "fee_amount": 0.002034,
    "fee_currency": "USDT",
    "commission": "0.00203400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263944741,
    "fee_amount": 0.002033,
    "fee_currency": "USDT",
    "commission": "0.00203300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263948265,
    "fee_amount": 0.0020358,
    "fee_currency": "USDT",
    "commission": "0.00203580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263955068,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263956966,
    "fee_amount": 0.002049,
    "fee_currency": "USDT",
    "commission": "0.00204900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263957053,
    "fee_amount": 0.0020498,
    "fee_currency": "USDT",
    "commission": "0.00204980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263957208,
    "fee_amount": 2.051e-05,
    "fee_currency": "USDT",
    "commission": "0.00002051 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263958609,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263958823,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263958993,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263959185,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263965019,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263966830,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263966896,
    "fee_amount": 0.08,
    "fee_currency": "USDT",
    "commission": "0.08000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263969429,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263970028,
    "fee_amount": 0.0020518,
    "fee_currency": "USDT",
    "commission": "0.00205180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263970028,
    "fee_amount": 0.0020528,
    "fee_currency": "USDT",
    "commission": "0.00205280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263970436,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263972301,
    "fee_amount": 0.00156013,
    "fee_currency": "USDT",
    "commission": "0.00156013 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263972303,
    "fee_amount": 0.00049267,
    "fee_currency": "USDT",
    "commission": "0.00049267 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263973982,
    "fee_amount": 0.004116,
    "fee_currency": "USDT",
    "commission": "0.00411600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774263978726,
    "fee_amount": 0.004136,
    "fee_currency": "USDT",
    "commission": "0.00413600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263985023,
    "fee_amount": 2.063e-05,
    "fee_currency": "USDT",
    "commission": "0.00002063 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263988737,
    "fee_amount": 2.069e-05,
    "fee_currency": "USDT",
    "commission": "0.00002069 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774263997826,
    "fee_amount": 0.0020692,
    "fee_currency": "USDT",
    "commission": "0.00206920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264006907,
    "fee_amount": 0.004148,
    "fee_currency": "USDT",
    "commission": "0.00414800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264007177,
    "fee_amount": 0.004152,
    "fee_currency": "USDT",
    "commission": "0.00415200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264011196,
    "fee_amount": 0.0020702,
    "fee_currency": "USDT",
    "commission": "0.00207020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264012370,
    "fee_amount": 0.002067,
    "fee_currency": "USDT",
    "commission": "0.00206700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264012377,
    "fee_amount": 0.002066,
    "fee_currency": "USDT",
    "commission": "0.00206600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264016800,
    "fee_amount": 0.0020754,
    "fee_currency": "USDT",
    "commission": "0.00207540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264031921,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264031927,
    "fee_amount": 0.0020836,
    "fee_currency": "USDT",
    "commission": "0.00208360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264032018,
    "fee_amount": 0.0020826,
    "fee_currency": "USDT",
    "commission": "0.00208260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264032023,
    "fee_amount": 0.0020816,
    "fee_currency": "USDT",
    "commission": "0.00208160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264032049,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264032213,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264052433,
    "fee_amount": 0.0020824,
    "fee_currency": "USDT",
    "commission": "0.00208240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264052438,
    "fee_amount": 0.00200006,
    "fee_currency": "USDT",
    "commission": "0.00200006 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264055098,
    "fee_amount": 0.0020762,
    "fee_currency": "USDT",
    "commission": "0.00207620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264055113,
    "fee_amount": 0.0020752,
    "fee_currency": "USDT",
    "commission": "0.00207520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264055245,
    "fee_amount": 0.002075,
    "fee_currency": "USDT",
    "commission": "0.00207500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264055262,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264055245,
    "fee_amount": 0.0020742,
    "fee_currency": "USDT",
    "commission": "0.00207420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264055245,
    "fee_amount": 0.002074,
    "fee_currency": "USDT",
    "commission": "0.00207400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264055381,
    "fee_amount": 0.25756,
    "fee_currency": "USDT",
    "commission": "0.25756000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264086567,
    "fee_amount": 0.0020636,
    "fee_currency": "USDT",
    "commission": "0.00206360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264089391,
    "fee_amount": 0.0020606,
    "fee_currency": "USDT",
    "commission": "0.00206060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264093918,
    "fee_amount": 0.00049406,
    "fee_currency": "USDT",
    "commission": "0.00049406 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264095882,
    "fee_amount": 0.00156454,
    "fee_currency": "USDT",
    "commission": "0.00156454 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264096576,
    "fee_amount": 0.0020588,
    "fee_currency": "USDT",
    "commission": "0.00205880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264097367,
    "fee_amount": 0.0020608,
    "fee_currency": "USDT",
    "commission": "0.00206080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264102939,
    "fee_amount": 0.0020622,
    "fee_currency": "USDT",
    "commission": "0.00206220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264103048,
    "fee_amount": 0.0020632,
    "fee_currency": "USDT",
    "commission": "0.00206320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264105102,
    "fee_amount": 0.004136,
    "fee_currency": "USDT",
    "commission": "0.00413600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264105096,
    "fee_amount": 0.0020654,
    "fee_currency": "USDT",
    "commission": "0.00206540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264107659,
    "fee_amount": 0.0020674,
    "fee_currency": "USDT",
    "commission": "0.00206740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264107661,
    "fee_amount": 0.0020678,
    "fee_currency": "USDT",
    "commission": "0.00206780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264107890,
    "fee_amount": 0.00414,
    "fee_currency": "USDT",
    "commission": "0.00414000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264115645,
    "fee_amount": 0.0020652,
    "fee_currency": "USDT",
    "commission": "0.00206520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264167121,
    "fee_amount": 0.004164,
    "fee_currency": "USDT",
    "commission": "0.00416400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264200160,
    "fee_amount": 0.0020818,
    "fee_currency": "USDT",
    "commission": "0.00208180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264306446,
    "fee_amount": 0.0020894,
    "fee_currency": "USDT",
    "commission": "0.00208940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264306461,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264313513,
    "fee_amount": 0.0020864,
    "fee_currency": "USDT",
    "commission": "0.00208640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264313515,
    "fee_amount": 0.0020862,
    "fee_currency": "USDT",
    "commission": "0.00208620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264313586,
    "fee_amount": 0.002086,
    "fee_currency": "USDT",
    "commission": "0.00208600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264334706,
    "fee_amount": 0.002082,
    "fee_currency": "USDT",
    "commission": "0.00208200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264334746,
    "fee_amount": 2.083e-05,
    "fee_currency": "USDT",
    "commission": "0.00002083 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264334872,
    "fee_amount": 0.00206217,
    "fee_currency": "USDT",
    "commission": "0.00206217 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264377325,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264381040,
    "fee_amount": 0.0021012,
    "fee_currency": "USDT",
    "commission": "0.00210120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264446881,
    "fee_amount": 0.00115302,
    "fee_currency": "USDT",
    "commission": "0.00115302 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264446881,
    "fee_amount": 0.00094338,
    "fee_currency": "USDT",
    "commission": "0.00094338 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264446978,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264467727,
    "fee_amount": 0.0020922,
    "fee_currency": "USDT",
    "commission": "0.00209220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264471215,
    "fee_amount": 0.002089,
    "fee_currency": "USDT",
    "commission": "0.00208900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264507718,
    "fee_amount": 0.0020962,
    "fee_currency": "USDT",
    "commission": "0.00209620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264507932,
    "fee_amount": 0.0020952,
    "fee_currency": "USDT",
    "commission": "0.00209520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264520198,
    "fee_amount": 2.107e-05,
    "fee_currency": "USDT",
    "commission": "0.00002107 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264530576,
    "fee_amount": 0.002101,
    "fee_currency": "USDT",
    "commission": "0.00210100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264530583,
    "fee_amount": 0.0021,
    "fee_currency": "USDT",
    "commission": "0.00210000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264551817,
    "fee_amount": 0.0021018,
    "fee_currency": "USDT",
    "commission": "0.00210180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264555406,
    "fee_amount": 0.0021008,
    "fee_currency": "USDT",
    "commission": "0.00210080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264560746,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264560733,
    "fee_amount": 0.0021016,
    "fee_currency": "USDT",
    "commission": "0.00210160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264667840,
    "fee_amount": 0.0020964,
    "fee_currency": "USDT",
    "commission": "0.00209640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264667994,
    "fee_amount": 0.0020954,
    "fee_currency": "USDT",
    "commission": "0.00209540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264794076,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264794076,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264794095,
    "fee_amount": 0.0020832,
    "fee_currency": "USDT",
    "commission": "0.00208320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264801082,
    "fee_amount": 0.00029134,
    "fee_currency": "USDT",
    "commission": "0.00029134 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264802429,
    "fee_amount": 0.002081,
    "fee_currency": "USDT",
    "commission": "0.00208100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264821061,
    "fee_amount": 0.0020798,
    "fee_currency": "USDT",
    "commission": "0.00207980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264827507,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264840725,
    "fee_amount": 0.002085,
    "fee_currency": "USDT",
    "commission": "0.00208500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264840725,
    "fee_amount": 0.0020852,
    "fee_currency": "USDT",
    "commission": "0.00208520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264851183,
    "fee_amount": 0.0020778,
    "fee_currency": "USDT",
    "commission": "0.00207780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774264930100,
    "fee_amount": 0.0020778,
    "fee_currency": "USDT",
    "commission": "0.00207780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774264953609,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774265003797,
    "fee_amount": 0.0020676,
    "fee_currency": "USDT",
    "commission": "0.00206760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774265003818,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774265723751,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774265859069,
    "fee_amount": 0.0020852,
    "fee_currency": "USDT",
    "commission": "0.00208520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774266080534,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774266615986,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774266615973,
    "fee_amount": 0.002089,
    "fee_currency": "USDT",
    "commission": "0.00208900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774266699686,
    "fee_amount": 0.0020902,
    "fee_currency": "USDT",
    "commission": "0.00209020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774266699728,
    "fee_amount": 0.0020914,
    "fee_currency": "USDT",
    "commission": "0.00209140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774266699747,
    "fee_amount": 0.004188,
    "fee_currency": "USDT",
    "commission": "0.00418800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774266699733,
    "fee_amount": 0.0020924,
    "fee_currency": "USDT",
    "commission": "0.00209240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774266699733,
    "fee_amount": 0.0020934,
    "fee_currency": "USDT",
    "commission": "0.00209340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774266699752,
    "fee_amount": 0.004192,
    "fee_currency": "USDT",
    "commission": "0.00419200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774266699752,
    "fee_amount": 0.004196,
    "fee_currency": "USDT",
    "commission": "0.00419600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774266699733,
    "fee_amount": 0.0020944,
    "fee_currency": "USDT",
    "commission": "0.00209440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774266699752,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774266891843,
    "fee_amount": 0.0021172,
    "fee_currency": "USDT",
    "commission": "0.00211720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774267315564,
    "fee_amount": 0.00209,
    "fee_currency": "USDT",
    "commission": "0.00209000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774267351849,
    "fee_amount": 0.002091,
    "fee_currency": "USDT",
    "commission": "0.00209100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774267352084,
    "fee_amount": 0.00209,
    "fee_currency": "USDT",
    "commission": "0.00209000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774267394033,
    "fee_amount": 0.0020816,
    "fee_currency": "USDT",
    "commission": "0.00208160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774268414029,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774268414017,
    "fee_amount": 0.0020938,
    "fee_currency": "USDT",
    "commission": "0.00209380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774268414017,
    "fee_amount": 0.0020946,
    "fee_currency": "USDT",
    "commission": "0.00209460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774268414017,
    "fee_amount": 0.0020948,
    "fee_currency": "USDT",
    "commission": "0.00209480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774268430114,
    "fee_amount": 0.0020978,
    "fee_currency": "USDT",
    "commission": "0.00209780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774269019334,
    "fee_amount": 0.0020814,
    "fee_currency": "USDT",
    "commission": "0.00208140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774269691075,
    "fee_amount": 0.002084,
    "fee_currency": "USDT",
    "commission": "0.00208400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774269691130,
    "fee_amount": 0.002083,
    "fee_currency": "USDT",
    "commission": "0.00208300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774269691154,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774269691145,
    "fee_amount": 0.002082,
    "fee_currency": "USDT",
    "commission": "0.00208200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774269696145,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774269712239,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774269713678,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774270399881,
    "fee_amount": 0.0020838,
    "fee_currency": "USDT",
    "commission": "0.00208380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774270399888,
    "fee_amount": 0.0020848,
    "fee_currency": "USDT",
    "commission": "0.00208480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774270399920,
    "fee_amount": 0.0020858,
    "fee_currency": "USDT",
    "commission": "0.00208580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774270399985,
    "fee_amount": 0.004176,
    "fee_currency": "USDT",
    "commission": "0.00417600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774270800770,
    "fee_amount": 0.0021024,
    "fee_currency": "USDT",
    "commission": "0.00210240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271132693,
    "fee_amount": 0.002092,
    "fee_currency": "USDT",
    "commission": "0.00209200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271132694,
    "fee_amount": 0.0020908,
    "fee_currency": "USDT",
    "commission": "0.00209080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774271132719,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271132702,
    "fee_amount": 0.0020898,
    "fee_currency": "USDT",
    "commission": "0.00208980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271384689,
    "fee_amount": 0.0020854,
    "fee_currency": "USDT",
    "commission": "0.00208540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774271384702,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271384689,
    "fee_amount": 0.0020844,
    "fee_currency": "USDT",
    "commission": "0.00208440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271384690,
    "fee_amount": 0.0020834,
    "fee_currency": "USDT",
    "commission": "0.00208340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271384690,
    "fee_amount": 0.0020832,
    "fee_currency": "USDT",
    "commission": "0.00208320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271384690,
    "fee_amount": 0.0020824,
    "fee_currency": "USDT",
    "commission": "0.00208240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774271384703,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774271384705,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271384814,
    "fee_amount": 0.0020834,
    "fee_currency": "USDT",
    "commission": "0.00208340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774271384816,
    "fee_amount": 0.0020832,
    "fee_currency": "USDT",
    "commission": "0.00208320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774273878063,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774273878063,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774273878065,
    "fee_amount": 0.004228,
    "fee_currency": "USDT",
    "commission": "0.00422800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774273878051,
    "fee_amount": 0.0021066,
    "fee_currency": "USDT",
    "commission": "0.00210660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774273878067,
    "fee_amount": 0.004232,
    "fee_currency": "USDT",
    "commission": "0.00423200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774273878051,
    "fee_amount": 0.0021076,
    "fee_currency": "USDT",
    "commission": "0.00210760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774273878052,
    "fee_amount": 0.0021086,
    "fee_currency": "USDT",
    "commission": "0.00210860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774273878052,
    "fee_amount": 0.0021098,
    "fee_currency": "USDT",
    "commission": "0.00210980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774273878053,
    "fee_amount": 0.0021108,
    "fee_currency": "USDT",
    "commission": "0.00211080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774273878076,
    "fee_amount": 0.004236,
    "fee_currency": "USDT",
    "commission": "0.00423600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774273878098,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774274726655,
    "fee_amount": 0.004272,
    "fee_currency": "USDT",
    "commission": "0.00427200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774274726661,
    "fee_amount": 0.00108722,
    "fee_currency": "USDT",
    "commission": "0.00108722 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774274858729,
    "fee_amount": 0.002148,
    "fee_currency": "USDT",
    "commission": "0.00214800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774274999050,
    "fee_amount": 0.0021394,
    "fee_currency": "USDT",
    "commission": "0.00213940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275166579,
    "fee_amount": 0.0021326,
    "fee_currency": "USDT",
    "commission": "0.00213260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275166622,
    "fee_amount": 0.00204941,
    "fee_currency": "USDT",
    "commission": "0.00204941 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275166623,
    "fee_amount": 8.539e-05,
    "fee_currency": "USDT",
    "commission": "0.00008539 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275166682,
    "fee_amount": 0.00204922,
    "fee_currency": "USDT",
    "commission": "0.00204922 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275166683,
    "fee_amount": 8.538e-05,
    "fee_currency": "USDT",
    "commission": "0.00008538 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275409384,
    "fee_amount": 0.002138,
    "fee_currency": "USDT",
    "commission": "0.00213800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275409388,
    "fee_amount": 0.002137,
    "fee_currency": "USDT",
    "commission": "0.00213700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275502001,
    "fee_amount": 0.0021336,
    "fee_currency": "USDT",
    "commission": "0.00213360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275502243,
    "fee_amount": 0.0021326,
    "fee_currency": "USDT",
    "commission": "0.00213260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774275503014,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275647435,
    "fee_amount": 0.0021248,
    "fee_currency": "USDT",
    "commission": "0.00212480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275822148,
    "fee_amount": 0.0021306,
    "fee_currency": "USDT",
    "commission": "0.00213060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275954314,
    "fee_amount": 0.00198109,
    "fee_currency": "USDT",
    "commission": "0.00198109 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774275954314,
    "fee_amount": 0.00014911,
    "fee_currency": "USDT",
    "commission": "0.00014911 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774275954335,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774275962216,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774275962216,
    "fee_amount": 0.21696,
    "fee_currency": "USDT",
    "commission": "0.21696000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774277153767,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277153755,
    "fee_amount": 0.0021358,
    "fee_currency": "USDT",
    "commission": "0.00213580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277153755,
    "fee_amount": 0.0021346,
    "fee_currency": "USDT",
    "commission": "0.00213460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277153755,
    "fee_amount": 0.0021336,
    "fee_currency": "USDT",
    "commission": "0.00213360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277153758,
    "fee_amount": 0.0021326,
    "fee_currency": "USDT",
    "commission": "0.00213260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277153808,
    "fee_amount": 0.0021314,
    "fee_currency": "USDT",
    "commission": "0.00213140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774277153828,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774277169355,
    "fee_amount": 0.00428,
    "fee_currency": "USDT",
    "commission": "0.00428000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774277236409,
    "fee_amount": 0.004292,
    "fee_currency": "USDT",
    "commission": "0.00429200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277236397,
    "fee_amount": 0.0021414,
    "fee_currency": "USDT",
    "commission": "0.00214140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774277236409,
    "fee_amount": 0.004296,
    "fee_currency": "USDT",
    "commission": "0.00429600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277236397,
    "fee_amount": 0.0021424,
    "fee_currency": "USDT",
    "commission": "0.00214240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277236402,
    "fee_amount": 0.0021434,
    "fee_currency": "USDT",
    "commission": "0.00214340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774277236415,
    "fee_amount": 0.0043,
    "fee_currency": "USDT",
    "commission": "0.00430000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277236408,
    "fee_amount": 0.0021444,
    "fee_currency": "USDT",
    "commission": "0.00214440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277286867,
    "fee_amount": 0.0021406,
    "fee_currency": "USDT",
    "commission": "0.00214060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277623232,
    "fee_amount": 0.0021384,
    "fee_currency": "USDT",
    "commission": "0.00213840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277623232,
    "fee_amount": 0.0021382,
    "fee_currency": "USDT",
    "commission": "0.00213820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277623232,
    "fee_amount": 0.0009621,
    "fee_currency": "USDT",
    "commission": "0.00096210 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277623235,
    "fee_amount": 0.0011759,
    "fee_currency": "USDT",
    "commission": "0.00117590 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277623873,
    "fee_amount": 0.0021382,
    "fee_currency": "USDT",
    "commission": "0.00213820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277623876,
    "fee_amount": 0.002138,
    "fee_currency": "USDT",
    "commission": "0.00213800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277640571,
    "fee_amount": 0.002139,
    "fee_currency": "USDT",
    "commission": "0.00213900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277640797,
    "fee_amount": 0.0021382,
    "fee_currency": "USDT",
    "commission": "0.00213820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277640797,
    "fee_amount": 0.00062002,
    "fee_currency": "USDT",
    "commission": "0.00062002 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277641113,
    "fee_amount": 0.0021386,
    "fee_currency": "USDT",
    "commission": "0.00213860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277642372,
    "fee_amount": 0.0021386,
    "fee_currency": "USDT",
    "commission": "0.00213860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277642415,
    "fee_amount": 0.0021382,
    "fee_currency": "USDT",
    "commission": "0.00213820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277642417,
    "fee_amount": 0.00151798,
    "fee_currency": "USDT",
    "commission": "0.00151798 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277651444,
    "fee_amount": 0.0021388,
    "fee_currency": "USDT",
    "commission": "0.00213880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277656985,
    "fee_amount": 0.0021386,
    "fee_currency": "USDT",
    "commission": "0.00213860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277656985,
    "fee_amount": 0.0021384,
    "fee_currency": "USDT",
    "commission": "0.00213840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277754743,
    "fee_amount": 0.0021348,
    "fee_currency": "USDT",
    "commission": "0.00213480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277754751,
    "fee_amount": 0.0014737,
    "fee_currency": "USDT",
    "commission": "0.00147370 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774277762314,
    "fee_amount": 0.0006621,
    "fee_currency": "USDT",
    "commission": "0.00066210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774278166357,
    "fee_amount": 0.00107213,
    "fee_currency": "USDT",
    "commission": "0.00107213 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774278166361,
    "fee_amount": 1.08e-06,
    "fee_currency": "USDT",
    "commission": "0.00000108 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774278520065,
    "fee_amount": 0.0010742,
    "fee_currency": "USDT",
    "commission": "0.00107420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774278922333,
    "fee_amount": 0.0010737,
    "fee_currency": "USDT",
    "commission": "0.00107370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774278922348,
    "fee_amount": 0.0010742,
    "fee_currency": "USDT",
    "commission": "0.00107420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774278922352,
    "fee_amount": 0.0010748,
    "fee_currency": "USDT",
    "commission": "0.00107480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774279199886,
    "fee_amount": 0.0010687,
    "fee_currency": "USDT",
    "commission": "0.00106870 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281604052,
    "fee_amount": 0.0010548,
    "fee_currency": "USDT",
    "commission": "0.00105480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281608306,
    "fee_amount": 0.0010558,
    "fee_currency": "USDT",
    "commission": "0.00105580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281608340,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281608380,
    "fee_amount": 0.0010568,
    "fee_currency": "USDT",
    "commission": "0.00105680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281639570,
    "fee_amount": 0.00105675,
    "fee_currency": "USDT",
    "commission": "0.00105675 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281772513,
    "fee_amount": 0.0010623,
    "fee_currency": "USDT",
    "commission": "0.00106230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281784171,
    "fee_amount": 0.0010644,
    "fee_currency": "USDT",
    "commission": "0.00106440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281846973,
    "fee_amount": 0.0010638,
    "fee_currency": "USDT",
    "commission": "0.00106380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281851770,
    "fee_amount": 0.0010648,
    "fee_currency": "USDT",
    "commission": "0.00106480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281851770,
    "fee_amount": 0.0010649,
    "fee_currency": "USDT",
    "commission": "0.00106490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281851772,
    "fee_amount": 0.0010654,
    "fee_currency": "USDT",
    "commission": "0.00106540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281851775,
    "fee_amount": 0.001066,
    "fee_currency": "USDT",
    "commission": "0.00106600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281890169,
    "fee_amount": 0.0010626,
    "fee_currency": "USDT",
    "commission": "0.00106260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281890169,
    "fee_amount": 0.001062,
    "fee_currency": "USDT",
    "commission": "0.00106200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281890169,
    "fee_amount": 0.0010615,
    "fee_currency": "USDT",
    "commission": "0.00106150 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281890170,
    "fee_amount": 0.001061,
    "fee_currency": "USDT",
    "commission": "0.00106100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281890170,
    "fee_amount": 0.0010604,
    "fee_currency": "USDT",
    "commission": "0.00106040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281893077,
    "fee_amount": 0.0010559,
    "fee_currency": "USDT",
    "commission": "0.00105590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281909068,
    "fee_amount": 0.0010559,
    "fee_currency": "USDT",
    "commission": "0.00105590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774281909074,
    "fee_amount": 0.0010564,
    "fee_currency": "USDT",
    "commission": "0.00105640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282151848,
    "fee_amount": 0.00105785,
    "fee_currency": "USDT",
    "commission": "0.00105785 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282151853,
    "fee_amount": 1.06e-06,
    "fee_currency": "USDT",
    "commission": "0.00000106 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282151888,
    "fee_amount": 0.0010594,
    "fee_currency": "USDT",
    "commission": "0.00105940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282151893,
    "fee_amount": 0.0010599,
    "fee_currency": "USDT",
    "commission": "0.00105990 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282176205,
    "fee_amount": 0.0010589,
    "fee_currency": "USDT",
    "commission": "0.00105890 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282406378,
    "fee_amount": 0.00052845,
    "fee_currency": "USDT",
    "commission": "0.00052845 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282478534,
    "fee_amount": 0.0010559,
    "fee_currency": "USDT",
    "commission": "0.00105590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282513521,
    "fee_amount": 0.0010559,
    "fee_currency": "USDT",
    "commission": "0.00105590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282664609,
    "fee_amount": 0.001055,
    "fee_currency": "USDT",
    "commission": "0.00105500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282872362,
    "fee_amount": 0.001051,
    "fee_currency": "USDT",
    "commission": "0.00105100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282936305,
    "fee_amount": 0.0010501,
    "fee_currency": "USDT",
    "commission": "0.00105010 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282947460,
    "fee_amount": 0.0010506,
    "fee_currency": "USDT",
    "commission": "0.00105060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774282966391,
    "fee_amount": 0.0010521,
    "fee_currency": "USDT",
    "commission": "0.00105210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774284442014,
    "fee_amount": 0.0010558,
    "fee_currency": "USDT",
    "commission": "0.00105580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774284980056,
    "fee_amount": 0.0010523,
    "fee_currency": "USDT",
    "commission": "0.00105230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774285139873,
    "fee_amount": 0.0010523,
    "fee_currency": "USDT",
    "commission": "0.00105230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774285207763,
    "fee_amount": 0.0010543,
    "fee_currency": "USDT",
    "commission": "0.00105430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774285250926,
    "fee_amount": 0.00105335,
    "fee_currency": "USDT",
    "commission": "0.00105335 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774285250931,
    "fee_amount": 1.06e-06,
    "fee_currency": "USDT",
    "commission": "0.00000106 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287212880,
    "fee_amount": 0.0010554,
    "fee_currency": "USDT",
    "commission": "0.00105540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287212928,
    "fee_amount": 0.00021128,
    "fee_currency": "USDT",
    "commission": "0.00021128 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287212935,
    "fee_amount": 0.00084512,
    "fee_currency": "USDT",
    "commission": "0.00084512 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287213051,
    "fee_amount": 0.0010562,
    "fee_currency": "USDT",
    "commission": "0.00105620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287213052,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287213104,
    "fee_amount": 0.0010569,
    "fee_currency": "USDT",
    "commission": "0.00105690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287368190,
    "fee_amount": 0.0010623,
    "fee_currency": "USDT",
    "commission": "0.00106230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287368258,
    "fee_amount": 0.0010617,
    "fee_currency": "USDT",
    "commission": "0.00106170 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287368314,
    "fee_amount": 0.0010617,
    "fee_currency": "USDT",
    "commission": "0.00106170 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287368314,
    "fee_amount": 0.0010612,
    "fee_currency": "USDT",
    "commission": "0.00106120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287368316,
    "fee_amount": 0.00022381,
    "fee_currency": "USDT",
    "commission": "0.00022381 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287368340,
    "fee_amount": 0.0008369,
    "fee_currency": "USDT",
    "commission": "0.00083690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774287368351,
    "fee_amount": 0.0010601,
    "fee_currency": "USDT",
    "commission": "0.00106010 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774288579061,
    "fee_amount": 0.0010648,
    "fee_currency": "USDT",
    "commission": "0.00106480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774289541061,
    "fee_amount": 0.0010653,
    "fee_currency": "USDT",
    "commission": "0.00106530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290303315,
    "fee_amount": 0.0010644,
    "fee_currency": "USDT",
    "commission": "0.00106440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290337795,
    "fee_amount": 0.0010601,
    "fee_currency": "USDT",
    "commission": "0.00106010 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290968386,
    "fee_amount": 0.0010659,
    "fee_currency": "USDT",
    "commission": "0.00106590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290968386,
    "fee_amount": 0.0010664,
    "fee_currency": "USDT",
    "commission": "0.00106640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290968389,
    "fee_amount": 0.001067,
    "fee_currency": "USDT",
    "commission": "0.00106700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290968392,
    "fee_amount": 0.0010675,
    "fee_currency": "USDT",
    "commission": "0.00106750 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290968420,
    "fee_amount": 0.0010677,
    "fee_currency": "USDT",
    "commission": "0.00106770 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774290968420,
    "fee_amount": 0.0010678,
    "fee_currency": "USDT",
    "commission": "0.00106780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774291600222,
    "fee_amount": 0.0010636,
    "fee_currency": "USDT",
    "commission": "0.00106360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774291600349,
    "fee_amount": 0.0010641,
    "fee_currency": "USDT",
    "commission": "0.00106410 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292292826,
    "fee_amount": 0.0010667,
    "fee_currency": "USDT",
    "commission": "0.00106670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292292826,
    "fee_amount": 0.0010672,
    "fee_currency": "USDT",
    "commission": "0.00106720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292427599,
    "fee_amount": 0.0010667,
    "fee_currency": "USDT",
    "commission": "0.00106670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292428867,
    "fee_amount": 0.00106574,
    "fee_currency": "USDT",
    "commission": "0.00106574 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292428951,
    "fee_amount": 1.07e-06,
    "fee_currency": "USDT",
    "commission": "0.00000107 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292428960,
    "fee_amount": 0.0010669,
    "fee_currency": "USDT",
    "commission": "0.00106690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292429041,
    "fee_amount": 0.0010671,
    "fee_currency": "USDT",
    "commission": "0.00106710 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292586599,
    "fee_amount": 0.0010629,
    "fee_currency": "USDT",
    "commission": "0.00106290 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292586625,
    "fee_amount": 0.0010634,
    "fee_currency": "USDT",
    "commission": "0.00106340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292761685,
    "fee_amount": 0.0010655,
    "fee_currency": "USDT",
    "commission": "0.00106550 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774292774935,
    "fee_amount": 0.001065,
    "fee_currency": "USDT",
    "commission": "0.00106500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296131563,
    "fee_amount": 0.0010511,
    "fee_currency": "USDT",
    "commission": "0.00105110 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296131599,
    "fee_amount": 0.0010516,
    "fee_currency": "USDT",
    "commission": "0.00105160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296138673,
    "fee_amount": 0.0010499,
    "fee_currency": "USDT",
    "commission": "0.00104990 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296833784,
    "fee_amount": 0.0010541,
    "fee_currency": "USDT",
    "commission": "0.00105410 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296833853,
    "fee_amount": 0.0010544,
    "fee_currency": "USDT",
    "commission": "0.00105440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296833855,
    "fee_amount": 0.00105355,
    "fee_currency": "USDT",
    "commission": "0.00105355 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296833856,
    "fee_amount": 1.06e-06,
    "fee_currency": "USDT",
    "commission": "0.00000106 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774296833905,
    "fee_amount": 0.0010552,
    "fee_currency": "USDT",
    "commission": "0.00105520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774301007253,
    "fee_amount": 0.00068102,
    "fee_currency": "USDT",
    "commission": "0.00068102 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774301007257,
    "fee_amount": 0.00037319,
    "fee_currency": "USDT",
    "commission": "0.00037319 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302063731,
    "fee_amount": 0.002115,
    "fee_currency": "USDT",
    "commission": "0.00211500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302063733,
    "fee_amount": 0.0021156,
    "fee_currency": "USDT",
    "commission": "0.00211560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302066831,
    "fee_amount": 0.0021152,
    "fee_currency": "USDT",
    "commission": "0.00211520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302074288,
    "fee_amount": 0.0021152,
    "fee_currency": "USDT",
    "commission": "0.00211520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302074290,
    "fee_amount": 0.0021154,
    "fee_currency": "USDT",
    "commission": "0.00211540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302074354,
    "fee_amount": 0.0021154,
    "fee_currency": "USDT",
    "commission": "0.00211540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302079415,
    "fee_amount": 0.0021156,
    "fee_currency": "USDT",
    "commission": "0.00211560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302085425,
    "fee_amount": 0.0021158,
    "fee_currency": "USDT",
    "commission": "0.00211580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302162125,
    "fee_amount": 0.002116,
    "fee_currency": "USDT",
    "commission": "0.00211600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302162355,
    "fee_amount": 0.002116,
    "fee_currency": "USDT",
    "commission": "0.00211600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302228023,
    "fee_amount": 0.0021162,
    "fee_currency": "USDT",
    "commission": "0.00211620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774302228023,
    "fee_amount": 0.0021172,
    "fee_currency": "USDT",
    "commission": "0.00211720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302242030,
    "fee_amount": 0.0010562,
    "fee_currency": "USDT",
    "commission": "0.00105620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302332564,
    "fee_amount": 0.00050064,
    "fee_currency": "USDT",
    "commission": "0.00050064 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302333065,
    "fee_amount": 0.00051966,
    "fee_currency": "USDT",
    "commission": "0.00051966 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302339065,
    "fee_amount": 0.00028201,
    "fee_currency": "USDT",
    "commission": "0.00028201 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302340065,
    "fee_amount": 0.00064746,
    "fee_currency": "USDT",
    "commission": "0.00064746 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302341066,
    "fee_amount": 0.00012675,
    "fee_currency": "USDT",
    "commission": "0.00012675 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302415316,
    "fee_amount": 0.0010571,
    "fee_currency": "USDT",
    "commission": "0.00105710 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302841677,
    "fee_amount": 0.0010596,
    "fee_currency": "USDT",
    "commission": "0.00105960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302841677,
    "fee_amount": 0.0010595,
    "fee_currency": "USDT",
    "commission": "0.00105950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302850057,
    "fee_amount": 0.0010596,
    "fee_currency": "USDT",
    "commission": "0.00105960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774302884131,
    "fee_amount": 0.0010596,
    "fee_currency": "USDT",
    "commission": "0.00105960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303144364,
    "fee_amount": 0.0010589,
    "fee_currency": "USDT",
    "commission": "0.00105890 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303145364,
    "fee_amount": 0.0010588,
    "fee_currency": "USDT",
    "commission": "0.00105880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303151569,
    "fee_amount": 0.0010588,
    "fee_currency": "USDT",
    "commission": "0.00105880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303154792,
    "fee_amount": 0.0010588,
    "fee_currency": "USDT",
    "commission": "0.00105880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303157464,
    "fee_amount": 0.00050077,
    "fee_currency": "USDT",
    "commission": "0.00050077 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303172221,
    "fee_amount": 0.0010587,
    "fee_currency": "USDT",
    "commission": "0.00105870 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303172221,
    "fee_amount": 0.0010586,
    "fee_currency": "USDT",
    "commission": "0.00105860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774303172213,
    "fee_amount": 0.002117,
    "fee_currency": "USDT",
    "commission": "0.00211700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303172225,
    "fee_amount": 0.0010582,
    "fee_currency": "USDT",
    "commission": "0.00105820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303186786,
    "fee_amount": 0.0010586,
    "fee_currency": "USDT",
    "commission": "0.00105860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303201722,
    "fee_amount": 0.0010586,
    "fee_currency": "USDT",
    "commission": "0.00105860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303206687,
    "fee_amount": 0.0010595,
    "fee_currency": "USDT",
    "commission": "0.00105950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303226077,
    "fee_amount": 0.0010595,
    "fee_currency": "USDT",
    "commission": "0.00105950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303235230,
    "fee_amount": 0.0010595,
    "fee_currency": "USDT",
    "commission": "0.00105950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303239649,
    "fee_amount": 0.0010595,
    "fee_currency": "USDT",
    "commission": "0.00105950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303239660,
    "fee_amount": 0.0010594,
    "fee_currency": "USDT",
    "commission": "0.00105940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303296425,
    "fee_amount": 0.0010568,
    "fee_currency": "USDT",
    "commission": "0.00105680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303303651,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303320200,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303321485,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774303321489,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303321489,
    "fee_amount": 0.0010558,
    "fee_currency": "USDT",
    "commission": "0.00105580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774303321478,
    "fee_amount": 0.0021122,
    "fee_currency": "USDT",
    "commission": "0.00211220 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303321489,
    "fee_amount": 0.0010557,
    "fee_currency": "USDT",
    "commission": "0.00105570 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303395566,
    "fee_amount": 0.00035002,
    "fee_currency": "USDT",
    "commission": "0.00035002 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303395567,
    "fee_amount": 0.00070109,
    "fee_currency": "USDT",
    "commission": "0.00070109 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303405651,
    "fee_amount": 0.0010501,
    "fee_currency": "USDT",
    "commission": "0.00105010 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303405653,
    "fee_amount": 0.0010496,
    "fee_currency": "USDT",
    "commission": "0.00104960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774303405644,
    "fee_amount": 0.0021,
    "fee_currency": "USDT",
    "commission": "0.00210000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774303405644,
    "fee_amount": 0.0020998,
    "fee_currency": "USDT",
    "commission": "0.00209980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303561644,
    "fee_amount": 0.0010489,
    "fee_currency": "USDT",
    "commission": "0.00104890 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303561644,
    "fee_amount": 0.0010488,
    "fee_currency": "USDT",
    "commission": "0.00104880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303570494,
    "fee_amount": 0.0010489,
    "fee_currency": "USDT",
    "commission": "0.00104890 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774303570494,
    "fee_amount": 0.0010488,
    "fee_currency": "USDT",
    "commission": "0.00104880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774306932363,
    "fee_amount": 0.0020794,
    "fee_currency": "USDT",
    "commission": "0.00207940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774307055975,
    "fee_amount": 0.0010388,
    "fee_currency": "USDT",
    "commission": "0.00103880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774308882708,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774308882712,
    "fee_amount": 0.0010398,
    "fee_currency": "USDT",
    "commission": "0.00103980 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774308882743,
    "fee_amount": 0.0010393,
    "fee_currency": "USDT",
    "commission": "0.00103930 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774308882739,
    "fee_amount": 0.00139306,
    "fee_currency": "USDT",
    "commission": "0.00139306 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774308882742,
    "fee_amount": 0.00068614,
    "fee_currency": "USDT",
    "commission": "0.00068614 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774308882787,
    "fee_amount": 0.0010388,
    "fee_currency": "USDT",
    "commission": "0.00103880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774308882889,
    "fee_amount": 0.0010388,
    "fee_currency": "USDT",
    "commission": "0.00103880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774308970677,
    "fee_amount": 0.0010376,
    "fee_currency": "USDT",
    "commission": "0.00103760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774308970680,
    "fee_amount": 0.0010371,
    "fee_currency": "USDT",
    "commission": "0.00103710 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774309262793,
    "fee_amount": 0.0020872,
    "fee_currency": "USDT",
    "commission": "0.00208720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774310112000,
    "fee_amount": 0.004212,
    "fee_currency": "USDT",
    "commission": "0.00421200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774310112003,
    "fee_amount": 0.004216,
    "fee_currency": "USDT",
    "commission": "0.00421600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774310112003,
    "fee_amount": 0.00422,
    "fee_currency": "USDT",
    "commission": "0.00422000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774310112007,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774310112016,
    "fee_amount": 0.004228,
    "fee_currency": "USDT",
    "commission": "0.00422800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310111988,
    "fee_amount": 0.0021014,
    "fee_currency": "USDT",
    "commission": "0.00210140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310111988,
    "fee_amount": 0.0021024,
    "fee_currency": "USDT",
    "commission": "0.00210240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310111988,
    "fee_amount": 0.0021034,
    "fee_currency": "USDT",
    "commission": "0.00210340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310111988,
    "fee_amount": 0.0021046,
    "fee_currency": "USDT",
    "commission": "0.00210460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310111988,
    "fee_amount": 0.0021056,
    "fee_currency": "USDT",
    "commission": "0.00210560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774310112000,
    "fee_amount": 0.0010508,
    "fee_currency": "USDT",
    "commission": "0.00105080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774310112000,
    "fee_amount": 0.0010513,
    "fee_currency": "USDT",
    "commission": "0.00105130 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774310112002,
    "fee_amount": 0.0010518,
    "fee_currency": "USDT",
    "commission": "0.00105180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774310112003,
    "fee_amount": 0.0010524,
    "fee_currency": "USDT",
    "commission": "0.00105240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774310112003,
    "fee_amount": 0.0010529,
    "fee_currency": "USDT",
    "commission": "0.00105290 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774310112093,
    "fee_amount": 0.00250323,
    "fee_currency": "USDT",
    "commission": "0.00250323 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774310112097,
    "fee_amount": 0.00172877,
    "fee_currency": "USDT",
    "commission": "0.00172877 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310112079,
    "fee_amount": 0.002111,
    "fee_currency": "USDT",
    "commission": "0.00211100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310112097,
    "fee_amount": 0.0006967,
    "fee_currency": "USDT",
    "commission": "0.00069670 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310112097,
    "fee_amount": 0.0014145,
    "fee_currency": "USDT",
    "commission": "0.00141450 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774310212338,
    "fee_amount": 0.001055,
    "fee_currency": "USDT",
    "commission": "0.00105500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310212326,
    "fee_amount": 0.0021102,
    "fee_currency": "USDT",
    "commission": "0.00211020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310335675,
    "fee_amount": 0.002117,
    "fee_currency": "USDT",
    "commission": "0.00211700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310660122,
    "fee_amount": 0.0021126,
    "fee_currency": "USDT",
    "commission": "0.00211260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310660126,
    "fee_amount": 0.0021136,
    "fee_currency": "USDT",
    "commission": "0.00211360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310846881,
    "fee_amount": 0.0021188,
    "fee_currency": "USDT",
    "commission": "0.00211880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310923448,
    "fee_amount": 0.002121,
    "fee_currency": "USDT",
    "commission": "0.00212100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310928061,
    "fee_amount": 0.0021222,
    "fee_currency": "USDT",
    "commission": "0.00212220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774310928061,
    "fee_amount": 0.002123,
    "fee_currency": "USDT",
    "commission": "0.00212300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311189010,
    "fee_amount": 0.0010599,
    "fee_currency": "USDT",
    "commission": "0.00105990 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774311188993,
    "fee_amount": 0.0021204,
    "fee_currency": "USDT",
    "commission": "0.00212040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774311188993,
    "fee_amount": 0.0021214,
    "fee_currency": "USDT",
    "commission": "0.00212140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774311189006,
    "fee_amount": 0.0021236,
    "fee_currency": "USDT",
    "commission": "0.00212360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311189013,
    "fee_amount": 0.0010605,
    "fee_currency": "USDT",
    "commission": "0.00106050 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311189014,
    "fee_amount": 0.001061,
    "fee_currency": "USDT",
    "commission": "0.00106100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774311189013,
    "fee_amount": 0.004248,
    "fee_currency": "USDT",
    "commission": "0.00424800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774311189017,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774311189017,
    "fee_amount": 0.004256,
    "fee_currency": "USDT",
    "commission": "0.00425600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774311189033,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311189108,
    "fee_amount": 0.0010619,
    "fee_currency": "USDT",
    "commission": "0.00106190 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311189115,
    "fee_amount": 0.001062,
    "fee_currency": "USDT",
    "commission": "0.00106200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311257449,
    "fee_amount": 0.001059,
    "fee_currency": "USDT",
    "commission": "0.00105900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311265594,
    "fee_amount": 0.001058,
    "fee_currency": "USDT",
    "commission": "0.00105800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311445416,
    "fee_amount": 0.0010597,
    "fee_currency": "USDT",
    "commission": "0.00105970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311445418,
    "fee_amount": 0.0010602,
    "fee_currency": "USDT",
    "commission": "0.00106020 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774311445420,
    "fee_amount": 0.004248,
    "fee_currency": "USDT",
    "commission": "0.00424800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774311445405,
    "fee_amount": 0.00212,
    "fee_currency": "USDT",
    "commission": "0.00212000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774311445498,
    "fee_amount": 0.0010601,
    "fee_currency": "USDT",
    "commission": "0.00106010 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774311512862,
    "fee_amount": 0.0021218,
    "fee_currency": "USDT",
    "commission": "0.00212180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774312064685,
    "fee_amount": 0.00111713,
    "fee_currency": "USDT",
    "commission": "0.00111713 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774312345850,
    "fee_amount": 0.0010516,
    "fee_currency": "USDT",
    "commission": "0.00105160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774312345831,
    "fee_amount": 0.0021036,
    "fee_currency": "USDT",
    "commission": "0.00210360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774312738057,
    "fee_amount": 0.0010555,
    "fee_currency": "USDT",
    "commission": "0.00105550 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774312738098,
    "fee_amount": 0.31364,
    "fee_currency": "USDT",
    "commission": "0.31364000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774312738109,
    "fee_amount": 0.08636,
    "fee_currency": "USDT",
    "commission": "0.08636000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774312738123,
    "fee_amount": 0.001055,
    "fee_currency": "USDT",
    "commission": "0.00105500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774312738125,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774312738141,
    "fee_amount": 0.0021114,
    "fee_currency": "USDT",
    "commission": "0.00211140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774312738410,
    "fee_amount": 0.00107681,
    "fee_currency": "USDT",
    "commission": "0.00107681 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774312738420,
    "fee_amount": 0.00103459,
    "fee_currency": "USDT",
    "commission": "0.00103459 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774312777166,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774312777164,
    "fee_amount": 0.0010508,
    "fee_currency": "USDT",
    "commission": "0.00105080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774312777152,
    "fee_amount": 0.002103,
    "fee_currency": "USDT",
    "commission": "0.00210300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774313576364,
    "fee_amount": 2.099e-05,
    "fee_currency": "USDT",
    "commission": "0.00002099 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774314210249,
    "fee_amount": 0.0021028,
    "fee_currency": "USDT",
    "commission": "0.00210280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774314543424,
    "fee_amount": 0.0010497,
    "fee_currency": "USDT",
    "commission": "0.00104970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774314752210,
    "fee_amount": 0.0010492,
    "fee_currency": "USDT",
    "commission": "0.00104920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774314756961,
    "fee_amount": 0.0020986,
    "fee_currency": "USDT",
    "commission": "0.00209860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774314764743,
    "fee_amount": 0.0010486,
    "fee_currency": "USDT",
    "commission": "0.00104860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774314764730,
    "fee_amount": 0.0020974,
    "fee_currency": "USDT",
    "commission": "0.00209740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774314905393,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774315096842,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774315096867,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774316374595,
    "fee_amount": 0.0020722,
    "fee_currency": "USDT",
    "commission": "0.00207220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774316374595,
    "fee_amount": 0.0020712,
    "fee_currency": "USDT",
    "commission": "0.00207120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316488966,
    "fee_amount": 0.001035,
    "fee_currency": "USDT",
    "commission": "0.00103500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774316519118,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774316533170,
    "fee_amount": 0.00159452,
    "fee_currency": "USDT",
    "commission": "0.00159452 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316664796,
    "fee_amount": 0.0010349,
    "fee_currency": "USDT",
    "commission": "0.00103490 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774316667218,
    "fee_amount": 0.0020688,
    "fee_currency": "USDT",
    "commission": "0.00206880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316667244,
    "fee_amount": 0.0010344,
    "fee_currency": "USDT",
    "commission": "0.00103440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316685902,
    "fee_amount": 0.0010338,
    "fee_currency": "USDT",
    "commission": "0.00103380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316685902,
    "fee_amount": 0.0010333,
    "fee_currency": "USDT",
    "commission": "0.00103330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316685916,
    "fee_amount": 0.0010327,
    "fee_currency": "USDT",
    "commission": "0.00103270 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316685916,
    "fee_amount": 0.0010322,
    "fee_currency": "USDT",
    "commission": "0.00103220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774316685934,
    "fee_amount": 0.00140597,
    "fee_currency": "USDT",
    "commission": "0.00140597 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774316685934,
    "fee_amount": 0.00066163,
    "fee_currency": "USDT",
    "commission": "0.00066163 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774316685943,
    "fee_amount": 0.00140529,
    "fee_currency": "USDT",
    "commission": "0.00140529 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316685981,
    "fee_amount": 0.0010333,
    "fee_currency": "USDT",
    "commission": "0.00103330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774316685981,
    "fee_amount": 0.0010332,
    "fee_currency": "USDT",
    "commission": "0.00103320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774318385769,
    "fee_amount": 0.004172,
    "fee_currency": "USDT",
    "commission": "0.00417200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774318385755,
    "fee_amount": 0.0020814,
    "fee_currency": "USDT",
    "commission": "0.00208140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774321704405,
    "fee_amount": 0.0010396,
    "fee_currency": "USDT",
    "commission": "0.00103960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774324570807,
    "fee_amount": 0.0010395,
    "fee_currency": "USDT",
    "commission": "0.00103950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774324570807,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774324689903,
    "fee_amount": 0.0020814,
    "fee_currency": "USDT",
    "commission": "0.00208140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774325164555,
    "fee_amount": 0.0020776,
    "fee_currency": "USDT",
    "commission": "0.00207760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774325164561,
    "fee_amount": 0.0020786,
    "fee_currency": "USDT",
    "commission": "0.00207860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774325164561,
    "fee_amount": 0.0020788,
    "fee_currency": "USDT",
    "commission": "0.00207880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774325164593,
    "fee_amount": 0.0010382,
    "fee_currency": "USDT",
    "commission": "0.00103820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774325164630,
    "fee_amount": 0.0010387,
    "fee_currency": "USDT",
    "commission": "0.00103870 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774325164617,
    "fee_amount": 0.0020796,
    "fee_currency": "USDT",
    "commission": "0.00207960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774325164634,
    "fee_amount": 0.0010388,
    "fee_currency": "USDT",
    "commission": "0.00103880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774325164700,
    "fee_amount": 0.00097751,
    "fee_currency": "USDT",
    "commission": "0.00097751 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774325164701,
    "fee_amount": 0.0009775,
    "fee_currency": "USDT",
    "commission": "0.00097750 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774325164704,
    "fee_amount": 0.00012479,
    "fee_currency": "USDT",
    "commission": "0.00012479 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774325168143,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774325390207,
    "fee_amount": 0.004156,
    "fee_currency": "USDT",
    "commission": "0.00415600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774332084311,
    "fee_amount": 0.0010367,
    "fee_currency": "USDT",
    "commission": "0.00103670 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774333378644,
    "fee_amount": 0.0020822,
    "fee_currency": "USDT",
    "commission": "0.00208220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774333378647,
    "fee_amount": 0.0020834,
    "fee_currency": "USDT",
    "commission": "0.00208340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774333378647,
    "fee_amount": 0.0020842,
    "fee_currency": "USDT",
    "commission": "0.00208420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774333378665,
    "fee_amount": 0.0010409,
    "fee_currency": "USDT",
    "commission": "0.00104090 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774333378665,
    "fee_amount": 0.0010414,
    "fee_currency": "USDT",
    "commission": "0.00104140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774333378667,
    "fee_amount": 0.0010419,
    "fee_currency": "USDT",
    "commission": "0.00104190 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774333378667,
    "fee_amount": 0.004172,
    "fee_currency": "USDT",
    "commission": "0.00417200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774333378669,
    "fee_amount": 0.004176,
    "fee_currency": "USDT",
    "commission": "0.00417600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334298197,
    "fee_amount": 0.0010389,
    "fee_currency": "USDT",
    "commission": "0.00103890 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334472202,
    "fee_amount": 0.00084359,
    "fee_currency": "USDT",
    "commission": "0.00084359 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334472202,
    "fee_amount": 0.00019532,
    "fee_currency": "USDT",
    "commission": "0.00019532 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774334487620,
    "fee_amount": 0.00418,
    "fee_currency": "USDT",
    "commission": "0.00418000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334487619,
    "fee_amount": 0.0010421,
    "fee_currency": "USDT",
    "commission": "0.00104210 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334487606,
    "fee_amount": 0.0020848,
    "fee_currency": "USDT",
    "commission": "0.00208480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334487608,
    "fee_amount": 0.0020858,
    "fee_currency": "USDT",
    "commission": "0.00208580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334487608,
    "fee_amount": 0.002087,
    "fee_currency": "USDT",
    "commission": "0.00208700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334487622,
    "fee_amount": 0.0010426,
    "fee_currency": "USDT",
    "commission": "0.00104260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334487610,
    "fee_amount": 0.002088,
    "fee_currency": "USDT",
    "commission": "0.00208800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334487622,
    "fee_amount": 0.0010431,
    "fee_currency": "USDT",
    "commission": "0.00104310 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334487628,
    "fee_amount": 0.0010436,
    "fee_currency": "USDT",
    "commission": "0.00104360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334487650,
    "fee_amount": 0.0010432,
    "fee_currency": "USDT",
    "commission": "0.00104320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774334487623,
    "fee_amount": 0.004184,
    "fee_currency": "USDT",
    "commission": "0.00418400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334487650,
    "fee_amount": 0.00073115,
    "fee_currency": "USDT",
    "commission": "0.00073115 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334487653,
    "fee_amount": 0.00073115,
    "fee_currency": "USDT",
    "commission": "0.00073115 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334487654,
    "fee_amount": 0.0006267,
    "fee_currency": "USDT",
    "commission": "0.00062670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334487749,
    "fee_amount": 0.0010433,
    "fee_currency": "USDT",
    "commission": "0.00104330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774334487760,
    "fee_amount": 0.004184,
    "fee_currency": "USDT",
    "commission": "0.00418400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334498041,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334640409,
    "fee_amount": 0.00077271,
    "fee_currency": "USDT",
    "commission": "0.00077271 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334640410,
    "fee_amount": 0.0002715,
    "fee_currency": "USDT",
    "commission": "0.00027150 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334640415,
    "fee_amount": 0.0010437,
    "fee_currency": "USDT",
    "commission": "0.00104370 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774334640403,
    "fee_amount": 0.0020894,
    "fee_currency": "USDT",
    "commission": "0.00208940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774334640423,
    "fee_amount": 0.0010432,
    "fee_currency": "USDT",
    "commission": "0.00104320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335265014,
    "fee_amount": 7.936e-05,
    "fee_currency": "USDT",
    "commission": "0.00007936 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335265115,
    "fee_amount": 0.0005753,
    "fee_currency": "USDT",
    "commission": "0.00057530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335270920,
    "fee_amount": 0.00038945,
    "fee_currency": "USDT",
    "commission": "0.00038945 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335278231,
    "fee_amount": 0.0010441,
    "fee_currency": "USDT",
    "commission": "0.00104410 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335278515,
    "fee_amount": 0.0010438,
    "fee_currency": "USDT",
    "commission": "0.00104380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335278615,
    "fee_amount": 0.0010438,
    "fee_currency": "USDT",
    "commission": "0.00104380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335278770,
    "fee_amount": 0.0010438,
    "fee_currency": "USDT",
    "commission": "0.00104380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335279971,
    "fee_amount": 0.0010435,
    "fee_currency": "USDT",
    "commission": "0.00104350 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335283910,
    "fee_amount": 0.0010437,
    "fee_currency": "USDT",
    "commission": "0.00104370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335284130,
    "fee_amount": 0.0010437,
    "fee_currency": "USDT",
    "commission": "0.00104370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335284180,
    "fee_amount": 6.054e-05,
    "fee_currency": "USDT",
    "commission": "0.00006054 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335284184,
    "fee_amount": 0.00098317,
    "fee_currency": "USDT",
    "commission": "0.00098317 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335286915,
    "fee_amount": 0.00012107,
    "fee_currency": "USDT",
    "commission": "0.00012107 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335286920,
    "fee_amount": 0.00092264,
    "fee_currency": "USDT",
    "commission": "0.00092264 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335286920,
    "fee_amount": 0.00085471,
    "fee_currency": "USDT",
    "commission": "0.00085471 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335287420,
    "fee_amount": 0.0001889,
    "fee_currency": "USDT",
    "commission": "0.00018890 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335295031,
    "fee_amount": 0.0010437,
    "fee_currency": "USDT",
    "commission": "0.00104370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335296080,
    "fee_amount": 0.0010437,
    "fee_currency": "USDT",
    "commission": "0.00104370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335299120,
    "fee_amount": 0.0010436,
    "fee_currency": "USDT",
    "commission": "0.00104360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335373114,
    "fee_amount": 0.0010446,
    "fee_currency": "USDT",
    "commission": "0.00104460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335480023,
    "fee_amount": 0.00057503,
    "fee_currency": "USDT",
    "commission": "0.00057503 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335480222,
    "fee_amount": 0.0010455,
    "fee_currency": "USDT",
    "commission": "0.00104550 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335481392,
    "fee_amount": 0.0010455,
    "fee_currency": "USDT",
    "commission": "0.00104550 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335484312,
    "fee_amount": 0.0010455,
    "fee_currency": "USDT",
    "commission": "0.00104550 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335485970,
    "fee_amount": 0.0010454,
    "fee_currency": "USDT",
    "commission": "0.00104540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335527910,
    "fee_amount": 0.00103082,
    "fee_currency": "USDT",
    "commission": "0.00103082 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335528831,
    "fee_amount": 0.00230379,
    "fee_currency": "USDT",
    "commission": "0.00230379 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335528932,
    "fee_amount": 0.00086939,
    "fee_currency": "USDT",
    "commission": "0.00086939 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335547361,
    "fee_amount": 0.0010444,
    "fee_currency": "USDT",
    "commission": "0.00104440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335547681,
    "fee_amount": 0.0010444,
    "fee_currency": "USDT",
    "commission": "0.00104440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335557195,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335559324,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335560221,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335560952,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335562412,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335562983,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335564015,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335565736,
    "fee_amount": 0.004204,
    "fee_currency": "USDT",
    "commission": "0.00420400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335565815,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335569595,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335569971,
    "fee_amount": 0.0010442,
    "fee_currency": "USDT",
    "commission": "0.00104420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335570115,
    "fee_amount": 0.00090846,
    "fee_currency": "USDT",
    "commission": "0.00090846 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335570915,
    "fee_amount": 0.0010442,
    "fee_currency": "USDT",
    "commission": "0.00104420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335575331,
    "fee_amount": 0.004204,
    "fee_currency": "USDT",
    "commission": "0.00420400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335576216,
    "fee_amount": 0.00368607,
    "fee_currency": "USDT",
    "commission": "0.00368607 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335576758,
    "fee_amount": 0.00051793,
    "fee_currency": "USDT",
    "commission": "0.00051793 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335579071,
    "fee_amount": 0.004204,
    "fee_currency": "USDT",
    "commission": "0.00420400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335582865,
    "fee_amount": 0.004204,
    "fee_currency": "USDT",
    "commission": "0.00420400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335584216,
    "fee_amount": 0.00230505,
    "fee_currency": "USDT",
    "commission": "0.00230505 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335584522,
    "fee_amount": 0.00189895,
    "fee_currency": "USDT",
    "commission": "0.00189895 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335588420,
    "fee_amount": 2.195e-05,
    "fee_currency": "USDT",
    "commission": "0.00002195 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335588614,
    "fee_amount": 2.09e-05,
    "fee_currency": "USDT",
    "commission": "0.00002090 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335588817,
    "fee_amount": 2.09e-05,
    "fee_currency": "USDT",
    "commission": "0.00002090 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335588932,
    "fee_amount": 2.091e-05,
    "fee_currency": "USDT",
    "commission": "0.00002091 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335588990,
    "fee_amount": 0.0010242,
    "fee_currency": "USDT",
    "commission": "0.00102420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335589615,
    "fee_amount": 0.0010451,
    "fee_currency": "USDT",
    "commission": "0.00104510 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335589780,
    "fee_amount": 0.0010451,
    "fee_currency": "USDT",
    "commission": "0.00104510 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335589780,
    "fee_amount": 0.00098126,
    "fee_currency": "USDT",
    "commission": "0.00098126 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335589941,
    "fee_amount": 0.0010451,
    "fee_currency": "USDT",
    "commission": "0.00104510 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335590092,
    "fee_amount": 0.0010451,
    "fee_currency": "USDT",
    "commission": "0.00104510 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335590601,
    "fee_amount": 0.001045,
    "fee_currency": "USDT",
    "commission": "0.00104500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335590801,
    "fee_amount": 0.001045,
    "fee_currency": "USDT",
    "commission": "0.00104500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335604369,
    "fee_amount": 0.0010444,
    "fee_currency": "USDT",
    "commission": "0.00104440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774335757691,
    "fee_amount": 0.002098,
    "fee_currency": "USDT",
    "commission": "0.00209800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335757708,
    "fee_amount": 0.004204,
    "fee_currency": "USDT",
    "commission": "0.00420400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335757840,
    "fee_amount": 0.00259129,
    "fee_currency": "USDT",
    "commission": "0.00259129 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335799541,
    "fee_amount": 0.00101859,
    "fee_currency": "USDT",
    "commission": "0.00101859 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335799564,
    "fee_amount": 0.00230194,
    "fee_currency": "USDT",
    "commission": "0.00230194 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335799614,
    "fee_amount": 0.00089548,
    "fee_currency": "USDT",
    "commission": "0.00089548 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335825730,
    "fee_amount": 0.00029512,
    "fee_currency": "USDT",
    "commission": "0.00029512 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335825814,
    "fee_amount": 0.00123613,
    "fee_currency": "USDT",
    "commission": "0.00123613 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335825915,
    "fee_amount": 0.00268475,
    "fee_currency": "USDT",
    "commission": "0.00268475 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335826115,
    "fee_amount": 0.00081959,
    "fee_currency": "USDT",
    "commission": "0.00081959 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335826255,
    "fee_amount": 0.00123697,
    "fee_currency": "USDT",
    "commission": "0.00123697 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335826315,
    "fee_amount": 0.00215944,
    "fee_currency": "USDT",
    "commission": "0.00215944 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335826914,
    "fee_amount": 0.00046502,
    "fee_currency": "USDT",
    "commission": "0.00046502 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335827873,
    "fee_amount": 0.00123613,
    "fee_currency": "USDT",
    "commission": "0.00123613 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335827913,
    "fee_amount": 0.00251484,
    "fee_currency": "USDT",
    "commission": "0.00251484 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335872483,
    "fee_amount": 0.00081699,
    "fee_currency": "USDT",
    "commission": "0.00081699 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335903678,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335905908,
    "fee_amount": 0.004228,
    "fee_currency": "USDT",
    "commission": "0.00422800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335916915,
    "fee_amount": 0.00346093,
    "fee_currency": "USDT",
    "commission": "0.00346093 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335917014,
    "fee_amount": 0.00077107,
    "fee_currency": "USDT",
    "commission": "0.00077107 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335917931,
    "fee_amount": 0.004232,
    "fee_currency": "USDT",
    "commission": "0.00423200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335920321,
    "fee_amount": 0.00420957,
    "fee_currency": "USDT",
    "commission": "0.00420957 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335920919,
    "fee_amount": 2.243e-05,
    "fee_currency": "USDT",
    "commission": "0.00002243 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335922324,
    "fee_amount": 0.00142576,
    "fee_currency": "USDT",
    "commission": "0.00142576 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335927315,
    "fee_amount": 0.00230263,
    "fee_currency": "USDT",
    "commission": "0.00230263 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335927416,
    "fee_amount": 0.00050361,
    "fee_currency": "USDT",
    "commission": "0.00050361 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335928325,
    "fee_amount": 0.00186547,
    "fee_currency": "USDT",
    "commission": "0.00186547 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335929325,
    "fee_amount": 0.00236653,
    "fee_currency": "USDT",
    "commission": "0.00236653 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335957547,
    "fee_amount": 0.004232,
    "fee_currency": "USDT",
    "commission": "0.00423200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335960880,
    "fee_amount": 0.004236,
    "fee_currency": "USDT",
    "commission": "0.00423600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335964067,
    "fee_amount": 0.0010569,
    "fee_currency": "USDT",
    "commission": "0.00105690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335964351,
    "fee_amount": 0.00057496,
    "fee_currency": "USDT",
    "commission": "0.00057496 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335965676,
    "fee_amount": 0.00048195,
    "fee_currency": "USDT",
    "commission": "0.00048195 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335965676,
    "fee_amount": 0.0010564,
    "fee_currency": "USDT",
    "commission": "0.00105640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774335965696,
    "fee_amount": 0.0021156,
    "fee_currency": "USDT",
    "commission": "0.00211560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335967404,
    "fee_amount": 0.00057596,
    "fee_currency": "USDT",
    "commission": "0.00057596 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335967627,
    "fee_amount": 0.00048085,
    "fee_currency": "USDT",
    "commission": "0.00048085 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335968588,
    "fee_amount": 0.00057591,
    "fee_currency": "USDT",
    "commission": "0.00057591 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335970244,
    "fee_amount": 8.877e-05,
    "fee_currency": "USDT",
    "commission": "0.00008877 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335970245,
    "fee_amount": 0.00096794,
    "fee_currency": "USDT",
    "commission": "0.00096794 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335970286,
    "fee_amount": 0.0010562,
    "fee_currency": "USDT",
    "commission": "0.00105620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774335970274,
    "fee_amount": 0.0021152,
    "fee_currency": "USDT",
    "commission": "0.00211520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774335970275,
    "fee_amount": 0.0021144,
    "fee_currency": "USDT",
    "commission": "0.00211440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335970289,
    "fee_amount": 0.0010557,
    "fee_currency": "USDT",
    "commission": "0.00105570 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774335983700,
    "fee_amount": 0.004248,
    "fee_currency": "USDT",
    "commission": "0.00424800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335988558,
    "fee_amount": 0.0010556,
    "fee_currency": "USDT",
    "commission": "0.00105560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774335988553,
    "fee_amount": 0.0021128,
    "fee_currency": "USDT",
    "commission": "0.00211280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335988570,
    "fee_amount": 0.001055,
    "fee_currency": "USDT",
    "commission": "0.00105500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774335988589,
    "fee_amount": 0.0021118,
    "fee_currency": "USDT",
    "commission": "0.00211180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774335988651,
    "fee_amount": 0.0010546,
    "fee_currency": "USDT",
    "commission": "0.00105460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774336248715,
    "fee_amount": 0.004256,
    "fee_currency": "USDT",
    "commission": "0.00425600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774336962141,
    "fee_amount": 0.00027731,
    "fee_currency": "USDT",
    "commission": "0.00027731 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774336962145,
    "fee_amount": 0.0007731,
    "fee_currency": "USDT",
    "commission": "0.00077310 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774336964089,
    "fee_amount": 0.0010499,
    "fee_currency": "USDT",
    "commission": "0.00104990 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337054487,
    "fee_amount": 0.0010528,
    "fee_currency": "USDT",
    "commission": "0.00105280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337083406,
    "fee_amount": 0.0010513,
    "fee_currency": "USDT",
    "commission": "0.00105130 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337086337,
    "fee_amount": 0.0010508,
    "fee_currency": "USDT",
    "commission": "0.00105080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774337469149,
    "fee_amount": 0.0021102,
    "fee_currency": "USDT",
    "commission": "0.00211020 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337490080,
    "fee_amount": 0.0010517,
    "fee_currency": "USDT",
    "commission": "0.00105170 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337490085,
    "fee_amount": 0.0010512,
    "fee_currency": "USDT",
    "commission": "0.00105120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774337490104,
    "fee_amount": 0.0021042,
    "fee_currency": "USDT",
    "commission": "0.00210420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337538121,
    "fee_amount": 0.0010491,
    "fee_currency": "USDT",
    "commission": "0.00104910 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337538121,
    "fee_amount": 0.0010486,
    "fee_currency": "USDT",
    "commission": "0.00104860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774337753282,
    "fee_amount": 0.0010501,
    "fee_currency": "USDT",
    "commission": "0.00105010 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338124052,
    "fee_amount": 4.952e-05,
    "fee_currency": "USDT",
    "commission": "0.00004952 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774338124040,
    "fee_amount": 0.0021088,
    "fee_currency": "USDT",
    "commission": "0.00210880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774338124052,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338124052,
    "fee_amount": 0.00100399,
    "fee_currency": "USDT",
    "commission": "0.00100399 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338124052,
    "fee_amount": 0.001053,
    "fee_currency": "USDT",
    "commission": "0.00105300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774338124085,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338247614,
    "fee_amount": 0.00052224,
    "fee_currency": "USDT",
    "commission": "0.00052224 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338247683,
    "fee_amount": 0.00053067,
    "fee_currency": "USDT",
    "commission": "0.00053067 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338247973,
    "fee_amount": 0.00034743,
    "fee_currency": "USDT",
    "commission": "0.00034743 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338247980,
    "fee_amount": 0.00070538,
    "fee_currency": "USDT",
    "commission": "0.00070538 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249020,
    "fee_amount": 0.0010528,
    "fee_currency": "USDT",
    "commission": "0.00105280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249095,
    "fee_amount": 0.0010528,
    "fee_currency": "USDT",
    "commission": "0.00105280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249221,
    "fee_amount": 0.0010523,
    "fee_currency": "USDT",
    "commission": "0.00105230 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774338249209,
    "fee_amount": 0.0021074,
    "fee_currency": "USDT",
    "commission": "0.00210740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249242,
    "fee_amount": 0.00057539,
    "fee_currency": "USDT",
    "commission": "0.00057539 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249291,
    "fee_amount": 0.00047652,
    "fee_currency": "USDT",
    "commission": "0.00047652 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249291,
    "fee_amount": 9.887e-05,
    "fee_currency": "USDT",
    "commission": "0.00009887 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249298,
    "fee_amount": 0.00057534,
    "fee_currency": "USDT",
    "commission": "0.00057534 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249341,
    "fee_amount": 0.0003776,
    "fee_currency": "USDT",
    "commission": "0.00037760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774338249341,
    "fee_amount": 0.00019772,
    "fee_currency": "USDT",
    "commission": "0.00019772 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774338689437,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774338691289,
    "fee_amount": 0.004256,
    "fee_currency": "USDT",
    "commission": "0.00425600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774338691800,
    "fee_amount": 0.002114,
    "fee_currency": "USDT",
    "commission": "0.00211400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339237371,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339237371,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774339237380,
    "fee_amount": 0.00106,
    "fee_currency": "USDT",
    "commission": "0.00106000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774339237359,
    "fee_amount": 0.0021238,
    "fee_currency": "USDT",
    "commission": "0.00212380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774339237360,
    "fee_amount": 0.0021228,
    "fee_currency": "USDT",
    "commission": "0.00212280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774339237383,
    "fee_amount": 0.0010595,
    "fee_currency": "USDT",
    "commission": "0.00105950 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774339237360,
    "fee_amount": 0.0021218,
    "fee_currency": "USDT",
    "commission": "0.00212180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774339237372,
    "fee_amount": 0.0021208,
    "fee_currency": "USDT",
    "commission": "0.00212080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339237399,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774339237405,
    "fee_amount": 0.0009997,
    "fee_currency": "USDT",
    "commission": "0.00099970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339500335,
    "fee_amount": 0.00146203,
    "fee_currency": "USDT",
    "commission": "0.00146203 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339500410,
    "fee_amount": 0.00279797,
    "fee_currency": "USDT",
    "commission": "0.00279797 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339501459,
    "fee_amount": 0.00219901,
    "fee_currency": "USDT",
    "commission": "0.00219901 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339501513,
    "fee_amount": 0.00206099,
    "fee_currency": "USDT",
    "commission": "0.00206099 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339505214,
    "fee_amount": 0.0037843,
    "fee_currency": "USDT",
    "commission": "0.00378430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339505315,
    "fee_amount": 0.0004797,
    "fee_currency": "USDT",
    "commission": "0.00047970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339506915,
    "fee_amount": 0.00227826,
    "fee_currency": "USDT",
    "commission": "0.00227826 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339506972,
    "fee_amount": 0.00198574,
    "fee_currency": "USDT",
    "commission": "0.00198574 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339522113,
    "fee_amount": 0.004268,
    "fee_currency": "USDT",
    "commission": "0.00426800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339524000,
    "fee_amount": 0.004268,
    "fee_currency": "USDT",
    "commission": "0.00426800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774339550836,
    "fee_amount": 0.002131,
    "fee_currency": "USDT",
    "commission": "0.00213100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774339590879,
    "fee_amount": 0.004272,
    "fee_currency": "USDT",
    "commission": "0.00427200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774340307556,
    "fee_amount": 0.0021166,
    "fee_currency": "USDT",
    "commission": "0.00211660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774340307556,
    "fee_amount": 0.0021156,
    "fee_currency": "USDT",
    "commission": "0.00211560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774341026325,
    "fee_amount": 0.00164767,
    "fee_currency": "USDT",
    "commission": "0.00164767 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774342057017,
    "fee_amount": 0.004248,
    "fee_currency": "USDT",
    "commission": "0.00424800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342057017,
    "fee_amount": 0.0010582,
    "fee_currency": "USDT",
    "commission": "0.00105820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774342057005,
    "fee_amount": 0.0021178,
    "fee_currency": "USDT",
    "commission": "0.00211780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342057020,
    "fee_amount": 0.0010587,
    "fee_currency": "USDT",
    "commission": "0.00105870 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774342057008,
    "fee_amount": 0.0021188,
    "fee_currency": "USDT",
    "commission": "0.00211880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774342057011,
    "fee_amount": 0.002119,
    "fee_currency": "USDT",
    "commission": "0.00211900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342057077,
    "fee_amount": 0.0010592,
    "fee_currency": "USDT",
    "commission": "0.00105920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342057094,
    "fee_amount": 0.00099998,
    "fee_currency": "USDT",
    "commission": "0.00099998 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342057098,
    "fee_amount": 5.933e-05,
    "fee_currency": "USDT",
    "commission": "0.00005933 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774342102423,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342102423,
    "fee_amount": 0.0010581,
    "fee_currency": "USDT",
    "commission": "0.00105810 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774342102408,
    "fee_amount": 0.0021168,
    "fee_currency": "USDT",
    "commission": "0.00211680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342102423,
    "fee_amount": 0.0010577,
    "fee_currency": "USDT",
    "commission": "0.00105770 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774342191380,
    "fee_amount": 0.0021146,
    "fee_currency": "USDT",
    "commission": "0.00211460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342422901,
    "fee_amount": 0.0010525,
    "fee_currency": "USDT",
    "commission": "0.00105250 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774342422890,
    "fee_amount": 0.0021054,
    "fee_currency": "USDT",
    "commission": "0.00210540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342422903,
    "fee_amount": 0.00088263,
    "fee_currency": "USDT",
    "commission": "0.00088263 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774342422903,
    "fee_amount": 0.00016938,
    "fee_currency": "USDT",
    "commission": "0.00016938 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774343521440,
    "fee_amount": 0.0015774,
    "fee_currency": "USDT",
    "commission": "0.00157740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774343521440,
    "fee_amount": 0.0005258,
    "fee_currency": "USDT",
    "commission": "0.00052580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774345253340,
    "fee_amount": 1.06e-06,
    "fee_currency": "USDT",
    "commission": "0.00000106 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774345254089,
    "fee_amount": 0.00053091,
    "fee_currency": "USDT",
    "commission": "0.00053091 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774345256915,
    "fee_amount": 0.00052774,
    "fee_currency": "USDT",
    "commission": "0.00052774 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774345256950,
    "fee_amount": 2.121e-05,
    "fee_currency": "USDT",
    "commission": "0.00002121 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774345256952,
    "fee_amount": 0.00076341,
    "fee_currency": "USDT",
    "commission": "0.00076341 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774345269093,
    "fee_amount": 1.07e-06,
    "fee_currency": "USDT",
    "commission": "0.00000107 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774345270302,
    "fee_amount": 0.00105964,
    "fee_currency": "USDT",
    "commission": "0.00105964 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774345270303,
    "fee_amount": 0.0010613,
    "fee_currency": "USDT",
    "commission": "0.00106130 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774345270290,
    "fee_amount": 0.00157087,
    "fee_currency": "USDT",
    "commission": "0.00157087 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774345270290,
    "fee_amount": 0.00055193,
    "fee_currency": "USDT",
    "commission": "0.00055193 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774345270293,
    "fee_amount": 0.0021238,
    "fee_currency": "USDT",
    "commission": "0.00212380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774345270448,
    "fee_amount": 0.0010618,
    "fee_currency": "USDT",
    "commission": "0.00106180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774348249633,
    "fee_amount": 0.0021046,
    "fee_currency": "USDT",
    "commission": "0.00210460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774348249633,
    "fee_amount": 0.0021036,
    "fee_currency": "USDT",
    "commission": "0.00210360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348462856,
    "fee_amount": 0.0010536,
    "fee_currency": "USDT",
    "commission": "0.00105360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348591315,
    "fee_amount": 0.0010515,
    "fee_currency": "USDT",
    "commission": "0.00105150 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348591389,
    "fee_amount": 0.001051,
    "fee_currency": "USDT",
    "commission": "0.00105100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774348591407,
    "fee_amount": 0.0021044,
    "fee_currency": "USDT",
    "commission": "0.00210440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348617325,
    "fee_amount": 0.0010514,
    "fee_currency": "USDT",
    "commission": "0.00105140 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774348759182,
    "fee_amount": 0.005275,
    "fee_currency": "USDT",
    "commission": "0.00527500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348784157,
    "fee_amount": 0.0010536,
    "fee_currency": "USDT",
    "commission": "0.00105360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348784161,
    "fee_amount": 0.001053,
    "fee_currency": "USDT",
    "commission": "0.00105300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348805324,
    "fee_amount": 0.0010533,
    "fee_currency": "USDT",
    "commission": "0.00105330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348841734,
    "fee_amount": 0.0010523,
    "fee_currency": "USDT",
    "commission": "0.00105230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348842709,
    "fee_amount": 0.0010518,
    "fee_currency": "USDT",
    "commission": "0.00105180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348842720,
    "fee_amount": 0.0010513,
    "fee_currency": "USDT",
    "commission": "0.00105130 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774348842720,
    "fee_amount": 0.0010512,
    "fee_currency": "USDT",
    "commission": "0.00105120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774349490571,
    "fee_amount": 0.0010561,
    "fee_currency": "USDT",
    "commission": "0.00105610 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774349490573,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774349490562,
    "fee_amount": 0.0021136,
    "fee_currency": "USDT",
    "commission": "0.00211360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774350269606,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774350269608,
    "fee_amount": 0.0010495,
    "fee_currency": "USDT",
    "commission": "0.00104950 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351152772,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351152774,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351287208,
    "fee_amount": 0.0016816,
    "fee_currency": "USDT",
    "commission": "0.00168160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351287920,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351579767,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774351581428,
    "fee_amount": 0.00036443,
    "fee_currency": "USDT",
    "commission": "0.00036443 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351581395,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774351581428,
    "fee_amount": 0.00068278,
    "fee_currency": "USDT",
    "commission": "0.00068278 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774351743143,
    "fee_amount": 0.0010467,
    "fee_currency": "USDT",
    "commission": "0.00104670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774351743144,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774351743144,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774351743144,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351743111,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774351743152,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351743111,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351743111,
    "fee_amount": 0.0016736,
    "fee_currency": "USDT",
    "commission": "0.00167360 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351743113,
    "fee_amount": 0.001672,
    "fee_currency": "USDT",
    "commission": "0.00167200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774351743116,
    "fee_amount": 0.0016704,
    "fee_currency": "USDT",
    "commission": "0.00167040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774351743143,
    "fee_amount": 0.0010457,
    "fee_currency": "USDT",
    "commission": "0.00104570 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774351743143,
    "fee_amount": 0.0010452,
    "fee_currency": "USDT",
    "commission": "0.00104520 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774351743110,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774351743110,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774351743110,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774351743110,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774351743132,
    "fee_amount": 0.0020948,
    "fee_currency": "USDT",
    "commission": "0.00209480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774351743132,
    "fee_amount": 0.0020938,
    "fee_currency": "USDT",
    "commission": "0.00209380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774351743132,
    "fee_amount": 0.0020928,
    "fee_currency": "USDT",
    "commission": "0.00209280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774351753141,
    "fee_amount": 0.0020904,
    "fee_currency": "USDT",
    "commission": "0.00209040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774351791220,
    "fee_amount": 0.0020924,
    "fee_currency": "USDT",
    "commission": "0.00209240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774351920766,
    "fee_amount": 0.0020936,
    "fee_currency": "USDT",
    "commission": "0.00209360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352066918,
    "fee_amount": 0.0010434,
    "fee_currency": "USDT",
    "commission": "0.00104340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774352066918,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774352066918,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352066918,
    "fee_amount": 0.0010428,
    "fee_currency": "USDT",
    "commission": "0.00104280 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352066886,
    "fee_amount": 0.0016704,
    "fee_currency": "USDT",
    "commission": "0.00167040 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352066886,
    "fee_amount": 0.0016688,
    "fee_currency": "USDT",
    "commission": "0.00166880 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352066888,
    "fee_amount": 0.0016672,
    "fee_currency": "USDT",
    "commission": "0.00166720 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352066895,
    "fee_amount": 0.0016656,
    "fee_currency": "USDT",
    "commission": "0.00166560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352066918,
    "fee_amount": 0.0010423,
    "fee_currency": "USDT",
    "commission": "0.00104230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352066919,
    "fee_amount": 0.0010418,
    "fee_currency": "USDT",
    "commission": "0.00104180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352066919,
    "fee_amount": 0.0010413,
    "fee_currency": "USDT",
    "commission": "0.00104130 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774352066884,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774352066884,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352066907,
    "fee_amount": 0.0020882,
    "fee_currency": "USDT",
    "commission": "0.00208820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352066907,
    "fee_amount": 0.0020872,
    "fee_currency": "USDT",
    "commission": "0.00208720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352066907,
    "fee_amount": 0.002086,
    "fee_currency": "USDT",
    "commission": "0.00208600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352066909,
    "fee_amount": 0.002085,
    "fee_currency": "USDT",
    "commission": "0.00208500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352066912,
    "fee_amount": 0.002084,
    "fee_currency": "USDT",
    "commission": "0.00208400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352067006,
    "fee_amount": 0.00081904,
    "fee_currency": "USDT",
    "commission": "0.00081904 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352067009,
    "fee_amount": 0.00022167,
    "fee_currency": "USDT",
    "commission": "0.00022167 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352067009,
    "fee_amount": 0.0010406,
    "fee_currency": "USDT",
    "commission": "0.00104060 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352067036,
    "fee_amount": 0.0016672,
    "fee_currency": "USDT",
    "commission": "0.00166720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352100466,
    "fee_amount": 0.001042,
    "fee_currency": "USDT",
    "commission": "0.00104200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352100454,
    "fee_amount": 0.00068818,
    "fee_currency": "USDT",
    "commission": "0.00068818 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352100454,
    "fee_amount": 0.00139722,
    "fee_currency": "USDT",
    "commission": "0.00139722 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352114001,
    "fee_amount": 0.0010421,
    "fee_currency": "USDT",
    "commission": "0.00104210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352114007,
    "fee_amount": 0.0010426,
    "fee_currency": "USDT",
    "commission": "0.00104260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352113996,
    "fee_amount": 0.0020856,
    "fee_currency": "USDT",
    "commission": "0.00208560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352113998,
    "fee_amount": 0.0020866,
    "fee_currency": "USDT",
    "commission": "0.00208660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352153784,
    "fee_amount": 0.00060715,
    "fee_currency": "USDT",
    "commission": "0.00060715 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352327763,
    "fee_amount": 0.00073143,
    "fee_currency": "USDT",
    "commission": "0.00073143 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352327767,
    "fee_amount": 0.00073143,
    "fee_currency": "USDT",
    "commission": "0.00073143 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352546587,
    "fee_amount": 0.0010453,
    "fee_currency": "USDT",
    "commission": "0.00104530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352546591,
    "fee_amount": 0.0010458,
    "fee_currency": "USDT",
    "commission": "0.00104580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774352546596,
    "fee_amount": 0.004196,
    "fee_currency": "USDT",
    "commission": "0.00419600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774352546597,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352546564,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352546596,
    "fee_amount": 0.0010463,
    "fee_currency": "USDT",
    "commission": "0.00104630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352546599,
    "fee_amount": 0.0010468,
    "fee_currency": "USDT",
    "commission": "0.00104680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352546568,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352546601,
    "fee_amount": 0.0010474,
    "fee_currency": "USDT",
    "commission": "0.00104740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774352546610,
    "fee_amount": 0.004204,
    "fee_currency": "USDT",
    "commission": "0.00420400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774352546577,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352546574,
    "fee_amount": 0.002092,
    "fee_currency": "USDT",
    "commission": "0.00209200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352546574,
    "fee_amount": 0.002093,
    "fee_currency": "USDT",
    "commission": "0.00209300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774352546648,
    "fee_amount": 0.00418654,
    "fee_currency": "USDT",
    "commission": "0.00418654 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774352546652,
    "fee_amount": 2.146e-05,
    "fee_currency": "USDT",
    "commission": "0.00002146 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352546585,
    "fee_amount": 0.002094,
    "fee_currency": "USDT",
    "commission": "0.00209400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352546585,
    "fee_amount": 0.0020952,
    "fee_currency": "USDT",
    "commission": "0.00209520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352546589,
    "fee_amount": 0.0020962,
    "fee_currency": "USDT",
    "commission": "0.00209620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352907915,
    "fee_amount": 0.0010517,
    "fee_currency": "USDT",
    "commission": "0.00105170 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774352908000,
    "fee_amount": 0.0010523,
    "fee_currency": "USDT",
    "commission": "0.00105230 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352907969,
    "fee_amount": 0.00111576,
    "fee_currency": "USDT",
    "commission": "0.00111576 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352907969,
    "fee_amount": 0.00098944,
    "fee_currency": "USDT",
    "commission": "0.00098944 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774352907977,
    "fee_amount": 0.0021062,
    "fee_currency": "USDT",
    "commission": "0.00210620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353347552,
    "fee_amount": 0.00422,
    "fee_currency": "USDT",
    "commission": "0.00422000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353347516,
    "fee_amount": 0.0016848,
    "fee_currency": "USDT",
    "commission": "0.00168480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353347550,
    "fee_amount": 0.0010509,
    "fee_currency": "USDT",
    "commission": "0.00105090 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353347552,
    "fee_amount": 0.0010514,
    "fee_currency": "USDT",
    "commission": "0.00105140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353347553,
    "fee_amount": 0.0010519,
    "fee_currency": "USDT",
    "commission": "0.00105190 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353347553,
    "fee_amount": 0.00061466,
    "fee_currency": "USDT",
    "commission": "0.00061466 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353347553,
    "fee_amount": 0.00043784,
    "fee_currency": "USDT",
    "commission": "0.00043784 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353347529,
    "fee_amount": 0.0016864,
    "fee_currency": "USDT",
    "commission": "0.00168640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353347591,
    "fee_amount": 0.001053,
    "fee_currency": "USDT",
    "commission": "0.00105300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353347603,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353347537,
    "fee_amount": 0.0021036,
    "fee_currency": "USDT",
    "commission": "0.00210360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353347537,
    "fee_amount": 0.0021046,
    "fee_currency": "USDT",
    "commission": "0.00210460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353347540,
    "fee_amount": 0.0021056,
    "fee_currency": "USDT",
    "commission": "0.00210560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353347542,
    "fee_amount": 0.0021066,
    "fee_currency": "USDT",
    "commission": "0.00210660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353347546,
    "fee_amount": 0.0021078,
    "fee_currency": "USDT",
    "commission": "0.00210780 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353347668,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353352529,
    "fee_amount": 0.004236,
    "fee_currency": "USDT",
    "commission": "0.00423600 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353352495,
    "fee_amount": 0.0016912,
    "fee_currency": "USDT",
    "commission": "0.00169120 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353352495,
    "fee_amount": 0.0016928,
    "fee_currency": "USDT",
    "commission": "0.00169280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353352530,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353352531,
    "fee_amount": 0.004244,
    "fee_currency": "USDT",
    "commission": "0.00424400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353352530,
    "fee_amount": 0.0010549,
    "fee_currency": "USDT",
    "commission": "0.00105490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353352530,
    "fee_amount": 0.0010553,
    "fee_currency": "USDT",
    "commission": "0.00105530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353352530,
    "fee_amount": 0.0010554,
    "fee_currency": "USDT",
    "commission": "0.00105540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353352532,
    "fee_amount": 0.004248,
    "fee_currency": "USDT",
    "commission": "0.00424800 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353352495,
    "fee_amount": 0.0016944,
    "fee_currency": "USDT",
    "commission": "0.00169440 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353352495,
    "fee_amount": 0.001696,
    "fee_currency": "USDT",
    "commission": "0.00169600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353352534,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353352530,
    "fee_amount": 0.0010558,
    "fee_currency": "USDT",
    "commission": "0.00105580 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353352495,
    "fee_amount": 0.0016976,
    "fee_currency": "USDT",
    "commission": "0.00169760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353352530,
    "fee_amount": 0.0010564,
    "fee_currency": "USDT",
    "commission": "0.00105640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353352567,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353352518,
    "fee_amount": 0.0021126,
    "fee_currency": "USDT",
    "commission": "0.00211260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353352518,
    "fee_amount": 0.0021136,
    "fee_currency": "USDT",
    "commission": "0.00211360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353352518,
    "fee_amount": 0.0021138,
    "fee_currency": "USDT",
    "commission": "0.00211380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353352518,
    "fee_amount": 0.0021148,
    "fee_currency": "USDT",
    "commission": "0.00211480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353352518,
    "fee_amount": 0.0021158,
    "fee_currency": "USDT",
    "commission": "0.00211580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353352673,
    "fee_amount": 0.004256,
    "fee_currency": "USDT",
    "commission": "0.00425600 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353357297,
    "fee_amount": 0.0016992,
    "fee_currency": "USDT",
    "commission": "0.00169920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.0010639,
    "fee_currency": "USDT",
    "commission": "0.00106390 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353358636,
    "fee_amount": 0.0017056,
    "fee_currency": "USDT",
    "commission": "0.00170560 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353358636,
    "fee_amount": 0.0017072,
    "fee_currency": "USDT",
    "commission": "0.00170720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.0010644,
    "fee_currency": "USDT",
    "commission": "0.00106440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.0010645,
    "fee_currency": "USDT",
    "commission": "0.00106450 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.0010649,
    "fee_currency": "USDT",
    "commission": "0.00106490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.001065,
    "fee_currency": "USDT",
    "commission": "0.00106500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.004272,
    "fee_currency": "USDT",
    "commission": "0.00427200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.004276,
    "fee_currency": "USDT",
    "commission": "0.00427600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353358673,
    "fee_amount": 0.00428,
    "fee_currency": "USDT",
    "commission": "0.00428000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353358675,
    "fee_amount": 0.004284,
    "fee_currency": "USDT",
    "commission": "0.00428400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353358636,
    "fee_amount": 0.0017088,
    "fee_currency": "USDT",
    "commission": "0.00170880 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353358636,
    "fee_amount": 0.0017104,
    "fee_currency": "USDT",
    "commission": "0.00171040 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353358636,
    "fee_amount": 0.001712,
    "fee_currency": "USDT",
    "commission": "0.00171200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353358677,
    "fee_amount": 0.004288,
    "fee_currency": "USDT",
    "commission": "0.00428800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353358660,
    "fee_amount": 0.0021302,
    "fee_currency": "USDT",
    "commission": "0.00213020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353358662,
    "fee_amount": 0.0021324,
    "fee_currency": "USDT",
    "commission": "0.00213240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353358662,
    "fee_amount": 0.0021332,
    "fee_currency": "USDT",
    "commission": "0.00213320 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353360460,
    "fee_amount": 0.0017056,
    "fee_currency": "USDT",
    "commission": "0.00170560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353363462,
    "fee_amount": 0.00106354,
    "fee_currency": "USDT",
    "commission": "0.00106354 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353363462,
    "fee_amount": 1.07e-06,
    "fee_currency": "USDT",
    "commission": "0.00000107 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353363441,
    "fee_amount": 0.0021316,
    "fee_currency": "USDT",
    "commission": "0.00213160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353373468,
    "fee_amount": 0.0017024,
    "fee_currency": "USDT",
    "commission": "0.00170240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353373490,
    "fee_amount": 0.00083008,
    "fee_currency": "USDT",
    "commission": "0.00083008 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353373491,
    "fee_amount": 0.00083007,
    "fee_currency": "USDT",
    "commission": "0.00083007 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353373625,
    "fee_amount": 0.0010631,
    "fee_currency": "USDT",
    "commission": "0.00106310 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353373611,
    "fee_amount": 0.00046825,
    "fee_currency": "USDT",
    "commission": "0.00046825 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353390118,
    "fee_amount": 0.001063,
    "fee_currency": "USDT",
    "commission": "0.00106300 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353390084,
    "fee_amount": 0.0017024,
    "fee_currency": "USDT",
    "commission": "0.00170240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353390106,
    "fee_amount": 0.0021282,
    "fee_currency": "USDT",
    "commission": "0.00212820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353390117,
    "fee_amount": 0.0021272,
    "fee_currency": "USDT",
    "commission": "0.00212720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353393865,
    "fee_amount": 0.0010624,
    "fee_currency": "USDT",
    "commission": "0.00106240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353393870,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353393831,
    "fee_amount": 0.0017008,
    "fee_currency": "USDT",
    "commission": "0.00170080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353393865,
    "fee_amount": 0.001062,
    "fee_currency": "USDT",
    "commission": "0.00106200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353393865,
    "fee_amount": 0.0010619,
    "fee_currency": "USDT",
    "commission": "0.00106190 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353393865,
    "fee_amount": 0.0010614,
    "fee_currency": "USDT",
    "commission": "0.00106140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353393853,
    "fee_amount": 0.0021262,
    "fee_currency": "USDT",
    "commission": "0.00212620 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353395806,
    "fee_amount": 0.0017008,
    "fee_currency": "USDT",
    "commission": "0.00170080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353400041,
    "fee_amount": 0.00059953,
    "fee_currency": "USDT",
    "commission": "0.00059953 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353400300,
    "fee_amount": 0.00046158,
    "fee_currency": "USDT",
    "commission": "0.00046158 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353400348,
    "fee_amount": 0.0016992,
    "fee_currency": "USDT",
    "commission": "0.00169920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353400571,
    "fee_amount": 0.0010608,
    "fee_currency": "USDT",
    "commission": "0.00106080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353400588,
    "fee_amount": 0.0010606,
    "fee_currency": "USDT",
    "commission": "0.00106060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353470953,
    "fee_amount": 0.0021252,
    "fee_currency": "USDT",
    "commission": "0.00212520 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353491136,
    "fee_amount": 0.0017056,
    "fee_currency": "USDT",
    "commission": "0.00170560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353500517,
    "fee_amount": 0.0010682,
    "fee_currency": "USDT",
    "commission": "0.00106820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353500517,
    "fee_amount": 0.0010687,
    "fee_currency": "USDT",
    "commission": "0.00106870 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353500517,
    "fee_amount": 0.0010688,
    "fee_currency": "USDT",
    "commission": "0.00106880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353500517,
    "fee_amount": 0.0010693,
    "fee_currency": "USDT",
    "commission": "0.00106930 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353500525,
    "fee_amount": 0.004288,
    "fee_currency": "USDT",
    "commission": "0.00428800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353500525,
    "fee_amount": 0.004292,
    "fee_currency": "USDT",
    "commission": "0.00429200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353500556,
    "fee_amount": 0.001712,
    "fee_currency": "USDT",
    "commission": "0.00171200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353500558,
    "fee_amount": 0.0017136,
    "fee_currency": "USDT",
    "commission": "0.00171360 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353500558,
    "fee_amount": 0.0017152,
    "fee_currency": "USDT",
    "commission": "0.00171520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353500601,
    "fee_amount": 0.004296,
    "fee_currency": "USDT",
    "commission": "0.00429600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353500605,
    "fee_amount": 0.0043,
    "fee_currency": "USDT",
    "commission": "0.00430000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353500569,
    "fee_amount": 0.0015331,
    "fee_currency": "USDT",
    "commission": "0.00153310 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353500576,
    "fee_amount": 0.00018369,
    "fee_currency": "USDT",
    "commission": "0.00018369 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353500540,
    "fee_amount": 0.0021386,
    "fee_currency": "USDT",
    "commission": "0.00213860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353500540,
    "fee_amount": 0.0021396,
    "fee_currency": "USDT",
    "commission": "0.00213960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353500540,
    "fee_amount": 0.0021398,
    "fee_currency": "USDT",
    "commission": "0.00213980 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353500540,
    "fee_amount": 0.0021406,
    "fee_currency": "USDT",
    "commission": "0.00214060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353500540,
    "fee_amount": 0.0021408,
    "fee_currency": "USDT",
    "commission": "0.00214080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353500681,
    "fee_amount": 0.0010724,
    "fee_currency": "USDT",
    "commission": "0.00107240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353500684,
    "fee_amount": 0.0010725,
    "fee_currency": "USDT",
    "commission": "0.00107250 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353502240,
    "fee_amount": 0.0017264,
    "fee_currency": "USDT",
    "commission": "0.00172640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353502277,
    "fee_amount": 0.0010768,
    "fee_currency": "USDT",
    "commission": "0.00107680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353502278,
    "fee_amount": 0.0010773,
    "fee_currency": "USDT",
    "commission": "0.00107730 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353502813,
    "fee_amount": 0.00066724,
    "fee_currency": "USDT",
    "commission": "0.00066724 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353502813,
    "fee_amount": 0.00148516,
    "fee_currency": "USDT",
    "commission": "0.00148516 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353502819,
    "fee_amount": 0.0021514,
    "fee_currency": "USDT",
    "commission": "0.00215140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353503450,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353503416,
    "fee_amount": 0.0017216,
    "fee_currency": "USDT",
    "commission": "0.00172160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353503439,
    "fee_amount": 0.0021504,
    "fee_currency": "USDT",
    "commission": "0.00215040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353503439,
    "fee_amount": 0.0021502,
    "fee_currency": "USDT",
    "commission": "0.00215020 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353503525,
    "fee_amount": 0.00104872,
    "fee_currency": "USDT",
    "commission": "0.00104872 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353504048,
    "fee_amount": 0.00172,
    "fee_currency": "USDT",
    "commission": "0.00172000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353504092,
    "fee_amount": 0.38212,
    "fee_currency": "USDT",
    "commission": "0.38212000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353504077,
    "fee_amount": 0.00109568,
    "fee_currency": "USDT",
    "commission": "0.00109568 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353504077,
    "fee_amount": 0.00064452,
    "fee_currency": "USDT",
    "commission": "0.00064452 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353504079,
    "fee_amount": 0.0004082,
    "fee_currency": "USDT",
    "commission": "0.00040820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353504203,
    "fee_amount": 0.0006444,
    "fee_currency": "USDT",
    "commission": "0.00064440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353505029,
    "fee_amount": 0.0010723,
    "fee_currency": "USDT",
    "commission": "0.00107230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353505033,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353505033,
    "fee_amount": 0.0010718,
    "fee_currency": "USDT",
    "commission": "0.00107180 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353504995,
    "fee_amount": 0.0017168,
    "fee_currency": "USDT",
    "commission": "0.00171680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353504995,
    "fee_amount": 0.0017152,
    "fee_currency": "USDT",
    "commission": "0.00171520 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353504996,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353504998,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353505017,
    "fee_amount": 0.002146,
    "fee_currency": "USDT",
    "commission": "0.00214600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353505018,
    "fee_amount": 0.0021448,
    "fee_currency": "USDT",
    "commission": "0.00214480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353505018,
    "fee_amount": 0.0021446,
    "fee_currency": "USDT",
    "commission": "0.00214460 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353505103,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353505145,
    "fee_amount": 0.0017136,
    "fee_currency": "USDT",
    "commission": "0.00171360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353534362,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353534357,
    "fee_amount": 0.0010685,
    "fee_currency": "USDT",
    "commission": "0.00106850 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353534365,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353534357,
    "fee_amount": 0.001068,
    "fee_currency": "USDT",
    "commission": "0.00106800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353534357,
    "fee_amount": 0.0010674,
    "fee_currency": "USDT",
    "commission": "0.00106740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353534357,
    "fee_amount": 0.0010669,
    "fee_currency": "USDT",
    "commission": "0.00106690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353534357,
    "fee_amount": 0.0010664,
    "fee_currency": "USDT",
    "commission": "0.00106640 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353534392,
    "fee_amount": 0.001712,
    "fee_currency": "USDT",
    "commission": "0.00171200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353534378,
    "fee_amount": 0.0021388,
    "fee_currency": "USDT",
    "commission": "0.00213880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353534417,
    "fee_amount": 0.0021378,
    "fee_currency": "USDT",
    "commission": "0.00213780 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353534531,
    "fee_amount": 0.00100402,
    "fee_currency": "USDT",
    "commission": "0.00100402 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353534537,
    "fee_amount": 0.0017104,
    "fee_currency": "USDT",
    "commission": "0.00171040 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353567403,
    "fee_amount": 0.0017088,
    "fee_currency": "USDT",
    "commission": "0.00170880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353567438,
    "fee_amount": 0.0010671,
    "fee_currency": "USDT",
    "commission": "0.00106710 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353567446,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353567404,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353567404,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353567428,
    "fee_amount": 0.002136,
    "fee_currency": "USDT",
    "commission": "0.00213600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353567563,
    "fee_amount": 0.0010666,
    "fee_currency": "USDT",
    "commission": "0.00106660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353567570,
    "fee_amount": 0.15592,
    "fee_currency": "USDT",
    "commission": "0.15592000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353567540,
    "fee_amount": 0.0017072,
    "fee_currency": "USDT",
    "commission": "0.00170720 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353567570,
    "fee_amount": 0.001066,
    "fee_currency": "USDT",
    "commission": "0.00106600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353567553,
    "fee_amount": 0.00066185,
    "fee_currency": "USDT",
    "commission": "0.00066185 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353567555,
    "fee_amount": 0.00066185,
    "fee_currency": "USDT",
    "commission": "0.00066185 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353567555,
    "fee_amount": 0.00066185,
    "fee_currency": "USDT",
    "commission": "0.00066185 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353567557,
    "fee_amount": 0.00014945,
    "fee_currency": "USDT",
    "commission": "0.00014945 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353567593,
    "fee_amount": 0.00066154,
    "fee_currency": "USDT",
    "commission": "0.00066154 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353574056,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353574052,
    "fee_amount": 0.0010639,
    "fee_currency": "USDT",
    "commission": "0.00106390 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353574056,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353574052,
    "fee_amount": 0.0010633,
    "fee_currency": "USDT",
    "commission": "0.00106330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353574052,
    "fee_amount": 0.0010628,
    "fee_currency": "USDT",
    "commission": "0.00106280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353579179,
    "fee_amount": 0.0010628,
    "fee_currency": "USDT",
    "commission": "0.00106280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353579179,
    "fee_amount": 0.0010627,
    "fee_currency": "USDT",
    "commission": "0.00106270 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353579179,
    "fee_amount": 0.0010622,
    "fee_currency": "USDT",
    "commission": "0.00106220 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353579223,
    "fee_amount": 0.0010627,
    "fee_currency": "USDT",
    "commission": "0.00106270 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353581187,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353581183,
    "fee_amount": 0.0010621,
    "fee_currency": "USDT",
    "commission": "0.00106210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353581183,
    "fee_amount": 0.0010617,
    "fee_currency": "USDT",
    "commission": "0.00106170 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353581183,
    "fee_amount": 0.0010616,
    "fee_currency": "USDT",
    "commission": "0.00106160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353581219,
    "fee_amount": 0.0010615,
    "fee_currency": "USDT",
    "commission": "0.00106150 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353581222,
    "fee_amount": 0.0017008,
    "fee_currency": "USDT",
    "commission": "0.00170080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353590896,
    "fee_amount": 0.0010647,
    "fee_currency": "USDT",
    "commission": "0.00106470 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353590899,
    "fee_amount": 0.004272,
    "fee_currency": "USDT",
    "commission": "0.00427200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353590864,
    "fee_amount": 0.0017056,
    "fee_currency": "USDT",
    "commission": "0.00170560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353590900,
    "fee_amount": 0.0010648,
    "fee_currency": "USDT",
    "commission": "0.00106480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353590900,
    "fee_amount": 0.0010652,
    "fee_currency": "USDT",
    "commission": "0.00106520 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353590865,
    "fee_amount": 0.0017072,
    "fee_currency": "USDT",
    "commission": "0.00170720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353590884,
    "fee_amount": 0.0021304,
    "fee_currency": "USDT",
    "commission": "0.00213040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353590886,
    "fee_amount": 0.0021314,
    "fee_currency": "USDT",
    "commission": "0.00213140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353590887,
    "fee_amount": 0.0021316,
    "fee_currency": "USDT",
    "commission": "0.00213160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353670736,
    "fee_amount": 0.0017152,
    "fee_currency": "USDT",
    "commission": "0.00171520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353670772,
    "fee_amount": 0.0010695,
    "fee_currency": "USDT",
    "commission": "0.00106950 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353670762,
    "fee_amount": 0.002142,
    "fee_currency": "USDT",
    "commission": "0.00214200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353721560,
    "fee_amount": 0.001072,
    "fee_currency": "USDT",
    "commission": "0.00107200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353726738,
    "fee_amount": 0.0010692,
    "fee_currency": "USDT",
    "commission": "0.00106920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353726743,
    "fee_amount": 0.0010687,
    "fee_currency": "USDT",
    "commission": "0.00106870 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353726743,
    "fee_amount": 0.0010686,
    "fee_currency": "USDT",
    "commission": "0.00106860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353726747,
    "fee_amount": 0.39404,
    "fee_currency": "USDT",
    "commission": "0.39404000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353726755,
    "fee_amount": 0.00596,
    "fee_currency": "USDT",
    "commission": "0.00596000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353726746,
    "fee_amount": 0.0010681,
    "fee_currency": "USDT",
    "commission": "0.00106810 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353726751,
    "fee_amount": 0.0010676,
    "fee_currency": "USDT",
    "commission": "0.00106760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353726721,
    "fee_amount": 0.0021406,
    "fee_currency": "USDT",
    "commission": "0.00214060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353726723,
    "fee_amount": 0.0021394,
    "fee_currency": "USDT",
    "commission": "0.00213940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353726726,
    "fee_amount": 0.0021384,
    "fee_currency": "USDT",
    "commission": "0.00213840 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353726780,
    "fee_amount": 0.001712,
    "fee_currency": "USDT",
    "commission": "0.00171200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353726779,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353748529,
    "fee_amount": 0.0010664,
    "fee_currency": "USDT",
    "commission": "0.00106640 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353748496,
    "fee_amount": 0.0017088,
    "fee_currency": "USDT",
    "commission": "0.00170880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353749105,
    "fee_amount": 4.279e-05,
    "fee_currency": "USDT",
    "commission": "0.00004279 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353749187,
    "fee_amount": 0.00209661,
    "fee_currency": "USDT",
    "commission": "0.00209661 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353796869,
    "fee_amount": 0.0010674,
    "fee_currency": "USDT",
    "commission": "0.00106740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353796877,
    "fee_amount": 0.0010668,
    "fee_currency": "USDT",
    "commission": "0.00106680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353796885,
    "fee_amount": 0.002137,
    "fee_currency": "USDT",
    "commission": "0.00213700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353796888,
    "fee_amount": 0.00108926,
    "fee_currency": "USDT",
    "commission": "0.00108926 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353796888,
    "fee_amount": 0.00104654,
    "fee_currency": "USDT",
    "commission": "0.00104654 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353796890,
    "fee_amount": 0.002135,
    "fee_currency": "USDT",
    "commission": "0.00213500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353796890,
    "fee_amount": 0.0021338,
    "fee_currency": "USDT",
    "commission": "0.00213380 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353796958,
    "fee_amount": 0.0017088,
    "fee_currency": "USDT",
    "commission": "0.00170880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353796996,
    "fee_amount": 0.0010664,
    "fee_currency": "USDT",
    "commission": "0.00106640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353796912,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353803145,
    "fee_amount": 0.0010623,
    "fee_currency": "USDT",
    "commission": "0.00106230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353803145,
    "fee_amount": 0.0010621,
    "fee_currency": "USDT",
    "commission": "0.00106210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353803148,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353803149,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353803113,
    "fee_amount": 0.0017008,
    "fee_currency": "USDT",
    "commission": "0.00170080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353803153,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353803145,
    "fee_amount": 0.0010618,
    "fee_currency": "USDT",
    "commission": "0.00106180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353803145,
    "fee_amount": 0.0010616,
    "fee_currency": "USDT",
    "commission": "0.00106160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353803116,
    "fee_amount": 0.0016992,
    "fee_currency": "USDT",
    "commission": "0.00169920 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353803120,
    "fee_amount": 0.0016976,
    "fee_currency": "USDT",
    "commission": "0.00169760 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353803116,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353803123,
    "fee_amount": 0.001696,
    "fee_currency": "USDT",
    "commission": "0.00169600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353803159,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353803118,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353803140,
    "fee_amount": 0.0016944,
    "fee_currency": "USDT",
    "commission": "0.00169440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353803176,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353803182,
    "fee_amount": 0.35964,
    "fee_currency": "USDT",
    "commission": "0.35964000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353803120,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353803190,
    "fee_amount": 0.04036,
    "fee_currency": "USDT",
    "commission": "0.04036000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353803143,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803135,
    "fee_amount": 0.0021266,
    "fee_currency": "USDT",
    "commission": "0.00212660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803137,
    "fee_amount": 0.0021262,
    "fee_currency": "USDT",
    "commission": "0.00212620 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353803167,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803138,
    "fee_amount": 0.0021254,
    "fee_currency": "USDT",
    "commission": "0.00212540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803138,
    "fee_amount": 0.0021252,
    "fee_currency": "USDT",
    "commission": "0.00212520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803139,
    "fee_amount": 0.002124,
    "fee_currency": "USDT",
    "commission": "0.00212400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803653,
    "fee_amount": 0.002122,
    "fee_currency": "USDT",
    "commission": "0.00212200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803673,
    "fee_amount": 0.0021228,
    "fee_currency": "USDT",
    "commission": "0.00212280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353803814,
    "fee_amount": 0.0021234,
    "fee_currency": "USDT",
    "commission": "0.00212340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353804141,
    "fee_amount": 0.0021244,
    "fee_currency": "USDT",
    "commission": "0.00212440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353805489,
    "fee_amount": 0.0021218,
    "fee_currency": "USDT",
    "commission": "0.00212180 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353809164,
    "fee_amount": 0.0016992,
    "fee_currency": "USDT",
    "commission": "0.00169920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809185,
    "fee_amount": 0.00065813,
    "fee_currency": "USDT",
    "commission": "0.00065813 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809185,
    "fee_amount": 0.00065813,
    "fee_currency": "USDT",
    "commission": "0.00065813 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809187,
    "fee_amount": 0.00080674,
    "fee_currency": "USDT",
    "commission": "0.00080674 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353809596,
    "fee_amount": 0.001056,
    "fee_currency": "USDT",
    "commission": "0.00105600 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353809561,
    "fee_amount": 0.0016944,
    "fee_currency": "USDT",
    "commission": "0.00169440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353809600,
    "fee_amount": 0.0010557,
    "fee_currency": "USDT",
    "commission": "0.00105570 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353809561,
    "fee_amount": 0.0016928,
    "fee_currency": "USDT",
    "commission": "0.00169280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353809599,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353809600,
    "fee_amount": 0.0010554,
    "fee_currency": "USDT",
    "commission": "0.00105540 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353809561,
    "fee_amount": 0.0016912,
    "fee_currency": "USDT",
    "commission": "0.00169120 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353809564,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353809564,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353809561,
    "fee_amount": 0.0016896,
    "fee_currency": "USDT",
    "commission": "0.00168960 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353809564,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353809640,
    "fee_amount": 0.0010549,
    "fee_currency": "USDT",
    "commission": "0.00105490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353809640,
    "fee_amount": 0.0010548,
    "fee_currency": "USDT",
    "commission": "0.00105480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809584,
    "fee_amount": 0.0021174,
    "fee_currency": "USDT",
    "commission": "0.00211740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809584,
    "fee_amount": 0.0021164,
    "fee_currency": "USDT",
    "commission": "0.00211640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809587,
    "fee_amount": 0.0021154,
    "fee_currency": "USDT",
    "commission": "0.00211540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809606,
    "fee_amount": 0.00063426,
    "fee_currency": "USDT",
    "commission": "0.00063426 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809607,
    "fee_amount": 0.00063426,
    "fee_currency": "USDT",
    "commission": "0.00063426 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809613,
    "fee_amount": 0.0006554,
    "fee_currency": "USDT",
    "commission": "0.00065540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809613,
    "fee_amount": 0.00019028,
    "fee_currency": "USDT",
    "commission": "0.00019028 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353809613,
    "fee_amount": 0.0021132,
    "fee_currency": "USDT",
    "commission": "0.00211320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353811627,
    "fee_amount": 0.0021152,
    "fee_currency": "USDT",
    "commission": "0.00211520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353813723,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353813723,
    "fee_amount": 0.0010567,
    "fee_currency": "USDT",
    "commission": "0.00105670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353813730,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353813723,
    "fee_amount": 0.0010568,
    "fee_currency": "USDT",
    "commission": "0.00105680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353813703,
    "fee_amount": 0.0016928,
    "fee_currency": "USDT",
    "commission": "0.00169280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353813725,
    "fee_amount": 0.00065577,
    "fee_currency": "USDT",
    "commission": "0.00065577 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353813726,
    "fee_amount": 0.00145963,
    "fee_currency": "USDT",
    "commission": "0.00145963 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353813726,
    "fee_amount": 0.0021156,
    "fee_currency": "USDT",
    "commission": "0.00211560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353813726,
    "fee_amount": 0.002116,
    "fee_currency": "USDT",
    "commission": "0.00211600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353813726,
    "fee_amount": 0.0021164,
    "fee_currency": "USDT",
    "commission": "0.00211640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353814706,
    "fee_amount": 0.0010536,
    "fee_currency": "USDT",
    "commission": "0.00105360 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353814672,
    "fee_amount": 0.0016864,
    "fee_currency": "USDT",
    "commission": "0.00168640 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353814672,
    "fee_amount": 0.0016848,
    "fee_currency": "USDT",
    "commission": "0.00168480 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353814672,
    "fee_amount": 0.0016832,
    "fee_currency": "USDT",
    "commission": "0.00168320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353814708,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353814712,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353814706,
    "fee_amount": 0.0010535,
    "fee_currency": "USDT",
    "commission": "0.00105350 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353814709,
    "fee_amount": 0.001053,
    "fee_currency": "USDT",
    "commission": "0.00105300 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353814710,
    "fee_amount": 0.0010525,
    "fee_currency": "USDT",
    "commission": "0.00105250 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353814710,
    "fee_amount": 0.0010524,
    "fee_currency": "USDT",
    "commission": "0.00105240 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353814676,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774353814676,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353814763,
    "fee_amount": 0.0010533,
    "fee_currency": "USDT",
    "commission": "0.00105330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353814766,
    "fee_amount": 0.001053,
    "fee_currency": "USDT",
    "commission": "0.00105300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353814695,
    "fee_amount": 0.0021092,
    "fee_currency": "USDT",
    "commission": "0.00210920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353814695,
    "fee_amount": 0.0021088,
    "fee_currency": "USDT",
    "commission": "0.00210880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353814775,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353814700,
    "fee_amount": 0.0021078,
    "fee_currency": "USDT",
    "commission": "0.00210780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353814723,
    "fee_amount": 0.0021068,
    "fee_currency": "USDT",
    "commission": "0.00210680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353814836,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353815379,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353815568,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353815736,
    "fee_amount": 0.0010538,
    "fee_currency": "USDT",
    "commission": "0.00105380 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353815700,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353815724,
    "fee_amount": 0.0021118,
    "fee_currency": "USDT",
    "commission": "0.00211180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353815994,
    "fee_amount": 0.0010537,
    "fee_currency": "USDT",
    "commission": "0.00105370 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353816014,
    "fee_amount": 0.0021116,
    "fee_currency": "USDT",
    "commission": "0.00211160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353816014,
    "fee_amount": 0.002112,
    "fee_currency": "USDT",
    "commission": "0.00211200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353817042,
    "fee_amount": 0.0010537,
    "fee_currency": "USDT",
    "commission": "0.00105370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353817048,
    "fee_amount": 0.0010538,
    "fee_currency": "USDT",
    "commission": "0.00105380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353817527,
    "fee_amount": 0.0010539,
    "fee_currency": "USDT",
    "commission": "0.00105390 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353817578,
    "fee_amount": 0.00067571,
    "fee_currency": "USDT",
    "commission": "0.00067571 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353817580,
    "fee_amount": 0.00143589,
    "fee_currency": "USDT",
    "commission": "0.00143589 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353817588,
    "fee_amount": 0.0021118,
    "fee_currency": "USDT",
    "commission": "0.00211180 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353817649,
    "fee_amount": 0.0016896,
    "fee_currency": "USDT",
    "commission": "0.00168960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353817689,
    "fee_amount": 0.004232,
    "fee_currency": "USDT",
    "commission": "0.00423200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353817685,
    "fee_amount": 0.0010541,
    "fee_currency": "USDT",
    "commission": "0.00105410 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353817686,
    "fee_amount": 0.0010542,
    "fee_currency": "USDT",
    "commission": "0.00105420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353817673,
    "fee_amount": 0.0021122,
    "fee_currency": "USDT",
    "commission": "0.00211220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353817673,
    "fee_amount": 0.0021124,
    "fee_currency": "USDT",
    "commission": "0.00211240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774353818159,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353819618,
    "fee_amount": 0.0006961,
    "fee_currency": "USDT",
    "commission": "0.00069610 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353819618,
    "fee_amount": 0.0006961,
    "fee_currency": "USDT",
    "commission": "0.00069610 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353819620,
    "fee_amount": 0.0007172,
    "fee_currency": "USDT",
    "commission": "0.00071720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353819620,
    "fee_amount": 0.00137124,
    "fee_currency": "USDT",
    "commission": "0.00137124 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353819621,
    "fee_amount": 0.00069617,
    "fee_currency": "USDT",
    "commission": "0.00069617 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353819621,
    "fee_amount": 4.219e-05,
    "fee_currency": "USDT",
    "commission": "0.00004219 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353820113,
    "fee_amount": 0.0021096,
    "fee_currency": "USDT",
    "commission": "0.00210960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353820776,
    "fee_amount": 0.0010529,
    "fee_currency": "USDT",
    "commission": "0.00105290 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353820755,
    "fee_amount": 0.00021096,
    "fee_currency": "USDT",
    "commission": "0.00021096 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353820757,
    "fee_amount": 0.00189864,
    "fee_currency": "USDT",
    "commission": "0.00189864 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353820757,
    "fee_amount": 0.00016878,
    "fee_currency": "USDT",
    "commission": "0.00016878 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353820759,
    "fee_amount": 0.00067514,
    "fee_currency": "USDT",
    "commission": "0.00067514 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353822473,
    "fee_amount": 0.00105205,
    "fee_currency": "USDT",
    "commission": "0.00105205 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353822474,
    "fee_amount": 1.06e-06,
    "fee_currency": "USDT",
    "commission": "0.00000106 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774353822443,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353822530,
    "fee_amount": 0.0010534,
    "fee_currency": "USDT",
    "commission": "0.00105340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353822461,
    "fee_amount": 0.0021096,
    "fee_currency": "USDT",
    "commission": "0.00210960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353822462,
    "fee_amount": 0.0006963,
    "fee_currency": "USDT",
    "commission": "0.00069630 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353822463,
    "fee_amount": 0.0014137,
    "fee_currency": "USDT",
    "commission": "0.00141370 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353822466,
    "fee_amount": 0.0021104,
    "fee_currency": "USDT",
    "commission": "0.00211040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353825100,
    "fee_amount": 0.00069663,
    "fee_currency": "USDT",
    "commission": "0.00069663 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353825101,
    "fee_amount": 0.00141437,
    "fee_currency": "USDT",
    "commission": "0.00141437 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353825101,
    "fee_amount": 0.00067558,
    "fee_currency": "USDT",
    "commission": "0.00067558 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835160,
    "fee_amount": 0.00069676,
    "fee_currency": "USDT",
    "commission": "0.00069676 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835162,
    "fee_amount": 0.00069676,
    "fee_currency": "USDT",
    "commission": "0.00069676 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835162,
    "fee_amount": 0.00069677,
    "fee_currency": "USDT",
    "commission": "0.00069677 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835163,
    "fee_amount": 2.111e-05,
    "fee_currency": "USDT",
    "commission": "0.00002111 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835165,
    "fee_amount": 0.00069683,
    "fee_currency": "USDT",
    "commission": "0.00069683 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835167,
    "fee_amount": 0.00069683,
    "fee_currency": "USDT",
    "commission": "0.00069683 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835169,
    "fee_amount": 0.00069682,
    "fee_currency": "USDT",
    "commission": "0.00069682 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353835170,
    "fee_amount": 2.112e-05,
    "fee_currency": "USDT",
    "commission": "0.00002112 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353840457,
    "fee_amount": 0.0021126,
    "fee_currency": "USDT",
    "commission": "0.00211260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353870702,
    "fee_amount": 0.00069722,
    "fee_currency": "USDT",
    "commission": "0.00069722 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353870702,
    "fee_amount": 0.00069723,
    "fee_currency": "USDT",
    "commission": "0.00069723 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353958075,
    "fee_amount": 0.0010544,
    "fee_currency": "USDT",
    "commission": "0.00105440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353958063,
    "fee_amount": 0.0021118,
    "fee_currency": "USDT",
    "commission": "0.00211180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774353959084,
    "fee_amount": 0.00202848,
    "fee_currency": "USDT",
    "commission": "0.00202848 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774353963053,
    "fee_amount": 2.321e-05,
    "fee_currency": "USDT",
    "commission": "0.00002321 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354020807,
    "fee_amount": 0.0010545,
    "fee_currency": "USDT",
    "commission": "0.00105450 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354020795,
    "fee_amount": 0.00071815,
    "fee_currency": "USDT",
    "commission": "0.00071815 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354020795,
    "fee_amount": 0.00139405,
    "fee_currency": "USDT",
    "commission": "0.00139405 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354020964,
    "fee_amount": 0.0010128,
    "fee_currency": "USDT",
    "commission": "0.00101280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354061076,
    "fee_amount": 0.0010546,
    "fee_currency": "USDT",
    "commission": "0.00105460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354141335,
    "fee_amount": 0.0010536,
    "fee_currency": "USDT",
    "commission": "0.00105360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354161431,
    "fee_amount": 0.0010546,
    "fee_currency": "USDT",
    "commission": "0.00105460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354266470,
    "fee_amount": 0.0010537,
    "fee_currency": "USDT",
    "commission": "0.00105370 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354272487,
    "fee_amount": 0.0010547,
    "fee_currency": "USDT",
    "commission": "0.00105470 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354272487,
    "fee_amount": 0.0010552,
    "fee_currency": "USDT",
    "commission": "0.00105520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354272491,
    "fee_amount": 0.0010553,
    "fee_currency": "USDT",
    "commission": "0.00105530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354272483,
    "fee_amount": 0.004236,
    "fee_currency": "USDT",
    "commission": "0.00423600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354272525,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354272525,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774354272515,
    "fee_amount": 0.0016896,
    "fee_currency": "USDT",
    "commission": "0.00168960 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774354272524,
    "fee_amount": 0.0016912,
    "fee_currency": "USDT",
    "commission": "0.00169120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354272502,
    "fee_amount": 0.0021126,
    "fee_currency": "USDT",
    "commission": "0.00211260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354272506,
    "fee_amount": 0.0021136,
    "fee_currency": "USDT",
    "commission": "0.00211360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354272506,
    "fee_amount": 0.0021138,
    "fee_currency": "USDT",
    "commission": "0.00211380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354273670,
    "fee_amount": 0.004244,
    "fee_currency": "USDT",
    "commission": "0.00424400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354273674,
    "fee_amount": 0.0010577,
    "fee_currency": "USDT",
    "commission": "0.00105770 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354279939,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354279939,
    "fee_amount": 0.0010591,
    "fee_currency": "USDT",
    "commission": "0.00105910 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354279939,
    "fee_amount": 0.0010596,
    "fee_currency": "USDT",
    "commission": "0.00105960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354332164,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354332168,
    "fee_amount": 0.0010592,
    "fee_currency": "USDT",
    "commission": "0.00105920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354332168,
    "fee_amount": 0.0010597,
    "fee_currency": "USDT",
    "commission": "0.00105970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354332168,
    "fee_amount": 0.004256,
    "fee_currency": "USDT",
    "commission": "0.00425600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354332168,
    "fee_amount": 0.00426,
    "fee_currency": "USDT",
    "commission": "0.00426000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354332168,
    "fee_amount": 0.0010603,
    "fee_currency": "USDT",
    "commission": "0.00106030 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354332168,
    "fee_amount": 0.0010608,
    "fee_currency": "USDT",
    "commission": "0.00106080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354332174,
    "fee_amount": 0.0010613,
    "fee_currency": "USDT",
    "commission": "0.00106130 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774354332200,
    "fee_amount": 0.0016976,
    "fee_currency": "USDT",
    "commission": "0.00169760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354332238,
    "fee_amount": 0.004264,
    "fee_currency": "USDT",
    "commission": "0.00426400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774354332201,
    "fee_amount": 0.0016992,
    "fee_currency": "USDT",
    "commission": "0.00169920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354332183,
    "fee_amount": 0.0021216,
    "fee_currency": "USDT",
    "commission": "0.00212160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774354332204,
    "fee_amount": 0.0017008,
    "fee_currency": "USDT",
    "commission": "0.00170080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354332183,
    "fee_amount": 0.0021228,
    "fee_currency": "USDT",
    "commission": "0.00212280 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354332183,
    "fee_amount": 0.0021238,
    "fee_currency": "USDT",
    "commission": "0.00212380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354332189,
    "fee_amount": 0.0021248,
    "fee_currency": "USDT",
    "commission": "0.00212480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354332189,
    "fee_amount": 0.002126,
    "fee_currency": "USDT",
    "commission": "0.00212600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354332652,
    "fee_amount": 0.0010632,
    "fee_currency": "USDT",
    "commission": "0.00106320 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774354398775,
    "fee_amount": 0.001696,
    "fee_currency": "USDT",
    "commission": "0.00169600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354398793,
    "fee_amount": 0.0021224,
    "fee_currency": "USDT",
    "commission": "0.00212240 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774354419516,
    "fee_amount": 0.0016944,
    "fee_currency": "USDT",
    "commission": "0.00169440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774354434297,
    "fee_amount": 0.004264,
    "fee_currency": "USDT",
    "commission": "0.00426400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354434363,
    "fee_amount": 0.0010617,
    "fee_currency": "USDT",
    "commission": "0.00106170 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354434368,
    "fee_amount": 0.0010622,
    "fee_currency": "USDT",
    "commission": "0.00106220 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354434374,
    "fee_amount": 0.0010627,
    "fee_currency": "USDT",
    "commission": "0.00106270 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774354434356,
    "fee_amount": 0.0021268,
    "fee_currency": "USDT",
    "commission": "0.00212680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354523775,
    "fee_amount": 0.0010604,
    "fee_currency": "USDT",
    "commission": "0.00106040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354523775,
    "fee_amount": 0.0010608,
    "fee_currency": "USDT",
    "commission": "0.00106080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774354523775,
    "fee_amount": 0.0010609,
    "fee_currency": "USDT",
    "commission": "0.00106090 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774355111611,
    "fee_amount": 0.00428,
    "fee_currency": "USDT",
    "commission": "0.00428000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774355111599,
    "fee_amount": 0.0010665,
    "fee_currency": "USDT",
    "commission": "0.00106650 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774355405272,
    "fee_amount": 0.001704,
    "fee_currency": "USDT",
    "commission": "0.00170400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774355433800,
    "fee_amount": 0.0016992,
    "fee_currency": "USDT",
    "commission": "0.00169920 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774358209527,
    "fee_amount": 0.01057,
    "fee_currency": "USDT",
    "commission": "0.01057000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774358832890,
    "fee_amount": 0.0021108,
    "fee_currency": "USDT",
    "commission": "0.00211080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359364026,
    "fee_amount": 0.0010497,
    "fee_currency": "USDT",
    "commission": "0.00104970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359364026,
    "fee_amount": 0.0010491,
    "fee_currency": "USDT",
    "commission": "0.00104910 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359364026,
    "fee_amount": 0.0010486,
    "fee_currency": "USDT",
    "commission": "0.00104860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359364026,
    "fee_amount": 0.0010481,
    "fee_currency": "USDT",
    "commission": "0.00104810 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359363997,
    "fee_amount": 0.0021004,
    "fee_currency": "USDT",
    "commission": "0.00210040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359559196,
    "fee_amount": 2.098e-05,
    "fee_currency": "USDT",
    "commission": "0.00002098 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359560226,
    "fee_amount": 0.0020972,
    "fee_currency": "USDT",
    "commission": "0.00209720 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774359560236,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359648901,
    "fee_amount": 0.0010445,
    "fee_currency": "USDT",
    "commission": "0.00104450 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774359648904,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774359648857,
    "fee_amount": 0.001672,
    "fee_currency": "USDT",
    "commission": "0.00167200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359648909,
    "fee_amount": 0.0010439,
    "fee_currency": "USDT",
    "commission": "0.00104390 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359648931,
    "fee_amount": 0.00070639,
    "fee_currency": "USDT",
    "commission": "0.00070639 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359648936,
    "fee_amount": 0.00033702,
    "fee_currency": "USDT",
    "commission": "0.00033702 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359648854,
    "fee_amount": 0.0020922,
    "fee_currency": "USDT",
    "commission": "0.00209220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359648867,
    "fee_amount": 0.002091,
    "fee_currency": "USDT",
    "commission": "0.00209100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359648893,
    "fee_amount": 0.00209,
    "fee_currency": "USDT",
    "commission": "0.00209000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774359649017,
    "fee_amount": 0.0010435,
    "fee_currency": "USDT",
    "commission": "0.00104350 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774359648876,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359648950,
    "fee_amount": 0.002089,
    "fee_currency": "USDT",
    "commission": "0.00208900 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774359649155,
    "fee_amount": 0.0016704,
    "fee_currency": "USDT",
    "commission": "0.00167040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359649069,
    "fee_amount": 2.088e-05,
    "fee_currency": "USDT",
    "commission": "0.00002088 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774359648984,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774359649396,
    "fee_amount": 0.0016688,
    "fee_currency": "USDT",
    "commission": "0.00166880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359710361,
    "fee_amount": 0.0020862,
    "fee_currency": "USDT",
    "commission": "0.00208620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774359913089,
    "fee_amount": 0.0015615,
    "fee_currency": "USDT",
    "commission": "0.00156150 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774360359648,
    "fee_amount": 0.0016688,
    "fee_currency": "USDT",
    "commission": "0.00166880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774360359665,
    "fee_amount": 0.002087,
    "fee_currency": "USDT",
    "commission": "0.00208700 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774360630912,
    "fee_amount": 0.0016736,
    "fee_currency": "USDT",
    "commission": "0.00167360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361280434,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361280427,
    "fee_amount": 0.0010518,
    "fee_currency": "USDT",
    "commission": "0.00105180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361280427,
    "fee_amount": 0.0010523,
    "fee_currency": "USDT",
    "commission": "0.00105230 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361280427,
    "fee_amount": 0.0010528,
    "fee_currency": "USDT",
    "commission": "0.00105280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361280427,
    "fee_amount": 0.0010533,
    "fee_currency": "USDT",
    "commission": "0.00105330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361280427,
    "fee_amount": 0.0010538,
    "fee_currency": "USDT",
    "commission": "0.00105380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361280443,
    "fee_amount": 0.004228,
    "fee_currency": "USDT",
    "commission": "0.00422800 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361280465,
    "fee_amount": 0.0016848,
    "fee_currency": "USDT",
    "commission": "0.00168480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361280509,
    "fee_amount": 0.0010549,
    "fee_currency": "USDT",
    "commission": "0.00105490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361280515,
    "fee_amount": 0.004232,
    "fee_currency": "USDT",
    "commission": "0.00423200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361280509,
    "fee_amount": 0.001055,
    "fee_currency": "USDT",
    "commission": "0.00105500 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361280471,
    "fee_amount": 0.0016864,
    "fee_currency": "USDT",
    "commission": "0.00168640 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361280472,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361280449,
    "fee_amount": 0.0021064,
    "fee_currency": "USDT",
    "commission": "0.00210640 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361280472,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361280449,
    "fee_amount": 0.0021074,
    "fee_currency": "USDT",
    "commission": "0.00210740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361280453,
    "fee_amount": 0.0021084,
    "fee_currency": "USDT",
    "commission": "0.00210840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361280455,
    "fee_amount": 0.0021094,
    "fee_currency": "USDT",
    "commission": "0.00210940 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361280622,
    "fee_amount": 0.004232,
    "fee_currency": "USDT",
    "commission": "0.00423200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361280782,
    "fee_amount": 0.0016896,
    "fee_currency": "USDT",
    "commission": "0.00168960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281249,
    "fee_amount": 0.00424,
    "fee_currency": "USDT",
    "commission": "0.00424000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361281244,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361281244,
    "fee_amount": 0.0010568,
    "fee_currency": "USDT",
    "commission": "0.00105680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281249,
    "fee_amount": 0.004244,
    "fee_currency": "USDT",
    "commission": "0.00424400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281249,
    "fee_amount": 0.004248,
    "fee_currency": "USDT",
    "commission": "0.00424800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361281244,
    "fee_amount": 0.0010569,
    "fee_currency": "USDT",
    "commission": "0.00105690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361281244,
    "fee_amount": 0.0010573,
    "fee_currency": "USDT",
    "commission": "0.00105730 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361281244,
    "fee_amount": 0.0010574,
    "fee_currency": "USDT",
    "commission": "0.00105740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281281,
    "fee_amount": 0.004252,
    "fee_currency": "USDT",
    "commission": "0.00425200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281286,
    "fee_amount": 0.004256,
    "fee_currency": "USDT",
    "commission": "0.00425600 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361281280,
    "fee_amount": 0.00152182,
    "fee_currency": "USDT",
    "commission": "0.00152182 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361281280,
    "fee_amount": 0.00017097,
    "fee_currency": "USDT",
    "commission": "0.00017097 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361281283,
    "fee_amount": 0.0016944,
    "fee_currency": "USDT",
    "commission": "0.00169440 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361281284,
    "fee_amount": 0.001696,
    "fee_currency": "USDT",
    "commission": "0.00169600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361281264,
    "fee_amount": 0.0021156,
    "fee_currency": "USDT",
    "commission": "0.00211560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361281264,
    "fee_amount": 0.0021166,
    "fee_currency": "USDT",
    "commission": "0.00211660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361281269,
    "fee_amount": 0.0021176,
    "fee_currency": "USDT",
    "commission": "0.00211760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361281269,
    "fee_amount": 0.0021178,
    "fee_currency": "USDT",
    "commission": "0.00211780 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361281270,
    "fee_amount": 0.0021188,
    "fee_currency": "USDT",
    "commission": "0.00211880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281369,
    "fee_amount": 0.004264,
    "fee_currency": "USDT",
    "commission": "0.00426400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361281744,
    "fee_amount": 0.0010654,
    "fee_currency": "USDT",
    "commission": "0.00106540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281769,
    "fee_amount": 0.00428,
    "fee_currency": "USDT",
    "commission": "0.00428000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361281865,
    "fee_amount": 0.004284,
    "fee_currency": "USDT",
    "commission": "0.00428400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361281835,
    "fee_amount": 0.0017088,
    "fee_currency": "USDT",
    "commission": "0.00170880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361281823,
    "fee_amount": 0.002136,
    "fee_currency": "USDT",
    "commission": "0.00213600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361281823,
    "fee_amount": 0.002137,
    "fee_currency": "USDT",
    "commission": "0.00213700 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361282086,
    "fee_amount": 0.0017056,
    "fee_currency": "USDT",
    "commission": "0.00170560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361282205,
    "fee_amount": 0.00204461,
    "fee_currency": "USDT",
    "commission": "0.00204461 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361282207,
    "fee_amount": 8.519e-05,
    "fee_currency": "USDT",
    "commission": "0.00008519 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361282785,
    "fee_amount": 0.0017024,
    "fee_currency": "USDT",
    "commission": "0.00170240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361282809,
    "fee_amount": 0.0021288,
    "fee_currency": "USDT",
    "commission": "0.00212880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361282972,
    "fee_amount": 0.0021274,
    "fee_currency": "USDT",
    "commission": "0.00212740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361283013,
    "fee_amount": 0.0021276,
    "fee_currency": "USDT",
    "commission": "0.00212760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361283617,
    "fee_amount": 0.0021294,
    "fee_currency": "USDT",
    "commission": "0.00212940 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361285103,
    "fee_amount": 0.0017008,
    "fee_currency": "USDT",
    "commission": "0.00170080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361285254,
    "fee_amount": 0.0010626,
    "fee_currency": "USDT",
    "commission": "0.00106260 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774361285220,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361285219,
    "fee_amount": 0.0021274,
    "fee_currency": "USDT",
    "commission": "0.00212740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361285219,
    "fee_amount": 0.0021272,
    "fee_currency": "USDT",
    "commission": "0.00212720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361285219,
    "fee_amount": 0.0021268,
    "fee_currency": "USDT",
    "commission": "0.00212680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361285219,
    "fee_amount": 0.0021266,
    "fee_currency": "USDT",
    "commission": "0.00212660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361285219,
    "fee_amount": 0.0021262,
    "fee_currency": "USDT",
    "commission": "0.00212620 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361300188,
    "fee_amount": 0.0016992,
    "fee_currency": "USDT",
    "commission": "0.00169920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361300212,
    "fee_amount": 0.0021244,
    "fee_currency": "USDT",
    "commission": "0.00212440 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361306032,
    "fee_amount": 0.0017008,
    "fee_currency": "USDT",
    "commission": "0.00170080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361306055,
    "fee_amount": 0.0021264,
    "fee_currency": "USDT",
    "commission": "0.00212640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361315832,
    "fee_amount": 0.0021242,
    "fee_currency": "USDT",
    "commission": "0.00212420 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361320988,
    "fee_amount": 0.0016976,
    "fee_currency": "USDT",
    "commission": "0.00169760 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774361320998,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361321011,
    "fee_amount": 0.002122,
    "fee_currency": "USDT",
    "commission": "0.00212200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361321020,
    "fee_amount": 0.002121,
    "fee_currency": "USDT",
    "commission": "0.00212100 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361482715,
    "fee_amount": 0.0016912,
    "fee_currency": "USDT",
    "commission": "0.00169120 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361551628,
    "fee_amount": 0.0016848,
    "fee_currency": "USDT",
    "commission": "0.00168480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361793501,
    "fee_amount": 0.001046,
    "fee_currency": "USDT",
    "commission": "0.00104600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361793509,
    "fee_amount": 0.0010465,
    "fee_currency": "USDT",
    "commission": "0.00104650 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361793509,
    "fee_amount": 0.001047,
    "fee_currency": "USDT",
    "commission": "0.00104700 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774361793503,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361793486,
    "fee_amount": 0.0020942,
    "fee_currency": "USDT",
    "commission": "0.00209420 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774361793567,
    "fee_amount": 0.00083808,
    "fee_currency": "USDT",
    "commission": "0.00083808 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361817391,
    "fee_amount": 0.0010449,
    "fee_currency": "USDT",
    "commission": "0.00104490 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774361817417,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774361817428,
    "fee_amount": 0.0010444,
    "fee_currency": "USDT",
    "commission": "0.00104440 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774362037607,
    "fee_amount": 0.0016688,
    "fee_currency": "USDT",
    "commission": "0.00166880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362037656,
    "fee_amount": 0.0010433,
    "fee_currency": "USDT",
    "commission": "0.00104330 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774362037608,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362060629,
    "fee_amount": 0.001045,
    "fee_currency": "USDT",
    "commission": "0.00104500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362069367,
    "fee_amount": 0.0010455,
    "fee_currency": "USDT",
    "commission": "0.00104550 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362069358,
    "fee_amount": 0.0020934,
    "fee_currency": "USDT",
    "commission": "0.00209340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362069922,
    "fee_amount": 0.001046,
    "fee_currency": "USDT",
    "commission": "0.00104600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362069936,
    "fee_amount": 0.0010465,
    "fee_currency": "USDT",
    "commission": "0.00104650 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362069941,
    "fee_amount": 0.001047,
    "fee_currency": "USDT",
    "commission": "0.00104700 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774362069904,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362069941,
    "fee_amount": 0.0010471,
    "fee_currency": "USDT",
    "commission": "0.00104710 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362069908,
    "fee_amount": 0.0020944,
    "fee_currency": "USDT",
    "commission": "0.00209440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362069924,
    "fee_amount": 0.0020954,
    "fee_currency": "USDT",
    "commission": "0.00209540 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362069926,
    "fee_amount": 0.0020964,
    "fee_currency": "USDT",
    "commission": "0.00209640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362069926,
    "fee_amount": 0.0020966,
    "fee_currency": "USDT",
    "commission": "0.00209660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362286634,
    "fee_amount": 0.0010443,
    "fee_currency": "USDT",
    "commission": "0.00104430 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362476811,
    "fee_amount": 0.0010454,
    "fee_currency": "USDT",
    "commission": "0.00104540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362476813,
    "fee_amount": 0.0010459,
    "fee_currency": "USDT",
    "commission": "0.00104590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774362476817,
    "fee_amount": 0.004196,
    "fee_currency": "USDT",
    "commission": "0.00419600 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774362476782,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362476817,
    "fee_amount": 0.0010464,
    "fee_currency": "USDT",
    "commission": "0.00104640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774362476819,
    "fee_amount": 0.0010469,
    "fee_currency": "USDT",
    "commission": "0.00104690 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362476801,
    "fee_amount": 0.0020934,
    "fee_currency": "USDT",
    "commission": "0.00209340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362476804,
    "fee_amount": 0.0015708,
    "fee_currency": "USDT",
    "commission": "0.00157080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362476804,
    "fee_amount": 0.0005236,
    "fee_currency": "USDT",
    "commission": "0.00052360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362765861,
    "fee_amount": 0.00152234,
    "fee_currency": "USDT",
    "commission": "0.00152234 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362765871,
    "fee_amount": 2.086e-05,
    "fee_currency": "USDT",
    "commission": "0.00002086 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362845674,
    "fee_amount": 0.00161038,
    "fee_currency": "USDT",
    "commission": "0.00161038 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774362845677,
    "fee_amount": 0.00048102,
    "fee_currency": "USDT",
    "commission": "0.00048102 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363058768,
    "fee_amount": 0.0016704,
    "fee_currency": "USDT",
    "commission": "0.00167040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363058790,
    "fee_amount": 0.00152512,
    "fee_currency": "USDT",
    "commission": "0.00152512 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363058790,
    "fee_amount": 0.00056408,
    "fee_currency": "USDT",
    "commission": "0.00056408 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363083476,
    "fee_amount": 0.0016688,
    "fee_currency": "USDT",
    "commission": "0.00166880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363083453,
    "fee_amount": 0.002087,
    "fee_currency": "USDT",
    "commission": "0.00208700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363124437,
    "fee_amount": 0.0020868,
    "fee_currency": "USDT",
    "commission": "0.00208680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363126537,
    "fee_amount": 0.0020868,
    "fee_currency": "USDT",
    "commission": "0.00208680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363126937,
    "fee_amount": 0.0020866,
    "fee_currency": "USDT",
    "commission": "0.00208660 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363127361,
    "fee_amount": 0.0016688,
    "fee_currency": "USDT",
    "commission": "0.00166880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363127337,
    "fee_amount": 0.0020866,
    "fee_currency": "USDT",
    "commission": "0.00208660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363199665,
    "fee_amount": 0.00148134,
    "fee_currency": "USDT",
    "commission": "0.00148134 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363199665,
    "fee_amount": 0.00060506,
    "fee_currency": "USDT",
    "commission": "0.00060506 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363543822,
    "fee_amount": 0.0010396,
    "fee_currency": "USDT",
    "commission": "0.00103960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363543822,
    "fee_amount": 0.0010391,
    "fee_currency": "USDT",
    "commission": "0.00103910 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363543822,
    "fee_amount": 0.001039,
    "fee_currency": "USDT",
    "commission": "0.00103900 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363543831,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363543841,
    "fee_amount": 0.00149846,
    "fee_currency": "USDT",
    "commission": "0.00149846 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363616987,
    "fee_amount": 0.0010447,
    "fee_currency": "USDT",
    "commission": "0.00104470 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363616993,
    "fee_amount": 0.004192,
    "fee_currency": "USDT",
    "commission": "0.00419200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363616987,
    "fee_amount": 0.0010452,
    "fee_currency": "USDT",
    "commission": "0.00104520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363616987,
    "fee_amount": 0.0010453,
    "fee_currency": "USDT",
    "commission": "0.00104530 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363616998,
    "fee_amount": 0.004196,
    "fee_currency": "USDT",
    "commission": "0.00419600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363616987,
    "fee_amount": 0.0010457,
    "fee_currency": "USDT",
    "commission": "0.00104570 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363617042,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363617024,
    "fee_amount": 0.0016736,
    "fee_currency": "USDT",
    "commission": "0.00167360 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363617028,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363617038,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617007,
    "fee_amount": 0.0020916,
    "fee_currency": "USDT",
    "commission": "0.00209160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617007,
    "fee_amount": 0.0020924,
    "fee_currency": "USDT",
    "commission": "0.00209240 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617007,
    "fee_amount": 0.0020926,
    "fee_currency": "USDT",
    "commission": "0.00209260 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617007,
    "fee_amount": 0.0020934,
    "fee_currency": "USDT",
    "commission": "0.00209340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617007,
    "fee_amount": 0.0020944,
    "fee_currency": "USDT",
    "commission": "0.00209440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363617515,
    "fee_amount": 0.0010496,
    "fee_currency": "USDT",
    "commission": "0.00104960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363617515,
    "fee_amount": 0.0010497,
    "fee_currency": "USDT",
    "commission": "0.00104970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363617521,
    "fee_amount": 0.004212,
    "fee_currency": "USDT",
    "commission": "0.00421200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363617521,
    "fee_amount": 0.004216,
    "fee_currency": "USDT",
    "commission": "0.00421600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363617515,
    "fee_amount": 0.0010501,
    "fee_currency": "USDT",
    "commission": "0.00105010 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363617521,
    "fee_amount": 0.00422,
    "fee_currency": "USDT",
    "commission": "0.00422000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363617553,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363617553,
    "fee_amount": 0.0016816,
    "fee_currency": "USDT",
    "commission": "0.00168160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363617556,
    "fee_amount": 0.0016832,
    "fee_currency": "USDT",
    "commission": "0.00168320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617535,
    "fee_amount": 0.0021008,
    "fee_currency": "USDT",
    "commission": "0.00210080 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617535,
    "fee_amount": 0.0021014,
    "fee_currency": "USDT",
    "commission": "0.00210140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617535,
    "fee_amount": 0.0021018,
    "fee_currency": "USDT",
    "commission": "0.00210180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363617535,
    "fee_amount": 0.0021026,
    "fee_currency": "USDT",
    "commission": "0.00210260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363617634,
    "fee_amount": 0.00422,
    "fee_currency": "USDT",
    "commission": "0.00422000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363617662,
    "fee_amount": 0.001052,
    "fee_currency": "USDT",
    "commission": "0.00105200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363617675,
    "fee_amount": 0.0010521,
    "fee_currency": "USDT",
    "commission": "0.00105210 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363617689,
    "fee_amount": 0.004224,
    "fee_currency": "USDT",
    "commission": "0.00422400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363617671,
    "fee_amount": 0.0016816,
    "fee_currency": "USDT",
    "commission": "0.00168160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363618036,
    "fee_amount": 0.0016832,
    "fee_currency": "USDT",
    "commission": "0.00168320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363618023,
    "fee_amount": 0.002106,
    "fee_currency": "USDT",
    "commission": "0.00210600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363618214,
    "fee_amount": 0.0010518,
    "fee_currency": "USDT",
    "commission": "0.00105180 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363618179,
    "fee_amount": 0.0016832,
    "fee_currency": "USDT",
    "commission": "0.00168320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363618200,
    "fee_amount": 0.002105,
    "fee_currency": "USDT",
    "commission": "0.00210500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363618208,
    "fee_amount": 0.0021048,
    "fee_currency": "USDT",
    "commission": "0.00210480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363618365,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363618329,
    "fee_amount": 0.0016816,
    "fee_currency": "USDT",
    "commission": "0.00168160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363618373,
    "fee_amount": 0.0010512,
    "fee_currency": "USDT",
    "commission": "0.00105120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363618352,
    "fee_amount": 0.002104,
    "fee_currency": "USDT",
    "commission": "0.00210400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774363618408,
    "fee_amount": 0.1184,
    "fee_currency": "PLUME",
    "commission": "0.11840000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363618463,
    "fee_amount": 0.0010508,
    "fee_currency": "USDT",
    "commission": "0.00105080 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774363618415,
    "fee_amount": 0.6816,
    "fee_currency": "PLUME",
    "commission": "0.68160000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363618689,
    "fee_amount": 0.0010508,
    "fee_currency": "USDT",
    "commission": "0.00105080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363618693,
    "fee_amount": 0.0010507,
    "fee_currency": "USDT",
    "commission": "0.00105070 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363618702,
    "fee_amount": 0.002103,
    "fee_currency": "USDT",
    "commission": "0.00210300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363618705,
    "fee_amount": 0.0021028,
    "fee_currency": "USDT",
    "commission": "0.00210280 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363620680,
    "fee_amount": 0.0016816,
    "fee_currency": "USDT",
    "commission": "0.00168160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363620703,
    "fee_amount": 0.002103,
    "fee_currency": "USDT",
    "commission": "0.00210300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363620715,
    "fee_amount": 0.002102,
    "fee_currency": "USDT",
    "commission": "0.00210200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363620717,
    "fee_amount": 0.0021018,
    "fee_currency": "USDT",
    "commission": "0.00210180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363620795,
    "fee_amount": 0.0010504,
    "fee_currency": "USDT",
    "commission": "0.00105040 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363620760,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363620798,
    "fee_amount": 0.00105,
    "fee_currency": "USDT",
    "commission": "0.00105000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363620800,
    "fee_amount": 0.0010499,
    "fee_currency": "USDT",
    "commission": "0.00104990 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363620783,
    "fee_amount": 0.0021006,
    "fee_currency": "USDT",
    "commission": "0.00210060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363621100,
    "fee_amount": 0.0010494,
    "fee_currency": "USDT",
    "commission": "0.00104940 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363621064,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363621021,
    "fee_amount": 0.0021006,
    "fee_currency": "USDT",
    "commission": "0.00210060 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363621088,
    "fee_amount": 0.0020996,
    "fee_currency": "USDT",
    "commission": "0.00209960 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363622319,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363622339,
    "fee_amount": 0.0020986,
    "fee_currency": "USDT",
    "commission": "0.00209860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363622441,
    "fee_amount": 0.0010483,
    "fee_currency": "USDT",
    "commission": "0.00104830 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363622441,
    "fee_amount": 0.001048,
    "fee_currency": "USDT",
    "commission": "0.00104800 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363622428,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774363622446,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363622469,
    "fee_amount": 0.0020974,
    "fee_currency": "USDT",
    "commission": "0.00209740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774363624076,
    "fee_amount": 0.0010473,
    "fee_currency": "USDT",
    "commission": "0.00104730 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363645449,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363651742,
    "fee_amount": 0.0016736,
    "fee_currency": "USDT",
    "commission": "0.00167360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363917937,
    "fee_amount": 4.202e-05,
    "fee_currency": "USDT",
    "commission": "0.00004202 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363918013,
    "fee_amount": 0.00205878,
    "fee_currency": "USDT",
    "commission": "0.00205878 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774363927242,
    "fee_amount": 6.312e-05,
    "fee_currency": "USDT",
    "commission": "0.00006312 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363945404,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774363954148,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774363954112,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774364128926,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774364128966,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774364237596,
    "fee_amount": 0.0010529,
    "fee_currency": "USDT",
    "commission": "0.00105290 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774364237683,
    "fee_amount": 0.0010534,
    "fee_currency": "USDT",
    "commission": "0.00105340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774364237671,
    "fee_amount": 0.0021076,
    "fee_currency": "USDT",
    "commission": "0.00210760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774364237674,
    "fee_amount": 0.0021088,
    "fee_currency": "USDT",
    "commission": "0.00210880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774364434420,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774364434388,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774364434385,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774364434411,
    "fee_amount": 0.0020976,
    "fee_currency": "USDT",
    "commission": "0.00209760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774364533045,
    "fee_amount": 0.0020982,
    "fee_currency": "USDT",
    "commission": "0.00209820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774365854553,
    "fee_amount": 0.0010529,
    "fee_currency": "USDT",
    "commission": "0.00105290 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774365854559,
    "fee_amount": 0.0010535,
    "fee_currency": "USDT",
    "commission": "0.00105350 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774365854524,
    "fee_amount": 0.0016864,
    "fee_currency": "USDT",
    "commission": "0.00168640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774365854561,
    "fee_amount": 0.0010539,
    "fee_currency": "USDT",
    "commission": "0.00105390 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774365854562,
    "fee_amount": 0.001054,
    "fee_currency": "USDT",
    "commission": "0.00105400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774365854531,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774365854547,
    "fee_amount": 0.0021076,
    "fee_currency": "USDT",
    "commission": "0.00210760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774365854553,
    "fee_amount": 0.0021088,
    "fee_currency": "USDT",
    "commission": "0.00210880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774365854553,
    "fee_amount": 0.0021096,
    "fee_currency": "USDT",
    "commission": "0.00210960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774366080265,
    "fee_amount": 0.0010542,
    "fee_currency": "USDT",
    "commission": "0.00105420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774366139660,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774366746340,
    "fee_amount": 0.0010518,
    "fee_currency": "USDT",
    "commission": "0.00105180 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774369659891,
    "fee_amount": 0.0016736,
    "fee_currency": "USDT",
    "commission": "0.00167360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774370359143,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774371937850,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774371937850,
    "fee_amount": 0.0010361,
    "fee_currency": "USDT",
    "commission": "0.00103610 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774371937817,
    "fee_amount": 0.0016576,
    "fee_currency": "USDT",
    "commission": "0.00165760 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774371937817,
    "fee_amount": 0.001656,
    "fee_currency": "USDT",
    "commission": "0.00165600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774371937850,
    "fee_amount": 0.0010355,
    "fee_currency": "USDT",
    "commission": "0.00103550 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774371937850,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774371937828,
    "fee_amount": 0.0016544,
    "fee_currency": "USDT",
    "commission": "0.00165440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774371937851,
    "fee_amount": 0.001035,
    "fee_currency": "USDT",
    "commission": "0.00103500 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774371937851,
    "fee_amount": 0.0010345,
    "fee_currency": "USDT",
    "commission": "0.00103450 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774371937856,
    "fee_amount": 0.001034,
    "fee_currency": "USDT",
    "commission": "0.00103400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774371937838,
    "fee_amount": 0.002074,
    "fee_currency": "USDT",
    "commission": "0.00207400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774371937838,
    "fee_amount": 0.002073,
    "fee_currency": "USDT",
    "commission": "0.00207300 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774371937838,
    "fee_amount": 0.0020718,
    "fee_currency": "USDT",
    "commission": "0.00207180 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774371937838,
    "fee_amount": 0.0020708,
    "fee_currency": "USDT",
    "commission": "0.00207080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774371937850,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774371937853,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774371937865,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774371937838,
    "fee_amount": 0.0020698,
    "fee_currency": "USDT",
    "commission": "0.00206980 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774371937815,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774371937815,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774371937815,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774371937815,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774371937960,
    "fee_amount": 0.0016528,
    "fee_currency": "USDT",
    "commission": "0.00165280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774371939555,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774371959474,
    "fee_amount": 0.0010336,
    "fee_currency": "USDT",
    "commission": "0.00103360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774371977019,
    "fee_amount": 0.0010336,
    "fee_currency": "USDT",
    "commission": "0.00103360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774371977004,
    "fee_amount": 0.0020688,
    "fee_currency": "USDT",
    "commission": "0.00206880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774371983193,
    "fee_amount": 0.002069,
    "fee_currency": "USDT",
    "commission": "0.00206900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774372224045,
    "fee_amount": 0.0020682,
    "fee_currency": "USDT",
    "commission": "0.00206820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774372291744,
    "fee_amount": 0.0010358,
    "fee_currency": "USDT",
    "commission": "0.00103580 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774372291953,
    "fee_amount": 0.0016592,
    "fee_currency": "USDT",
    "commission": "0.00165920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774372291975,
    "fee_amount": 0.0020732,
    "fee_currency": "USDT",
    "commission": "0.00207320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774372497505,
    "fee_amount": 0.0010358,
    "fee_currency": "USDT",
    "commission": "0.00103580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774372987402,
    "fee_amount": 0.0010359,
    "fee_currency": "USDT",
    "commission": "0.00103590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774373855138,
    "fee_amount": 0.00103387,
    "fee_currency": "USDT",
    "commission": "0.00103387 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774373855141,
    "fee_amount": 1.04e-06,
    "fee_currency": "USDT",
    "commission": "0.00000104 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774373855173,
    "fee_amount": 2.072e-05,
    "fee_currency": "USDT",
    "commission": "0.00002072 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774374745646,
    "fee_amount": 0.0016544,
    "fee_currency": "USDT",
    "commission": "0.00165440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774374745669,
    "fee_amount": 0.0020686,
    "fee_currency": "USDT",
    "commission": "0.00206860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774374745819,
    "fee_amount": 0.00103247,
    "fee_currency": "USDT",
    "commission": "0.00103247 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774374745850,
    "fee_amount": 1.04e-06,
    "fee_currency": "USDT",
    "commission": "0.00000104 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774374929399,
    "fee_amount": 0.001032,
    "fee_currency": "USDT",
    "commission": "0.00103200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774374929376,
    "fee_amount": 0.0020658,
    "fee_currency": "USDT",
    "commission": "0.00206580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774374929523,
    "fee_amount": 0.002066,
    "fee_currency": "USDT",
    "commission": "0.00206600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774374937519,
    "fee_amount": 0.0010326,
    "fee_currency": "USDT",
    "commission": "0.00103260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774375229005,
    "fee_amount": 0.00011704,
    "fee_currency": "USDT",
    "commission": "0.00011704 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774377579358,
    "fee_amount": 0.0010412,
    "fee_currency": "USDT",
    "commission": "0.00104120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774378516302,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774378516303,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774378516294,
    "fee_amount": 0.0020658,
    "fee_currency": "USDT",
    "commission": "0.00206580 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774378516272,
    "fee_amount": 0.0016528,
    "fee_currency": "USDT",
    "commission": "0.00165280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774378516306,
    "fee_amount": 0.001033,
    "fee_currency": "USDT",
    "commission": "0.00103300 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774378516268,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774378516268,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774378516312,
    "fee_amount": 0.0010325,
    "fee_currency": "USDT",
    "commission": "0.00103250 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774378689445,
    "fee_amount": 0.001031,
    "fee_currency": "USDT",
    "commission": "0.00103100 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774378689430,
    "fee_amount": 0.0020638,
    "fee_currency": "USDT",
    "commission": "0.00206380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774378689447,
    "fee_amount": 0.0010315,
    "fee_currency": "USDT",
    "commission": "0.00103150 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774380708361,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774380708362,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774380708328,
    "fee_amount": 0.0016448,
    "fee_currency": "USDT",
    "commission": "0.00164480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774380708362,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774380708332,
    "fee_amount": 0.0016432,
    "fee_currency": "USDT",
    "commission": "0.00164320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774380708365,
    "fee_amount": 0.0010279,
    "fee_currency": "USDT",
    "commission": "0.00102790 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774380708350,
    "fee_amount": 0.0020586,
    "fee_currency": "USDT",
    "commission": "0.00205860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774380708370,
    "fee_amount": 0.0010274,
    "fee_currency": "USDT",
    "commission": "0.00102740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774380708350,
    "fee_amount": 0.0020576,
    "fee_currency": "USDT",
    "commission": "0.00205760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774380708370,
    "fee_amount": 0.0010269,
    "fee_currency": "USDT",
    "commission": "0.00102690 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774380708350,
    "fee_amount": 0.0020566,
    "fee_currency": "USDT",
    "commission": "0.00205660 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774380708327,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774380708327,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774380708448,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774380708472,
    "fee_amount": 0.0016416,
    "fee_currency": "USDT",
    "commission": "0.00164160 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774380721724,
    "fee_amount": 0.0010277,
    "fee_currency": "USDT",
    "commission": "0.00102770 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774380835222,
    "fee_amount": 0.0010267,
    "fee_currency": "USDT",
    "commission": "0.00102670 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774380932518,
    "fee_amount": 0.0010277,
    "fee_currency": "USDT",
    "commission": "0.00102770 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774381534106,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774381534141,
    "fee_amount": 0.0010256,
    "fee_currency": "USDT",
    "commission": "0.00102560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774381534143,
    "fee_amount": 0.0876,
    "fee_currency": "USDT",
    "commission": "0.08760000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774381534147,
    "fee_amount": 0.3124,
    "fee_currency": "USDT",
    "commission": "0.31240000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774381534152,
    "fee_amount": 0.00025628,
    "fee_currency": "USDT",
    "commission": "0.00025628 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774381534159,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774381534152,
    "fee_amount": 0.00076883,
    "fee_currency": "USDT",
    "commission": "0.00076883 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774381534156,
    "fee_amount": 0.0010246,
    "fee_currency": "USDT",
    "commission": "0.00102460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774381534156,
    "fee_amount": 0.001024,
    "fee_currency": "USDT",
    "commission": "0.00102400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774381534156,
    "fee_amount": 0.0010235,
    "fee_currency": "USDT",
    "commission": "0.00102350 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774381534174,
    "fee_amount": 0.0020528,
    "fee_currency": "USDT",
    "commission": "0.00205280 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774381534145,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774381534204,
    "fee_amount": 0.00049243,
    "fee_currency": "USDT",
    "commission": "0.00049243 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774381534191,
    "fee_amount": 0.0016416,
    "fee_currency": "USDT",
    "commission": "0.00164160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_spot_makerv3-000",
    "venue": "BINANCE_SPOT",
    "instrument_id": "PLUMEUSDT.BINANCE_SPOT",
    "event_ts_ms": 1774381534247,
    "fee_amount": 0.8,
    "fee_currency": "PLUME",
    "commission": "0.80000000 PLUME"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774381583470,
    "fee_amount": 0.0010264,
    "fee_currency": "USDT",
    "commission": "0.00102640 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383302137,
    "fee_amount": 0.0020566,
    "fee_currency": "USDT",
    "commission": "0.00205660 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383610455,
    "fee_amount": 0.0020576,
    "fee_currency": "USDT",
    "commission": "0.00205760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383614288,
    "fee_amount": 0.0020586,
    "fee_currency": "USDT",
    "commission": "0.00205860 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383614290,
    "fee_amount": 0.0020588,
    "fee_currency": "USDT",
    "commission": "0.00205880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383614939,
    "fee_amount": 0.00102798,
    "fee_currency": "USDT",
    "commission": "0.00102798 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383614940,
    "fee_amount": 1.03e-06,
    "fee_currency": "USDT",
    "commission": "0.00000103 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383615096,
    "fee_amount": 0.00206,
    "fee_currency": "USDT",
    "commission": "0.00206000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383617099,
    "fee_amount": 0.0020608,
    "fee_currency": "USDT",
    "commission": "0.00206080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383617120,
    "fee_amount": 0.0010295,
    "fee_currency": "USDT",
    "commission": "0.00102950 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383617136,
    "fee_amount": 0.0010296,
    "fee_currency": "USDT",
    "commission": "0.00102960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383617126,
    "fee_amount": 0.0020612,
    "fee_currency": "USDT",
    "commission": "0.00206120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383617159,
    "fee_amount": 0.002062,
    "fee_currency": "USDT",
    "commission": "0.00206200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774383617140,
    "fee_amount": 0.001648,
    "fee_currency": "USDT",
    "commission": "0.00164800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383617179,
    "fee_amount": 0.0010301,
    "fee_currency": "USDT",
    "commission": "0.00103010 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383617300,
    "fee_amount": 0.0010307,
    "fee_currency": "USDT",
    "commission": "0.00103070 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383617318,
    "fee_amount": 0.0020638,
    "fee_currency": "USDT",
    "commission": "0.00206380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383617332,
    "fee_amount": 0.0010308,
    "fee_currency": "USDT",
    "commission": "0.00103080 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774383617323,
    "fee_amount": 0.0016496,
    "fee_currency": "USDT",
    "commission": "0.00164960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383729387,
    "fee_amount": 0.0010345,
    "fee_currency": "USDT",
    "commission": "0.00103450 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774383729352,
    "fee_amount": 0.001656,
    "fee_currency": "USDT",
    "commission": "0.00165600 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383729396,
    "fee_amount": 0.0010349,
    "fee_currency": "USDT",
    "commission": "0.00103490 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383729382,
    "fee_amount": 0.002072,
    "fee_currency": "USDT",
    "commission": "0.00207200 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383729382,
    "fee_amount": 0.0020722,
    "fee_currency": "USDT",
    "commission": "0.00207220 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774383729364,
    "fee_amount": 0.0016576,
    "fee_currency": "USDT",
    "commission": "0.00165760 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383890077,
    "fee_amount": 0.0010391,
    "fee_currency": "USDT",
    "commission": "0.00103910 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383890077,
    "fee_amount": 0.0010392,
    "fee_currency": "USDT",
    "commission": "0.00103920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383890077,
    "fee_amount": 0.0010396,
    "fee_currency": "USDT",
    "commission": "0.00103960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383890077,
    "fee_amount": 0.0010397,
    "fee_currency": "USDT",
    "commission": "0.00103970 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383890077,
    "fee_amount": 0.0010402,
    "fee_currency": "USDT",
    "commission": "0.00104020 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383890096,
    "fee_amount": 0.0020796,
    "fee_currency": "USDT",
    "commission": "0.00207960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383890097,
    "fee_amount": 0.0020804,
    "fee_currency": "USDT",
    "commission": "0.00208040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383890098,
    "fee_amount": 0.00056176,
    "fee_currency": "USDT",
    "commission": "0.00056176 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383890100,
    "fee_amount": 0.00056176,
    "fee_currency": "USDT",
    "commission": "0.00056176 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383890102,
    "fee_amount": 0.00056177,
    "fee_currency": "USDT",
    "commission": "0.00056177 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774383890112,
    "fee_amount": 0.0016624,
    "fee_currency": "USDT",
    "commission": "0.00166240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383899674,
    "fee_amount": 0.0010409,
    "fee_currency": "USDT",
    "commission": "0.00104090 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383975299,
    "fee_amount": 0.0010404,
    "fee_currency": "USDT",
    "commission": "0.00104040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383975313,
    "fee_amount": 0.0020832,
    "fee_currency": "USDT",
    "commission": "0.00208320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383975330,
    "fee_amount": 0.0010409,
    "fee_currency": "USDT",
    "commission": "0.00104090 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383975335,
    "fee_amount": 0.00093944,
    "fee_currency": "USDT",
    "commission": "0.00093944 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774383975318,
    "fee_amount": 0.0020842,
    "fee_currency": "USDT",
    "commission": "0.00208420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383975335,
    "fee_amount": 0.00010207,
    "fee_currency": "USDT",
    "commission": "0.00010207 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774383975368,
    "fee_amount": 0.001042,
    "fee_currency": "USDT",
    "commission": "0.00104200 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774383975368,
    "fee_amount": 0.0016656,
    "fee_currency": "USDT",
    "commission": "0.00166560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774384010687,
    "fee_amount": 0.0010442,
    "fee_currency": "USDT",
    "commission": "0.00104420 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774384010693,
    "fee_amount": 0.0010447,
    "fee_currency": "USDT",
    "commission": "0.00104470 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774384016580,
    "fee_amount": 0.0010452,
    "fee_currency": "USDT",
    "commission": "0.00104520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774384772263,
    "fee_amount": 0.0020774,
    "fee_currency": "USDT",
    "commission": "0.00207740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774384772355,
    "fee_amount": 0.0010375,
    "fee_currency": "USDT",
    "commission": "0.00103750 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774384772362,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774384772430,
    "fee_amount": 0.0020764,
    "fee_currency": "USDT",
    "commission": "0.00207640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386145172,
    "fee_amount": 0.0010349,
    "fee_currency": "USDT",
    "commission": "0.00103490 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774386150732,
    "fee_amount": 0.001664,
    "fee_currency": "USDT",
    "commission": "0.00166400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386150772,
    "fee_amount": 0.0010396,
    "fee_currency": "USDT",
    "commission": "0.00103960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774386150756,
    "fee_amount": 0.00126904,
    "fee_currency": "USDT",
    "commission": "0.00126904 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774386150756,
    "fee_amount": 0.00062412,
    "fee_currency": "USDT",
    "commission": "0.00062412 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774386150758,
    "fee_amount": 0.00018724,
    "fee_currency": "USDT",
    "commission": "0.00018724 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774386150759,
    "fee_amount": 0.0020814,
    "fee_currency": "USDT",
    "commission": "0.00208140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386154591,
    "fee_amount": 0.0010375,
    "fee_currency": "USDT",
    "commission": "0.00103750 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774386154600,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386154884,
    "fee_amount": 0.001037,
    "fee_currency": "USDT",
    "commission": "0.00103700 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386227780,
    "fee_amount": 0.0010406,
    "fee_currency": "USDT",
    "commission": "0.00104060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386228317,
    "fee_amount": 0.0010406,
    "fee_currency": "USDT",
    "commission": "0.00104060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386228317,
    "fee_amount": 0.0010407,
    "fee_currency": "USDT",
    "commission": "0.00104070 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386228317,
    "fee_amount": 0.0010411,
    "fee_currency": "USDT",
    "commission": "0.00104110 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774386228520,
    "fee_amount": 0.0010407,
    "fee_currency": "USDT",
    "commission": "0.00104070 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774386488424,
    "fee_amount": 0.001656,
    "fee_currency": "USDT",
    "commission": "0.00165600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389700231,
    "fee_amount": 0.0020718,
    "fee_currency": "USDT",
    "commission": "0.00207180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774389700252,
    "fee_amount": 0.0010353,
    "fee_currency": "USDT",
    "commission": "0.00103530 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774389700227,
    "fee_amount": 0.0016576,
    "fee_currency": "USDT",
    "commission": "0.00165760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389700248,
    "fee_amount": 0.0020728,
    "fee_currency": "USDT",
    "commission": "0.00207280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774389700269,
    "fee_amount": 0.0010358,
    "fee_currency": "USDT",
    "commission": "0.00103580 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389700251,
    "fee_amount": 0.002074,
    "fee_currency": "USDT",
    "commission": "0.00207400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389700433,
    "fee_amount": 0.000415,
    "fee_currency": "USDT",
    "commission": "0.00041500 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389700437,
    "fee_amount": 0.00083,
    "fee_currency": "USDT",
    "commission": "0.00083000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389700439,
    "fee_amount": 0.00083,
    "fee_currency": "USDT",
    "commission": "0.00083000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774389700526,
    "fee_amount": 0.0016608,
    "fee_currency": "USDT",
    "commission": "0.00166080 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774389726009,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774389730178,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774389833519,
    "fee_amount": 0.0009996,
    "fee_currency": "USDT",
    "commission": "0.00099960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833503,
    "fee_amount": 0.0020784,
    "fee_currency": "USDT",
    "commission": "0.00207840 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774389833519,
    "fee_amount": 3.841e-05,
    "fee_currency": "USDT",
    "commission": "0.00003841 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774389833520,
    "fee_amount": 0.004164,
    "fee_currency": "USDT",
    "commission": "0.00416400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774389833489,
    "fee_amount": 0.0016624,
    "fee_currency": "USDT",
    "commission": "0.00166240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774389833530,
    "fee_amount": 0.0010385,
    "fee_currency": "USDT",
    "commission": "0.00103850 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833562,
    "fee_amount": 0.00039493,
    "fee_currency": "USDT",
    "commission": "0.00039493 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833564,
    "fee_amount": 0.00039494,
    "fee_currency": "USDT",
    "commission": "0.00039494 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833566,
    "fee_amount": 0.00039493,
    "fee_currency": "USDT",
    "commission": "0.00039493 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833568,
    "fee_amount": 0.00039494,
    "fee_currency": "USDT",
    "commission": "0.00039494 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833569,
    "fee_amount": 0.00039493,
    "fee_currency": "USDT",
    "commission": "0.00039493 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833571,
    "fee_amount": 0.00010393,
    "fee_currency": "USDT",
    "commission": "0.00010393 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774389833595,
    "fee_amount": 0.001039,
    "fee_currency": "USDT",
    "commission": "0.00103900 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833579,
    "fee_amount": 0.0020794,
    "fee_currency": "USDT",
    "commission": "0.00207940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774389833579,
    "fee_amount": 0.0020796,
    "fee_currency": "USDT",
    "commission": "0.00207960 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774389833561,
    "fee_amount": 0.001664,
    "fee_currency": "USDT",
    "commission": "0.00166400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774390291099,
    "fee_amount": 0.3542,
    "fee_currency": "USDT",
    "commission": "0.35420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774390291101,
    "fee_amount": 0.0458,
    "fee_currency": "USDT",
    "commission": "0.04580000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774390291065,
    "fee_amount": 0.0016672,
    "fee_currency": "USDT",
    "commission": "0.00166720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774390291087,
    "fee_amount": 0.0020868,
    "fee_currency": "USDT",
    "commission": "0.00208680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774390291087,
    "fee_amount": 0.0020858,
    "fee_currency": "USDT",
    "commission": "0.00208580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774390291107,
    "fee_amount": 0.001042,
    "fee_currency": "USDT",
    "commission": "0.00104200 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774390293177,
    "fee_amount": 0.0010441,
    "fee_currency": "USDT",
    "commission": "0.00104410 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774391235737,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774391235776,
    "fee_amount": 0.0010419,
    "fee_currency": "USDT",
    "commission": "0.00104190 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774391235777,
    "fee_amount": 0.0010414,
    "fee_currency": "USDT",
    "commission": "0.00104140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774391235760,
    "fee_amount": 0.0020844,
    "fee_currency": "USDT",
    "commission": "0.00208440 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774391235810,
    "fee_amount": 0.0010409,
    "fee_currency": "USDT",
    "commission": "0.00104090 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774391235772,
    "fee_amount": 0.0016672,
    "fee_currency": "USDT",
    "commission": "0.00166720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774391235798,
    "fee_amount": 0.0020834,
    "fee_currency": "USDT",
    "commission": "0.00208340 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774391235774,
    "fee_amount": 0.0016656,
    "fee_currency": "USDT",
    "commission": "0.00166560 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774391235776,
    "fee_amount": 0.001664,
    "fee_currency": "USDT",
    "commission": "0.00166400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774391414089,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774394498079,
    "fee_amount": 0.0010479,
    "fee_currency": "USDT",
    "commission": "0.00104790 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774394498106,
    "fee_amount": 0.0020982,
    "fee_currency": "USDT",
    "commission": "0.00209820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774394498287,
    "fee_amount": 0.0010484,
    "fee_currency": "USDT",
    "commission": "0.00104840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774395186834,
    "fee_amount": 0.002091,
    "fee_currency": "USDT",
    "commission": "0.00209100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774397914653,
    "fee_amount": 0.0010469,
    "fee_currency": "USDT",
    "commission": "0.00104690 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774397914656,
    "fee_amount": 0.0010475,
    "fee_currency": "USDT",
    "commission": "0.00104750 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774397914656,
    "fee_amount": 0.0042,
    "fee_currency": "USDT",
    "commission": "0.00420000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397914623,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397914640,
    "fee_amount": 0.0020964,
    "fee_currency": "USDT",
    "commission": "0.00209640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774397914660,
    "fee_amount": 0.004204,
    "fee_currency": "USDT",
    "commission": "0.00420400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397914642,
    "fee_amount": 0.0020974,
    "fee_currency": "USDT",
    "commission": "0.00209740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397914642,
    "fee_amount": 0.0020984,
    "fee_currency": "USDT",
    "commission": "0.00209840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397914642,
    "fee_amount": 0.0020994,
    "fee_currency": "USDT",
    "commission": "0.00209940 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397914646,
    "fee_amount": 0.0021006,
    "fee_currency": "USDT",
    "commission": "0.00210060 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774397914662,
    "fee_amount": 0.004208,
    "fee_currency": "USDT",
    "commission": "0.00420800 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397914623,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774397914656,
    "fee_amount": 0.001048,
    "fee_currency": "USDT",
    "commission": "0.00104800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774397914656,
    "fee_amount": 0.0010485,
    "fee_currency": "USDT",
    "commission": "0.00104850 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774397914656,
    "fee_amount": 0.001049,
    "fee_currency": "USDT",
    "commission": "0.00104900 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397914628,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397914632,
    "fee_amount": 0.0016816,
    "fee_currency": "USDT",
    "commission": "0.00168160 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397914640,
    "fee_amount": 0.0016832,
    "fee_currency": "USDT",
    "commission": "0.00168320 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397914691,
    "fee_amount": 0.0016864,
    "fee_currency": "USDT",
    "commission": "0.00168640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774397914800,
    "fee_amount": 0.004228,
    "fee_currency": "USDT",
    "commission": "0.00422800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397915608,
    "fee_amount": 0.0021092,
    "fee_currency": "USDT",
    "commission": "0.00210920 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397915633,
    "fee_amount": 0.0016848,
    "fee_currency": "USDT",
    "commission": "0.00168480 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774397915672,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397915663,
    "fee_amount": 0.00050597,
    "fee_currency": "USDT",
    "commission": "0.00050597 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397915665,
    "fee_amount": 0.00050597,
    "fee_currency": "USDT",
    "commission": "0.00050597 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774397915677,
    "fee_amount": 0.0010532,
    "fee_currency": "USDT",
    "commission": "0.00105320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397915665,
    "fee_amount": 0.00109626,
    "fee_currency": "USDT",
    "commission": "0.00109626 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397915681,
    "fee_amount": 0.00103253,
    "fee_currency": "USDT",
    "commission": "0.00103253 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774397915660,
    "fee_amount": 0.0016832,
    "fee_currency": "USDT",
    "commission": "0.00168320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774397915714,
    "fee_amount": 0.00107467,
    "fee_currency": "USDT",
    "commission": "0.00107467 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774398001047,
    "fee_amount": 0.0021036,
    "fee_currency": "USDT",
    "commission": "0.00210360 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774398025931,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774398025953,
    "fee_amount": 0.0021026,
    "fee_currency": "USDT",
    "commission": "0.00210260 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774398026111,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774398026438,
    "fee_amount": 0.0021014,
    "fee_currency": "USDT",
    "commission": "0.00210140 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774398150134,
    "fee_amount": 0.00068604,
    "fee_currency": "USDT",
    "commission": "0.00068604 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774398150139,
    "fee_amount": 0.00036617,
    "fee_currency": "USDT",
    "commission": "0.00036617 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774398290334,
    "fee_amount": 0.0021002,
    "fee_currency": "USDT",
    "commission": "0.00210020 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774398290313,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774398622832,
    "fee_amount": 0.0021,
    "fee_currency": "USDT",
    "commission": "0.00210000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774398622788,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774398622851,
    "fee_amount": 0.0010489,
    "fee_currency": "USDT",
    "commission": "0.00104890 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774398625793,
    "fee_amount": 0.0016768,
    "fee_currency": "USDT",
    "commission": "0.00167680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774398626103,
    "fee_amount": 0.0020968,
    "fee_currency": "USDT",
    "commission": "0.00209680 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774398626125,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774398626134,
    "fee_amount": 0.0010474,
    "fee_currency": "USDT",
    "commission": "0.00104740 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774398626152,
    "fee_amount": 0.0016752,
    "fee_currency": "USDT",
    "commission": "0.00167520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774398626411,
    "fee_amount": 0.0020958,
    "fee_currency": "USDT",
    "commission": "0.00209580 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774398626513,
    "fee_amount": 0.0016736,
    "fee_currency": "USDT",
    "commission": "0.00167360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774399550276,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774399550241,
    "fee_amount": 0.0016912,
    "fee_currency": "USDT",
    "commission": "0.00169120 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774399550262,
    "fee_amount": 0.0021174,
    "fee_currency": "USDT",
    "commission": "0.00211740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774399855839,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774399855915,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774399855880,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774399855903,
    "fee_amount": 0.0021032,
    "fee_currency": "USDT",
    "commission": "0.00210320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774399855915,
    "fee_amount": 0.0010505,
    "fee_currency": "USDT",
    "commission": "0.00105050 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774400459360,
    "fee_amount": 0.0020992,
    "fee_currency": "USDT",
    "commission": "0.00209920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402815369,
    "fee_amount": 0.0010647,
    "fee_currency": "USDT",
    "commission": "0.00106470 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402819903,
    "fee_amount": 0.0010652,
    "fee_currency": "USDT",
    "commission": "0.00106520 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774402819893,
    "fee_amount": 0.0021324,
    "fee_currency": "USDT",
    "commission": "0.00213240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402823947,
    "fee_amount": 0.0010662,
    "fee_currency": "USDT",
    "commission": "0.00106620 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402841124,
    "fee_amount": 0.00059882,
    "fee_currency": "USDT",
    "commission": "0.00059882 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402841124,
    "fee_amount": 0.00046859,
    "fee_currency": "USDT",
    "commission": "0.00046859 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774402841121,
    "fee_amount": 0.0021368,
    "fee_currency": "USDT",
    "commission": "0.00213680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774402841121,
    "fee_amount": 0.00025654,
    "fee_currency": "USDT",
    "commission": "0.00025654 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402861623,
    "fee_amount": 0.0010679,
    "fee_currency": "USDT",
    "commission": "0.00106790 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402861639,
    "fee_amount": 0.0010684,
    "fee_currency": "USDT",
    "commission": "0.00106840 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774402861648,
    "fee_amount": 0.0017088,
    "fee_currency": "USDT",
    "commission": "0.00170880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774402861677,
    "fee_amount": 0.002138,
    "fee_currency": "USDT",
    "commission": "0.00213800 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774402865390,
    "fee_amount": 0.20648,
    "fee_currency": "USDT",
    "commission": "0.20648000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774402865391,
    "fee_amount": 0.19352,
    "fee_currency": "USDT",
    "commission": "0.19352000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402865419,
    "fee_amount": 0.0010657,
    "fee_currency": "USDT",
    "commission": "0.00106570 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774402865455,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774402865438,
    "fee_amount": 0.0021336,
    "fee_currency": "USDT",
    "commission": "0.00213360 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774402865544,
    "fee_amount": 0.001704,
    "fee_currency": "USDT",
    "commission": "0.00170400 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402932438,
    "fee_amount": 0.0010659,
    "fee_currency": "USDT",
    "commission": "0.00106590 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774402943763,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774402943769,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774402943734,
    "fee_amount": 0.0017024,
    "fee_currency": "USDT",
    "commission": "0.00170240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774402943781,
    "fee_amount": 0.0010646,
    "fee_currency": "USDT",
    "commission": "0.00106460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774402943758,
    "fee_amount": 0.00038365,
    "fee_currency": "USDT",
    "commission": "0.00038365 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774402943761,
    "fee_amount": 0.00174775,
    "fee_currency": "USDT",
    "commission": "0.00174775 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774404001768,
    "fee_amount": 0.0010588,
    "fee_currency": "USDT",
    "commission": "0.00105880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774404575091,
    "fee_amount": 2.138e-05,
    "fee_currency": "USDT",
    "commission": "0.00002138 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774404584603,
    "fee_amount": 0.0010683,
    "fee_currency": "USDT",
    "commission": "0.00106830 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774404584603,
    "fee_amount": 0.0010688,
    "fee_currency": "USDT",
    "commission": "0.00106880 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774404614458,
    "fee_amount": 0.0010709,
    "fee_currency": "USDT",
    "commission": "0.00107090 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774404614424,
    "fee_amount": 0.0017136,
    "fee_currency": "USDT",
    "commission": "0.00171360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774404614446,
    "fee_amount": 0.0021438,
    "fee_currency": "USDT",
    "commission": "0.00214380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774404769417,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774404769420,
    "fee_amount": 0.0010657,
    "fee_currency": "USDT",
    "commission": "0.00106570 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774404769419,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774404769387,
    "fee_amount": 0.001704,
    "fee_currency": "USDT",
    "commission": "0.00170400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774404769405,
    "fee_amount": 0.0021334,
    "fee_currency": "USDT",
    "commission": "0.00213340 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774404769428,
    "fee_amount": 0.0010651,
    "fee_currency": "USDT",
    "commission": "0.00106510 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774404769407,
    "fee_amount": 0.0021322,
    "fee_currency": "USDT",
    "commission": "0.00213220 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774404769429,
    "fee_amount": 0.0010646,
    "fee_currency": "USDT",
    "commission": "0.00106460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774404769411,
    "fee_amount": 0.0021312,
    "fee_currency": "USDT",
    "commission": "0.00213120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774404798295,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774405559917,
    "fee_amount": 0.25484,
    "fee_currency": "USDT",
    "commission": "0.25484000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774405559917,
    "fee_amount": 0.14516,
    "fee_currency": "USDT",
    "commission": "0.14516000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774405560067,
    "fee_amount": 0.0010574,
    "fee_currency": "USDT",
    "commission": "0.00105740 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774406697674,
    "fee_amount": 0.0010546,
    "fee_currency": "USDT",
    "commission": "0.00105460 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774407900570,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774408424855,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774408492167,
    "fee_amount": 0.0010486,
    "fee_currency": "USDT",
    "commission": "0.00104860 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774408492132,
    "fee_amount": 0.0016784,
    "fee_currency": "USDT",
    "commission": "0.00167840 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774408492174,
    "fee_amount": 0.0010491,
    "fee_currency": "USDT",
    "commission": "0.00104910 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774408492202,
    "fee_amount": 0.0010496,
    "fee_currency": "USDT",
    "commission": "0.00104960 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774408492167,
    "fee_amount": 0.00168,
    "fee_currency": "USDT",
    "commission": "0.00168000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774408494649,
    "fee_amount": 0.0020968,
    "fee_currency": "USDT",
    "commission": "0.00209680 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774408494649,
    "fee_amount": 0.0020958,
    "fee_currency": "USDT",
    "commission": "0.00209580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774409532496,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774409532588,
    "fee_amount": 0.0021066,
    "fee_currency": "USDT",
    "commission": "0.00210660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774409532647,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774409532612,
    "fee_amount": 0.0016832,
    "fee_currency": "USDT",
    "commission": "0.00168320 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774409532748,
    "fee_amount": 0.0005474,
    "fee_currency": "USDT",
    "commission": "0.00054740 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774409532750,
    "fee_amount": 0.00054741,
    "fee_currency": "USDT",
    "commission": "0.00054741 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774410407469,
    "fee_amount": 0.0010571,
    "fee_currency": "USDT",
    "commission": "0.00105710 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774410407460,
    "fee_amount": 0.0021158,
    "fee_currency": "USDT",
    "commission": "0.00211580 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774410411285,
    "fee_amount": 0.0010586,
    "fee_currency": "USDT",
    "commission": "0.00105860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774410420234,
    "fee_amount": 0.0010602,
    "fee_currency": "USDT",
    "commission": "0.00106020 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774410420261,
    "fee_amount": 0.0010607,
    "fee_currency": "USDT",
    "commission": "0.00106070 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774410420412,
    "fee_amount": 0.0016976,
    "fee_currency": "USDT",
    "commission": "0.00169760 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774410424128,
    "fee_amount": 0.00059287,
    "fee_currency": "USDT",
    "commission": "0.00059287 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774410426772,
    "fee_amount": 0.00152453,
    "fee_currency": "USDT",
    "commission": "0.00152453 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774410489383,
    "fee_amount": 0.00059315,
    "fee_currency": "USDT",
    "commission": "0.00059315 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774410489383,
    "fee_amount": 0.00152525,
    "fee_currency": "USDT",
    "commission": "0.00152525 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411021313,
    "fee_amount": 0.00180115,
    "fee_currency": "USDT",
    "commission": "0.00180115 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411021413,
    "fee_amount": 0.00031785,
    "fee_currency": "USDT",
    "commission": "0.00031785 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411026523,
    "fee_amount": 0.0021182,
    "fee_currency": "USDT",
    "commission": "0.00211820 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_spot_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-SPOT.BYBIT",
    "event_ts_ms": 1774411074217,
    "fee_amount": 0.4,
    "fee_currency": "USDT",
    "commission": "0.40000000 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411074217,
    "fee_amount": 0.0010584,
    "fee_currency": "USDT",
    "commission": "0.00105840 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411074217,
    "fee_amount": 0.0010579,
    "fee_currency": "USDT",
    "commission": "0.00105790 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774411074184,
    "fee_amount": 0.0016912,
    "fee_currency": "USDT",
    "commission": "0.00169120 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411074218,
    "fee_amount": 0.0010573,
    "fee_currency": "USDT",
    "commission": "0.00105730 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774411074184,
    "fee_amount": 0.0016896,
    "fee_currency": "USDT",
    "commission": "0.00168960 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411074218,
    "fee_amount": 0.0010568,
    "fee_currency": "USDT",
    "commission": "0.00105680 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774411074188,
    "fee_amount": 0.001688,
    "fee_currency": "USDT",
    "commission": "0.00168800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411074204,
    "fee_amount": 0.002118,
    "fee_currency": "USDT",
    "commission": "0.00211800 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411074204,
    "fee_amount": 0.002117,
    "fee_currency": "USDT",
    "commission": "0.00211700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411074204,
    "fee_amount": 0.002116,
    "fee_currency": "USDT",
    "commission": "0.00211600 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411074204,
    "fee_amount": 0.0021148,
    "fee_currency": "USDT",
    "commission": "0.00211480 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411074206,
    "fee_amount": 0.0021138,
    "fee_currency": "USDT",
    "commission": "0.00211380 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411074219,
    "fee_amount": 0.0010563,
    "fee_currency": "USDT",
    "commission": "0.00105630 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774411074190,
    "fee_amount": 0.0016864,
    "fee_currency": "USDT",
    "commission": "0.00168640 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411523786,
    "fee_amount": 0.0010522,
    "fee_currency": "USDT",
    "commission": "0.00105220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774411523806,
    "fee_amount": 0.0021054,
    "fee_currency": "USDT",
    "commission": "0.00210540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411532348,
    "fee_amount": 0.0010532,
    "fee_currency": "USDT",
    "commission": "0.00105320 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774411532348,
    "fee_amount": 0.0010533,
    "fee_currency": "USDT",
    "commission": "0.00105330 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774412306916,
    "fee_amount": 0.0010554,
    "fee_currency": "USDT",
    "commission": "0.00105540 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774412306897,
    "fee_amount": 0.0016896,
    "fee_currency": "USDT",
    "commission": "0.00168960 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774412306921,
    "fee_amount": 0.0021118,
    "fee_currency": "USDT",
    "commission": "0.00211180 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774526905506,
    "fee_amount": 0.0010115,
    "fee_currency": "USDT",
    "commission": "0.00101150 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774526905493,
    "fee_amount": 0.002024,
    "fee_currency": "USDT",
    "commission": "0.00202400 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774533109452,
    "fee_amount": 0.0020338,
    "fee_currency": "USDT",
    "commission": "0.00203380 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774533132822,
    "fee_amount": 0.002034,
    "fee_currency": "USDT",
    "commission": "0.00203400 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774533133734,
    "fee_amount": 0.0016272,
    "fee_currency": "USDT",
    "commission": "0.00162720 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774533140908,
    "fee_amount": 0.0016288,
    "fee_currency": "USDT",
    "commission": "0.00162880 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774533140933,
    "fee_amount": 0.0020362,
    "fee_currency": "USDT",
    "commission": "0.00203620 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774533142077,
    "fee_amount": 0.0020382,
    "fee_currency": "USDT",
    "commission": "0.00203820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774533142077,
    "fee_amount": 0.0020384,
    "fee_currency": "USDT",
    "commission": "0.00203840 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774533142092,
    "fee_amount": 0.0016304,
    "fee_currency": "USDT",
    "commission": "0.00163040 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774533142207,
    "fee_amount": 0.0020394,
    "fee_currency": "USDT",
    "commission": "0.00203940 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774533956592,
    "fee_amount": 0.0016256,
    "fee_currency": "USDT",
    "commission": "0.00162560 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774534011908,
    "fee_amount": 0.0016272,
    "fee_currency": "USDT",
    "commission": "0.00162720 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774534018928,
    "fee_amount": 0.0016288,
    "fee_currency": "USDT",
    "commission": "0.00162880 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774534502684,
    "fee_amount": 0.0016256,
    "fee_currency": "USDT",
    "commission": "0.00162560 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774534502707,
    "fee_amount": 0.002033,
    "fee_currency": "USDT",
    "commission": "0.00203300 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774534544168,
    "fee_amount": 0.0016272,
    "fee_currency": "USDT",
    "commission": "0.00162720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774534544405,
    "fee_amount": 0.0020352,
    "fee_currency": "USDT",
    "commission": "0.00203520 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774534902847,
    "fee_amount": 0.0010128,
    "fee_currency": "USDT",
    "commission": "0.00101280 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774534902922,
    "fee_amount": 0.0010127,
    "fee_currency": "USDT",
    "commission": "0.00101270 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774534902909,
    "fee_amount": 0.0020272,
    "fee_currency": "USDT",
    "commission": "0.00202720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774534902911,
    "fee_amount": 0.002027,
    "fee_currency": "USDT",
    "commission": "0.00202700 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774534903031,
    "fee_amount": 0.002026,
    "fee_currency": "USDT",
    "commission": "0.00202600 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774534903024,
    "fee_amount": 0.0016192,
    "fee_currency": "USDT",
    "commission": "0.00161920 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774534949232,
    "fee_amount": 0.0016192,
    "fee_currency": "USDT",
    "commission": "0.00161920 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774535372068,
    "fee_amount": 0.0010121,
    "fee_currency": "USDT",
    "commission": "0.00101210 USDT"
  },
  {
    "strategy_id": "plumeusdt_binance_perp_makerv3-000",
    "venue": "BINANCE_PERP",
    "instrument_id": "PLUMEUSDT-PERP.BINANCE_PERP",
    "event_ts_ms": 1774535723420,
    "fee_amount": 0.0016256,
    "fee_currency": "USDT",
    "commission": "0.00162560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774538241845,
    "fee_amount": 0.001001,
    "fee_currency": "USDT",
    "commission": "0.00100100 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774538637542,
    "fee_amount": 0.0010086,
    "fee_currency": "USDT",
    "commission": "0.00100860 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774539904996,
    "fee_amount": 0.0010051,
    "fee_currency": "USDT",
    "commission": "0.00100510 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541251519,
    "fee_amount": 0.0010051,
    "fee_currency": "USDT",
    "commission": "0.00100510 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541251520,
    "fee_amount": 0.0010056,
    "fee_currency": "USDT",
    "commission": "0.00100560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541251520,
    "fee_amount": 0.0010061,
    "fee_currency": "USDT",
    "commission": "0.00100610 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541251521,
    "fee_amount": 0.0010066,
    "fee_currency": "USDT",
    "commission": "0.00100660 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541251523,
    "fee_amount": 0.0010071,
    "fee_currency": "USDT",
    "commission": "0.00100710 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541251509,
    "fee_amount": 0.0020116,
    "fee_currency": "USDT",
    "commission": "0.00201160 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541251511,
    "fee_amount": 0.0020136,
    "fee_currency": "USDT",
    "commission": "0.00201360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541251511,
    "fee_amount": 0.0020146,
    "fee_currency": "USDT",
    "commission": "0.00201460 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541251511,
    "fee_amount": 0.0020156,
    "fee_currency": "USDT",
    "commission": "0.00201560 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541283272,
    "fee_amount": 0.0010054,
    "fee_currency": "USDT",
    "commission": "0.00100540 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541283275,
    "fee_amount": 0.0010049,
    "fee_currency": "USDT",
    "commission": "0.00100490 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541283252,
    "fee_amount": 0.0020124,
    "fee_currency": "USDT",
    "commission": "0.00201240 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541283280,
    "fee_amount": 0.0010044,
    "fee_currency": "USDT",
    "commission": "0.00100440 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541283256,
    "fee_amount": 0.0020114,
    "fee_currency": "USDT",
    "commission": "0.00201140 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541283261,
    "fee_amount": 0.0020104,
    "fee_currency": "USDT",
    "commission": "0.00201040 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541283552,
    "fee_amount": 0.0010039,
    "fee_currency": "USDT",
    "commission": "0.00100390 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774541283552,
    "fee_amount": 0.0010034,
    "fee_currency": "USDT",
    "commission": "0.00100340 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541283534,
    "fee_amount": 0.0020092,
    "fee_currency": "USDT",
    "commission": "0.00200920 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541283534,
    "fee_amount": 0.0020082,
    "fee_currency": "USDT",
    "commission": "0.00200820 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774541283543,
    "fee_amount": 0.0020072,
    "fee_currency": "USDT",
    "commission": "0.00200720 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774543667232,
    "fee_amount": 0.0020036,
    "fee_currency": "USDT",
    "commission": "0.00200360 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774543787610,
    "fee_amount": 0.0020036,
    "fee_currency": "USDT",
    "commission": "0.00200360 USDT"
  },
  {
    "strategy_id": "plumeusdt_bybit_perp_makerv3-000",
    "venue": "BYBIT",
    "instrument_id": "PLUMEUSDT-LINEAR.BYBIT",
    "event_ts_ms": 1774543806677,
    "fee_amount": 0.0010022,
    "fee_currency": "USDT",
    "commission": "0.00100220 USDT"
  },
  {
    "strategy_id": "plumeusdt_okx_perp_makerv3-000",
    "venue": "OKX",
    "instrument_id": "PLUME-USDT-SWAP.OKX",
    "event_ts_ms": 1774544079458,
    "fee_amount": 0.0020038,
    "fee_currency": "USDT",
    "commission": "0.00200380 USDT"
  }
]

FROZEN_FUNDING_ROWS = [
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773748800000,
    "funding_amount": 0.08228166,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_240509953718_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.08228166",
      "orderLinkId": "",
      "orderId": "26f27cc4-3052-4dcb-8cb3-f955a45148ed",
      "fee": "0",
      "change": "0.08228166",
      "cashFlow": "0",
      "transactionTime": "1773748800000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000073",
      "bonusChange": "0",
      "size": "87589",
      "qty": "87589",
      "cashBalance": "30446.0651519",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_240509953718_0",
      "category": "linear",
      "tradePrice": "0.012888",
      "extraFees": "",
      "tradeId": "aa62b9a2-5bf1-4c36-ad31-96fb49b043c1"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773748800551,
    "funding_amount": -0.0064537311829248,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3397265462562201606",
    "raw": {
      "bal": "810.8452905359054765",
      "balChg": "-0.0064537311829248",
      "billId": "3397265462562201606",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773748800551",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0064537311829248",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.012891",
      "subType": "173",
      "sz": "1842",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773748800551",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773763200000,
    "funding_amount": -0.06203784,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_240598045358_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06203784",
      "orderLinkId": "",
      "orderId": "170dfb35-1a8d-4d03-b041-2f4948867d87",
      "fee": "0",
      "change": "-0.06203784",
      "cashFlow": "0",
      "transactionTime": "1773763200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "96587",
      "qty": "96587",
      "cashBalance": "30458.21026892",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_240598045358_0",
      "category": "linear",
      "tradePrice": "0.012846",
      "extraFees": "",
      "tradeId": "46ddb5cd-f6aa-4690-a8d2-fa66c53ab84c"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773763201109,
    "funding_amount": -0.011831166,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3397748665106374658",
    "raw": {
      "bal": "810.8334593699054765",
      "balChg": "-0.0118311660000000",
      "billId": "3397748665106374658",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773763201109",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.011831166",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.012846",
      "subType": "173",
      "sz": "1842",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773763201109",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773777600000,
    "funding_amount": -0.05978338,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_240689999638_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05978338",
      "orderLinkId": "",
      "orderId": "0d1a4917-b5ce-40bb-8b52-c1b72afb52ff",
      "fee": "0",
      "change": "-0.05978338",
      "cashFlow": "0",
      "transactionTime": "1773777600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "93587",
      "qty": "93587",
      "cashBalance": "30482.84149777",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_240689999638_0",
      "category": "linear",
      "tradePrice": "0.012776",
      "extraFees": "",
      "tradeId": "bbcf0c5d-1cd7-4fdd-b0ef-ce2e6ec26d6b"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773777600593,
    "funding_amount": 0.0218014019613495,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3398231831613087750",
    "raw": {
      "bal": "812.047231334814826",
      "balChg": "0.0218014019613495",
      "billId": "3398231831613087750",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773777600593",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0218014019613495",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.012789",
      "subType": "174",
      "sz": "1542",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773777600593",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773792000000,
    "funding_amount": 0.10582018,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_240748105706_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.10582018",
      "orderLinkId": "",
      "orderId": "155fb479-209f-48e4-8bfa-53476a9f16cf",
      "fee": "0",
      "change": "0.10582018",
      "cashFlow": "0",
      "transactionTime": "1773792000000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000076",
      "bonusChange": "0",
      "size": "111587",
      "qty": "111587",
      "cashBalance": "30407.97351126",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_240748105706_0",
      "category": "linear",
      "tradePrice": "0.012529",
      "extraFees": "",
      "tradeId": "7852c88b-45a3-40bc-b88a-48330f3ad04a"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773792001145,
    "funding_amount": 0.0248018451345344,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3398715033955934210",
    "raw": {
      "bal": "812.0312015799493604",
      "balChg": "0.0248018451345344",
      "billId": "3398715033955934210",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773792001145",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0248018451345344",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.012531",
      "subType": "174",
      "sz": "3142",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773792001145",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773806400000,
    "funding_amount": -0.05857094,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_240823845053_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05857094",
      "orderLinkId": "",
      "orderId": "87b7f76b-6a9a-4367-af9e-1f4d6bc07af1",
      "fee": "0",
      "change": "-0.05857094",
      "cashFlow": "0",
      "transactionTime": "1773806400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "95587",
      "qty": "95587",
      "cashBalance": "30499.68326168",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_240823845053_0",
      "category": "linear",
      "tradePrice": "0.012255",
      "extraFees": "",
      "tradeId": "ed4f6827-0cdc-46d2-8302-d4dae3dfede4"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773806400597,
    "funding_amount": -0.018041815,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3399198199388905478",
    "raw": {
      "bal": "809.2601975335583604",
      "balChg": "-0.0180418150000000",
      "billId": "3399198199388905478",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773806400597",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.018041815",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.012265",
      "subType": "173",
      "sz": "2942",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773806400597",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773820800000,
    "funding_amount": -0.05864763,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_240884342358_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05864763",
      "orderLinkId": "",
      "orderId": "bd0fc161-f77d-45e8-aca4-d7b3dc4f24b1",
      "fee": "0",
      "change": "-0.05864763",
      "cashFlow": "0",
      "transactionTime": "1773820800000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "96587",
      "qty": "96587",
      "cashBalance": "30499.81355587",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_240884342358_0",
      "category": "linear",
      "tradePrice": "0.012144",
      "extraFees": "",
      "tradeId": "a44f7653-a1c4-46d9-9716-a5d878f8a12b"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773820801192,
    "funding_amount": -0.017880005,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3399681403174592516",
    "raw": {
      "bal": "808.8650220413060604",
      "balChg": "-0.0178800050000000",
      "billId": "3399681403174592516",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773820801192",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.017880005",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.012155",
      "subType": "173",
      "sz": "2942",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773820801192",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773835200000,
    "funding_amount": -0.05787976,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_240951826742_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05787976",
      "orderLinkId": "",
      "orderId": "56949896-dc17-4608-b413-1c0452620c6e",
      "fee": "0",
      "change": "-0.05787976",
      "cashFlow": "0",
      "transactionTime": "1773835200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "96587",
      "qty": "96587",
      "cashBalance": "30487.20878231",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_240951826742_0",
      "category": "linear",
      "tradePrice": "0.011985",
      "extraFees": "",
      "tradeId": "05131a72-c4f5-4e5d-9acf-2f48e3c930e4"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773835200547,
    "funding_amount": -0.014633685,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3400164565352783880",
    "raw": {
      "bal": "805.0171259710353604",
      "balChg": "-0.0146336850000000",
      "billId": "3400164565352783880",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773835200547",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.014633685",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011985",
      "subType": "173",
      "sz": "2442",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773835200547",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773849600000,
    "funding_amount": -0.0630131,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241038075867_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.0630131",
      "orderLinkId": "",
      "orderId": "e90ad8d7-4e82-4de0-a3d3-c006033cfbc7",
      "fee": "0",
      "change": "-0.0630131",
      "cashFlow": "0",
      "transactionTime": "1773849600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "109588",
      "qty": "109588",
      "cashBalance": "30453.86023314",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241038075867_0",
      "category": "linear",
      "tradePrice": "0.0115",
      "extraFees": "",
      "tradeId": "739aeb61-fd15-4c16-90ae-e26b1d9b371c"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773849601259,
    "funding_amount": -0.010005666,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3400647773064339460",
    "raw": {
      "bal": "782.0296224268248604",
      "balChg": "-0.0100056660000000",
      "billId": "3400647773064339460",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773849601259",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.010005666",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011514",
      "subType": "173",
      "sz": "1738",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773849601259",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773864000000,
    "funding_amount": -0.06328939,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241108399820_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06328939",
      "orderLinkId": "",
      "orderId": "c27a385a-d5f0-4c0e-8fee-25b179ae9e10",
      "fee": "0",
      "change": "-0.06328939",
      "cashFlow": "0",
      "transactionTime": "1773864000000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "110366",
      "qty": "110366",
      "cashBalance": "30679.2734901",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241108399820_0",
      "category": "linear",
      "tradePrice": "0.011469",
      "extraFees": "",
      "tradeId": "83c4f9b8-4b28-4f99-984a-aa66cab9322d"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773864000550,
    "funding_amount": -0.010549201,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3401130933095047176",
    "raw": {
      "bal": "780.0507224593334604",
      "balChg": "-0.0105492010000000",
      "billId": "3401130933095047176",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773864000550",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.010549201",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011479",
      "subType": "173",
      "sz": "1838",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773864000550",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773878400000,
    "funding_amount": -0.06458236,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241161764088_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06458236",
      "orderLinkId": "",
      "orderId": "3c4f0717-0306-4b37-8cd2-cab7565c5609",
      "fee": "0",
      "change": "-0.06458236",
      "cashFlow": "0",
      "transactionTime": "1773878400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "112366",
      "qty": "112366",
      "cashBalance": "30785.47247624",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241161764088_0",
      "category": "linear",
      "tradePrice": "0.011495",
      "extraFees": "",
      "tradeId": "bf9708fc-fb59-47d4-8492-7f90844840a0"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773878401282,
    "funding_amount": -0.0169234425,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3401614141477691396",
    "raw": {
      "bal": "779.1254330101598604",
      "balChg": "-0.0169234425000000",
      "billId": "3401614141477691396",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773878401282",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0169234425",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011493",
      "subType": "173",
      "sz": "2945",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773878401282",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773892800000,
    "funding_amount": 0.60271485,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241229223357_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.60271485",
      "orderLinkId": "",
      "orderId": "a9196318-079b-4d84-ae5a-7ba0c564f83a",
      "fee": "0",
      "change": "0.60271485",
      "cashFlow": "0",
      "transactionTime": "1773892800000",
      "type": "SETTLEMENT",
      "feeRate": "-0.00042",
      "bonusChange": "0",
      "size": "127366",
      "qty": "127366",
      "cashBalance": "30660.96818059",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241229223357_0",
      "category": "linear",
      "tradePrice": "0.011274",
      "extraFees": "",
      "tradeId": "1b2edac7-2263-41ed-a594-f87c1625dab0"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773892800546,
    "funding_amount": -0.0194298,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3402097300602429448",
    "raw": {
      "bal": "777.9016064800497604",
      "balChg": "-0.0194298000000000",
      "billId": "3402097300602429448",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773892800546",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0194298",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.01128",
      "subType": "173",
      "sz": "3445",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773892800546",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773907200000,
    "funding_amount": 0.08223989,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241311106210_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.08223989",
      "orderLinkId": "",
      "orderId": "b7cf1772-3e05-416c-a72c-c3fb6eb26e1f",
      "fee": "0",
      "change": "0.08223989",
      "cashFlow": "0",
      "transactionTime": "1773907200000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000058",
      "bonusChange": "0",
      "size": "127468",
      "qty": "127468",
      "cashBalance": "30656.26259317",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241311106210_0",
      "category": "linear",
      "tradePrice": "0.011244",
      "extraFees": "",
      "tradeId": "c267126c-afc2-4307-897e-f1bc97577b82"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773907201351,
    "funding_amount": 0.117447184273343,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3402580511434547204",
    "raw": {
      "bal": "774.3010754377411034",
      "balChg": "0.1174471842733430",
      "billId": "3402580511434547204",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773907201351",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.117447184273343",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011251",
      "subType": "174",
      "sz": "3845",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773907201351",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773921600000,
    "funding_amount": -0.04444324,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241378179108_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.04444324",
      "orderLinkId": "",
      "orderId": "ee760264-c965-4835-a660-ffb894fee5d4",
      "fee": "0",
      "change": "-0.04444324",
      "cashFlow": "0",
      "transactionTime": "1773921600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "78835",
      "qty": "78835",
      "cashBalance": "30751.89454757",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241378179108_0",
      "category": "linear",
      "tradePrice": "0.011275",
      "extraFees": "",
      "tradeId": "8fde7d77-93ff-47a2-b718-a1744b1b7a81"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773921600568,
    "funding_amount": -0.00648485,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3403063668982226952",
    "raw": {
      "bal": "772.0914030469615534",
      "balChg": "-0.0064848500000000",
      "billId": "3403063668982226952",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773921600568",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.00648485",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011278",
      "subType": "173",
      "sz": "1150",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773921600568",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773936000000,
    "funding_amount": -0.04347751,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241448540676_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.04347751",
      "orderLinkId": "",
      "orderId": "38089a96-0e63-4cee-80ec-f35793c806bc",
      "fee": "0",
      "change": "-0.04347751",
      "cashFlow": "0",
      "transactionTime": "1773936000000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "78835",
      "qty": "78835",
      "cashBalance": "30784.62800462",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241448540676_0",
      "category": "linear",
      "tradePrice": "0.01103",
      "extraFees": "",
      "tradeId": "d6d29f67-0d47-44dd-8246-f95f9b75845b"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773936001437,
    "funding_amount": -0.01020275,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3403546881961828354",
    "raw": {
      "bal": "770.4427908247422534",
      "balChg": "-0.0102027500000000",
      "billId": "3403546881961828354",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773936001437",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.01020275",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.01103",
      "subType": "173",
      "sz": "1850",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773936001437",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773950400000,
    "funding_amount": -0.04643868,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241497684347_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.04643868",
      "orderLinkId": "",
      "orderId": "3329c244-6f70-4fa3-be09-8444f0b137c6",
      "fee": "0",
      "change": "-0.04643868",
      "cashFlow": "0",
      "transactionTime": "1773950400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "84835",
      "qty": "84835",
      "cashBalance": "30728.27095765",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241497684347_0",
      "category": "linear",
      "tradePrice": "0.010948",
      "extraFees": "",
      "tradeId": "6840e6f1-4373-491d-af5d-ef66782e683c"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773950400631,
    "funding_amount": -0.016703325,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3404030038737756166",
    "raw": {
      "bal": "769.6974388729642534",
      "balChg": "-0.0167033250000000",
      "billId": "3404030038737756166",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773950400631",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.016703325",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010953",
      "subType": "173",
      "sz": "3050",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773950400631",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773964800000,
    "funding_amount": -0.04577877,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241535219229_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.04577877",
      "orderLinkId": "",
      "orderId": "5833691e-03e5-49ff-a882-295a34e1d5bc",
      "fee": "0",
      "change": "-0.04577877",
      "cashFlow": "0",
      "transactionTime": "1773964800000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "82835",
      "qty": "82835",
      "cashBalance": "30678.65142326",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241535219229_0",
      "category": "linear",
      "tradePrice": "0.011053",
      "extraFees": "",
      "tradeId": "c2a5e799-1933-4688-9df0-f45b2136b9f6"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773964801447,
    "funding_amount": -0.00635835,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3404513249938972674",
    "raw": {
      "bal": "768.2853683320944534",
      "balChg": "-0.0063583500000000",
      "billId": "3404513249938972674",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773964801447",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.00635835",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011058",
      "subType": "173",
      "sz": "1150",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773964801447",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773979200000,
    "funding_amount": -0.05003632,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241581254805_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05003632",
      "orderLinkId": "",
      "orderId": "ed7cb09f-1e3a-4886-bb83-411738aec384",
      "fee": "0",
      "change": "-0.05003632",
      "cashFlow": "0",
      "transactionTime": "1773979200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "88835",
      "qty": "88835",
      "cashBalance": "30571.8065776",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241581254805_0",
      "category": "linear",
      "tradePrice": "0.011265",
      "extraFees": "",
      "tradeId": "ec71c2ef-c6ce-48b4-bb54-013d630c74ed"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773979200687,
    "funding_amount": -0.00755546,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3404996408258404358",
    "raw": {
      "bal": "768.9882871714100334",
      "balChg": "-0.0075554600000000",
      "billId": "3404996408258404358",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773979200687",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.00755546",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.01126",
      "subType": "173",
      "sz": "1342",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773979200687",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1773993600000,
    "funding_amount": -0.05100128,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241629661546_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05100128",
      "orderLinkId": "",
      "orderId": "8a970f37-c214-4c7f-916a-f357850dc4be",
      "fee": "0",
      "change": "-0.05100128",
      "cashFlow": "0",
      "transactionTime": "1773993600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "92085",
      "qty": "92085",
      "cashBalance": "30568.81523487",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241629661546_0",
      "category": "linear",
      "tradePrice": "0.011077",
      "extraFees": "",
      "tradeId": "25afa4d8-ed4c-4077-b1cc-43c86aeccaad"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1773993601555,
    "funding_amount": -0.005768512,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3405479621204451330",
    "raw": {
      "bal": "765.4191240374549334",
      "balChg": "-0.0057685120000000",
      "billId": "3405479621204451330",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1773993601555",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.005768512",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011072",
      "subType": "173",
      "sz": "1042",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1773993601555",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774008000000,
    "funding_amount": -0.05271037,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241680644155_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05271037",
      "orderLinkId": "",
      "orderId": "347ab337-7848-4ea3-b6ad-f261306996ac",
      "fee": "0",
      "change": "-0.05271037",
      "cashFlow": "0",
      "transactionTime": "1774008000000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "95085",
      "qty": "95085",
      "cashBalance": "30565.16174394",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241680644155_0",
      "category": "linear",
      "tradePrice": "0.011087",
      "extraFees": "",
      "tradeId": "29afbbc4-2a1c-42af-937e-bf9c2b693de6"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774008000780,
    "funding_amount": -0.0101150625,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3405962779020566534",
    "raw": {
      "bal": "764.1406534040788084",
      "balChg": "-0.0101150625000000",
      "billId": "3405962779020566534",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774008000780",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0101150625",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011085",
      "subType": "173",
      "sz": "1825",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774008000780",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774022400000,
    "funding_amount": -0.05143148,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241750981845_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05143148",
      "orderLinkId": "",
      "orderId": "7d26cda6-7156-4131-8771-0059883ae469",
      "fee": "0",
      "change": "-0.05143148",
      "cashFlow": "0",
      "transactionTime": "1774022400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "95085",
      "qty": "95085",
      "cashBalance": "30494.27081558",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241750981845_0",
      "category": "linear",
      "tradePrice": "0.010818",
      "extraFees": "",
      "tradeId": "16f0a811-cb92-4447-b773-d0074cab3268"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774022401629,
    "funding_amount": -0.0136552,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3406445991329079298",
    "raw": {
      "bal": "762.7112858514756084",
      "balChg": "-0.0136552000000000",
      "billId": "3406445991329079298",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774022401629",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0136552",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010816",
      "subType": "173",
      "sz": "2525",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774022401629",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774036800000,
    "funding_amount": -0.06216494,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241801317146_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06216494",
      "orderLinkId": "",
      "orderId": "8fc2b60f-3de1-4811-b894-0366159ede56",
      "fee": "0",
      "change": "-0.06216494",
      "cashFlow": "0",
      "transactionTime": "1774036800000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "114632",
      "qty": "114632",
      "cashBalance": "30488.81403429",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241801317146_0",
      "category": "linear",
      "tradePrice": "0.010846",
      "extraFees": "",
      "tradeId": "5b2f85e5-a15a-4ef5-888a-86c9850da084"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774036800828,
    "funding_amount": -0.0061025625,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3406929148272779270",
    "raw": {
      "bal": "761.2578903211599084",
      "balChg": "-0.0061025625000000",
      "billId": "3406929148272779270",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774036800828",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0061025625",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010849",
      "subType": "173",
      "sz": "1125",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774036800828",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774051200000,
    "funding_amount": -0.05970011,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241836699762_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05970011",
      "orderLinkId": "",
      "orderId": "76710f2b-668b-46ad-a0cc-8b757b3b30a9",
      "fee": "0",
      "change": "-0.05970011",
      "cashFlow": "0",
      "transactionTime": "1774051200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "109632",
      "qty": "109632",
      "cashBalance": "30498.15997546",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241836699762_0",
      "category": "linear",
      "tradePrice": "0.010891",
      "extraFees": "",
      "tradeId": "fab1e07c-1ef6-4582-9fe1-8d56007015e4"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774051201638,
    "funding_amount": 0.1201271867850423,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3407412359272669186",
    "raw": {
      "bal": "761.4201132503471507",
      "balChg": "0.1201271867850423",
      "billId": "3407412359272669186",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774051201638",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.1201271867850423",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010889",
      "subType": "174",
      "sz": "2706",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774051201638",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774065600000,
    "funding_amount": -0.06040732,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241872761704_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06040732",
      "orderLinkId": "",
      "orderId": "cf238102-58cb-4b9b-a1c4-6145202b4153",
      "fee": "0",
      "change": "-0.06040732",
      "cashFlow": "0",
      "transactionTime": "1774065600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "109662",
      "qty": "109662",
      "cashBalance": "30395.77045406",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241872761704_0",
      "category": "linear",
      "tradePrice": "0.011017",
      "extraFees": "",
      "tradeId": "7cf3e288-60d0-4ecd-99b0-cbfe44fb78bc"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774065600911,
    "funding_amount": -0.009093708,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3407895518699397126",
    "raw": {
      "bal": "765.5077043440364507",
      "balChg": "-0.0090937080000000",
      "billId": "3407895518699397126",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774065600911",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.009093708",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011016",
      "subType": "173",
      "sz": "1651",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774065600911",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774080000000,
    "funding_amount": -0.05857878,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241913335469_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05857878",
      "orderLinkId": "",
      "orderId": "fbebf8fe-3306-4358-a52f-5969e79fde74",
      "fee": "0",
      "change": "-0.05857878",
      "cashFlow": "0",
      "transactionTime": "1774080000000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "106662",
      "qty": "106662",
      "cashBalance": "30505.4167625",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241913335469_0",
      "category": "linear",
      "tradePrice": "0.010984",
      "extraFees": "",
      "tradeId": "73a2762d-89f7-4673-98d6-d10afe78285c"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774080001688,
    "funding_amount": 0.0354563312170479,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3408378728591990786",
    "raw": {
      "bal": "765.4625002797232986",
      "balChg": "0.0354563312170479",
      "billId": "3408378728591990786",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774080001688",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0354563312170479",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010985",
      "subType": "174",
      "sz": "1651",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774080001688",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774094400000,
    "funding_amount": -0.07292569,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_241954719751_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.07292569",
      "orderLinkId": "",
      "orderId": "0ccf5d7e-5cf2-4bc3-a8fc-a9b68b0fbcc8",
      "fee": "0",
      "change": "-0.07292569",
      "cashFlow": "0",
      "transactionTime": "1774094400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "131849",
      "qty": "131849",
      "cashBalance": "31045.2062873",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_241954719751_0",
      "category": "linear",
      "tradePrice": "0.011062",
      "extraFees": "",
      "tradeId": "23bdc059-a5f8-46fe-999d-d1e416cf2416"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774094400927,
    "funding_amount": -0.0224081065,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3408861886877868038",
    "raw": {
      "bal": "764.8461041309722986",
      "balChg": "-0.0224081065000000",
      "billId": "3408861886877868038",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774094400927",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0224081065",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011063",
      "subType": "173",
      "sz": "4051",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774094400927",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774108800000,
    "funding_amount": 0.17085685,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242005724649_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.17085685",
      "orderLinkId": "",
      "orderId": "318ca075-e3ed-4e72-af30-a468a2c23695",
      "fee": "0",
      "change": "0.17085685",
      "cashFlow": "0",
      "transactionTime": "1774108800000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000117",
      "bonusChange": "0",
      "size": "132849",
      "qty": "132849",
      "cashBalance": "30978.09594292",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242005724649_0",
      "category": "linear",
      "tradePrice": "0.011013",
      "extraFees": "",
      "tradeId": "bd5ed369-94db-4b51-ad40-5417fab28ea8"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774108801793,
    "funding_amount": -0.014599057,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3409345099756806146",
    "raw": {
      "bal": "763.3811126027410986",
      "balChg": "-0.0145990570000000",
      "billId": "3409345099756806146",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774108801793",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.014599057",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.011014",
      "subType": "173",
      "sz": "2651",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774108801793",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774123200000,
    "funding_amount": -0.07378264,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242041482326_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.07378264",
      "orderLinkId": "",
      "orderId": "e126ea46-ca90-4aea-99fa-5dce97153756",
      "fee": "0",
      "change": "-0.07378264",
      "cashFlow": "0",
      "transactionTime": "1774123200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "134849",
      "qty": "134849",
      "cashBalance": "30965.86697337",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242041482326_0",
      "category": "linear",
      "tradePrice": "0.010943",
      "extraFees": "",
      "tradeId": "eb1974b0-8c3c-412f-9966-136d6372f206"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774123200992,
    "funding_amount": -0.010683676,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3409828256700506118",
    "raw": {
      "bal": "761.6496063714602986",
      "balChg": "-0.0106836760000000",
      "billId": "3409828256700506118",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774123200992",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.010683676",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010952",
      "subType": "173",
      "sz": "1951",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774123200992",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774137600000,
    "funding_amount": -0.07741827,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242072824042_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.07741827",
      "orderLinkId": "",
      "orderId": "c294177f-e4b5-41e4-b132-ad076b4f4caf",
      "fee": "0",
      "change": "-0.07741827",
      "cashFlow": "0",
      "transactionTime": "1774137600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "146848",
      "qty": "146848",
      "cashBalance": "30719.07967486",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242072824042_0",
      "category": "linear",
      "tradePrice": "0.010544",
      "extraFees": "",
      "tradeId": "9992fb51-6788-40c4-bf8d-ebd238da2852"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774137601810,
    "funding_amount": 0.0002586955,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3410311467968831490",
    "raw": {
      "bal": "753.8391985277106536",
      "balChg": "0.0002586955000000",
      "billId": "3410311467968831490",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774137601810",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0002586955",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010559",
      "subType": "174",
      "sz": "49",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774137601810",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774152000000,
    "funding_amount": -0.07599567,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242122379445_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.07599567",
      "orderLinkId": "",
      "orderId": "96a60b26-a9ac-4cb9-9ff0-7ed04eb6847d",
      "fee": "0",
      "change": "-0.07599567",
      "cashFlow": "0",
      "transactionTime": "1774152000000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "142849",
      "qty": "142849",
      "cashBalance": "30794.32156813",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242122379445_0",
      "category": "linear",
      "tradePrice": "0.01064",
      "extraFees": "",
      "tradeId": "4cb6194c-30f0-4e47-a0e5-d1323b5a2076"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774152001036,
    "funding_amount": -0.005062173,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3410794625818501126",
    "raw": {
      "bal": "754.6832627622480536",
      "balChg": "-0.0050621730000000",
      "billId": "3410794625818501126",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774152001036",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.005062173",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010646",
      "subType": "173",
      "sz": "951",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774152001036",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774166400000,
    "funding_amount": 0.07634358,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242179720343_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.07634358",
      "orderLinkId": "",
      "orderId": "f68b5ee2-fdfc-4bb6-8041-e78615586ec7",
      "fee": "0",
      "change": "0.07634358",
      "cashFlow": "0",
      "transactionTime": "1774166400000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000047",
      "bonusChange": "0",
      "size": "153721",
      "qty": "153721",
      "cashBalance": "30814.99779641",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242179720343_0",
      "category": "linear",
      "tradePrice": "0.010569",
      "extraFees": "",
      "tradeId": "906145c0-a135-4592-a14e-e08832dd5cb6"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774166401881,
    "funding_amount": 0.0452613765963801,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3411277837992796162",
    "raw": {
      "bal": "755.0583166826399337",
      "balChg": "0.0452613765963801",
      "billId": "3411277837992796162",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774166401881",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0452613765963801",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010581",
      "subType": "174",
      "sz": "1351",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774166401881",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774180800000,
    "funding_amount": -0.08213873,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242236327578_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.08213873",
      "orderLinkId": "",
      "orderId": "e6648b58-73a0-4263-8950-9ecae1a757ba",
      "fee": "0",
      "change": "-0.08213873",
      "cashFlow": "0",
      "transactionTime": "1774180800000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "157868",
      "qty": "157868",
      "cashBalance": "30792.16252251",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242236327578_0",
      "category": "linear",
      "tradePrice": "0.010406",
      "extraFees": "",
      "tradeId": "6401080f-e495-4f7c-875a-c426f41bb108"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774180801129,
    "funding_amount": 0.0011185694691483,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3411760996580663302",
    "raw": {
      "bal": "754.016293127108082",
      "balChg": "0.0011185694691483",
      "billId": "3411760996580663302",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774180801129",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0011185694691483",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010418",
      "subType": "174",
      "sz": "1551",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774180801129",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774195200000,
    "funding_amount": -0.05621501,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242294309776_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05621501",
      "orderLinkId": "",
      "orderId": "bf123ce7-33e6-480a-bbd4-a72c45fcbde6",
      "fee": "0",
      "change": "-0.05621501",
      "cashFlow": "0",
      "transactionTime": "1774195200000",
      "type": "SETTLEMENT",
      "feeRate": "0.000036",
      "bonusChange": "0",
      "size": "151868",
      "qty": "151868",
      "cashBalance": "30810.47296116",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242294309776_0",
      "category": "linear",
      "tradePrice": "0.010386",
      "extraFees": "",
      "tradeId": "c349c619-de55-41ce-bb98-b1af702be465"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774195200118,
    "funding_amount": -0.0044252,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3412244146477932546",
    "raw": {
      "bal": "753.717925358279682",
      "balChg": "-0.004425200000000",
      "billId": "3412244146477932546",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774195200118",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0044252",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.0104",
      "subType": "173",
      "sz": "851",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774195200118",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774209600000,
    "funding_amount": 0.19179169,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242376860711_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.19179169",
      "orderLinkId": "",
      "orderId": "b057502a-06ed-45a2-bd0a-057689784559",
      "fee": "0",
      "change": "0.19179169",
      "cashFlow": "0",
      "transactionTime": "1774209600000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000137",
      "bonusChange": "0",
      "size": "137869",
      "qty": "137869",
      "cashBalance": "30779.0982122",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242376860711_0",
      "category": "linear",
      "tradePrice": "0.010166",
      "extraFees": "",
      "tradeId": "ba913b8c-fef5-442f-8e0f-e1b262032606"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774209601144,
    "funding_amount": -0.0005837988678611,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3412727364725579782",
    "raw": {
      "bal": "753.0247138113590209",
      "balChg": "-0.0005837988678611",
      "billId": "3412727364725579782",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774209601144",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0005837988678611",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010169",
      "subType": "173",
      "sz": "1051",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774209601144",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774224000000,
    "funding_amount": 0.25229833,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242432091302_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.25229833",
      "orderLinkId": "",
      "orderId": "e7ed2e82-59a3-421a-bf34-71eea62cae05",
      "fee": "0",
      "change": "0.25229833",
      "cashFlow": "0",
      "transactionTime": "1774224000000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000186",
      "bonusChange": "0",
      "size": "133633",
      "qty": "133633",
      "cashBalance": "30657.00834349",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242432091302_0",
      "category": "linear",
      "tradePrice": "0.010168",
      "extraFees": "",
      "tradeId": "8e0638e7-2fb7-4933-9f0d-b64e0e2701a7"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774224000187,
    "funding_amount": -0.008402072,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3413210516434788354",
    "raw": {
      "bal": "751.1507344039275209",
      "balChg": "-0.0084020720000000",
      "billId": "3413210516434788354",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774224000187",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.008402072",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010172",
      "subType": "173",
      "sz": "1652",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774224000187",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774238400000,
    "funding_amount": -0.05309165,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242486923690_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05309165",
      "orderLinkId": "",
      "orderId": "1593712e-abcb-4b51-a71d-0bc79f884423",
      "fee": "0",
      "change": "-0.05309165",
      "cashFlow": "0",
      "transactionTime": "1774238400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "105634",
      "qty": "105634",
      "cashBalance": "30659.64314496",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242486923690_0",
      "category": "linear",
      "tradePrice": "0.010052",
      "extraFees": "",
      "tradeId": "c79353dd-34d0-4658-832e-072056ffcc93"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774238400141,
    "funding_amount": -0.0037788,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3413693698712084488",
    "raw": {
      "bal": "750.5649740695041209",
      "balChg": "-0.0037788000000000",
      "billId": "3413693698712084488",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774238400141",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0037788",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.01005",
      "subType": "173",
      "sz": "752",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774238400141",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774252800000,
    "funding_amount": 0.08733226,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242542357195_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.08733226",
      "orderLinkId": "",
      "orderId": "45f46f1e-403b-4452-8be8-8511bf567444",
      "fee": "0",
      "change": "0.08733226",
      "cashFlow": "0",
      "transactionTime": "1774252800000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000078",
      "bonusChange": "0",
      "size": "114014",
      "qty": "114014",
      "cashBalance": "30711.73825086",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242542357195_0",
      "category": "linear",
      "tradePrice": "0.009922",
      "extraFees": "",
      "tradeId": "d69faff7-b6ab-49a8-aad4-e98014c116fe"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774252800263,
    "funding_amount": -0.008705398,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3414176886626525186",
    "raw": {
      "bal": "749.8000713033759209",
      "balChg": "-0.0087053980000000",
      "billId": "3414176886626525186",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774252800263",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.008705398",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.009932",
      "subType": "173",
      "sz": "1753",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774252800263",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774267200000,
    "funding_amount": -0.04306925,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242625481282_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.04306925",
      "orderLinkId": "",
      "orderId": "55083040-fa86-4b48-ab16-06780d2bc7b4",
      "fee": "0",
      "change": "-0.04306925",
      "cashFlow": "0",
      "transactionTime": "1774267200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "81982",
      "qty": "81982",
      "cashBalance": "30624.97599994",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242625481282_0",
      "category": "linear",
      "tradePrice": "0.010507",
      "extraFees": "",
      "tradeId": "9b1c180a-d391-4996-8319-e7a89305b468"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774267200220,
    "funding_amount": -0.01019025,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3414660069004484616",
    "raw": {
      "bal": "754.8653462901864249",
      "balChg": "-0.0101902500000000",
      "billId": "3414660069004484616",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774267200220",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.01019025",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.0105",
      "subType": "173",
      "sz": "1941",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774267200220",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774281600000,
    "funding_amount": -0.06058349,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242703971389_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06058349",
      "orderLinkId": "",
      "orderId": "86fc1670-6c81-4e6f-9f3e-0009cc6287b7",
      "fee": "0",
      "change": "-0.06058349",
      "cashFlow": "0",
      "transactionTime": "1774281600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "114981",
      "qty": "114981",
      "cashBalance": "30623.35223847",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242703971389_0",
      "category": "linear",
      "tradePrice": "0.010538",
      "extraFees": "",
      "tradeId": "18a524e0-1277-47f4-805e-6189ee518c40"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774281600310,
    "funding_amount": 0.0742752516908019,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3415143255845183490",
    "raw": {
      "bal": "758.0031334838295468",
      "balChg": "0.0742752516908019",
      "billId": "3415143255845183490",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774281600310",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0742752516908019",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010543",
      "subType": "174",
      "sz": "4190",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774281600310",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774296000000,
    "funding_amount": -0.03649195,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242757345315_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.03649195",
      "orderLinkId": "",
      "orderId": "37647e27-d3f4-4a49-a5b6-9ae18475f89e",
      "fee": "0",
      "change": "-0.03649195",
      "cashFlow": "0",
      "transactionTime": "1774296000000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "69482",
      "qty": "69482",
      "cashBalance": "30628.37752966",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242757345315_0",
      "category": "linear",
      "tradePrice": "0.010504",
      "extraFees": "",
      "tradeId": "faa32946-803f-4b97-bfb6-f3c699110e63"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774296000271,
    "funding_amount": 0.0324792820200599,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3415626438357360648",
    "raw": {
      "bal": "758.0356127658496067",
      "balChg": "0.0324792820200599",
      "billId": "3415626438357360648",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774296000271",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0324792820200599",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010506",
      "subType": "174",
      "sz": "4190",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774296000271",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774310400000,
    "funding_amount": -0.05398246,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242803307520_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05398246",
      "orderLinkId": "",
      "orderId": "fa7b3d80-1770-4f1e-88b2-a1ef52082062",
      "fee": "0",
      "change": "-0.05398246",
      "cashFlow": "0",
      "transactionTime": "1774310400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "101921",
      "qty": "101921",
      "cashBalance": "30670.65404537",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242803307520_0",
      "category": "linear",
      "tradePrice": "0.010593",
      "extraFees": "",
      "tradeId": "59f8cf57-61b7-48cd-945f-3e2bebfd0391"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774310400357,
    "funding_amount": -0.01583803,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3416109625063841794",
    "raw": {
      "bal": "757.6473854965748067",
      "balChg": "-0.0158380300000000",
      "billId": "3416109625063841794",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774310400357",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.01583803",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010594",
      "subType": "173",
      "sz": "2990",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774310400357",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774324800000,
    "funding_amount": -0.05920475,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242895189928_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05920475",
      "orderLinkId": "",
      "orderId": "ce80d6ed-43b6-4659-90d8-bf39c4640294",
      "fee": "0",
      "change": "-0.05920475",
      "cashFlow": "0",
      "transactionTime": "1774324800000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "113921",
      "qty": "113921",
      "cashBalance": "30650.88130687",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242895189928_0",
      "category": "linear",
      "tradePrice": "0.010394",
      "extraFees": "",
      "tradeId": "98b71eb6-8334-466c-be5c-549708a5c8e3"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774324800346,
    "funding_amount": -0.0165843945,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3416592808515543046",
    "raw": {
      "bal": "757.7197957930646067",
      "balChg": "-0.0165843945000000",
      "billId": "3416592808515543046",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774324800346",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0165843945",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010401",
      "subType": "173",
      "sz": "3189",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774324800346",
      "type": "8"
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774339200000,
    "funding_amount": -0.09461962,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_242964374464_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.09461962",
      "orderLinkId": "",
      "orderId": "c8e4eeac-4d4b-442b-a1c1-b880049b9f9c",
      "fee": "0",
      "change": "-0.09461962",
      "cashFlow": "0",
      "transactionTime": "1774339200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "178074",
      "qty": "178074",
      "cashBalance": "30984.541872",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_242964374464_0",
      "category": "linear",
      "tradePrice": "0.010627",
      "extraFees": "",
      "tradeId": "9d02454a-3269-41e4-b0cc-3a7486fd012c"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774339201013,
    "funding_amount": -0.015363702,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3417076014717149186",
    "raw": {
      "bal": "756.2647119782276067",
      "balChg": "-0.0153637020000000",
      "billId": "3417076014717149186",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774339201013",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.015363702",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010636",
      "subType": "173",
      "sz": "2889",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774339201013",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774353600000,
    "funding_amount": 0.01502889,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "887943431762698880",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "0.01502889",
      "asset": "USDT",
      "time": 1774353600000,
      "info": "FUNDING_FEE",
      "tranId": 887943431762698880,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774353600000,
    "funding_amount": 0.20308702,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243015574837_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.20308702",
      "orderLinkId": "",
      "orderId": "ce69c03e-eac1-439c-9039-ca083fb749d9",
      "fee": "0",
      "change": "0.20308702",
      "cashFlow": "0",
      "transactionTime": "1774353600000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000097",
      "bonusChange": "0",
      "size": "197352",
      "qty": "197352",
      "cashBalance": "31080.77216571",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243015574837_0",
      "category": "linear",
      "tradePrice": "0.010655",
      "extraFees": "",
      "tradeId": "9654708b-76cb-48f7-b285-0a4f4586255d"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774353600586,
    "funding_amount": 0.1707056087998436,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3417559184210206726",
    "raw": {
      "bal": "757.3623200380538443",
      "balChg": "0.1707056087998436",
      "billId": "3417559184210206726",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774353600586",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.1707056087998436",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010673",
      "subType": "174",
      "sz": "2921",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774353600586",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774368000000,
    "funding_amount": -0.0172755,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "888547412042014720",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.01727550",
      "asset": "USDT",
      "time": 1774368000000,
      "info": "FUNDING_FEE",
      "tranId": 888547412042014720,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774368000000,
    "funding_amount": -0.0860477,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243071402173_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.0860477",
      "orderLinkId": "",
      "orderId": "f367e8cc-d14a-418c-b183-1ac5db4a3d30",
      "fee": "0",
      "change": "-0.0860477",
      "cashFlow": "0",
      "transactionTime": "1774368000000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "164370",
      "qty": "164370",
      "cashBalance": "31219.83832222",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243071402173_0",
      "category": "linear",
      "tradePrice": "0.01047",
      "extraFees": "",
      "tradeId": "a2001223-c8fc-4174-b4bc-06d37352c621"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774368001030,
    "funding_amount": -0.0118666115,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3418042382929174530",
    "raw": {
      "bal": "754.6959235754327053",
      "balChg": "-0.0118666115000000",
      "billId": "3418042382929174530",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774368001030",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.0118666115",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010469",
      "subType": "173",
      "sz": "2267",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774368001030",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774382400000,
    "funding_amount": -0.02058,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "889151391859957980",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.02058000",
      "asset": "USDT",
      "time": 1774382400000,
      "info": "FUNDING_FEE",
      "tranId": 889151391859957980,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774382400000,
    "funding_amount": -0.0839141,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243114230674_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.0839141",
      "orderLinkId": "",
      "orderId": "77fa66a1-728c-431d-8c86-a923dd0e32e0",
      "fee": "0",
      "change": "-0.0839141",
      "cashFlow": "0",
      "transactionTime": "1774382400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "163257",
      "qty": "163257",
      "cashBalance": "31061.64609621",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243114230674_0",
      "category": "linear",
      "tradePrice": "0.01028",
      "extraFees": "",
      "tradeId": "af22d83e-9e6f-40c6-8b7b-a302bedf7dd9"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774382400569,
    "funding_amount": -0.01281105,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3418525551281381382",
    "raw": {
      "bal": "753.4588779290316953",
      "balChg": "-0.0128110500000000",
      "billId": "3418525551281381382",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774382400569",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.01281105",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.01029",
      "subType": "173",
      "sz": "2490",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774382400569",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774396800000,
    "funding_amount": -0.0172755,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "889755372978135370",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.01727550",
      "asset": "USDT",
      "time": 1774396800000,
      "info": "FUNDING_FEE",
      "tranId": 889755372978135370,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774396800000,
    "funding_amount": -0.07138505,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243153308440_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.07138505",
      "orderLinkId": "",
      "orderId": "ffbeabe4-2f59-4951-bfeb-34b64441af0e",
      "fee": "0",
      "change": "-0.07138505",
      "cashFlow": "0",
      "transactionTime": "1774396800000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "136257",
      "qty": "136257",
      "cashBalance": "30995.15618177",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243153308440_0",
      "category": "linear",
      "tradePrice": "0.010478",
      "extraFees": "",
      "tradeId": "edb9ffd8-b6ae-456a-8765-3ccf88666fe2"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774396801002,
    "funding_amount": -0.002667669,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3419008749631250434",
    "raw": {
      "bal": "751.2595015117099593",
      "balChg": "-0.0026676690000000",
      "billId": "3419008749631250434",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774396801002",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.002667669",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010482",
      "subType": "173",
      "sz": "509",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774396801002",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774411200000,
    "funding_amount": -0.02052755,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "890359349860065790",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.02052755",
      "asset": "USDT",
      "time": 1774411200000,
      "info": "FUNDING_FEE",
      "tranId": 890359349860065790,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774411200000,
    "funding_amount": -0.06648694,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243226813957_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06648694",
      "orderLinkId": "",
      "orderId": "e1096355-be27-4ff6-8636-98ae3d7e31de",
      "fee": "0",
      "change": "-0.06648694",
      "cashFlow": "0",
      "transactionTime": "1774411200000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "126257",
      "qty": "126257",
      "cashBalance": "30838.09136774",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243226813957_0",
      "category": "linear",
      "tradePrice": "0.010532",
      "extraFees": "",
      "tradeId": "00f69a2e-39a1-4a3f-ad86-d210b08120d5"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774411200579,
    "funding_amount": -0.012361046,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3419491919258525702",
    "raw": {
      "bal": "752.1287460001590273",
      "balChg": "-0.0123610460000000",
      "billId": "3419491919258525702",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774411200579",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.012361046",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010529",
      "subType": "173",
      "sz": "2348",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774411200579",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774425600000,
    "funding_amount": -0.0198875,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "890963331229901200",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.01988750",
      "asset": "USDT",
      "time": 1774425600000,
      "info": "FUNDING_FEE",
      "tranId": 890963331229901200,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774425600000,
    "funding_amount": 0.27024437,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243297850771_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.27024437",
      "orderLinkId": "",
      "orderId": "1f7a6865-8c53-4705-a600-7a0ba03d9f29",
      "fee": "0",
      "change": "0.27024437",
      "cashFlow": "0",
      "transactionTime": "1774425600000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000206",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30838.47741457",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243297850771_0",
      "category": "linear",
      "tradePrice": "0.010745",
      "extraFees": "",
      "tradeId": "9ad4a6b3-70c3-4a7b-b437-4a7529719c35"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774425600999,
    "funding_amount": -0.011544426,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3419975117172187138",
    "raw": {
      "bal": "752.0608799093256273",
      "balChg": "-0.0115444260000000",
      "billId": "3419975117172187138",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774425600999",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.011544426",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010749",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774425600999",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774440000000,
    "funding_amount": -0.01961,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "891567311509217870",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.01961000",
      "asset": "USDT",
      "time": 1774440000000,
      "info": "FUNDING_FEE",
      "tranId": 891567311509217870,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774440000000,
    "funding_amount": 0.05440355,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243374284900_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.05440355",
      "orderLinkId": "",
      "orderId": "d4102f00-9387-4244-94fb-faa24e1251aa",
      "fee": "0",
      "change": "0.05440355",
      "cashFlow": "0",
      "transactionTime": "1774440000000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000043",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30838.53181812",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243374284900_0",
      "category": "linear",
      "tradePrice": "0.010585",
      "extraFees": "",
      "tradeId": "f1f3dbdd-93a9-4230-bec0-a2c723c52fc8"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774440000603,
    "funding_amount": -0.01138977,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3420458287705432070",
    "raw": {
      "bal": "752.0494901393256273",
      "balChg": "-0.0113897700000000",
      "billId": "3420458287705432070",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774440000603",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.01138977",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010605",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774440000603",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774454400000,
    "funding_amount": -0.02036158,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "892171290991616000",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.02036158",
      "asset": "USDT",
      "time": 1774454400000,
      "info": "FUNDING_FEE",
      "tranId": 892171290991616000,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774454400000,
    "funding_amount": -0.06382427,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243449381022_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06382427",
      "orderLinkId": "",
      "orderId": "7aae9de3-43e8-4952-8512-513a71d073b6",
      "fee": "0",
      "change": "-0.06382427",
      "cashFlow": "0",
      "transactionTime": "1774454400000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30838.46799385",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243449381022_0",
      "category": "linear",
      "tradePrice": "0.010441",
      "extraFees": "",
      "tradeId": "9e24b01d-85ee-478e-bb68-e5fb57b9b2bf"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774454401055,
    "funding_amount": -0.011225448,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3420941486692835330",
    "raw": {
      "bal": "752.0382646913256273",
      "balChg": "-0.0112254480000000",
      "billId": "3420941486692835330",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774454401055",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.011225448",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010452",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774454401055",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774468800000,
    "funding_amount": -0.02086499,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "892775271145103550",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.02086499",
      "asset": "USDT",
      "time": 1774468800000,
      "info": "FUNDING_FEE",
      "tranId": 892775271145103550,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774468800000,
    "funding_amount": 0.24143107,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243541955335_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.24143107",
      "orderLinkId": "",
      "orderId": "e72656b1-2589-41e7-b2f7-c6e956464d5c",
      "fee": "0",
      "change": "0.24143107",
      "cashFlow": "0",
      "transactionTime": "1774468800000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000185",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30838.70942492",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243541955335_0",
      "category": "linear",
      "tradePrice": "0.010693",
      "extraFees": "",
      "tradeId": "d01070ef-db7d-491a-bc09-80d4021257dd"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774468800615,
    "funding_amount": -0.011488578,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3421424655749685254",
    "raw": {
      "bal": "752.0267761133256273",
      "balChg": "-0.0114885780000000",
      "billId": "3421424655749685254",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774468800615",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.011488578",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010697",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774468800615",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774483200000,
    "funding_amount": -0.0209625,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "893379248530350470",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.02096250",
      "asset": "USDT",
      "time": 1774483200000,
      "info": "FUNDING_FEE",
      "tranId": 893379248530350470,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774483200000,
    "funding_amount": -0.05444785,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243607520557_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.05444785",
      "orderLinkId": "",
      "orderId": "5661a011-8f36-4825-a5d4-5c0582bc4ca1",
      "fee": "0",
      "change": "-0.05444785",
      "cashFlow": "0",
      "transactionTime": "1774483200000",
      "type": "SETTLEMENT",
      "feeRate": "0.000042",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30838.65497707",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243607520557_0",
      "category": "linear",
      "tradePrice": "0.010747",
      "extraFees": "",
      "tradeId": "69e73bc4-ee3f-4455-94e2-878f7a405cdf"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774483201006,
    "funding_amount": -0.011541204,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3421907852690268162",
    "raw": {
      "bal": "752.0152349093256273",
      "balChg": "-0.0115412040000000",
      "billId": "3421907852690268162",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774483201006",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.011541204",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010746",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774483201006",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774497600000,
    "funding_amount": -0.020514,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "893983230277673340",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.02051400",
      "asset": "USDT",
      "time": 1774497600000,
      "info": "FUNDING_FEE",
      "tranId": 893983230277673340,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774497600000,
    "funding_amount": -0.06420938,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243679956694_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "-0.06420938",
      "orderLinkId": "",
      "orderId": "ff058a1f-2736-4a0d-ae88-584d755947d4",
      "fee": "0",
      "change": "-0.06420938",
      "cashFlow": "0",
      "transactionTime": "1774497600000",
      "type": "SETTLEMENT",
      "feeRate": "0.00005",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30838.59076769",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243679956694_0",
      "category": "linear",
      "tradePrice": "0.010504",
      "extraFees": "",
      "tradeId": "df2e7c17-57ed-4759-b073-eedff6689ee1"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774497600854,
    "funding_amount": -0.011292036,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3422391031410794502",
    "raw": {
      "bal": "752.0039428733256273",
      "balChg": "-0.0112920360000000",
      "billId": "3422391031410794502",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774497600854",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.011292036",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010514",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774497600854",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774512000000,
    "funding_amount": -0.01058254,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "894587208627610270",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.01058254",
      "asset": "USDT",
      "time": 1774512000000,
      "info": "FUNDING_FEE",
      "tranId": 894587208627610270,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774512000000,
    "funding_amount": 0.30687173,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243743833906_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.30687173",
      "orderLinkId": "",
      "orderId": "b45624d2-34b2-4634-9ffe-b16115bb0ce2",
      "fee": "0",
      "change": "0.30687173",
      "cashFlow": "0",
      "transactionTime": "1774512000000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000245",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30838.89763942",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243743833906_0",
      "category": "linear",
      "tradePrice": "0.010277",
      "extraFees": "",
      "tradeId": "da8b30e0-2b54-42f1-b1ce-4a416b1742c4"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774512001016,
    "funding_amount": -0.011047164,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3422874220667412482",
    "raw": {
      "bal": "751.9928957093256273",
      "balChg": "-0.0110471640000000",
      "billId": "3422874220667412482",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774512001016",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.011047164",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010286",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774512001016",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774526400000,
    "funding_amount": 0.01105151,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "895191189871616190",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "0.01105151",
      "asset": "USDT",
      "time": 1774526400000,
      "info": "FUNDING_FEE",
      "tranId": 895191189871616190,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774526400000,
    "funding_amount": 0.17894281,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243806359427_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.17894281",
      "orderLinkId": "",
      "orderId": "a47a934a-c9a7-4b49-9e5d-a6c18e40e809",
      "fee": "0",
      "change": "0.17894281",
      "cashFlow": "0",
      "transactionTime": "1774526400000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000145",
      "bonusChange": "0",
      "size": "122257",
      "qty": "122257",
      "cashBalance": "30839.07658223",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243806359427_0",
      "category": "linear",
      "tradePrice": "0.010097",
      "extraFees": "",
      "tradeId": "0a9b7076-cb8d-4a87-8788-23208e50412b"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774526401047,
    "funding_amount": -0.010846326,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3423357405528399878",
    "raw": {
      "bal": "751.9820493833256273",
      "balChg": "-0.0108463260000000",
      "billId": "3423357405528399878",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774526401047",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "-0.010846326",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010099",
      "subType": "173",
      "sz": "2148",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774526401047",
      "type": "8"
    }
  },
  {
    "venue": "BINANCE_PERP",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774540800000,
    "funding_amount": -0.015045,
    "currency": "USDT",
    "source": "binance_papi_um_income",
    "raw_id": "895795169102356590",
    "raw": {
      "symbol": "PLUMEUSDT",
      "incomeType": "FUNDING_FEE",
      "income": "-0.01504500",
      "asset": "USDT",
      "time": 1774540800000,
      "info": "FUNDING_FEE",
      "tranId": 895795169102356590,
      "tradeId": ""
    }
  },
  {
    "venue": "BYBIT",
    "symbol": "PLUMEUSDT",
    "event_ts_ms": 1774540800000,
    "funding_amount": 0.31452155,
    "currency": "USDT",
    "source": "bybit_transaction_log_settlement",
    "raw_id": "506253878_PLUMEUSDT_243879547555_0",
    "raw": {
      "transSubType": "",
      "symbol": "PLUMEUSDT",
      "side": "Buy",
      "funding": "0.31452155",
      "orderLinkId": "",
      "orderId": "2870f81b-e763-4620-8153-c9e691282699",
      "fee": "0",
      "change": "0.31452155",
      "cashFlow": "0",
      "transactionTime": "1774540800000",
      "type": "SETTLEMENT",
      "feeRate": "-0.000259",
      "bonusChange": "0",
      "size": "121257",
      "qty": "121257",
      "cashBalance": "30827.6144178",
      "currency": "USDT",
      "id": "506253878_PLUMEUSDT_243879547555_0",
      "category": "linear",
      "tradePrice": "0.010028",
      "extraFees": "",
      "tradeId": "417ff8cd-f9d8-4273-a712-a626e42045ea"
    }
  },
  {
    "venue": "OKX",
    "symbol": "PLUME-USDT-SWAP",
    "event_ts_ms": 1774540801016,
    "funding_amount": 0.0015970634301562,
    "currency": "USDT",
    "source": "okx_bills_archive_type8",
    "raw_id": "3423840588309012482",
    "raw": {
      "bal": "748.3987839550054835",
      "balChg": "0.0015970634301562",
      "billId": "3423840588309012482",
      "ccy": "USDT",
      "clOrdId": "",
      "earnAmt": "",
      "earnApr": "",
      "execType": "",
      "fee": "0",
      "fillFwdPx": "",
      "fillIdxPx": "",
      "fillMarkPx": "",
      "fillMarkVol": "",
      "fillPxUsd": "",
      "fillPxVol": "",
      "fillTime": "1774540801016",
      "from": "",
      "instId": "PLUME-USDT-SWAP",
      "instType": "SWAP",
      "interest": "0",
      "mgnMode": "cross",
      "notes": "",
      "ordId": "",
      "pnl": "0.0015970634301562",
      "posBal": "0",
      "posBalChg": "0",
      "px": "0.010031",
      "subType": "174",
      "sz": "1548",
      "tag": "",
      "to": "",
      "tradeId": "0",
      "ts": "1774540801016",
      "type": "8"
    }
  }
]

FROZEN_LEDGER_METADATA = {
  "fetched_at_utc": "2026-03-26T17:08:04.460086+00:00",
  "window_start_utc": "2026-03-17T08:23:30.037000+00:00",
  "window_end_utc": "2026-03-26T16:54:39.458000+00:00",
  "funding_sources": {
    "BINANCE_PERP": "binance portfolio margin income history (/papi/v1/um/income, incomeType=FUNDING_FEE)",
    "BYBIT": "bybit transaction log settlements (/v5/account/transaction-log, type=SETTLEMENT)",
    "OKX": "okx bills archive funding fee (/api/v5/account/bills-archive, type=8)"
  },
  "fee_source": "execution_fill.commission from /var/lib/nautilus/telemetry/tokenmm/fills.sqlite",
  "note": "Frozen one-off extraction embedded into the notebook so reruns do not require live API credentials."
}


In [ ]:
fee_frame = pd.DataFrame(FROZEN_FEE_ROWS)
funding_frame = pd.DataFrame(FROZEN_FUNDING_ROWS)
funding_meta = pd.DataFrame([FROZEN_LEDGER_METADATA])

if not fee_frame.empty:
    fee_frame["event_ts_utc"] = pd.to_datetime(fee_frame["event_ts_ms"], unit="ms", utc=True)
    fee_frame["fee_amount"] = pd.to_numeric(fee_frame["fee_amount"], errors="coerce").fillna(0.0)

if not funding_frame.empty:
    funding_frame["event_ts_utc"] = pd.to_datetime(funding_frame["event_ts_ms"], unit="ms", utc=True)
    funding_frame["funding_amount"] = pd.to_numeric(funding_frame["funding_amount"], errors="coerce").fillna(0.0)

fee_window = fee_frame.loc[
    (fee_frame["event_ts_utc"] >= requested_start)
    & (fee_frame["event_ts_utc"] <= requested_end)
].copy()
funding_window = funding_frame.loc[
    (funding_frame["event_ts_utc"] >= requested_start)
    & (funding_frame["event_ts_utc"] <= requested_end)
].copy()

if EXCLUDED_VENUES:
    fee_window = fee_window.loc[~fee_window["venue"].isin(sorted(EXCLUDED_VENUES))].copy()
    funding_window = funding_window.loc[~funding_window["venue"].isin(sorted(EXCLUDED_VENUES))].copy()

trading_realized = float(global_summary["realized_pnl"].sum())
fees_total = float(fee_window["fee_amount"].sum()) if not fee_window.empty else 0.0
funding_total = float(funding_window["funding_amount"].sum()) if not funding_window.empty else 0.0

net_summary = pd.DataFrame(
    [
        {
            "trading_realized_pnl": trading_realized,
            "fees": fees_total,
            "funding": funding_total,
            "trading_less_fees": trading_realized - fees_total,
            "trading_less_fees_plus_funding": trading_realized - fees_total + funding_total,
            "frozen_fee_rows": len(fee_window),
            "frozen_funding_rows": len(funding_window),
        }
    ]
)
net_summary_display = net_summary.assign(
    trading_realized_pnl=format_signed_number(net_summary["trading_realized_pnl"], digits=6),
    fees=format_signed_number(-net_summary["fees"], digits=6),
    funding=format_signed_number(net_summary["funding"], digits=6),
    trading_less_fees=format_signed_number(net_summary["trading_less_fees"], digits=6),
    trading_less_fees_plus_funding=format_signed_number(net_summary["trading_less_fees_plus_funding"], digits=6),
)
display(net_summary_display)

trading_by_venue = local_summary.groupby("venue", dropna=False)["realized_pnl"].sum().rename("trading_realized_pnl")
fees_by_venue = fee_window.groupby("venue", dropna=False)["fee_amount"].sum().rename("fees") if not fee_window.empty else pd.Series(dtype="float64", name="fees")
funding_by_venue = funding_window.groupby("venue", dropna=False)["funding_amount"].sum().rename("funding") if not funding_window.empty else pd.Series(dtype="float64", name="funding")
venue_net = pd.concat([trading_by_venue, fees_by_venue, funding_by_venue], axis=1).fillna(0.0).reset_index()
venue_net["trading_less_fees_plus_funding"] = venue_net["trading_realized_pnl"] - venue_net["fees"] + venue_net["funding"]
venue_net_display = venue_net.assign(
    trading_realized_pnl=format_signed_number(venue_net["trading_realized_pnl"], digits=6),
    fees=format_signed_number(-venue_net["fees"], digits=6),
    funding=format_signed_number(venue_net["funding"], digits=6),
    trading_less_fees_plus_funding=format_signed_number(venue_net["trading_less_fees_plus_funding"], digits=6),
)
display(venue_net_display)

funding_window_display = funding_window.loc[:, ["event_ts_utc", "venue", "symbol", "funding_amount", "currency", "source"]].copy() if not funding_window.empty else funding_window.copy()
if not funding_window_display.empty:
    funding_window_display = funding_window_display.sort_values(["event_ts_utc", "venue"]).assign(
        event_ts_utc=format_utc(funding_window_display["event_ts_utc"]),
        funding_amount=format_signed_number(funding_window_display["funding_amount"], digits=8),
    )
funding_window_display.tail(40)

funding_meta
